# TDCS comprehensive experiments — S12 paired crops plus the fixed ranking loss

**Only S12 fits on Kaggle in this notebook.** All prior tests stay labelled
below and their completed receipts are reused. Local Run All exercises
synthetic data only. This is a TDCS research run, not a submission.

S10 (typed auxiliary data) reached 0.2107 ungated Walking AP; M22 (typed
plus binary Event labels) reached 0.2440. S11's unpaired ranking loss reached
0.2074 and saw usable pairs in just 6.51% of microbatches. Two matched tests
check whether the weakness is lack of paired training examples or ranking
itself, without assuming either change will improve the score.

> **Public reproduction copy.** Run All defaults to a synthetic wiring test. These outputs do not measure TDCS performance. The retained predictions, private feature cache, and per-record gate are not distributed here. For the real-data inputs, opt-in, and audited aggregate results, see [the v31 release notes](../docs/tdcs-v31-public-release.md).

Historical experiment labels and the latest v31 paired-crop test are preserved.

## Current protocol — one paired microbatch, six unchanged natural crops

Both runs refit the M22 weak Event warmup (512 updates; 91 typed and 46
binary-labelled training recordings; Valid AND Task; no assumed subtypes).
Keep the same 50-Hz architecture, 1,536 TDCS updates, optimizer, loss weights,
normalization, EMA, scoring and inference. Reuse original natural crops 2–7
from each batch of eight with their original augmentation stream.

Replace crops 0–1 with a Walking example and, alternately, a Turn or
StartHesitation example. For the requested subtype, choose a fitting subject
uniformly, then one of that subject's contiguous valid events (at least 3
samples), then an interior center. Use a private seeded RNG for both added
crops. The first two crops share a microbatch, giving at least one usable
pair each update. **M23** keeps the original cross entropy; **S12** adds the
same 0.2 softplus rank term (0.2 log-odds margin, at most sixteen hardest
valid other-FOG negatives per microbatch) as S11. Both save actual crop
traces and pair counts. Warmups are matched refits, not claimed byte-identical.

Only fitting subjects supply training crops. All 6,959,762 native rows,
815 recordings and 61 subjects remain in the same five outer folds, with
all negative recordings included. Report ungated native AP and the fixed
one-gate Walking ablation. No held-out stopping or automatic promotion.

Below, older protocol blocks describe their original runs; they are not
instructions to retrain those tests or an updated independent test score.

> Historical protocol from the previous notebook version; the fitted result is reused here.

## S11 — rank Walking above the other FOG types

**Only S11 fits on Kaggle.** The 11 completed S00–S10 results remain labelled
below and are reused, not retrained. This is the same sensor notebook; the
other notebook's M22 weak-Event transfer test is not changed.

Current exploratory Walking AP is 0.279789 from the retained/M19 blend.
At a diagnostic 50% Walking recall, 279,724 of its 295,577 false-positive
rows are annotated Turn or StartHesitation. These held-out labels describe
mistakes; their thresholds, predictions and rows are not fitting inputs.

S11 refits S10's typed-only DeFOG warmup (512 updates, 91 typed files with
Valid AND Task) with the same seed, then 1,536 TDCS updates per fold. The
architecture, natural crops, base four-state cross entropy, EMA 0.995,
output heads, and 100-second inference context are unchanged. Warmup is a
matched refit, not a claim of byte-identical GPU weights. In fine-tuning
only, add 0.2 times a softplus pairwise ranking loss with margin 0.2 in
Walking log-odds. Pair every valid Walking row in a microbatch with up to
16 highest-scoring valid SH/Turn negatives. Batches without both sides
contribute zero rank loss. Background still trains through the base loss.
All ranking labels and scores come from the current outer-training model
and its fitting rows only. Save actual pair coverage alongside each run.

Score all 815 recordings / 61 subjects / 6,959,762 native rows, including
all negatives, exactly as before. Walking AP is the primary decision;
also report SH and two-target mAP. Ungated inference is primary; the same
fixed Walking gate is a secondary ablation. Fold membership, preprocessing,
active models, retained Turn/DeFOG and the 0.6 goal are unchanged. Local
notebook outputs are synthetic wiring checks, not measured TDCS scores.

## Historical S10 reader recovery — completed in sensor v5

Sensor version 4 completed as a notebook, but S10 failed **before fitting**:
it assumed every one of 137 DeFOG metadata IDs had a train/defog CSV.
Official inventory has **91 subtype-labelled train/defog files (38 subjects)**
and **46 train/notype files**, all covered by the 137-row / 45-subject table.
Some IDs also appear in public test examples; **no test file is read**.
The 46 notype records are excluded from this fixed four-state objective;
their missing subtype labels are not invented or called background.

Select actual train/defog CSVs, validate all 91 IDs against unique metadata,
and exclude every TDCS-metadata subject before fitting. A separately pinned
replacement reader preserves the original model, rate/unit conversion,
Valid AND Task mask, seeds, optimizer, 512/1536-update budgets and all folds.
The composite source hash binds both frozen model and reader revisions.
**S10 completed and is now reused alongside S00–S09.**
M21 and its original run remain untouched. See the saved S10 audit for measured results.

## Historical v28 — matched raw-signal control versus cross-domain transfer

**M21:** TDCS-only warmup. **S10:** independent labelled-DeFOG warmup.
Both use anti-aliased 50 Hz accelerometry, an eight-channel raw/standardized
input, CNN stem and three residual bidirectional GRU blocks (hidden 96).
Train 20-second windows, infer overlapping 100-second windows. Eight inputs:
three raw g axes, three per-record standardized axes, standardized dynamic
magnitude and within-record position. Whole-record moments use raw sensor
values only; this is offline, not streaming. No clinical or recording fields,
Subject IDs, events or task boundaries are predictors.

Each outer model gets 512 warmup and 1536 fine-tune updates, effective batch
eight, microbatch two, FP32 and final EMA 0.995. Both start from seed 2851;
their TDCS fine-tune crop seeds match per fold. S10's independent pretraining
checkpoint is computed once and shared: physical GPU work is therefore lower,
while each outer model inherits the same declared 512-update warmup budget.
Optimizer and EMA reset before fine-tuning; no held-out stopping or selection.
Natural valid-center crops, subject-balanced sampling, time warp, small yaw,
dynamic gain and noise are identical across the pair. Four training states
(NoFOG, SH, Turn, Walking) let Turn be auxiliary supervision only; reported
metrics and the target stay SH/Walking and the retained Turn head is unchanged.

TDCS: 128 Hz, m/s² converted to g. DeFOG: 100 Hz, already g.
Auxiliary loss uses **Valid AND Task**, never unannotated background.
All TDCS-metadata subject IDs are excluded from auxiliary pretraining.
The metadata table has 137 IDs / 45 subjects, but only 91 train/defog files / 38 subjects; 46 IDs are train/notype. The typed subject IDs have no
overlap against 62 TDCS subject IDs; exclusion is still enforced at runtime.
This verifies provided IDs, not an independent biometric identity audit.

**Primary:** ungated pooled native-row AP. **Secondary declared ablation:**
the existing frozen Walking gate, applied once after native interpolation.
Both are recorded; no hidden gate tuning or automatic promotion. Five frozen
subject folds retain all 815 TDCS recordings / 61 subjects / 6,959,762 native
rows, including negatives. Auxiliary samples are never added to that metric.
Each notebook has a six-hour internal budget and no real-data CPU fallback.

## Historical v27 followups — completed and reused

All five v27 methods completed and passed saved-model / native-row
validation. This notebook reuses its 10 prior results;
the historical S10 fit is also reused. S11 is the only new real-data fit. Historical single-head
tests have undefined other-target AP and two-target mAP, not zeros.
The large reused development-fold search remains exploratory.

## Historical version 2 — Parquet recovery completed
Version 1 completed five sensor tests. S03/S04 did not train because the
loader searched for CSV but all 65 official daily inputs are Parquet.
Version 2 retrained **only S03 and S04**; this version reuses those results, keeping all seven Markdown labels.
The other five results are explicitly marked `complete_reused_v1`, with their
version-1 artifacts preserved. No prior predictor or best registry is edited.
Bounded Arrow batches avoid loading entire week-long files. Rates, units,
folds, teacher thresholds and all training budgets are unchanged.
Local Run All still executes all seven synthetic tests, not real training.

## Setup and fixed evaluation protocol
Use the competition input and the completed private CatBoost v8 output
attached in kernel metadata. No API token belongs in a notebook.
Five frozen subject-disjoint folds; all 815 eligible recordings, 61 subjects
and 6,959,762 native rows, including negative recordings. Public example
subject 4dc2f8 stays excluded. Historical v26/v27 models train only SH/Walking. New v28 uses Turn as an auxiliary training state;
Turn and DeFOG stay protected. These are offline, not streaming models.

Trees: 1000 rounds; seed 27103 + 1009 × fold + head; all fitting positives
and a seeded quarter of negatives with inverse-sampling correction and
count^-0.5 subject weights. This is a new matched suite, not the historical
seed-5107 recipe. Networks: width 64, 12 × 64 updates, batch 16, windows 256
at 8 Hz, AdamW, final EMA 0.995; fit preprocessing only on training subjects.
M18 alone uses a training-only Kinetic auxiliary target.

### Key assumptions and limits
No outer-fold labels select checkpoints, features, thresholds or routing.
Interpolate to every native row; apply the existing cross-fitted sensor-only
Walking presence gate exactly once (except M12: no gate). Scores are not
calibrated confidence. Reused development-fold AP is exploratory, not
independently established generalization. The retained full ensemble has
historical metadata ancestry and is not a pure metadata-free control.

Six-hour internal budget per suite, checked between methods/folds and
during neural updates; a single tree fit cannot be interrupted by that
guard. Completed folds have hash-verified checkpoints; partial folds are
preserved and refused rather than silently overwritten. Each method catches
and records its own failure so later methods can continue. Nothing is
promoted, submitted or written to the retained best-model registries.

### 1. Configure the runtime

In [1]:
import hashlib, importlib.util, json, os, subprocess, sys, types, uuid
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from IPython.display import display
SUITE = 'sensor'
IS_KAGGLE = Path('/kaggle/working').exists()
RUN_FULL = os.environ.get('TDCS_RUN_FULL', '0') == '1'
PROJECT_ROOT = Path('D:/Dissertation') if Path('D:/Dissertation/data').exists() else Path.cwd()
WORK_ROOT = (Path('/kaggle/working') / 'tdcs_sensor_comprehensive_v26_v31_paired_crops' if RUN_FULL else
    PROJECT_ROOT / 'tmp' / ('tdcs_sensor_comprehensive_v26_v31_paired_crops' + '_smoke_' + uuid.uuid4().hex[:8]))
WORK_ROOT.mkdir(parents=True, exist_ok=True)
if RUN_FULL and (not IS_KAGGLE or not torch.cuda.is_available()):
    raise RuntimeError('Real-data fitting requires Kaggle CUDA; no local or CPU fallback')
if not RUN_FULL and SUITE == 'sensor' and importlib.util.find_spec('lightgbm') is None:
    if not RUN_FULL:
        raise RuntimeError('Install LightGBM 4.6.0 in an isolated local smoke runtime first')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--no-deps',
        '--index-url', 'https://pypi.org/simple', 'lightgbm==4.6.0'])
print({'suite': SUITE, 'synthetic_only': not RUN_FULL, 'CUDA': torch.cuda.is_available(),
    'output': str(WORK_ROOT)}, flush=True)

{'suite': 'sensor', 'synthetic_only': True, 'CUDA': False, 'output': 'D:\\Dissertation\\tmp\\tdcs_sensor_comprehensive_v26_v31_paired_crops_smoke_14b34482'}


### 2. Load pinned sources and the immutable feature-cache recipe

In [2]:
SOURCES = {'tdcs_multiscale_signal_experiment': '"""TDCS-only multiscale sensor model on the frozen subject folds.\n\nNotebook companion.  Features use only accelerometer measurements and their\ntiming.  Three independent histogram gradient boosting classifiers predict\nStartHesitation, Turn and Walking.  All hyperparameters are fixed before scoring.\n"""\n\n# %% Imports and frozen experiment settings\nfrom __future__ import annotations\n\nimport gc\nimport hashlib\nimport json\nimport platform\nimport time\nfrom dataclasses import asdict, dataclass\nfrom pathlib import Path\n\nimport joblib\nimport matplotlib\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nimport scipy\nimport sklearn\nfrom scipy.ndimage import maximum_filter1d, minimum_filter1d, uniform_filter1d\nfrom scipy.signal import butter, sosfiltfilt\nfrom sklearn.ensemble import HistGradientBoostingClassifier\nfrom sklearn.metrics import average_precision_score\nfrom threadpoolctl import threadpool_limits\n\nTARGETS = ("StartHesitation", "Turn", "Walking")\nAXES = ("AccV", "AccML", "AccAP")\n\n# Exact TDCS subject assignment from the earlier immutable_subject_folds.csv.\n# The public-example subject 4dc2f8 remains excluded for a like-for-like comparison.\nFROZEN_SUBJECT_FOLDS = {\n    "02bc69": 2, "07285e": 0, "082f01": 1, "194d1d": 3, "19ea47": 1,\n    "220a17": 3, "231c3b": 0, "242a3e": 4, "24a59d": 3, "251738": 0,\n    "2a39f8": 0, "2c98f7": 2, "2d57c2": 0, "301ada": 4, "312788": 1,\n    "31d269": 0, "364459": 0, "3b2403": 1, "3b2b7a": 4, "48fd62": 2,\n    "4b39ac": 2, "4ba1d3": 3, "4bb5d0": 3, "4ca9b3": 4, "4f13b4": 2,\n    "51574c": 1, "516a67": 2, "54ee6e": 2, "59f492": 0, "5c0b8a": 4,\n    "66341b": 0, "69cc45": 3, "6a3e93": 4, "743f4e": 1, "7688c1": 4,\n    "79011a": 1, "7eb666": 2, "7fcee9": 3, "87174c": 3, "8db7dd": 2,\n    "93f49f": 1, "9f85da": 4, "a03db7": 4, "a80ae4": 1, "af82b2": 4,\n    "b19f77": 2, "bc3908": 0, "c7fee4": 1, "c85fdf": 2, "c8e721": 0,\n    "c95ab0": 1, "d8836b": 3, "d9312a": 1, "e39bc5": 4, "e8919c": 3,\n    "e9fc55": 2, "eeaff0": 3, "f2c8aa": 0, "f62eec": 1, "f686f0": 3,\n    "fa8764": 4,\n}\n\n\n@dataclass(frozen=True)\nclass ExperimentConfig:\n    feature_version: str = "multiscale_signal_v1"\n    sample_hz: int = 128\n    training_stride: int = 32\n    windows: tuple[int, ...] = (64, 256, 1024)\n    band_windows: tuple[int, ...] = (256, 1024)\n    max_iter: int = 220\n    learning_rate: float = 0.07\n    max_leaf_nodes: int = 31\n    min_samples_leaf: int = 100\n    l2_regularization: float = 10.0\n    max_bins: int = 128\n    positive_weight_cap: float = 8.0\n    seed: int = 42\n    cpu_threads: int = 4\n\n\nCFG = ExperimentConfig()\n\n\ndef locate_data() -> tuple[Path, Path, Path]:\n    """Locate raw competition data locally or beneath nested Kaggle mounts."""\n    candidates = [Path.cwd(), *Path.cwd().parents, Path("D:/Dissertation")]\n    kaggle = Path("/kaggle/input")\n    if kaggle.exists():\n        candidates = [path.parent for path in kaggle.rglob("tdcsfog_metadata.csv")] + candidates\n    for base in candidates:\n        for raw, meta in ((base, base), (base / "data" / "raw", base / "data" / "metadata"),\n                          (base / "raw", base / "metadata")):\n            if (raw / "train" / "tdcsfog").exists() and (meta / "tdcsfog_metadata.csv").exists():\n                if Path("/kaggle/working").exists():\n                    output = Path("/kaggle/working") / "tdcs_multiscale_signal"\n                else:\n                    output = base / "results" / "tdcs_multiscale_signal"\n                return raw / "train" / "tdcsfog", meta / "tdcsfog_metadata.csv", output\n    raise FileNotFoundError(\n        "Attach Parkinson\'s Freezing of Gait Prediction data to this notebook, "\n        "or use the local data/raw/train/tdcsfog and data/metadata layout."\n    )\n\n\n# %% Recording manifest and input audit\ndef build_manifest(raw_root: Path, metadata_path: Path) -> pd.DataFrame:\n    metadata = pd.read_csv(metadata_path, dtype={"Id": str, "Subject": str})\n    if metadata["Id"].duplicated().any():\n        raise ValueError("Duplicate metadata recording IDs")\n    metadata = metadata.set_index("Id")\n    rows = []\n    for path in sorted(raw_root.glob("*.csv")):\n        if path.stem not in metadata.index:\n            raise ValueError(f"Missing subject metadata for {path.stem}")\n        subject = str(metadata.loc[path.stem, "Subject"])\n        if subject == "4dc2f8":\n            continue\n        if subject not in FROZEN_SUBJECT_FOLDS:\n            raise ValueError(f"Unexpected subject {subject}; do not silently change the folds")\n        stat = path.stat()\n        rows.append({"recording_id": path.stem, "subject": subject,\n                     "fold": FROZEN_SUBJECT_FOLDS[subject], "path": str(path),\n                     "source_bytes": stat.st_size, "source_mtime_ns": stat.st_mtime_ns})\n    manifest = pd.DataFrame(rows)\n    if len(manifest) != 815 or manifest.subject.nunique() != 61:\n        raise ValueError(f"Expected 815 recordings / 61 subjects; found {len(manifest)} / {manifest.subject.nunique()}")\n    if manifest.groupby("subject").fold.nunique().max() != 1:\n        raise ValueError("Subject crosses folds")\n    for fold in range(5):\n        train_subjects = set(manifest.loc[manifest.fold.ne(fold), "subject"])\n        val_subjects = set(manifest.loc[manifest.fold.eq(fold), "subject"])\n        assert train_subjects.isdisjoint(val_subjects)\n    return manifest\n\n\ndef input_fingerprint(manifest: pd.DataFrame) -> str:\n    source_columns = ["recording_id", "subject", "fold", "source_bytes", "source_mtime_ns"]\n    payload = json.dumps(asdict(CFG), sort_keys=True) + manifest[source_columns].to_csv(index=False)\n    return hashlib.sha256(payload.encode("utf-8")).hexdigest()[:20]\n\n\n# %% Label-free multiscale sensor features\ndef sensor_features(raw: np.ndarray, cfg: ExperimentConfig = CFG) -> tuple[np.ndarray, np.ndarray, list[str]]:\n    """Feature rows at a sparse grid; centered windows are for offline detection."""\n    raw = np.asarray(raw, np.float64)\n    if len(raw) < 2 or raw.shape[1] != 3 or not np.isfinite(raw).all():\n        raise ValueError("Need a finite accelerometer series with at least two rows and three axes")\n    n_rows = len(raw)\n    grid = np.unique(np.r_[np.arange(0, n_rows, cfg.training_stride), n_rows - 1]).astype(np.int32)\n    center = raw.mean(axis=0)\n    scale = np.maximum(raw.std(axis=0), 0.05)\n    z = (raw - center) / scale\n    dynamic_magnitude = np.linalg.norm(z, axis=1)\n    signals = np.column_stack([z, dynamic_magnitude])\n    names = [*AXES, "dynamic_magnitude"]\n    jerk = np.diff(signals, axis=0, prepend=signals[:1])\n    columns: list[np.ndarray] = []\n    feature_names: list[str] = []\n\n    def add(values: np.ndarray, labels: list[str]) -> None:\n        values = np.asarray(values)\n        if values.ndim == 1:\n            values = values[:, None]\n        if values.shape[1] != len(labels):\n            raise ValueError("Feature-name mismatch")\n        columns.append(values.astype(np.float32))\n        feature_names.extend(labels)\n\n    add(raw[grid], [f"raw_{name}" for name in AXES])\n    add(signals[grid], [f"normalized_{name}" for name in names])\n    add(jerk[grid], [f"first_difference_{name}" for name in names])\n    add(np.broadcast_to(np.r_[center, scale], (len(grid), 6)),\n        [f"series_mean_{name}" for name in AXES] + [f"series_std_{name}" for name in AXES])\n    seconds = grid / cfg.sample_hz\n    add(np.column_stack([np.log1p(seconds), np.log1p((n_rows - 1 - grid) / cfg.sample_hz),\n                         grid / max(n_rows - 1, 1)]),\n        ["log_elapsed_seconds", "log_remaining_seconds", "relative_recording_position"])\n\n    for window in cfg.windows:\n        mean = uniform_filter1d(signals, window, axis=0, mode="nearest")\n        variance = np.maximum(uniform_filter1d(signals ** 2, window, axis=0, mode="nearest") - mean ** 2, 0)\n        std = np.sqrt(variance)\n        value_range = maximum_filter1d(signals, window, axis=0, mode="nearest") - minimum_filter1d(signals, window, axis=0, mode="nearest")\n        jerk_rms = np.sqrt(np.maximum(uniform_filter1d(jerk ** 2, window, axis=0, mode="nearest"), 0))\n        for statistic, values in (("mean", mean), ("std", std), ("range", value_range), ("jerk_rms", jerk_rms)):\n            add(values[grid], [f"{statistic}_{name}_w{window}" for name in names])\n        for a, b in ((0, 1), (0, 2), (1, 2)):\n            covariance = uniform_filter1d(signals[:, a] * signals[:, b], window, mode="nearest") - mean[:, a] * mean[:, b]\n            correlation = covariance / np.maximum(std[:, a] * std[:, b], 1e-5)\n            add(np.clip(correlation[grid], -1, 1), [f"correlation_{AXES[a]}_{AXES[b]}_w{window}"])\n\n    bands = {}\n    for band_name, limits in (("low", (0.5, 3.0)), ("mid", (3.0, 8.0)), ("high", (8.0, 15.0))):\n        sos = butter(3, limits, btype="bandpass", fs=cfg.sample_hz, output="sos")\n        filtered = sosfiltfilt(sos, z, axis=0, padlen=min(24, n_rows - 1))\n        bands[band_name] = filtered ** 2\n    for window in cfg.band_windows:\n        energy = {name: uniform_filter1d(values, window, axis=0, mode="nearest") for name, values in bands.items()}\n        for band_name, values in energy.items():\n            add(np.log1p(np.maximum(values[grid], 0)), [f"log_band_energy_{band_name}_{name}_w{window}" for name in AXES])\n        ratio = np.log((np.maximum(energy["mid"], 0) + 1e-5) / (np.maximum(energy["low"], 0) + 1e-5))\n        add(ratio[grid], [f"log_mid_low_energy_ratio_{name}_w{window}" for name in AXES])\n\n    features = np.concatenate(columns, axis=1)\n    if not np.isfinite(features).all():\n        raise ValueError("Non-finite sensor features")\n    return grid, features, feature_names\n\n\ndef prepare_arrays(manifest: pd.DataFrame, run_dir: Path) -> dict[str, np.ndarray | list[str]]:\n    cache_path = run_dir / "sampled_features.npz"\n    if cache_path.exists():\n        with np.load(cache_path, allow_pickle=False) as payload:\n            return {name: payload[name].copy() for name in payload.files}\n    x_parts, y_parts, fold_parts, record_parts, grid_parts, row_counts = [], [], [], [], [], []\n    feature_names = None\n    for record_index, row in enumerate(manifest.itertuples(index=False)):\n        frame = pd.read_csv(row.path, usecols=[*AXES, *TARGETS])\n        y_native = frame.loc[:, list(TARGETS)].to_numpy(np.uint8)\n        if not np.isin(y_native, [0, 1]).all():\n            raise ValueError(f"Non-binary labels in {row.recording_id}")\n        grid, x, names = sensor_features(frame.loc[:, list(AXES)].to_numpy())\n        if feature_names is not None and feature_names != names:\n            raise ValueError("Feature names changed between recordings")\n        feature_names = names\n        x_parts.append(x)\n        y_parts.append(y_native[grid])\n        fold_parts.append(np.full(len(grid), row.fold, np.int8))\n        record_parts.append(np.full(len(grid), record_index, np.int16))\n        grid_parts.append(grid)\n        row_counts.append(len(frame))\n        if (record_index + 1) % 100 == 0:\n            print(f"Features: {record_index + 1}/{len(manifest)} recordings", flush=True)\n    arrays = {"x": np.concatenate(x_parts), "y": np.concatenate(y_parts),\n              "fold": np.concatenate(fold_parts), "record": np.concatenate(record_parts),\n              "grid": np.concatenate(grid_parts), "row_counts": np.asarray(row_counts, np.int32),\n              "feature_names": np.asarray(feature_names, dtype=str)}\n    np.savez_compressed(cache_path, **arrays)\n    return arrays\n\n\n# %% Independent target classifiers on the frozen subject folds\ndef score_targets(y: np.ndarray, p: np.ndarray) -> dict[str, float]:\n    values = {target: float(average_precision_score(y[:, index], p[:, index]))\n              if np.unique(y[:, index]).size == 2 else float("nan")\n              for index, target in enumerate(TARGETS)}\n    values["mAP"] = float(np.nanmean(list(values.values())))\n    return values\n\n\ndef train_and_predict(arrays: dict, run_dir: Path) -> np.ndarray:\n    x, y, folds = arrays["x"], arrays["y"], arrays["fold"]\n    predictions = np.full(y.shape, np.nan, np.float32)\n    training_log = []\n    with threadpool_limits(limits=CFG.cpu_threads):\n        for fold in range(5):\n            fitting, validation = folds != fold, folds == fold\n            fold_dir = run_dir / f"fold_{fold}"\n            fold_dir.mkdir(exist_ok=True)\n            for target_index, target in enumerate(TARGETS):\n                model_path = fold_dir / f"{target}.joblib"\n                prediction_path = fold_dir / f"{target}_sampled_prediction.npy"\n                start = time.monotonic()\n                labels = y[fitting, target_index]\n                positives = int(labels.sum())\n                if positives == 0 or positives == len(labels):\n                    raise ValueError(f"Fold {fold} / {target} lacks both classes in training")\n                positive_weight = min(CFG.positive_weight_cap, np.sqrt((len(labels) - positives) / positives))\n                reused = model_path.exists() and prediction_path.exists()\n                if reused:\n                    p = np.load(prediction_path, allow_pickle=False)\n                else:\n                    model = HistGradientBoostingClassifier(\n                        learning_rate=CFG.learning_rate, max_iter=CFG.max_iter,\n                        max_leaf_nodes=CFG.max_leaf_nodes, min_samples_leaf=CFG.min_samples_leaf,\n                        l2_regularization=CFG.l2_regularization, max_bins=CFG.max_bins,\n                        early_stopping=False, random_state=CFG.seed + fold,\n                    )\n                    sample_weight = np.where(labels == 1, positive_weight, 1.0)\n                    model.fit(x[fitting], labels, sample_weight=sample_weight)\n                    p = model.predict_proba(x[validation])[:, 1].astype(np.float32)\n                    joblib.dump(model, model_path)\n                    np.save(prediction_path, p, allow_pickle=False)\n                    del model\n                    gc.collect()\n                if len(p) != int(validation.sum()) or not np.isfinite(p).all():\n                    raise ValueError(f"Invalid cached prediction for fold {fold} / {target}")\n                predictions[validation, target_index] = p\n                training_log.append({"fold": fold, "target": target, "training_rows": int(fitting.sum()),\n                                     "validation_rows": int(validation.sum()), "training_positive_rows": positives,\n                                     "positive_weight": positive_weight, "elapsed_seconds": time.monotonic() - start,\n                                     "reused": reused})\n                print(f"Fold {fold} / {target}: complete ({training_log[-1][\'elapsed_seconds\']:.1f}s)", flush=True)\n    pd.DataFrame(training_log).to_csv(run_dir / "training_log.csv", index=False)\n    if not np.isfinite(predictions).all():\n        raise ValueError("OOF predictions are incomplete")\n    np.save(run_dir / "sampled_oof_prediction.npy", predictions, allow_pickle=False)\n    return predictions\n\n\n# %% Native-row evaluation and baseline comparison\ndef evaluate_native(manifest: pd.DataFrame, arrays: dict, sampled_predictions: np.ndarray,\n                    run_dir: Path) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:\n    prediction_dir = run_dir / "oof"\n    prediction_dir.mkdir(exist_ok=True)\n    y_parts, p_parts, folds_parts, diagnostic_rows = [], [], [], []\n    for record_index, row in enumerate(manifest.itertuples(index=False)):\n        y = pd.read_csv(row.path, usecols=list(TARGETS), dtype={name: np.uint8 for name in TARGETS}).loc[:, list(TARGETS)].to_numpy()\n        mask = arrays["record"] == record_index\n        grid, sparse = arrays["grid"][mask], sampled_predictions[mask]\n        if len(y) != arrays["row_counts"][record_index]:\n            raise ValueError("Row count changed since feature extraction")\n        p = np.column_stack([np.interp(np.arange(len(y)), grid, sparse[:, index]) for index in range(3)]).astype(np.float32)\n        np.savez_compressed(prediction_dir / f"{row.recording_id}.npz", prediction=p)\n        y_parts.append(y)\n        p_parts.append(p)\n        folds_parts.append(np.full(len(y), row.fold, np.int8))\n        for target_index, target in enumerate(TARGETS):\n            diagnostic_rows.append({"recording_id": row.recording_id, "subject": row.subject,\n                                    "fold": row.fold, "target": target,\n                                    "positive_rows": int(y[:, target_index].sum()), "n_rows": len(y),\n                                    "AP": float(average_precision_score(y[:, target_index], p[:, target_index]))\n                                    if np.unique(y[:, target_index]).size == 2 else float("nan")})\n    y, p, folds = np.concatenate(y_parts), np.concatenate(p_parts), np.concatenate(folds_parts)\n    metrics = pd.DataFrame([{"domain": "tdcsfog", "method": "multiscale_sensor_hgb", **score_targets(y, p)}])\n    fold_metrics = pd.DataFrame([{"fold": fold, **score_targets(y[folds == fold], p[folds == fold])} for fold in range(5)])\n    recording_metrics = pd.DataFrame(diagnostic_rows)\n    metrics.to_csv(run_dir / "native_oof_metrics.csv", index=False)\n    fold_metrics.to_csv(run_dir / "native_fold_metrics.csv", index=False)\n    recording_metrics.to_csv(run_dir / "recording_metrics.csv", index=False)\n    return metrics, fold_metrics, recording_metrics\n\n\ndef comparison_table(metrics: pd.DataFrame) -> pd.DataFrame:\n    # Previously executed TDCS blend metrics, recomputed from the downloaded OOF.\n    baseline = {"method": "previous_tdcs_blend", "StartHesitation": 0.1868209824890754,\n                "Turn": 0.7161279460203974, "Walking": 0.15061415066447398, "mAP": 0.35118769305798225}\n    candidate = metrics.iloc[0][["method", *TARGETS, "mAP"]].to_dict()\n    return pd.DataFrame([baseline, candidate])\n\n\ndef plot_experiment(comparison: pd.DataFrame, fold_metrics: pd.DataFrame, run_dir: Path) -> list[Path]:\n    plt.rcParams.update({"font.size": 11, "axes.spines.top": False, "axes.spines.right": False})\n    colors = {"previous_tdcs_blend": "#8b929c", "multiscale_sensor_hgb": "#2463a0"}\n    fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), gridspec_kw={"width_ratios": [2, 1]}, layout="constrained")\n    positions = np.arange(3)\n    for index, row in comparison.iterrows():\n        offset = (index - 0.5) * 0.34\n        label = "Previous TDCS blend" if index == 0 else "Multiscale sensor model"\n        axes[0].bar(positions + offset, row.loc[list(TARGETS)].to_numpy(float), 0.34, label=label, color=colors[row.method])\n    axes[0].set_xticks(positions, ["Start hesitation", "Turn", "Walking"])\n    axes[0].set_ylim(0, 1)\n    axes[0].set_ylabel("Native-row average precision")\n    axes[0].set_title("TDCS: class performance")\n    axes[0].legend(loc="upper left", frameon=False)\n    axes[0].grid(axis="y", alpha=0.2)\n    bars = axes[1].bar(["Previous\\nblend", "Sensor\\nmodel"], comparison.mAP, color=[colors[name] for name in comparison.method])\n    axes[1].bar_label(bars, fmt="%.4f", padding=4)\n    axes[1].set_ylim(0, 1)\n    axes[1].set_title("TDCS: pooled OOF mAP")\n    axes[1].grid(axis="y", alpha=0.2)\n    first = run_dir / "tdcs_model_comparison.png"\n    fig.savefig(first, dpi=150)\n    plt.close(fig)\n\n    fig, ax = plt.subplots(figsize=(10, 5), layout="constrained")\n    for target, color in zip(TARGETS, ["#2463a0", "#a26916", "#7a537d"]):\n        ax.plot(fold_metrics.fold, fold_metrics[target], "o-", label=target, color=color)\n    ax.set_xticks(range(5))\n    ax.set_ylim(0, 1)\n    ax.set_xlabel("Held-out subject fold")\n    ax.set_ylabel("Native-row average precision")\n    ax.set_title("TDCS multiscale sensor model: subject-fold variation")\n    ax.legend(loc="upper right", frameon=False)\n    ax.grid(axis="y", alpha=0.2)\n    second = run_dir / "tdcs_subject_fold_performance.png"\n    fig.savefig(second, dpi=150)\n    plt.close(fig)\n    return [first, second]\n\n\n# %% Run and retain complete experiment artifacts\ndef run_experiment() -> dict:\n    raw_root, metadata_path, output_root = locate_data()\n    manifest = build_manifest(raw_root, metadata_path)\n    fingerprint = input_fingerprint(manifest)\n    run_dir = output_root / fingerprint\n    run_dir.mkdir(parents=True, exist_ok=True)\n    manifest.to_csv(run_dir / "frozen_tdcs_manifest.csv", index=False)\n    audit = {"domain": "tdcsfog", "configuration": asdict(CFG), "fingerprint": fingerprint,\n             "recordings": len(manifest), "subjects": int(manifest.subject.nunique()),\n             "fold_recordings": manifest.groupby("fold").recording_id.count().to_dict(),\n             "validation": "five frozen subject-disjoint folds; fixed settings; no random-row early stopping",\n             "feature_inputs": "accelerometer values and recording timing only",\n             "feature_assumption": "offline detection; centered filters and whole-record normalization",\n             "training_grain": f"one label every {CFG.training_stride} native rows plus final row",\n             "evaluation_grain": "all native rows; linear interpolation within each recording",\n             "excluded_subjects": ["4dc2f8"], "retained_defog_reported_oof_mAP": 0.817,\n             "python": platform.python_version(), "numpy": np.__version__, "scipy": scipy.__version__,\n             "sklearn": sklearn.__version__}\n    print(f"TDCS experiment: {len(manifest)} recordings, {manifest.subject.nunique()} subjects", flush=True)\n    print(f"Artifacts: {run_dir}", flush=True)\n    arrays = prepare_arrays(manifest, run_dir)\n    audit["native_rows"] = int(arrays["row_counts"].sum())\n    audit["sampled_training_rows"] = int(len(arrays["x"]))\n    audit["feature_count"] = int(arrays["x"].shape[1])\n    (run_dir / "audit.json").write_text(json.dumps(audit, indent=2), encoding="utf-8")\n    if audit["native_rows"] != 6959762:\n        raise ValueError("Native row total differs from the earlier TDCS baseline")\n    print(f"Prepared {audit[\'sampled_training_rows\']:,} rows with {audit[\'feature_count\']} features", flush=True)\n    sampled_predictions = train_and_predict(arrays, run_dir)\n    metrics, fold_metrics, recording_metrics = evaluate_native(manifest, arrays, sampled_predictions, run_dir)\n    comparison = comparison_table(metrics)\n    comparison.to_csv(run_dir / "baseline_comparison.csv", index=False)\n    figures = plot_experiment(comparison, fold_metrics, run_dir)\n    delta = float(comparison.iloc[1].mAP - comparison.iloc[0].mAP)\n    decision = {"candidate_tdcs_oof_mAP": float(comparison.iloc[1].mAP),\n                "previous_tdcs_oof_mAP": float(comparison.iloc[0].mAP), "delta": delta,\n                "decision": "retain as a separate candidate" if delta > 0.002 else "retain previous TDCS baseline",\n                "defog_reported_oof_mAP_retained": 0.817,\n                "score_scope": "TDCS local OOF only; no competition submission or leaderboard score"}\n    (run_dir / "decision.json").write_text(json.dumps(decision, indent=2), encoding="utf-8")\n    (output_root / "latest_run.json").write_text(json.dumps({"run_dir": str(run_dir), **decision}, indent=2), encoding="utf-8")\n    print(comparison.to_string(index=False), flush=True)\n    print(f"TDCS mAP delta: {delta:+.4f}", flush=True)\n    return {"run_dir": run_dir, "manifest": manifest, "audit": audit, "metrics": metrics,\n            "fold_metrics": fold_metrics, "recording_metrics": recording_metrics,\n            "comparison": comparison, "figures": figures, "decision": decision}\n\n\nif __name__ == "__main__":\n    run_experiment()\n', 'tdcs_two_target_neural': '"""Fresh two-output TDCS sequence models, with frozen subject-held-out evaluation.\n\nNo Turn head, no prior neural checkpoint, no leaderboard/submission call. Models\nuse event/subject-balanced training and longer context only at inference. Fixed\nepochs and EMA checkpoints avoid selecting epochs on outer evaluation labels.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport copy\nimport hashlib\nimport json\nimport math\nimport os\nimport random\nimport sys\nimport time\nfrom dataclasses import asdict, dataclass, replace\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom sklearn.metrics import average_precision_score\n\nfrom tdcs_multiscale_signal_experiment import AXES, FROZEN_SUBJECT_FOLDS\n\nTARGETS = (\'StartHesitation\', \'Walking\')\nROOT = Path(__file__).resolve().parents[1]\n\n\n@dataclass(frozen=True)\nclass Config:\n    name: str = \'two_target_sequence_v1\'\n    targets: tuple = TARGETS\n    variants: tuple = (\'bigru_bce\', \'gated_bigru_rank\')\n    folds: tuple = (0, 1, 2, 3, 4)\n    epochs: int = 24\n    steps_per_epoch: int = 144\n    batch_size: int = 32\n    train_length: int = 1000\n    inference_length: int = 5000\n    inference_batch: int = 4\n    hidden: int = 128\n    gru_layers: int = 3\n    width: int = 64\n    dropout: float = 0.15\n    learning_rate: float = 0.001\n    weight_decay: float = 0.0001\n    event_fraction: float = 0.5\n    negative_fraction: float = 0.25\n    rank_weight: float = 0.1\n    ema_decay: float = 0.995\n    seed: int = 731\n    goal_two_target_map: float = 0.6\n\n\nCFG = Config()\n\n\ndef write_json(path: Path, payload: dict):\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(\'.tmp.json\')\n    temporary.write_text(json.dumps(payload, indent=2, default=str), encoding=\'utf-8\')\n    os.replace(temporary, path)\n\n\ndef locate_data():\n    candidates = [ROOT / \'data/metadata\', Path.cwd() / \'data/metadata\']\n    mount = Path(\'/kaggle/input\')\n    if mount.exists():\n        candidates = [p.parent for p in sorted(mount.rglob(\'tdcsfog_metadata.csv\'))] + candidates\n    for metadata_root in candidates:\n        metadata_path = metadata_root / \'tdcsfog_metadata.csv\'\n        for raw in [metadata_root / \'train/tdcsfog\', metadata_root.parent / \'raw/train/tdcsfog\']:\n            if metadata_path.exists() and raw.is_dir():\n                return raw, metadata_path\n    raise FileNotFoundError(\'Attach the competition data: need tdcsfog_metadata.csv and train/tdcsfog\')\n\n\ndef build_manifest(raw_root: Path, metadata_path: Path):\n    metadata = pd.read_csv(metadata_path, dtype={\'Id\': str, \'Subject\': str})\n    if metadata.Id.duplicated().any():\n        raise ValueError(\'Duplicate recording metadata\')\n    subject_by_id = metadata.set_index(\'Id\').Subject.to_dict()\n    rows = []\n    for path in sorted(raw_root.glob(\'*.csv\')):\n        subject = subject_by_id.get(path.stem)\n        if subject == \'4dc2f8\':\n            continue\n        if subject not in FROZEN_SUBJECT_FOLDS:\n            raise ValueError(f\'Unexpected/missing subject for recording {path.stem}\')\n        rows.append({\'recording_id\': path.stem, \'subject\': subject,\n                     \'fold\': FROZEN_SUBJECT_FOLDS[subject], \'path\': str(path), \'source_bytes\': path.stat().st_size})\n    manifest = pd.DataFrame(rows)\n    if len(manifest) != 815 or manifest.subject.nunique() != 61:\n        raise ValueError(\'Frozen development population must be 815 recordings / 61 subjects\')\n    if manifest.groupby(\'subject\').fold.nunique().max() != 1:\n        raise ValueError(\'Subject leakage between folds\')\n    return manifest\n\n\ndef sensor_features(raw: np.ndarray):\n    raw = np.asarray(raw, np.float32)\n    if raw.ndim != 2 or raw.shape[1] != 3 or len(raw) < 2 or not np.isfinite(raw).all():\n        raise ValueError(\'Need a finite recording with at least two rows and three accelerometer axes\')\n    lag = np.vstack([np.zeros((1, 3), np.float32), np.diff(raw, axis=0)])\n    lead = np.vstack([np.diff(raw, axis=0), np.zeros((1, 3), np.float32)])\n    integrated = np.cumsum(raw - np.median(raw, axis=0, keepdims=True), axis=0, dtype=np.float64).astype(np.float32)\n    values = np.concatenate([raw, lag, lead, integrated], axis=1)\n    median = np.median(values, axis=0)\n    q25, q75 = np.quantile(values, [0.25, 0.75], axis=0)\n    values = (values - median) / np.maximum(q75 - q25, 1e-4)\n    position = np.arange(len(raw), dtype=np.float64)\n    seconds = position / 128.0\n    timing = np.column_stack([position / max(len(raw) - 1, 1), np.log1p(seconds) / 8,\n                              np.log1p((len(raw) - 1 - position) / 128.0) / 8])\n    features = np.column_stack([np.clip(values, -20, 20), timing]).astype(np.float32)\n    if not np.isfinite(features).all():\n        raise ValueError(\'Invalid robust features\')\n    return features\n\n\ndef event_centers(labels: np.ndarray):\n    labels = np.asarray(labels, bool)\n    starts = np.flatnonzero(labels & ~np.r_[False, labels[:-1]])\n    stops = np.flatnonzero(labels & ~np.r_[labels[1:], False]) + 1\n    return ((starts + stops - 1) // 2).astype(np.int64)\n\n\ndef prepare_records(manifest: pd.DataFrame):\n    records = []\n    for index, row in enumerate(manifest.itertuples(index=False)):\n        frame = pd.read_csv(row.path, usecols=[*AXES, *TARGETS])\n        y = frame.loc[:, list(TARGETS)].to_numpy()\n        if not np.isin(y, [0, 1]).all():\n            raise ValueError(\'Targets must be binary\')\n        x = sensor_features(frame.loc[:, list(AXES)].to_numpy(np.float32))\n        negative = np.flatnonzero(y.sum(axis=1) == 0)\n        negative = negative[::max(1, len(negative) // 2000)]\n        records.append({\'recording_id\': row.recording_id, \'subject\': row.subject, \'fold\': row.fold,\n                        \'x\': x, \'y\': y.astype(np.float32), \'negative\': negative,\n                        \'centers\': [event_centers(y[:, i]) for i in range(2)]})\n        if (index + 1) % 100 == 0:\n            print(f\'Prepared sensor records: {index + 1}/{len(manifest)}\', flush=True)\n    if sum(len(r[\'y\']) for r in records) != 6959762:\n        raise ValueError(\'Native population changed\')\n    return records\n\n\ndef padded_window(x, y, start, length):\n    indices = np.arange(start, start + length)\n    valid = (indices >= 0) & (indices < len(x))\n    window = x[np.clip(indices, 0, len(x) - 1)].copy()\n    labels = np.zeros((length, 2), np.float32)\n    labels[valid] = y[indices[valid]]\n    return window, labels, np.repeat(valid[:, None], 2, axis=1).astype(np.float32)\n\n\nclass EventSampler:\n    def __init__(self, records, held_out_fold, cfg):\n        self.cfg = cfg\n        self.records = [r for r in records if r[\'fold\'] != held_out_fold]\n        self.held_out_fold = held_out_fold\n        if not self.records or any(r[\'fold\'] == held_out_fold for r in self.records):\n            raise ValueError(\'Training records cross the held-out fold\')\n        self.subjects = {}\n        self.positive = [{}, {}]\n        self.negative = {}\n        for record in self.records:\n            subject = record[\'subject\']\n            self.subjects.setdefault(subject, []).append(record)\n            if len(record[\'negative\']):\n                self.negative.setdefault(subject, []).append(record)\n            for target in range(2):\n                if len(record[\'centers\'][target]):\n                    self.positive[target].setdefault(subject, []).append(record)\n        self.available_targets = [t for t in range(2) if self.positive[t]]\n        if len(self.available_targets) != 2:\n            raise ValueError(\'Both targets require training events\')\n\n    @staticmethod\n    def choose_record(pool, rng):\n        subjects = sorted(pool)\n        subject = subjects[int(rng.integers(len(subjects)))]\n        choices = pool[subject]\n        return choices[int(rng.integers(len(choices)))]\n\n    def plan(self, rng):\n        draw = rng.random()\n        if draw < self.cfg.event_fraction:\n            target = int(rng.choice(self.available_targets))\n            record = self.choose_record(self.positive[target], rng)\n            center = int(rng.choice(record[\'centers\'][target]))\n            center += int(rng.integers(-self.cfg.train_length // 5, self.cfg.train_length // 5 + 1))\n        elif draw < self.cfg.event_fraction + self.cfg.negative_fraction and self.negative:\n            record = self.choose_record(self.negative, rng)\n            center = int(rng.choice(record[\'negative\']))\n        else:\n            record = self.choose_record(self.subjects, rng)\n            center = int(rng.integers(len(record[\'x\'])))\n        center = int(np.clip(center, 0, len(record[\'x\']) - 1))\n        return record, center - self.cfg.train_length // 2\n\n    def batch(self, rng):\n        x_parts, y_parts, masks = [], [], []\n        for _ in range(self.cfg.batch_size):\n            record, start = self.plan(rng)\n            x, y, mask = padded_window(record[\'x\'], record[\'y\'], start, self.cfg.train_length)\n            # Mild amplitude/axis perturbation, applied coherently to all four 3-axis blocks.\n            gains = rng.uniform(0.95, 1.05, 3).astype(np.float32)\n            x[:, :12] *= np.tile(gains, 4)\n            theta = float(rng.uniform(-0.10, 0.10))\n            rotation = np.array([[1, 0, 0], [0, np.cos(theta), -np.sin(theta)],\n                                 [0, np.sin(theta), np.cos(theta)]], np.float32)\n            for begin in range(0, 12, 3):\n                x[:, begin:begin + 3] = x[:, begin:begin + 3] @ rotation.T\n            x_parts.append(x)\n            y_parts.append(y)\n            masks.append(mask)\n        return tuple(torch.from_numpy(np.stack(parts)) for parts in [x_parts, y_parts, masks])\n\n\nclass GatedBlock(nn.Module):\n    def __init__(self, width, dilation):\n        super().__init__()\n        self.filter = nn.Conv1d(width, width, 3, padding=dilation, dilation=dilation)\n        self.gate = nn.Conv1d(width, width, 3, padding=dilation, dilation=dilation)\n        self.output = nn.Conv1d(width, width, 1)\n        self.norm = nn.GroupNorm(8, width)\n\n    def forward(self, x):\n        h = torch.tanh(self.filter(x)) * torch.sigmoid(self.gate(x))\n        return x + self.output(self.norm(h))\n\n\nclass TwoTargetGRU(nn.Module):\n    def __init__(self, cfg=CFG, variant=\'bigru_bce\'):\n        super().__init__()\n        if variant not in (\'bigru_bce\', \'gated_bigru_rank\'):\n            raise ValueError(\'Unknown neural variant\')\n        self.input = nn.Sequential(nn.Linear(15, cfg.width), nn.LayerNorm(cfg.width), nn.GELU())\n        self.gated = nn.Sequential(*[GatedBlock(cfg.width, d) for d in (1, 2, 4, 8)]) if variant.startswith(\'gated\') else None\n        self.gru = nn.GRU(cfg.width, cfg.hidden, num_layers=cfg.gru_layers,\n                          dropout=cfg.dropout if cfg.gru_layers > 1 else 0,\n                          batch_first=True, bidirectional=True)\n        self.head = nn.Sequential(nn.Linear(2 * cfg.hidden, 128), nn.LayerNorm(128), nn.GELU(),\n                                  nn.Dropout(cfg.dropout), nn.Linear(128, 2))\n\n    def forward(self, x):\n        h = self.input(x)\n        if self.gated is not None:\n            h = self.gated(h.transpose(1, 2)).transpose(1, 2)\n        h, _ = self.gru(h.contiguous())\n        return self.head(h)\n\n\ndef sequence_loss(logits, labels, mask, rank_weight=0.0):\n    # Sampling already increases event frequency: do NOT also apply inverse-frequency positive weights.\n    bce = (F.binary_cross_entropy_with_logits(logits.float(), labels.float(), reduction=\'none\') * mask).sum()\n    loss = bce / mask.sum().clamp_min(1)\n    rank_terms = []\n    if rank_weight:\n        for target in range(2):\n            valid = mask[..., target] > 0\n            pos = logits[..., target][valid & (labels[..., target] > 0.5)].float()\n            neg = logits[..., target][valid & (labels[..., target] < 0.5)].float()\n            if pos.numel() and neg.numel():\n                pos = pos[torch.randperm(len(pos), device=pos.device)[:192]]\n                # Hard negatives are selected from this TRAINING batch, never from outer OOF labels.\n                neg = torch.topk(neg, min(192, len(neg))).values\n                rank_terms.append(F.softplus(neg[None, :] - pos[:, None]).mean())\n    if rank_terms:\n        loss = loss + rank_weight * torch.stack(rank_terms).mean()\n    return loss\n\n\ndef core_model(model):\n    return model.module if isinstance(model, nn.DataParallel) else model\n\n\ndef learning_rate_multiplier(step, total_steps):\n    warmup = max(1, int(total_steps * 0.05))\n    if step < warmup:\n        return max(0.001, (step + 1) / warmup)\n    return max(0.02, (total_steps - step) / max(total_steps - warmup, 1))\n\n\ndef train_model(records, fold, variant, cfg, run_dir, device, fingerprint):\n    fold_dir = run_dir / variant / f\'fold_{fold}\'\n    fold_dir.mkdir(parents=True, exist_ok=True)\n    final_path = fold_dir / \'final_ema.pt\'\n    seed = cfg.seed + 1009 * fold + (1 if variant.startswith(\'gated\') else 0)\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n    network = TwoTargetGRU(cfg, variant).to(device)\n    if final_path.exists():\n        checkpoint = torch.load(final_path, map_location=device, weights_only=False)\n        if checkpoint[\'fingerprint\'] != fingerprint:\n            raise ValueError(\'Cannot reuse a checkpoint from another configuration\')\n        network.load_state_dict(checkpoint[\'model\'])\n        print(f\'Reusing completed {variant} fold {fold}\', flush=True)\n        return network\n    sampler = EventSampler(records, fold, cfg)\n    training_subjects = sorted(sampler.subjects)\n    held_out_subjects = sorted({r[\'subject\'] for r in records if r[\'fold\'] == fold})\n    if set(training_subjects) & set(held_out_subjects):\n        raise ValueError(\'Outer subject leakage\')\n    write_json(fold_dir / \'training_receipt.json\', {\'targets\': TARGETS, \'training_subjects\': training_subjects,\n        \'held_out_subjects\': held_out_subjects, \'outer_labels_used_for_checkpoint_selection\': False,\n        \'event_fraction\': cfg.event_fraction, \'negative_fraction\': cfg.negative_fraction,\n        \'positive_class_weights\': [1, 1], \'fixed_epochs\': cfg.epochs, \'seed\': seed})\n    ema = copy.deepcopy(network).eval()\n    ema.requires_grad_(False)\n    optimizer = torch.optim.AdamW(network.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay)\n    total_steps = cfg.epochs * cfg.steps_per_epoch\n    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lambda step: learning_rate_multiplier(step, total_steps))\n    use_amp = device.type == \'cuda\'\n    scaler = torch.amp.GradScaler(\'cuda\', enabled=use_amp)\n    start_epoch = 0\n    rng = np.random.default_rng(seed)\n    resume_path = fold_dir / \'resume.pt\'\n    if resume_path.exists():\n        saved = torch.load(resume_path, map_location=device, weights_only=False)\n        if saved[\'fingerprint\'] != fingerprint:\n            raise ValueError(\'Stale training resume state\')\n        network.load_state_dict(saved[\'model\'])\n        ema.load_state_dict(saved[\'ema\'])\n        optimizer.load_state_dict(saved[\'optimizer\'])\n        scheduler.load_state_dict(saved[\'scheduler\'])\n        scaler.load_state_dict(saved[\'scaler\'])\n        rng.bit_generator.state = saved[\'numpy_rng\']\n        torch.set_rng_state(saved[\'torch_rng\'].cpu())\n        if use_amp and saved[\'cuda_rng\'] is not None:\n            torch.cuda.set_rng_state_all([state.cpu() for state in saved[\'cuda_rng\']])\n        start_epoch = saved[\'next_epoch\']\n    if use_amp and torch.cuda.device_count() > 1:\n        network = nn.DataParallel(network)\n    history = []\n    history_path = fold_dir / \'training_log.csv\'\n    if history_path.exists():\n        history = pd.read_csv(history_path).to_dict(\'records\')\n    for epoch in range(start_epoch, cfg.epochs):\n        network.train()\n        started = time.monotonic()\n        losses, event_rows, valid_rows = [], np.zeros(2, np.float64), np.zeros(2, np.float64)\n        print(f\'{variant} fold {fold} epoch {epoch + 1}/{cfg.epochs}: training\', flush=True)\n        for step in range(cfg.steps_per_epoch):\n            x, y, mask = sampler.batch(rng)\n            event_rows += (y.numpy() * mask.numpy()).sum(axis=(0, 1))\n            valid_rows += mask.numpy().sum(axis=(0, 1))\n            if use_amp:\n                x, y, mask = [v.pin_memory().to(device, non_blocking=True) for v in (x, y, mask)]\n            else:\n                x, y, mask = [v.to(device) for v in (x, y, mask)]\n            optimizer.zero_grad(set_to_none=True)\n            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):\n                logits = network(x)\n                loss = sequence_loss(logits, y, mask, cfg.rank_weight if variant.startswith(\'gated\') else 0)\n            if not torch.isfinite(loss):\n                raise FloatingPointError(\'Non-finite training loss\')\n            scaler.scale(loss).backward()\n            scaler.unscale_(optimizer)\n            nn.utils.clip_grad_norm_(network.parameters(), 2.0)\n            scaler.step(optimizer)\n            scaler.update()\n            scheduler.step()\n            with torch.no_grad():\n                for destination, source in zip(ema.parameters(), core_model(network).parameters()):\n                    destination.lerp_(source.detach(), 1 - cfg.ema_decay)\n            losses.append(float(loss.detach().cpu()))\n        history.append({\'epoch\': epoch + 1, \'loss\': float(np.mean(losses)),\n                        \'StartHesitation_sampled_fraction\': float(event_rows[0] / valid_rows[0]),\n                        \'Walking_sampled_fraction\': float(event_rows[1] / valid_rows[1]),\n                        \'learning_rate\': scheduler.get_last_lr()[0], \'seconds\': time.monotonic() - started})\n        pd.DataFrame(history).to_csv(history_path, index=False)\n        state = {\'fingerprint\': fingerprint, \'model\': core_model(network).state_dict(), \'ema\': ema.state_dict(),\n                 \'optimizer\': optimizer.state_dict(), \'scheduler\': scheduler.state_dict(), \'scaler\': scaler.state_dict(),\n                 \'next_epoch\': epoch + 1, \'numpy_rng\': rng.bit_generator.state, \'torch_rng\': torch.get_rng_state(),\n                 \'cuda_rng\': torch.cuda.get_rng_state_all() if use_amp else None}\n        temporary = fold_dir / \'resume.tmp.pt\'\n        torch.save(state, temporary)\n        os.replace(temporary, resume_path)\n        write_json(run_dir / \'progress.json\', {\'stage\': \'training\', \'variant\': variant, \'fold\': fold,\n                   \'epoch_completed\': epoch + 1, \'epochs\': cfg.epochs, \'latest_training_loss\': history[-1][\'loss\']})\n        print(f\'{variant} fold {fold}: epoch {epoch + 1}, loss {history[-1]["loss"]:.5f}, {history[-1]["seconds"]:.1f}s\', flush=True)\n    torch.save({\'fingerprint\': fingerprint, \'model\': ema.state_dict(), \'targets\': TARGETS}, final_path)\n    return ema\n\n\ndef inference_starts(n, length):\n    if n <= length:\n        return [0]\n    starts = list(range(0, n - length + 1, max(1, length // 2)))\n    if starts[-1] != n - length:\n        starts.append(n - length)\n    return starts\n\n\ndef inference_weight(length, first, last):\n    weight = np.zeros(length, np.float32)\n    quarter = length // 4\n    weight[quarter:length - quarter] = 1\n    if first:\n        weight[:quarter] = 1\n    if last:\n        weight[length - quarter:] = 1\n    return weight\n\n\ndef predict_record(model, features, cfg, device):\n    n, length = len(features), cfg.inference_length\n    starts = inference_starts(n, length)\n    total = np.zeros((n, 2), np.float32)\n    denominator = np.zeros(n, np.float32)\n    model.eval()\n    with torch.inference_mode():\n        for begin in range(0, len(starts), cfg.inference_batch):\n            selected = starts[begin:begin + cfg.inference_batch]\n            windows = [features[np.clip(np.arange(start, start + length), 0, n - 1)] for start in selected]\n            x = torch.from_numpy(np.stack(windows)).to(device)\n            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == \'cuda\'):\n                p = torch.sigmoid(model(x).float()).cpu().numpy()\n            for i, start in enumerate(selected):\n                count = min(length, n - start)\n                weights = inference_weight(length, start == 0, start + length >= n)[:count]\n                total[start:start + count] += p[i, :count] * weights[:, None]\n                denominator[start:start + count] += weights\n    if (denominator <= 0).any():\n        raise ValueError(\'Inference windows left native rows uncovered\')\n    prediction = total / denominator[:, None]\n    if not np.isfinite(prediction).all():\n        raise ValueError(\'Non-finite prediction\')\n    return np.clip(prediction, 1e-6, 1 - 1e-6)\n\n\ndef average_precision(y, p):\n    result = {target: float(average_precision_score(y[:, i], p[:, i])) for i, target in enumerate(TARGETS)}\n    result[\'two_target_mAP\'] = float(np.mean(list(result.values())))\n    return result\n\n\ndef run_experiment(cfg=CFG, output_root=None, synthetic_records=None):\n    torch.set_num_threads(4)\n    torch.backends.cudnn.benchmark = True\n    device = torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n    if synthetic_records is None and device.type != \'cuda\':\n        raise RuntimeError(\'Full sequence training requires a GPU; run the smoke mode locally instead\')\n    if output_root is None:\n        output_root = Path(\'/kaggle/working/tdcs_two_target_neural\') if Path(\'/kaggle/working\').exists() else ROOT / \'results/tdcs_two_target_neural\'\n    output_root = Path(output_root)\n    if synthetic_records is None:\n        raw_root, metadata_path = locate_data()\n        manifest = build_manifest(raw_root, metadata_path)\n        source_population = manifest[[\'recording_id\', \'subject\', \'fold\', \'source_bytes\']].to_csv(index=False)\n        metadata_sha = hashlib.sha256(metadata_path.read_bytes()).hexdigest()\n    else:\n        manifest = pd.DataFrame([{key: record[key] for key in [\'recording_id\', \'subject\', \'fold\']} for record in synthetic_records])\n        source_population = manifest.to_csv(index=False)\n        metadata_sha = \'synthetic smoke only\'\n    source_sha = globals().get(\'SOURCE_SHA256\') or hashlib.sha256(Path(__file__).read_bytes()).hexdigest()\n    config = {\'configuration\': asdict(cfg), \'source_sha256\': source_sha,\n              \'runtime_versions\': {\'torch\': str(torch.__version__), \'numpy\': np.__version__, \'pandas\': pd.__version__},\n              \'frozen_fold_sha256\': hashlib.sha256(json.dumps(FROZEN_SUBJECT_FOLDS, sort_keys=True).encode()).hexdigest(),\n              \'metadata_sha256\': metadata_sha, \'population_sha256\': hashlib.sha256(source_population.encode()).hexdigest(),\n              \'validation\': \'frozen outer subject folds; all native rows; no outer-label checkpoint selection\',\n              \'confidence\': \'independent sigmoid scores; event oversampling means not calibrated probability\',\n              \'leaderboard_scoring_used\': False, \'competition_submission_created\': False,\n              \'previous_two_target_reference\': 0.19629988455476777, \'goal_is_measured_not_guaranteed\': 0.6,\n              \'smoke_only\': synthetic_records is not None}\n    fingerprint = hashlib.sha256(json.dumps(config, sort_keys=True).encode()).hexdigest()[:20]\n    run_dir = output_root / fingerprint\n    run_dir.mkdir(parents=True, exist_ok=True)\n    write_json(run_dir / \'configuration.json\', config)\n    manifest.to_csv(run_dir / \'frozen_tdcs_manifest.csv\', index=False)\n    write_json(output_root / \'latest_run.json\', {\'run_dir\': str(run_dir), \'status\': \'running\'})\n    print(f\'Fresh TDCS neural run: {run_dir}; torch {torch.__version__}; {device}; GPUs {torch.cuda.device_count()}\', flush=True)\n    records = prepare_records(manifest) if synthetic_records is None else synthetic_records\n    model_metrics = []\n    for variant in cfg.variants:\n        for fold in cfg.folds:\n            model = train_model(records, fold, variant, cfg, run_dir, device, fingerprint)\n            oof_dir = run_dir / variant / \'oof\'\n            oof_dir.mkdir(parents=True, exist_ok=True)\n            ys, ps = [], []\n            for record in records:\n                if record[\'fold\'] != fold:\n                    continue\n                path = oof_dir / f\'{record["recording_id"]}.npz\'\n                if path.exists():\n                    with np.load(path, allow_pickle=False) as payload:\n                        p = payload[\'prediction\']\n                else:\n                    p = predict_record(model, record[\'x\'], cfg, device)\n                    np.savez_compressed(path, prediction=p, targets=np.asarray(TARGETS))\n                if p.shape != record[\'y\'].shape or not np.isfinite(p).all():\n                    raise ValueError(\'Native-row OOF alignment failure\')\n                ys.append(record[\'y\'])\n                ps.append(p)\n            score = {\'variant\': variant, \'fold\': fold, **average_precision(np.concatenate(ys), np.concatenate(ps))}\n            model_metrics.append(score)\n            pd.DataFrame(model_metrics).to_csv(run_dir / \'fold_metrics.csv\', index=False)\n            write_json(run_dir / \'progress.json\', {\'stage\': \'fold_scored\', **score})\n            print(json.dumps(score), flush=True)\n            del model\n            if device.type == \'cuda\':\n                torch.cuda.empty_cache()\n    y = np.concatenate([record[\'y\'] for record in records if record[\'fold\'] in cfg.folds])\n    predictions = {}\n    for variant in cfg.variants:\n        parts = []\n        for record in records:\n            if record[\'fold\'] in cfg.folds:\n                with np.load(run_dir / variant / \'oof\' / f\'{record["recording_id"]}.npz\', allow_pickle=False) as payload:\n                    parts.append(payload[\'prediction\'])\n        predictions[variant] = np.concatenate(parts)\n    predictions[\'neural_equal_ensemble\'] = sum(predictions.values()) / len(predictions)\n    metrics = pd.DataFrame([{\'method\': method, **average_precision(y, p)} for method, p in predictions.items()])\n    metrics.to_csv(run_dir / \'candidate_metrics.csv\', index=False)\n    selected = metrics.sort_values(\'two_target_mAP\', ascending=False).iloc[0]\n    selected_dir = run_dir / \'selected_candidate\'\n    (selected_dir / \'oof\').mkdir(parents=True, exist_ok=True)\n    manifest.to_csv(selected_dir / \'frozen_tdcs_manifest.csv\', index=False)\n    pd.DataFrame([selected]).to_csv(selected_dir / \'native_oof_metrics.csv\', index=False)\n    offset = 0\n    for record in records:\n        if record[\'fold\'] not in cfg.folds:\n            continue\n        n = len(record[\'y\'])\n        np.savez_compressed(selected_dir / \'oof\' / f\'{record["recording_id"]}.npz\',\n                            prediction=predictions[str(selected.method)][offset:offset + n], targets=np.asarray(TARGETS))\n        offset += n\n    decision = {\'run_dir\': str(run_dir), \'selected_candidate_dir\': str(selected_dir), \'status\': \'complete\',\n                \'selected_method\': str(selected.method), \'two_target_mAP\': float(selected.two_target_mAP),\n                \'class_average_precision\': {target: float(selected[target]) for target in TARGETS},\n                \'reference_two_target_map\': 0.19629988455476777,\n                \'target_two_target_map\': 0.6, \'target_reached\': bool(selected.two_target_mAP >= 0.6) and synthetic_records is None,\n                \'smoke_only\': synthetic_records is not None, \'native_rows_scored\': len(y),\n                \'no_outer_labels_in_training\': True, \'competition_submission_created\': False,\n                \'leaderboard_scoring_used\': False, \'independent_generalization_established\': False,\n                \'retained_defog_reported_oof_map\': 0.817}\n    write_json(run_dir / \'decision.json\', decision)\n    write_json(output_root / \'latest_run.json\', decision)\n    write_json(run_dir / \'progress.json\', {\'stage\': \'complete\', **decision})\n    print(metrics.to_string(index=False), flush=True)\n    print(json.dumps(decision, indent=2), flush=True)\n    return decision\n\n\ndef smoke_experiment(output_root=None):\n    cfg = replace(CFG, name=\'synthetic_smoke_only\', epochs=1, steps_per_epoch=2, batch_size=2,\n                  train_length=32, inference_length=64, inference_batch=2, width=16, hidden=8, gru_layers=1)\n    rng = np.random.default_rng(1)\n    records = []\n    for fold in range(5):\n        raw = rng.normal(size=(96 + fold, 3)).astype(np.float32)\n        y = np.zeros((len(raw), 2), np.float32)\n        y[16:28, 0] = 1\n        y[55:67, 1] = 1\n        records.append({\'recording_id\': f\'smoke_{fold}\', \'subject\': f\'smoke_subject_{fold}\', \'fold\': fold,\n                        \'x\': sensor_features(raw), \'y\': y, \'negative\': np.flatnonzero(y.sum(axis=1) == 0),\n                        \'centers\': [event_centers(y[:, target]) for target in range(2)]})\n    return run_experiment(cfg, output_root or ROOT / \'tmp/tdcs_neural_smoke\', records)\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--smoke\', action=\'store_true\', help=\'Run synthetic CPU wiring checks, not a performance experiment\')\n    parser.add_argument(\'--output-root\', type=Path)\n    args = parser.parse_args()\n    if args.smoke:\n        smoke_experiment(args.output_root)\n    else:\n        run_experiment(output_root=args.output_root)\n', 'tdcs_alternative_architectures': '"""Independent CNN / patch-attention TDCS experiments; never overwrite retained heads.\n\nDense three-target adaptations inspired by InceptionTime and patch-based attention,\nnot reproductions of either paper. Two-target and three-target mAP stay separate.\n"""\nfrom __future__ import annotations\nimport argparse\nimport copy\nimport hashlib\nimport json\nimport math\nimport os\nimport time\nfrom dataclasses import asdict, dataclass, replace\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom sklearn.metrics import average_precision_score\n\nimport tdcs_two_target_neural as common\n\nROOT = Path(__file__).resolve().parents[1]\nTARGETS = (\'StartHesitation\', \'Turn\', \'Walking\')\nARCHITECTURES = (\'multiscale_cnn\', \'patch_transformer\')\n\n\n@dataclass(frozen=True)\nclass Config:\n    name: str = \'tdcs_alternatives_v2\'\n    targets: tuple = TARGETS\n    folds: tuple = (0, 1, 2, 3, 4)\n    epochs: int = 24\n    steps_per_epoch: int = 96\n    batch_size: int = 16\n    window: int = 2048\n    inference_batch: int = 4\n    width: int = 64\n    dilations: tuple = (1, 2, 4, 8, 16, 32)\n    patch: int = 32\n    attention_width: int = 96\n    attention_layers: int = 3\n    dropout: float = 0.25\n    weight_decay: float = 0.01\n    learning_rate: float = 0.0007\n    event_fraction: float = 0.5\n    negative_fraction: float = 0.25\n    ema_decay: float = 0.995\n    seed: int = 1207\n\n\nCFG = Config()\n\n\ndef prepare_records(manifest):\n    records = []\n    for index, row in enumerate(manifest.itertuples(index=False)):\n        frame = pd.read_csv(row.path, usecols=[*common.AXES, *TARGETS])\n        y = frame.loc[:, list(TARGETS)].to_numpy(np.float32)\n        assert np.isin(y, [0, 1]).all()\n        # Sensor-only: remove the three absolute recording-progress features.\n        x = common.sensor_features(frame.loc[:, list(common.AXES)].to_numpy(np.float32))[:, :12].copy()\n        negatives = np.flatnonzero(y.sum(axis=1) == 0)\n        negatives = negatives[::max(1, len(negatives) // 2000)]\n        records.append({\'recording_id\': row.recording_id, \'subject\': row.subject, \'fold\': row.fold,\n            \'x\': x, \'y\': y, \'negative\': negatives, \'centers\': [common.event_centers(y[:, i]) for i in range(3)]})\n        if (index + 1) % 100 == 0:\n            print(f\'Prepared {index+1}/{len(manifest)} recordings\', flush=True)\n    assert sum(len(record[\'y\']) for record in records) == 6959762\n    return records\n\n\ndef window(record, start, length):\n    indices = np.arange(start, start + length)\n    valid = (indices >= 0) & (indices < len(record[\'x\']))\n    x = record[\'x\'][np.clip(indices, 0, len(record[\'x\']) - 1)].copy()\n    y = np.zeros((length, 3), np.float32)\n    y[valid] = record[\'y\'][indices[valid]]\n    return x, y, np.repeat(valid[:, None], 3, axis=1).astype(np.float32)\n\n\nclass Sampler:\n    def __init__(self, records, fold, cfg):\n        self.cfg = cfg\n        self.records = [record for record in records if record[\'fold\'] != fold]\n        self.subjects, self.negative = {}, {}\n        self.positive = [{}, {}, {}]\n        for record in self.records:\n            subject = record[\'subject\']\n            self.subjects.setdefault(subject, []).append(record)\n            if len(record[\'negative\']):\n                self.negative.setdefault(subject, []).append(record)\n            for target in range(3):\n                if len(record[\'centers\'][target]):\n                    self.positive[target].setdefault(subject, []).append(record)\n        if not self.subjects or any(not pool for pool in self.positive):\n            raise ValueError(\'All targets require training events; no held-out records may be sampled\')\n\n    def plan(self, rng):\n        draw = rng.random()\n        if draw < self.cfg.event_fraction:\n            target = int(rng.integers(3))\n            record = common.EventSampler.choose_record(self.positive[target], rng)\n            center = int(rng.choice(record[\'centers\'][target]))\n            center += int(rng.integers(-self.cfg.window // 4, self.cfg.window // 4 + 1))\n        elif draw < self.cfg.event_fraction + self.cfg.negative_fraction and self.negative:\n            record = common.EventSampler.choose_record(self.negative, rng)\n            center = int(rng.choice(record[\'negative\']))\n        else:\n            record = common.EventSampler.choose_record(self.subjects, rng)\n            center = int(rng.integers(len(record[\'x\'])))\n        center = int(np.clip(center, 0, len(record[\'x\']) - 1))\n        return record, center - self.cfg.window // 2\n\n    def batch(self, rng):\n        xs, ys, masks = [], [], []\n        for _ in range(self.cfg.batch_size):\n            record, start = self.plan(rng)\n            x, y, mask = window(record, start, self.cfg.window)\n            gains = np.tile(rng.uniform(.9, 1.1, 3).astype(np.float32), 4)\n            x *= gains\n            angle = rng.uniform(-.15, .15)\n            rotation = np.asarray([[1, 0, 0], [0, np.cos(angle), -np.sin(angle)],\n                                    [0, np.sin(angle), np.cos(angle)]], np.float32)\n            for begin in range(0, 12, 3):\n                x[:, begin:begin+3] = x[:, begin:begin+3] @ rotation.T\n            if rng.random() < .25:\n                begin = int(rng.integers(len(x)))\n                x[begin:begin + min(64, len(x) // 8)] = 0\n            xs.append(x)\n            ys.append(y)\n            masks.append(mask)\n        return tuple(torch.from_numpy(np.stack(parts)) for parts in [xs, ys, masks])\n\n\nclass MultiscaleBlock(nn.Module):\n    def __init__(self, width, dilation, dropout):\n        super().__init__()\n        self.branches = nn.ModuleList([nn.Conv1d(width, width, kernel, groups=width,\n            dilation=dilation, padding=dilation * (kernel - 1) // 2) for kernel in (3, 7, 15)])\n        self.mix = nn.Sequential(nn.Conv1d(width * 3, width, 1), nn.GroupNorm(8, width),\n                                 nn.GELU(), nn.Dropout(dropout))\n\n    def forward(self, x):\n        return x + self.mix(torch.cat([branch(x) for branch in self.branches], dim=1))\n\n\nclass AlternativeNetwork(nn.Module):\n    def __init__(self, architecture, cfg=CFG):\n        super().__init__()\n        if architecture not in ARCHITECTURES:\n            raise ValueError(\'Unknown architecture\')\n        self.architecture = architecture\n        self.stem = nn.Sequential(nn.Conv1d(12, cfg.width, 7, padding=3), nn.GroupNorm(8, cfg.width), nn.GELU())\n        if architecture == \'multiscale_cnn\':\n            self.temporal = nn.Sequential(*[MultiscaleBlock(cfg.width, dilation, cfg.dropout) for dilation in cfg.dilations])\n            self.head = nn.Sequential(nn.Dropout(cfg.dropout), nn.Conv1d(cfg.width, 3, 1))\n        else:\n            self.patch = nn.Conv1d(cfg.width, cfg.attention_width, cfg.patch,\n                                    stride=max(1, cfg.patch // 2), padding=cfg.patch // 2)\n            layer = nn.TransformerEncoderLayer(cfg.attention_width, 4, cfg.attention_width * 3,\n                dropout=cfg.dropout, activation=\'gelu\', batch_first=True, norm_first=True)\n            self.temporal = nn.TransformerEncoder(layer, cfg.attention_layers, enable_nested_tensor=False)\n            self.head = nn.Sequential(nn.Conv1d(cfg.width + cfg.attention_width, cfg.width, 3, padding=1),\n                                      nn.GELU(), nn.Dropout(cfg.dropout), nn.Conv1d(cfg.width, 3, 1))\n\n    def forward(self, x):\n        h = self.stem(x.transpose(1, 2))\n        if self.architecture == \'multiscale_cnn\':\n            return self.head(self.temporal(h)).transpose(1, 2)\n        patches = self.patch(h).transpose(1, 2)\n        position = torch.arange(patches.shape[1], device=x.device, dtype=torch.float32)[:, None]\n        frequency = torch.exp(torch.arange(0, patches.shape[2], 2, device=x.device) * (-math.log(10000) / patches.shape[2]))\n        encoding = torch.zeros(patches.shape[1:], device=x.device, dtype=torch.float32)\n        encoding[:, 0::2], encoding[:, 1::2] = torch.sin(position * frequency), torch.cos(position * frequency)\n        patches = self.temporal(patches + encoding.to(patches.dtype))\n        context = F.interpolate(patches.transpose(1, 2), size=h.shape[2], mode=\'linear\', align_corners=False)\n        return self.head(torch.cat([h, context], dim=1)).transpose(1, 2)\n\n\ndef masked_loss(logits, y, mask):\n    loss = F.binary_cross_entropy_with_logits(logits.float(), y.float(), reduction=\'none\')\n    return (loss * mask).sum() / mask.sum().clamp_min(1)\n\n\ndef train_model(records, fold, architecture, cfg, directory, device, fingerprint):\n    directory.mkdir(parents=True, exist_ok=True)\n    seed = cfg.seed + 1009 * fold\n    torch.manual_seed(seed)\n    if device.type == \'cuda\':\n        torch.cuda.manual_seed_all(seed)\n    model = AlternativeNetwork(architecture, cfg).to(device)\n    final = directory / \'final_ema.pt\'\n    if final.exists():\n        state = torch.load(final, map_location=device, weights_only=False)\n        assert state[\'fingerprint\'] == fingerprint and tuple(state[\'targets\']) == TARGETS\n        model.load_state_dict(state[\'model\'])\n        return model\n    sampler = Sampler(records, fold, cfg)\n    training = sorted(sampler.subjects)\n    held_out = sorted({record[\'subject\'] for record in records if record[\'fold\'] == fold})\n    assert set(training).isdisjoint(held_out)\n    common.write_json(directory / \'training_receipt.json\', {\'targets\': TARGETS, \'training_subjects\': training,\n        \'held_out_subjects\': held_out, \'fixed_epochs\': cfg.epochs, \'outer_labels_used_for_checkpoint_selection\': False,\n        \'architecture\': architecture, \'sensor_features\': 12, \'training_window\': cfg.window,\n        \'inference_window\': cfg.window, \'previous_models_overwritten\': False, \'seed\': seed})\n    ema = copy.deepcopy(model).eval().requires_grad_(False)\n    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay)\n    schedule = torch.optim.lr_scheduler.LambdaLR(optimizer,\n        lambda step: common.learning_rate_multiplier(step, cfg.epochs * cfg.steps_per_epoch))\n    scaler = torch.amp.GradScaler(\'cuda\', enabled=device.type == \'cuda\')\n    rng = np.random.default_rng(seed)\n    resume, history, first = directory / \'resume.pt\', [], 0\n    if resume.exists():\n        state = torch.load(resume, map_location=device, weights_only=False)\n        assert state[\'fingerprint\'] == fingerprint\n        model.load_state_dict(state[\'model\'])\n        ema.load_state_dict(state[\'ema\'])\n        optimizer.load_state_dict(state[\'optimizer\'])\n        schedule.load_state_dict(state[\'scheduler\'])\n        scaler.load_state_dict(state[\'scaler\'])\n        rng.bit_generator.state = state[\'numpy_rng\']\n        torch.set_rng_state(state[\'torch_rng\'].cpu())\n        if device.type == \'cuda\':\n            torch.cuda.set_rng_state_all([value.cpu() for value in state[\'cuda_rng\']])\n        first, history = state[\'next_epoch\'], state[\'history\']\n    if device.type == \'cuda\' and torch.cuda.device_count() > 1:\n        model = nn.DataParallel(model)\n    for epoch in range(first, cfg.epochs):\n        started, losses = time.monotonic(), []\n        model.train()\n        print(f\'{architecture} fold {fold} epoch {epoch+1}/{cfg.epochs}\', flush=True)\n        for _ in range(cfg.steps_per_epoch):\n            x, y, mask = [value.to(device) for value in sampler.batch(rng)]\n            optimizer.zero_grad(set_to_none=True)\n            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == \'cuda\'):\n                loss = masked_loss(model(x), y, mask)\n            if not torch.isfinite(loss):\n                raise FloatingPointError(\'Non-finite training loss\')\n            scaler.scale(loss).backward()\n            scaler.unscale_(optimizer)\n            nn.utils.clip_grad_norm_(model.parameters(), 2)\n            scaler.step(optimizer)\n            scaler.update()\n            schedule.step()\n            with torch.no_grad():\n                for destination, source in zip(ema.parameters(), common.core_model(model).parameters()):\n                    destination.lerp_(source.detach(), 1-cfg.ema_decay)\n            losses.append(float(loss.detach().cpu()))\n        history.append({\'epoch\': epoch+1, \'loss\': float(np.mean(losses)), \'seconds\': time.monotonic()-started,\n                        \'learning_rate\': schedule.get_last_lr()[0]})\n        pd.DataFrame(history).to_csv(directory / \'training_log.csv\', index=False)\n        state = {\'fingerprint\': fingerprint, \'model\': common.core_model(model).state_dict(), \'ema\': ema.state_dict(),\n            \'optimizer\': optimizer.state_dict(), \'scheduler\': schedule.state_dict(), \'scaler\': scaler.state_dict(),\n            \'numpy_rng\': rng.bit_generator.state, \'torch_rng\': torch.get_rng_state(),\n            \'cuda_rng\': torch.cuda.get_rng_state_all() if device.type == \'cuda\' else [],\n            \'next_epoch\': epoch+1, \'history\': history}\n        temporary = directory / \'resume.tmp.pt\'\n        torch.save(state, temporary)\n        os.replace(temporary, resume)\n        common.write_json(directory.parent / \'progress.json\', {\'stage\': \'training\', \'architecture\': architecture,\n            \'fold\': fold, \'epoch\': epoch+1, \'fixed_epochs\': cfg.epochs, \'training_loss\': history[-1][\'loss\']})\n    torch.save({\'fingerprint\': fingerprint, \'model\': ema.state_dict(), \'targets\': TARGETS}, final)\n    return ema\n\n\ndef predict_record(model, features, cfg, device):\n    length, n = cfg.window, len(features)\n    starts = common.inference_starts(n, length)\n    total, counts = np.zeros((n, 3), np.float32), np.zeros(n, np.float32)\n    model.eval()\n    with torch.inference_mode():\n        for begin in range(0, len(starts), cfg.inference_batch):\n            batch = starts[begin:begin+cfg.inference_batch]\n            windows = [features[np.clip(np.arange(start, start+length), 0, n-1)] for start in batch]\n            x = torch.from_numpy(np.stack(windows)).to(device)\n            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == \'cuda\'):\n                p = torch.sigmoid(model(x).float()).cpu().numpy()\n            for i, start in enumerate(batch):\n                count = min(length, n-start)\n                weight = common.inference_weight(length, start == 0, start+length >= n)[:count]\n                total[start:start+count] += p[i, :count] * weight[:, None]\n                counts[start:start+count] += weight\n    if (counts <= 0).any():\n        raise ValueError(\'Uncovered native rows\')\n    result = np.clip(total / counts[:, None], 1e-6, 1-1e-6)\n    assert np.isfinite(result).all()\n    return result\n\n\ndef score(y, p):\n    assert y.shape == p.shape and y.shape[1] == 3\n    values = {target: float(average_precision_score(y[:, i], p[:, i])) for i, target in enumerate(TARGETS)}\n    return {**values, \'three_target_mAP\': float(np.mean(list(values.values()))),\n            \'two_target_mAP\': float((values[\'StartHesitation\'] + values[\'Walking\']) / 2)}\n\n\ndef run_experiment(architecture, cfg=CFG, output_root=None, synthetic_records=None):\n    torch.set_num_threads(4)\n    if architecture not in ARCHITECTURES or tuple(cfg.targets) != TARGETS:\n        raise ValueError(\'Unexpected architecture or target order\')\n    device = torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n    if synthetic_records is None and device.type != \'cuda\':\n        raise RuntimeError(\'Full training requires a CUDA GPU; no automatic smoke fallback\')\n    manifest = None\n    if synthetic_records is None:\n        raw, metadata = common.locate_data()\n        manifest = common.build_manifest(raw, metadata)\n        population = manifest[[\'recording_id\', \'subject\', \'fold\', \'source_bytes\']].to_csv(index=False, lineterminator=\'\\n\')\n    else:\n        population = \'synthetic wiring only; not TDCS performance\'\n    configuration = {\'configuration\': asdict(cfg), \'architecture\': architecture, \'targets\': TARGETS,\n        \'source_sha256\': globals().get(\'SOURCE_SHA256\') or hashlib.sha256(Path(__file__).read_bytes()).hexdigest(),\n        \'common_source_sha256\': globals().get(\'COMMON_SOURCE_SHA256\') or hashlib.sha256(Path(common.__file__).read_bytes()).hexdigest(),\n        \'population_sha256\': hashlib.sha256(population.encode()).hexdigest(), \'smoke_only\': synthetic_records is not None,\n        \'runtime\': {\'torch\': str(torch.__version__), \'numpy\': np.__version__, \'gpu_count\': torch.cuda.device_count()},\n        \'walking_policy\': \'candidate only; retained Walking is not changed by training\',\n        \'checkpoint_selection\': \'fixed final EMA; no outer-label epoch selection\',\n        \'inference_mode\': \'offline, whole-record robust sensor scaling\', \'confidence\': \'uncalibrated sigmoid scores\',\n        \'references\': [\'https://arxiv.org/abs/1909.04939\', \'https://arxiv.org/abs/2211.14730\'],\n        \'competition_submission_created\': False}\n    fingerprint = hashlib.sha256(json.dumps(configuration, sort_keys=True).encode()).hexdigest()[:20]\n    output_root = Path(output_root or ROOT / \'results/tdcs_alternative_architectures\')\n    run = output_root / fingerprint\n    run.mkdir(parents=True, exist_ok=True)\n    common.write_json(run / \'configuration.json\', configuration)\n    if manifest is not None:\n        manifest.to_csv(run / \'frozen_tdcs_manifest.csv\', index=False)\n    records = prepare_records(manifest) if synthetic_records is None else synthetic_records\n    rows, labels, predictions = [], [], []\n    (run / \'oof\').mkdir(exist_ok=True)\n    for fold in cfg.folds:\n        model = train_model(records, fold, architecture, cfg, run / f\'fold_{fold}\', device, fingerprint)\n        ys, ps = [], []\n        for record in records:\n            if record[\'fold\'] != fold:\n                continue\n            pred = predict_record(model, record[\'x\'], cfg, device)\n            assert pred.shape == record[\'y\'].shape\n            np.savez_compressed(run / \'oof\' / f\'{record["recording_id"]}.npz\', prediction=pred, targets=np.asarray(TARGETS))\n            ys.append(record[\'y\'])\n            ps.append(pred)\n        rows.append({\'fold\': fold, **score(np.concatenate(ys), np.concatenate(ps))})\n        pd.DataFrame(rows).to_csv(run / \'fold_metrics.csv\', index=False)\n        labels.extend(ys)\n        predictions.extend(ps)\n        del model\n        if device.type == \'cuda\':\n            torch.cuda.empty_cache()\n    metrics = score(np.concatenate(labels), np.concatenate(predictions))\n    pd.DataFrame([metrics]).to_csv(run / \'native_oof_metrics.csv\', index=False)\n    decision = {\'run_dir\': str(run), \'status\': \'complete\', \'architecture\': architecture, **metrics,\n        \'smoke_only\': synthetic_records is not None, \'native_rows\': sum(len(y) for y in labels),\n        \'goal_two_target_mAP\': .6, \'goal_reached\': metrics[\'two_target_mAP\'] >= .6 and synthetic_records is None,\n        \'retained_heads_unchanged\': True, \'independent_generalization_established\': False,\n        \'competition_submission_created\': False}\n    common.write_json(run / \'decision.json\', decision)\n    common.write_json(run / \'progress.json\', decision)\n    common.write_json(output_root / \'latest_run.json\', decision)\n    print(json.dumps(decision, indent=2), flush=True)\n    return decision\n\n\ndef smoke_experiment(architecture, output_root=None):\n    cfg = replace(CFG, name=\'synthetic_wiring_only\', epochs=1, steps_per_epoch=1, batch_size=2,\n        window=64, width=16, dilations=(1, 2), patch=8, attention_width=32, attention_layers=1)\n    rng, records = np.random.default_rng(1), []\n    for fold in range(5):\n        x = common.sensor_features(rng.normal(size=(96+fold, 3)).astype(np.float32))[:, :12].copy()\n        y = np.zeros((len(x), 3), np.float32)\n        for target, start in enumerate((12, 35, 62)):\n            y[start:start+12, target] = 1\n        records.append({\'recording_id\': f\'smoke_{fold}\', \'subject\': f\'synthetic_{fold}\', \'fold\': fold,\n            \'x\': x, \'y\': y, \'negative\': np.flatnonzero(y.sum(axis=1) == 0),\n            \'centers\': [common.event_centers(y[:, i]) for i in range(3)]})\n    return run_experiment(architecture, cfg, output_root or ROOT / \'tmp\' / f\'tdcs_{architecture}_smoke\', records)\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--architecture\', choices=ARCHITECTURES, required=True)\n    parser.add_argument(\'--smoke\', action=\'store_true\')\n    parser.add_argument(\'--output-root\', type=Path)\n    args = parser.parse_args()\n    smoke_experiment(args.architecture, args.output_root) if args.smoke else run_experiment(args.architecture, output_root=args.output_root)\n', 'tdcs_resource_safe_experiment': '"""Lower-memory three-target TDCS candidates; immutable v2 helpers are reused.\n\nThese are new pooled-context candidates, not a claim that the v2 kernel-death\ncause is known. Retained heads and DeFOG are never read for training or replaced.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport copy\nimport faulthandler\nimport hashlib\nimport json\nimport os\nimport threading\nimport time\nfrom dataclasses import asdict, dataclass, replace\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\nimport tdcs_alternative_architectures as legacy\n\ncommon = legacy.common\nROOT = Path(__file__).resolve().parents[1]\nTARGETS = legacy.TARGETS\nARCHITECTURES = (\'pooled_cnn\', \'gated_gru32\')\n\n\n@dataclass(frozen=True)\nclass Config:\n    name: str = \'tdcs_resource_safe_v3\'\n    targets: tuple = TARGETS\n    folds: tuple = (0, 1, 2, 3, 4)\n    epochs: int = 24\n    steps_per_epoch: int = 96\n    batch_size: int = 16\n    micro_batch: int = 2\n    window: int = 2048\n    inference_batch: int = 1\n    width: int = 48\n    pool_stride: int = 4\n    dilations: tuple = (1, 2, 4, 8)\n    hidden: int = 64\n    gru_layers: int = 2\n    dropout: float = .25\n    weight_decay: float = .01\n    learning_rate: float = .0007\n    event_fraction: float = .5\n    negative_fraction: float = .25\n    ema_decay: float = .995\n    gate_aux_weight: float = .2\n    rank_weight: float = .03\n    checkpoint_steps: int = 12\n    max_seconds: int = 21600\n    seed: int = 1701\n\n\nCFG = Config()\n\n\ndef optional_text(path):\n    try:\n        return Path(path).read_text()\n    except (OSError, UnicodeError):\n        return None\n\n\ndef check_config(cfg):\n    if tuple(cfg.targets) != TARGETS or tuple(cfg.folds) != (0, 1, 2, 3, 4):\n        raise ValueError(\'All three targets and all five frozen folds are required\')\n    if cfg.micro_batch < 1 or cfg.batch_size % cfg.micro_batch:\n        raise ValueError(\'Effective batch must be divisible by the microbatch\')\n    if cfg.width % 8 or min(cfg.window, cfg.pool_stride, cfg.epochs, cfg.steps_per_epoch,\n                            cfg.checkpoint_steps, cfg.max_seconds) < 1:\n        raise ValueError(\'Invalid width, context or training budget\')\n    if min(cfg.event_fraction, cfg.negative_fraction) < 0 or cfg.event_fraction + cfg.negative_fraction > 1:\n        raise ValueError(\'Invalid sampling fractions\')\n\n\nclass ResourceLog:\n    """Best-effort host/GPU allocator observations plus a native-stack watchdog."""\n    def __init__(self, directory, device, interval=30):\n        self.directory, self.device, self.interval = Path(directory), device, interval\n        self.directory.mkdir(parents=True, exist_ok=True)\n        self.stage, self.details = \'initializing\', {}\n        self.lock, self.stop = threading.Lock(), threading.Event()\n\n    def emit(self, stage=None, **details):\n        with self.lock:\n            if stage is not None:\n                self.stage, self.details = stage, details\n            row = {\'utc\': time.strftime(\'%Y-%m-%dT%H:%M:%SZ\', time.gmtime()),\n                   \'stage\': self.stage, **self.details}\n            status = optional_text(\'/proc/self/status\')\n            if status is not None:\n                for line in status.splitlines():\n                    if line.startswith((\'VmRSS:\', \'VmHWM:\', \'VmSwap:\')):\n                        key, value = line.split(\':\', 1)\n                        row[key + \'_KiB\'] = int(value.split()[0])\n            for name in (\'memory.current\', \'memory.max\', \'memory.events\'):\n                value = optional_text(Path(\'/sys/fs/cgroup\') / name)\n                if value is not None:\n                    row[\'cgroup_\' + name] = value.strip()\n            if self.device.type == \'cuda\':\n                row.update(gpu_allocated_bytes=torch.cuda.memory_allocated(self.device),\n                           gpu_reserved_bytes=torch.cuda.memory_reserved(self.device),\n                           gpu_peak_allocated_bytes=torch.cuda.max_memory_allocated(self.device))\n            with (self.directory / \'resources.jsonl\').open(\'a\', encoding=\'utf-8\') as stream:\n                stream.write(json.dumps(row) + \'\\n\')\n                stream.flush()\n\n    def _watch(self):\n        while not self.stop.wait(self.interval):\n            try:\n                self.emit()\n            except Exception:\n                # A missing optional telemetry source must not stop training.\n                pass\n\n    def __enter__(self):\n        self.trace = (self.directory / \'native_stacks.log\').open(\'a\', encoding=\'utf-8\')\n        self.watchdog_enabled = False\n        try:\n            faulthandler.enable(file=self.trace, all_threads=True)\n            faulthandler.dump_traceback_later(180, repeat=True, file=self.trace)\n            self.watchdog_enabled = True\n        except (RuntimeError, OSError):\n            pass\n        self.emit(\'starting\')\n        self.thread = threading.Thread(target=self._watch, daemon=True)\n        self.thread.start()\n        return self\n\n    def __exit__(self, kind, value, traceback):\n        self.stop.set()\n        self.thread.join(timeout=2)\n        if self.watchdog_enabled:\n            faulthandler.cancel_dump_traceback_later()\n            faulthandler.disable()\n        self.trace.close()\n\n\nclass Network(nn.Module):\n    def __init__(self, architecture, cfg=CFG):\n        super().__init__()\n        if architecture not in ARCHITECTURES:\n            raise ValueError(\'Unknown candidate\')\n        self.architecture = architecture\n        self.stem = nn.Sequential(nn.Conv1d(12, cfg.width, 7, padding=3),\n                                  nn.GroupNorm(8, cfg.width), nn.GELU())\n        self.pool = nn.AvgPool1d(cfg.pool_stride, cfg.pool_stride, ceil_mode=True,\n                                 count_include_pad=False)\n        if architecture == \'pooled_cnn\':\n            self.temporal = nn.Sequential(*[legacy.MultiscaleBlock(cfg.width, d, cfg.dropout)\n                                             for d in cfg.dilations])\n            context_width = cfg.width\n        else:\n            self.temporal = nn.GRU(cfg.width, cfg.hidden, cfg.gru_layers, batch_first=True,\n                bidirectional=True, dropout=cfg.dropout if cfg.gru_layers > 1 else 0)\n            context_width = cfg.hidden * 2\n        self.fuse = nn.Sequential(nn.Conv1d(cfg.width + context_width, cfg.width, 1),\n                                  nn.GroupNorm(8, cfg.width), nn.GELU(), nn.Dropout(cfg.dropout))\n        self.types = nn.Conv1d(cfg.width, 3, 1)\n        self.gate = nn.Conv1d(cfg.width, 1, 1) if architecture == \'gated_gru32\' else None\n\n    def parts(self, x):\n        local = self.stem(x.transpose(1, 2))\n        pooled = self.pool(local)\n        if self.architecture == \'pooled_cnn\':\n            context = self.temporal(pooled)\n        else:\n            context, _ = self.temporal(pooled.transpose(1, 2))\n            context = context.transpose(1, 2)\n        context = F.interpolate(context, size=local.shape[2], mode=\'linear\', align_corners=False)\n        fused = self.fuse(torch.cat([local, context], dim=1))\n        logits = self.types(fused).transpose(1, 2).float()\n        gate = self.gate(fused).transpose(1, 2).float() if self.gate is not None else None\n        if gate is not None:\n            probability = (torch.sigmoid(logits) * torch.sigmoid(gate)).clamp(1e-6, 1-1e-6)\n            logits = torch.logit(probability)\n        return logits, gate\n\n    def forward(self, x):\n        return self.parts(x)[0]\n\n\ndef ranking_loss(logits, y, mask):\n    terms = []\n    for target in (0, 2):\n        valid = mask[..., target].bool()\n        positives = logits[..., target][valid & (y[..., target] > .5)]\n        negatives = logits[..., target][valid & (y[..., target] < .5)]\n        if len(positives) and len(negatives):\n            pi = torch.linspace(0, len(positives)-1, min(256, len(positives)), device=logits.device).long()\n            ni = torch.linspace(0, len(negatives)-1, min(256, len(negatives)), device=logits.device).long()\n            count = min(len(pi), len(ni))\n            terms.append(F.softplus(negatives[ni[:count]] - positives[pi[:count]]).mean())\n    return torch.stack(terms).mean() if terms else logits.sum() * 0\n\n\ndef loss_for_parts(logits, gate, y, mask, cfg):\n    value = legacy.masked_loss(logits, y, mask)\n    if gate is not None:\n        gate_mask, gate_y = mask.amax(-1, keepdim=True), y.amax(-1, keepdim=True)\n        value = value + cfg.gate_aux_weight * legacy.masked_loss(gate, gate_y, gate_mask)\n        if cfg.rank_weight:\n            value = value + cfg.rank_weight * ranking_loss(logits, y, mask)\n    return value\n\n\ndef backward_microbatches(model, batch, cfg, device, scaler):\n    """One optimizer batch, weighted by valid rows even when padding differs."""\n    total_valid = batch[2].sum().clamp_min(1)\n    value = 0.\n    for start in range(0, len(batch[0]), cfg.micro_batch):\n        x, y, mask = [part[start:start+cfg.micro_batch].to(device) for part in batch]\n        weight = float(mask.sum().detach().cpu() / total_valid)\n        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == \'cuda\'):\n            logits, gate = model.parts(x)\n            loss = loss_for_parts(logits, gate, y, mask, cfg) * weight\n        if not torch.isfinite(loss):\n            raise FloatingPointError(\'Non-finite microbatch loss\')\n        scaler.scale(loss).backward()\n        value += float(loss.detach().cpu())\n    return value\n\n\ndef cpu_state(value):\n    if torch.is_tensor(value):\n        return value.detach().cpu().clone()\n    if isinstance(value, dict):\n        return {key: cpu_state(item) for key, item in value.items()}\n    if isinstance(value, (list, tuple)):\n        return type(value)(cpu_state(item) for item in value)\n    return value\n\n\ndef atomic_checkpoint(state, path):\n    temporary = path.with_suffix(\'.tmp.pt\')\n    torch.save(cpu_state(state), temporary)\n    os.replace(temporary, path)\n\n\ndef train_model(records, fold, architecture, cfg, directory, device, fingerprint, health, deadline):\n    directory.mkdir(parents=True, exist_ok=True)\n    seed = cfg.seed + 1009 * fold\n    torch.manual_seed(seed)\n    if device.type == \'cuda\':\n        torch.cuda.manual_seed(seed)\n    model = Network(architecture, cfg).to(device)\n    final = directory / \'final_ema.pt\'\n    if final.is_file():\n        state = torch.load(final, map_location=\'cpu\', weights_only=False)\n        assert state[\'fingerprint\'] == fingerprint and tuple(state[\'targets\']) == TARGETS\n        model.load_state_dict(state[\'model\'])\n        return model\n    sampler = legacy.Sampler(records, fold, cfg)\n    training = sorted(sampler.subjects)\n    heldout = sorted({record[\'subject\'] for record in records if record[\'fold\'] == fold})\n    assert set(training).isdisjoint(heldout)\n    common.write_json(directory / \'training_receipt.json\', {\n        \'targets\': TARGETS, \'architecture\': architecture, \'training_subjects\': training,\n        \'held_out_subjects\': heldout, \'fixed_epochs\': cfg.epochs,\n        \'outer_labels_used_for_checkpoint_selection\': False,\n        \'training_window\': cfg.window, \'inference_window\': cfg.window,\n        \'effective_batch\': cfg.batch_size, \'micro_batch\': cfg.micro_batch,\n        \'gradient_accumulation\': cfg.batch_size // cfg.micro_batch,\n        \'data_parallel\': False, \'device\': str(device), \'seed\': seed,\n        \'previous_models_overwritten\': False})\n    ema = copy.deepcopy(model).eval().requires_grad_(False)\n    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay)\n    schedule = torch.optim.lr_scheduler.LambdaLR(optimizer,\n        lambda step: common.learning_rate_multiplier(step, cfg.epochs * cfg.steps_per_epoch))\n    scaler = torch.amp.GradScaler(\'cuda\', enabled=device.type == \'cuda\')\n    rng = np.random.default_rng(seed)\n    resume, history, first_epoch, first_step, partial_sum = directory/\'resume.pt\', [], 0, 0, 0.\n    if resume.is_file():\n        state = torch.load(resume, map_location=\'cpu\', weights_only=False)\n        assert state[\'fingerprint\'] == fingerprint and tuple(state[\'targets\']) == TARGETS\n        model.load_state_dict(state[\'model\'])\n        ema.load_state_dict(state[\'ema\'])\n        optimizer.load_state_dict(state[\'optimizer\'])\n        schedule.load_state_dict(state[\'scheduler\'])\n        scaler.load_state_dict(state[\'scaler\'])\n        rng.bit_generator.state = state[\'numpy_rng\']\n        torch.set_rng_state(state[\'torch_rng\'])\n        if device.type == \'cuda\':\n            torch.cuda.set_rng_state(state[\'cuda_rng\'], device)\n        first_epoch, first_step = state[\'next_epoch\'], state[\'next_step\']\n        history, partial_sum = state[\'history\'], state[\'partial_loss_sum\']\n    for epoch in range(first_epoch, cfg.epochs):\n        started, loss_sum = time.monotonic(), partial_sum if epoch == first_epoch else 0.\n        begin = first_step if epoch == first_epoch else 0\n        model.train()\n        print(f\'{architecture} fold {fold} epoch {epoch+1}/{cfg.epochs}\', flush=True)\n        for step in range(begin, cfg.steps_per_epoch):\n            if time.monotonic() >= deadline:\n                raise TimeoutError(\'Six-hour safety budget reached; latest checkpoint is retained\')\n            health.emit(\'training\', fold=fold, epoch=epoch+1, step=step+1)\n            optimizer.zero_grad(set_to_none=True)\n            loss_sum += backward_microbatches(model, sampler.batch(rng), cfg, device, scaler)\n            scaler.unscale_(optimizer)\n            nn.utils.clip_grad_norm_(model.parameters(), 2)\n            scaler.step(optimizer)\n            scaler.update()\n            schedule.step()\n            with torch.no_grad():\n                for destination, source in zip(ema.parameters(), model.parameters()):\n                    destination.lerp_(source.detach(), 1-cfg.ema_decay)\n            last = step+1 == cfg.steps_per_epoch\n            if last:\n                history.append({\'epoch\': epoch+1, \'loss\': loss_sum/cfg.steps_per_epoch,\n                    \'seconds\': time.monotonic()-started, \'learning_rate\': schedule.get_last_lr()[0]})\n                pd.DataFrame(history).to_csv(directory/\'training_log.csv\', index=False)\n            if last or (step+1) % cfg.checkpoint_steps == 0:\n                health.emit(\'checkpoint\', fold=fold, epoch=epoch+1, step=step+1)\n                atomic_checkpoint({\'fingerprint\': fingerprint, \'targets\': TARGETS,\n                    \'model\': model.state_dict(), \'ema\': ema.state_dict(),\n                    \'optimizer\': optimizer.state_dict(), \'scheduler\': schedule.state_dict(),\n                    \'scaler\': scaler.state_dict(), \'numpy_rng\': rng.bit_generator.state,\n                    \'torch_rng\': torch.get_rng_state(),\n                    \'cuda_rng\': torch.cuda.get_rng_state(device) if device.type == \'cuda\' else None,\n                    \'next_epoch\': epoch+1 if last else epoch, \'next_step\': 0 if last else step+1,\n                    \'partial_loss_sum\': 0. if last else loss_sum, \'history\': history}, resume)\n                common.write_json(directory.parent/\'progress.json\', {\'stage\': \'training\',\n                    \'architecture\': architecture, \'fold\': fold, \'epoch\': epoch+1,\n                    \'step\': step+1, \'fixed_epochs\': cfg.epochs, \'checkpoint\': str(resume)})\n    atomic_checkpoint({\'fingerprint\': fingerprint, \'targets\': TARGETS, \'model\': ema.state_dict()}, final)\n    return ema\n\n\ndef source_hash(module, override):\n    return globals().get(override) or hashlib.sha256(Path(module.__file__).read_bytes()).hexdigest()\n\n\ndef run_experiment(architecture, cfg=CFG, output_root=None, synthetic_records=None):\n    check_config(cfg)\n    if architecture not in ARCHITECTURES:\n        raise ValueError(\'Unknown architecture\')\n    torch.set_num_threads(4)\n    torch.backends.cudnn.benchmark = False\n    device = torch.device(\'cuda:0\' if torch.cuda.is_available() else \'cpu\')\n    if synthetic_records is None and device.type != \'cuda\':\n        raise RuntimeError(\'Full training requires CUDA; no silent synthetic fallback\')\n    manifest = None\n    if synthetic_records is None:\n        raw, metadata = common.locate_data()\n        manifest = common.build_manifest(raw, metadata)\n        population = manifest[[\'recording_id\',\'subject\',\'fold\',\'source_bytes\']].to_csv(index=False, lineterminator=\'\\n\')\n    else:\n        population = \'synthetic wiring only; not TDCS performance\'\n    configuration = {\'configuration\': asdict(cfg), \'architecture\': architecture, \'targets\': TARGETS,\n        \'source_sha256\': globals().get(\'SOURCE_SHA256\') or hashlib.sha256(Path(__file__).read_bytes()).hexdigest(),\n        \'legacy_source_sha256\': source_hash(legacy, \'LEGACY_SOURCE_SHA256\'),\n        \'common_source_sha256\': source_hash(common, \'COMMON_SOURCE_SHA256\'),\n        \'population_sha256\': hashlib.sha256(population.encode()).hexdigest(),\n        \'smoke_only\': synthetic_records is not None,\n        \'runtime\': {\'torch\': str(torch.__version__), \'numpy\': np.__version__,\n                    \'visible_gpu_count\': torch.cuda.device_count(), \'used_gpu_indices\': [0] if device.type==\'cuda\' else []},\n        \'checkpoint_selection\': \'fixed final EMA; no outer-label epoch selection\',\n        \'inference_mode\': \'offline; native-row output with pooled sensor context and local skip\',\n        \'confidence\': \'uncalibrated independent sigmoid scores\',\n        \'failure_cause_of_v2_confirmed\': False, \'retained_heads_overwritten\': False,\n        \'competition_submission_created\': False}\n    fingerprint = hashlib.sha256(json.dumps(configuration, sort_keys=True).encode()).hexdigest()[:20]\n    output_root = Path(output_root or ROOT/\'results/tdcs_resource_safe\')\n    run = output_root/fingerprint\n    run.mkdir(parents=True, exist_ok=True)\n    common.write_json(run/\'configuration.json\', configuration)\n    if manifest is not None:\n        manifest.to_csv(run/\'frozen_tdcs_manifest.csv\', index=False)\n    (run/\'oof\').mkdir(exist_ok=True)\n    deadline = time.monotonic() + cfg.max_seconds\n    with ResourceLog(run, device) as health:\n        health.emit(\'preparing_records\')\n        records = legacy.prepare_records(manifest) if synthetic_records is None else synthetic_records\n        rows, labels, predictions = [], [], []\n        for fold in cfg.folds:\n            model = train_model(records, fold, architecture, cfg, run/f\'fold_{fold}\', device, fingerprint, health, deadline)\n            ys, ps = [], []\n            for record in records:\n                if record[\'fold\'] != fold:\n                    continue\n                if time.monotonic() >= deadline:\n                    raise TimeoutError(\'Six-hour safety budget reached during inference\')\n                health.emit(\'inference\', fold=fold, recording_id=record[\'recording_id\'])\n                pred = legacy.predict_record(model, record[\'x\'], cfg, device)\n                assert pred.shape == record[\'y\'].shape\n                np.savez_compressed(run/\'oof\'/f\'{record["recording_id"]}.npz\', prediction=pred, targets=np.asarray(TARGETS))\n                ys.append(record[\'y\'])\n                ps.append(pred)\n            rows.append({\'fold\': fold, **legacy.score(np.concatenate(ys), np.concatenate(ps))})\n            pd.DataFrame(rows).to_csv(run/\'fold_metrics.csv\', index=False)\n            labels.extend(ys)\n            predictions.extend(ps)\n            del model\n            if device.type == \'cuda\':\n                torch.cuda.empty_cache()\n        metrics = legacy.score(np.concatenate(labels), np.concatenate(predictions))\n        pd.DataFrame([metrics]).to_csv(run/\'native_oof_metrics.csv\', index=False)\n        result = {\'run_dir\': str(run), \'status\': \'complete\', \'architecture\': architecture, **metrics,\n            \'smoke_only\': synthetic_records is not None, \'native_rows\': sum(len(y) for y in labels),\n            \'goal_two_target_mAP\': .6,\n            \'goal_reached\': metrics[\'two_target_mAP\'] >= .6 and synthetic_records is None,\n            \'retained_heads_unchanged\': True, \'independent_generalization_established\': False,\n            \'competition_submission_created\': False}\n        common.write_json(run/\'decision.json\', result)\n        common.write_json(run/\'progress.json\', result)\n        common.write_json(output_root/\'latest_run.json\', result)\n        health.emit(\'complete\', native_rows=result[\'native_rows\'])\n    print(json.dumps(result, indent=2), flush=True)\n    return result\n\n\ndef smoke_experiment(architecture, output_root):\n    cfg = replace(CFG, name=\'synthetic_resource_safe_wiring\', epochs=1, steps_per_epoch=2,\n        batch_size=4, micro_batch=2, window=64, width=16, dilations=(1,2),\n        hidden=8, gru_layers=1, checkpoint_steps=1)\n    rng, records = np.random.default_rng(1), []\n    for fold in range(5):\n        x = common.sensor_features(rng.normal(size=(96+fold,3)).astype(np.float32))[:,:12].copy()\n        y = np.zeros((len(x),3), np.float32)\n        for target, begin in enumerate((12,35,62)):\n            y[begin:begin+12,target] = 1\n        records.append({\'recording_id\':f\'smoke_{fold}\', \'subject\':f\'synthetic_{fold}\',\n            \'fold\':fold, \'x\':x, \'y\':y, \'negative\':np.flatnonzero(y.sum(1)==0),\n            \'centers\':[common.event_centers(y[:,i]) for i in range(3)]})\n    return run_experiment(architecture, cfg, output_root, records)\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--architecture\', choices=ARCHITECTURES, required=True)\n    parser.add_argument(\'--output-root\', type=Path, required=True)\n    parser.add_argument(\'--smoke\', action=\'store_true\')\n    args = parser.parse_args()\n    if args.smoke:\n        smoke_experiment(args.architecture, args.output_root)\n    else:\n        run_experiment(args.architecture, output_root=args.output_root)\n', 'tdcs_native_context_experiment': '"""Matched native-rate gated-GRU candidates: 15 features versus 27 features.\n\nRestore the stronger v1 temporal encoder/timing/context recipe, but train all\nthree targets. The spectral variant adds only label-free rotation-invariant\nband-energy features. Reuse immutable v3 recovery utilities, not its network.\n"""\nfrom __future__ import annotations\nimport argparse\nimport copy\nimport hashlib\nimport json\nimport time\nfrom dataclasses import asdict, dataclass, replace\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom scipy.ndimage import uniform_filter1d\nfrom scipy.signal import butter, sosfiltfilt\n\nimport tdcs_resource_safe_experiment as safe\n\ncommon,legacy=safe.common,safe.legacy\nROOT=Path(__file__).resolve().parents[1]\nTARGETS=legacy.TARGETS\nARCHITECTURES=(\'native_clock\',\'spectral_clock\')\nBANDS=((.5,3.),(3.,8.),(8.,16.))\nENERGY_WINDOWS=(128,512)\n\n\n@dataclass(frozen=True)\nclass Config:\n    name: str=\'tdcs_native_context_v4\'\n    targets: tuple=TARGETS\n    folds: tuple=(0,1,2,3,4)\n    epochs: int=24\n    steps_per_epoch: int=144\n    batch_size: int=32\n    micro_batch: int=4\n    train_length: int=1000\n    inference_length: int=5000\n    inference_batch: int=1\n    width: int=64\n    hidden: int=128\n    gru_layers: int=3\n    dropout: float=.15\n    learning_rate: float=.001\n    weight_decay: float=.0001\n    event_fraction: float=.625\n    negative_fraction: float=.25\n    target_sampling: tuple=(.4,.2,.4)\n    rank_weight: float=.1\n    ema_decay: float=.995\n    checkpoint_steps: int=12\n    max_seconds: int=21600\n    seed: int=2107\n\n\nCFG=Config()\n\n\ndef check_config(cfg):\n    if tuple(cfg.targets)!=TARGETS or tuple(cfg.folds)!=(0,1,2,3,4):\n        raise ValueError(\'All three targets and all five frozen folds are required\')\n    if cfg.micro_batch<1 or cfg.batch_size%cfg.micro_batch:\n        raise ValueError(\'Invalid microbatch accumulation\')\n    if cfg.width%8 or min(cfg.epochs,cfg.steps_per_epoch,cfg.train_length,cfg.inference_length,cfg.checkpoint_steps,cfg.max_seconds)<1:\n        raise ValueError(\'Invalid network or training budget\')\n    if len(cfg.target_sampling)!=3 or min(cfg.target_sampling)<0 or not np.isclose(sum(cfg.target_sampling),1):\n        raise ValueError(\'Three target-sampling probabilities must sum to one\')\n    if min(cfg.event_fraction,cfg.negative_fraction)<0 or cfg.event_fraction+cfg.negative_fraction>1:\n        raise ValueError(\'Invalid event/negative/natural mixture\')\n\n\ndef spectral_features(raw):\n    """Twelve offline energy features; no labels, subject ID or fitted parameters."""\n    raw=np.asarray(raw,np.float64)\n    if raw.ndim!=2 or raw.shape[1]!=3 or len(raw)<2 or not np.isfinite(raw).all():\n        raise ValueError(\'Finite native accelerometer axes required\')\n    centered=raw-np.median(raw,axis=0,keepdims=True)\n    energies=[]\n    for low,high in BANDS:\n        sos=butter(2,(low,high),btype=\'bandpass\',fs=128.,output=\'sos\')\n        filtered=sosfiltfilt(sos,centered,axis=0,padlen=min(15,len(raw)-1))\n        energies.append(np.square(filtered).sum(axis=1))\n    energies=np.column_stack(energies)\n    parts=[]\n    for window in ENERGY_WINDOWS:\n        powers=np.maximum(uniform_filter1d(energies,size=window,axis=0,mode=\'nearest\'),0)\n        total=powers.sum(axis=1)\n        ratio=np.clip(np.log(powers[:,1]+1e-6)-np.log(powers[:,0]+1e-6),-12,12)\n        fraction=powers[:,1]/np.maximum(total,1e-6)\n        parts.extend([np.log1p(powers),ratio[:,None],fraction[:,None],np.log1p(total)[:,None]])\n    result=np.column_stack(parts).astype(np.float32)\n    assert result.shape==(len(raw),12) and np.isfinite(result).all()\n    return result\n\n\ndef features(raw,architecture):\n    if architecture not in ARCHITECTURES:\n        raise ValueError(\'Unknown feature comparison\')\n    original=common.sensor_features(raw)\n    if architecture==\'native_clock\':\n        return original\n    return np.column_stack([original,spectral_features(raw)]).astype(np.float32)\n\n\ndef prepare_records(manifest,architecture):\n    records=[]\n    for number,row in enumerate(manifest.itertuples(index=False),1):\n        frame=pd.read_csv(row.path,usecols=[*common.AXES,*TARGETS])\n        y=frame.loc[:,list(TARGETS)].to_numpy(np.float32)\n        assert np.isin(y,[0,1]).all()\n        x=features(frame.loc[:,list(common.AXES)].to_numpy(np.float32),architecture)\n        negative=np.flatnonzero(y.sum(axis=1)==0)\n        negative=negative[::max(1,len(negative)//2000)]\n        records.append({\'recording_id\':row.recording_id,\'subject\':row.subject,\'fold\':row.fold,\n            \'x\':x,\'y\':y,\'negative\':negative,\'centers\':[common.event_centers(y[:,i]) for i in range(3)]})\n        if number%100==0:\n            print(f\'Prepared {number}/{len(manifest)} native records\',flush=True)\n    assert sum(len(record[\'y\']) for record in records)==6959762\n    return records\n\n\nclass Sampler(legacy.Sampler):\n    def plan(self,rng):\n        draw=rng.random()\n        if draw<self.cfg.event_fraction:\n            target=int(rng.choice(3,p=self.cfg.target_sampling))\n            record=common.EventSampler.choose_record(self.positive[target],rng)\n            center=int(rng.choice(record[\'centers\'][target]))\n            center+=int(rng.integers(-self.cfg.train_length//5,self.cfg.train_length//5+1))\n        elif draw<self.cfg.event_fraction+self.cfg.negative_fraction and self.negative:\n            record=common.EventSampler.choose_record(self.negative,rng)\n            center=int(rng.choice(record[\'negative\']))\n        else:\n            record=common.EventSampler.choose_record(self.subjects,rng)\n            center=int(rng.integers(len(record[\'x\'])))\n        center=int(np.clip(center,0,len(record[\'x\'])-1))\n        return record,center-self.cfg.train_length//2\n\n    def batch(self,rng):\n        xs,ys,masks=[],[],[]\n        for _ in range(self.cfg.batch_size):\n            record,start=self.plan(rng)\n            x,y,mask=legacy.window(record,start,self.cfg.train_length)\n            angle=float(rng.uniform(-.1,.1))\n            rotation=np.asarray([[1,0,0],[0,np.cos(angle),-np.sin(angle)],\n                                 [0,np.sin(angle),np.cos(angle)]],np.float32)\n            for begin in range(0,12,3):\n                x[:,begin:begin+3]=x[:,begin:begin+3]@rotation.T\n            # Timing and rotation-invariant band powers are not perturbed.\n            # No anisotropic gain/cutout is used in either matched candidate.\n            xs.append(x)\n            ys.append(y)\n            masks.append(mask)\n        return tuple(torch.from_numpy(np.stack(parts)) for parts in (xs,ys,masks))\n\n\nclass Network(common.TwoTargetGRU):\n    def __init__(self,architecture,cfg=CFG):\n        if architecture not in ARCHITECTURES:\n            raise ValueError(\'Unknown feature comparison\')\n        super().__init__(cfg,variant=\'gated_bigru_rank\')\n        self.head[-1]=nn.Linear(128,3)\n        self.architecture=architecture\n        self.input_channels=15 if architecture==\'native_clock\' else 27\n        if self.input_channels==27:\n            original=self.input[0]\n            # Shared weights/RNG start identically; new feature weights learn\n            # from zero rather than perturbing the entire initial network.\n            with torch.random.fork_rng(devices=[]):\n                replacement=nn.Linear(27,cfg.width)\n            with torch.no_grad():\n                replacement.weight[:,:15].copy_(original.weight)\n                replacement.weight[:,15:].zero_()\n                replacement.bias.copy_(original.bias)\n            self.input[0]=replacement\n\n\ndef sequence_loss(logits,y,mask,rank_weight):\n    value=legacy.masked_loss(logits,y,mask)\n    terms=[]\n    if rank_weight:\n        for target in (0,2):\n            valid=mask[...,target]>0\n            positive=logits[...,target][valid&(y[...,target]>.5)].float()\n            negative=logits[...,target][valid&(y[...,target]<.5)].float()\n            if positive.numel() and negative.numel():\n                positive=positive[torch.randperm(len(positive),device=positive.device)[:192]]\n                negative=torch.topk(negative,min(192,len(negative))).values\n                terms.append(F.softplus(negative[None,:]-positive[:,None]).mean())\n    if terms:\n        value=value+rank_weight*torch.stack(terms).mean()\n    return value\n\n\ndef backward_batch(model,batch,cfg,device,scaler):\n    total_valid=batch[2].sum().clamp_min(1)\n    value=0.\n    for start in range(0,len(batch[0]),cfg.micro_batch):\n        x,y,mask=[part[start:start+cfg.micro_batch].to(device) for part in batch]\n        weight=float(mask.sum().detach().cpu()/total_valid)\n        with torch.autocast(device_type=device.type,dtype=torch.float16,enabled=device.type==\'cuda\'):\n            loss=sequence_loss(model(x),y,mask,cfg.rank_weight)*weight\n        if not torch.isfinite(loss):\n            raise FloatingPointError(\'Non-finite native-context loss\')\n        scaler.scale(loss).backward()\n        value+=float(loss.detach().cpu())\n    return value\n\n\ndef train_model(records,fold,architecture,cfg,directory,device,fingerprint,health,deadline):\n    directory.mkdir(parents=True,exist_ok=True)\n    seed=cfg.seed+1009*fold\n    torch.manual_seed(seed)\n    if device.type==\'cuda\':\n        torch.cuda.manual_seed(seed)\n    model=Network(architecture,cfg).to(device)\n    final=directory/\'final_ema.pt\'\n    if final.is_file():\n        state=torch.load(final,map_location=\'cpu\',weights_only=False)\n        assert state[\'fingerprint\']==fingerprint and tuple(state[\'targets\'])==TARGETS\n        model.load_state_dict(state[\'model\'])\n        return model\n    sampler=Sampler(records,fold,cfg)\n    training=sorted(sampler.subjects)\n    heldout=sorted({record[\'subject\'] for record in records if record[\'fold\']==fold})\n    assert set(training).isdisjoint(heldout)\n    common.write_json(directory/\'training_receipt.json\',{\'targets\':TARGETS,\'architecture\':architecture,\n        \'training_subjects\':training,\'held_out_subjects\':heldout,\'fixed_epochs\':cfg.epochs,\n        \'outer_labels_used_for_checkpoint_selection\':False,\'effective_batch\':cfg.batch_size,\n        \'micro_batch\':cfg.micro_batch,\'gradient_accumulation\':cfg.batch_size//cfg.micro_batch,\n        \'data_parallel\':False,\'training_window\':cfg.train_length,\'inference_window\':cfg.inference_length,\n        \'input_features\':model.input_channels,\'target_sampling\':cfg.target_sampling,\n        \'previous_models_overwritten\':False,\'device\':str(device),\'seed\':seed})\n    ema=copy.deepcopy(model).eval().requires_grad_(False)\n    optimizer=torch.optim.AdamW(model.parameters(),lr=cfg.learning_rate,weight_decay=cfg.weight_decay)\n    scheduler=torch.optim.lr_scheduler.LambdaLR(optimizer,\n        lambda step:common.learning_rate_multiplier(step,cfg.epochs*cfg.steps_per_epoch))\n    scaler=torch.amp.GradScaler(\'cuda\',enabled=device.type==\'cuda\')\n    rng=np.random.default_rng(seed)\n    resume,history,first_epoch,first_step,partial=directory/\'resume.pt\',[],0,0,0.\n    if resume.is_file():\n        state=torch.load(resume,map_location=\'cpu\',weights_only=False)\n        assert state[\'fingerprint\']==fingerprint and tuple(state[\'targets\'])==TARGETS\n        model.load_state_dict(state[\'model\'])\n        ema.load_state_dict(state[\'ema\'])\n        optimizer.load_state_dict(state[\'optimizer\'])\n        scheduler.load_state_dict(state[\'scheduler\'])\n        scaler.load_state_dict(state[\'scaler\'])\n        rng.bit_generator.state=state[\'numpy_rng\']\n        torch.set_rng_state(state[\'torch_rng\'])\n        if device.type==\'cuda\':\n            torch.cuda.set_rng_state(state[\'cuda_rng\'],device)\n        first_epoch,first_step,partial,history=state[\'next_epoch\'],state[\'next_step\'],state[\'partial_loss_sum\'],state[\'history\']\n    for epoch in range(first_epoch,cfg.epochs):\n        begin=first_step if epoch==first_epoch else 0\n        loss_sum=partial if epoch==first_epoch else 0.\n        started=time.monotonic()\n        model.train()\n        print(f\'{architecture} fold {fold} epoch {epoch+1}/{cfg.epochs}\',flush=True)\n        for step in range(begin,cfg.steps_per_epoch):\n            if time.monotonic()>=deadline:\n                raise TimeoutError(\'Six-hour safety budget reached; latest checkpoint retained\')\n            health.emit(\'training\',fold=fold,epoch=epoch+1,step=step+1)\n            optimizer.zero_grad(set_to_none=True)\n            loss_sum+=backward_batch(model,sampler.batch(rng),cfg,device,scaler)\n            scaler.unscale_(optimizer)\n            nn.utils.clip_grad_norm_(model.parameters(),2)\n            scaler.step(optimizer)\n            scaler.update()\n            scheduler.step()\n            with torch.no_grad():\n                for destination,source in zip(ema.parameters(),model.parameters()):\n                    destination.lerp_(source.detach(),1-cfg.ema_decay)\n            last=step+1==cfg.steps_per_epoch\n            if last:\n                history.append({\'epoch\':epoch+1,\'loss\':loss_sum/cfg.steps_per_epoch,\n                    \'seconds\':time.monotonic()-started,\'learning_rate\':scheduler.get_last_lr()[0]})\n                pd.DataFrame(history).to_csv(directory/\'training_log.csv\',index=False)\n            if last or (step+1)%cfg.checkpoint_steps==0:\n                health.emit(\'checkpoint\',fold=fold,epoch=epoch+1,step=step+1)\n                safe.atomic_checkpoint({\'fingerprint\':fingerprint,\'targets\':TARGETS,\n                    \'model\':model.state_dict(),\'ema\':ema.state_dict(),\'optimizer\':optimizer.state_dict(),\n                    \'scheduler\':scheduler.state_dict(),\'scaler\':scaler.state_dict(),\n                    \'numpy_rng\':rng.bit_generator.state,\'torch_rng\':torch.get_rng_state(),\n                    \'cuda_rng\':torch.cuda.get_rng_state(device) if device.type==\'cuda\' else None,\n                    \'next_epoch\':epoch+1 if last else epoch,\'next_step\':0 if last else step+1,\n                    \'partial_loss_sum\':0. if last else loss_sum,\'history\':history},resume)\n                common.write_json(directory.parent/\'progress.json\',{\'stage\':\'training\',\'architecture\':architecture,\n                    \'fold\':fold,\'epoch\':epoch+1,\'step\':step+1,\'fixed_epochs\':cfg.epochs})\n    safe.atomic_checkpoint({\'fingerprint\':fingerprint,\'targets\':TARGETS,\'model\':ema.state_dict()},final)\n    return ema\n\n\ndef run_experiment(architecture,cfg=CFG,output_root=None,synthetic_records=None):\n    check_config(cfg)\n    if architecture not in ARCHITECTURES:\n        raise ValueError(\'Unknown feature comparison\')\n    torch.set_num_threads(4)\n    torch.backends.cudnn.benchmark=False\n    device=torch.device(\'cuda:0\' if torch.cuda.is_available() else \'cpu\')\n    if synthetic_records is None and device.type!=\'cuda\':\n        raise RuntimeError(\'Full training requires CUDA; no silent synthetic fallback\')\n    manifest=None\n    if synthetic_records is None:\n        raw,metadata=common.locate_data()\n        manifest=common.build_manifest(raw,metadata)\n        population=manifest[[\'recording_id\',\'subject\',\'fold\',\'source_bytes\']].to_csv(index=False,lineterminator=\'\\n\')\n    else:\n        population=\'synthetic wiring only; not TDCS performance\'\n    def digest(module,override):\n        return globals().get(override) or hashlib.sha256(Path(module.__file__).read_bytes()).hexdigest()\n    configuration={\'configuration\':asdict(cfg),\'architecture\':architecture,\'targets\':TARGETS,\n        \'input_features\':15 if architecture==\'native_clock\' else 27,\n        \'source_sha256\':globals().get(\'SOURCE_SHA256\') or hashlib.sha256(Path(__file__).read_bytes()).hexdigest(),\n        \'common_source_sha256\':digest(common,\'COMMON_SOURCE_SHA256\'),\n        \'legacy_source_sha256\':digest(legacy,\'LEGACY_SOURCE_SHA256\'),\n        \'recovery_source_sha256\':digest(safe,\'RECOVERY_SOURCE_SHA256\'),\n        \'population_sha256\':hashlib.sha256(population.encode()).hexdigest(),\n        \'smoke_only\':synthetic_records is not None,\n        \'runtime\':{\'torch\':str(torch.__version__),\'numpy\':np.__version__,\n            \'visible_gpu_count\':torch.cuda.device_count(),\'used_gpu_indices\':[0] if device.type==\'cuda\' else []},\n        \'augmentation\':\'matched small rotations only; timing and invariant band energies unchanged\',\n        \'ranking\':\'train-only hard-negative ranking for SH/Walking, computed within microbatches\',\n        \'checkpoint_selection\':\'fixed final EMA; no outer-label epoch selection\',\n        \'inference_mode\':\'offline; native 128-Hz GRU, 1000-row training and 5000-row inference as the stronger v1 recipe\',\n        \'matched_comparison\':\'only the 12 additional band-energy features and their zero-initialized input weights differ\',\n        \'retained_heads_overwritten\':False,\'competition_submission_created\':False}\n    fingerprint=hashlib.sha256(json.dumps(configuration,sort_keys=True).encode()).hexdigest()[:20]\n    output_root=Path(output_root or ROOT/\'results/tdcs_native_context\')\n    run=output_root/fingerprint\n    run.mkdir(parents=True,exist_ok=True)\n    common.write_json(run/\'configuration.json\',configuration)\n    if manifest is not None:\n        manifest.to_csv(run/\'frozen_tdcs_manifest.csv\',index=False)\n    (run/\'oof\').mkdir(exist_ok=True)\n    deadline=time.monotonic()+cfg.max_seconds\n    with safe.ResourceLog(run,device) as health:\n        health.emit(\'preparing_records\')\n        records=prepare_records(manifest,architecture) if synthetic_records is None else synthetic_records\n        rows,labels,predictions=[],[],[]\n        for fold in cfg.folds:\n            model=train_model(records,fold,architecture,cfg,run/f\'fold_{fold}\',device,fingerprint,health,deadline)\n            ys,ps=[],[]\n            for record in records:\n                if record[\'fold\']!=fold:\n                    continue\n                if time.monotonic()>=deadline:\n                    raise TimeoutError(\'Six-hour safety budget reached during inference\')\n                health.emit(\'inference\',fold=fold,recording_id=record[\'recording_id\'])\n                inference_cfg=replace(safe.CFG,window=cfg.inference_length,inference_batch=cfg.inference_batch)\n                pred=legacy.predict_record(model,record[\'x\'],inference_cfg,device)\n                assert pred.shape==record[\'y\'].shape\n                np.savez_compressed(run/\'oof\'/f\'{record["recording_id"]}.npz\',prediction=pred,targets=np.asarray(TARGETS))\n                ys.append(record[\'y\'])\n                ps.append(pred)\n            rows.append({\'fold\':fold,**legacy.score(np.concatenate(ys),np.concatenate(ps))})\n            pd.DataFrame(rows).to_csv(run/\'fold_metrics.csv\',index=False)\n            labels.extend(ys)\n            predictions.extend(ps)\n            del model\n            if device.type==\'cuda\':\n                torch.cuda.empty_cache()\n        metrics=legacy.score(np.concatenate(labels),np.concatenate(predictions))\n        pd.DataFrame([metrics]).to_csv(run/\'native_oof_metrics.csv\',index=False)\n        result={\'run_dir\':str(run),\'status\':\'complete\',\'architecture\':architecture,**metrics,\n            \'smoke_only\':synthetic_records is not None,\'native_rows\':sum(len(y) for y in labels),\n            \'goal_two_target_mAP\':.6,\'goal_reached\':metrics[\'two_target_mAP\']>=.6 and synthetic_records is None,\n            \'retained_heads_unchanged\':True,\'independent_generalization_established\':False,\n            \'competition_submission_created\':False}\n        common.write_json(run/\'decision.json\',result)\n        common.write_json(run/\'progress.json\',result)\n        common.write_json(output_root/\'latest_run.json\',result)\n        health.emit(\'complete\',native_rows=result[\'native_rows\'])\n    print(json.dumps(result,indent=2),flush=True)\n    return result\n\n\ndef smoke_experiment(architecture,output_root):\n    cfg=replace(CFG,name=\'synthetic_native_context_wiring\',epochs=1,steps_per_epoch=2,\n        batch_size=4,micro_batch=2,train_length=64,inference_length=128,\n        width=16,hidden=8,gru_layers=1,checkpoint_steps=1)\n    rng,records=np.random.default_rng(1),[]\n    for fold in range(5):\n        x=features(rng.normal(size=(96+fold,3)).astype(np.float32),architecture)\n        y=np.zeros((len(x),3),np.float32)\n        for target,start in enumerate((12,35,62)):\n            y[start:start+12,target]=1\n        records.append({\'recording_id\':f\'smoke_{fold}\',\'subject\':f\'synthetic_{fold}\',\'fold\':fold,\n            \'x\':x,\'y\':y,\'negative\':np.flatnonzero(y.sum(1)==0),\n            \'centers\':[common.event_centers(y[:,i]) for i in range(3)]})\n    return run_experiment(architecture,cfg,output_root,records)\n\n\nif __name__==\'__main__\':\n    parser=argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--architecture\',choices=ARCHITECTURES,required=True)\n    parser.add_argument(\'--output-root\',type=Path,required=True)\n    parser.add_argument(\'--smoke\',action=\'store_true\')\n    args=parser.parse_args()\n    if args.smoke:\n        smoke_experiment(args.architecture,args.output_root)\n    else:\n        run_experiment(args.architecture,output_root=args.output_root)\n', 'tdcs_gpu_boosted_features': '"""Reuse the frozen 213 local + 112 recording sensor-feature recipes on Kaggle."""\nfrom __future__ import annotations\nimport ast\nimport hashlib\nfrom dataclasses import replace\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom numpy.lib.stride_tricks import sliding_window_view\n\nimport tdcs_multiscale_signal_experiment as base\nimport tdcs_native_context_experiment as native\n\nROOT = Path(__file__).resolve().parents[1]\nFROZEN_SOURCES = {\n    "tdcs_dense_context_experiment.py": "7877338e575f9cc824a5ab9227f7ec1ce6ce4d7b1f240e71bfba3a07bff96ba4",\n    "tdcs_recording_context.py": "7501dfdd906f90ded1c691c9f712ad721f10f25b77d5efda0f436bb6f1db4c3d"}\nFEATURE_CFG = replace(base.CFG, feature_version="dense_temporal_spectral_v2", training_stride=16)\nDENSE = ROOT / "results/tdcs_dense_context/e1844437c340c0b75d39/dense_context_features.npz"\nGLOBAL = ROOT / "results/tdcs_recording_context/a92c6da6e016faa1cc85/recording_features.npz"\nCACHE_HASHES = {\n    "dense": "19935bb84e9456616de70e2174a7839ac80fc8aa0e285bae3e1db4e2e0cd6f50",\n    "global": "1e79e1b74b9c2b3f07a6222f4406c5240e463efb73ac3e3d588a5adea084b23d"}\n\n\ndef recipe_functions():\n    """Compile only unchanged feature functions, not unrelated model pipelines."""\n    namespace = {"np": np, "base": base, "native": native, "common": native.common,\n        "sliding_window_view": sliding_window_view, "FEATURE_CFG": FEATURE_CFG,\n        "FFT_WINDOWS": (256, 1024), "CONTEXT_SHIFTS": (-256, -64, 64, 256)}\n    selections = {\n        "tdcs_dense_context_experiment.py": ("spectral_features", "temporal_context", "expanded_features"),\n        "tdcs_recording_context.py": ("recording_features",)}\n    embedded = globals().get("FROZEN_FEATURE_SOURCES", {})\n    for filename, names in selections.items():\n        source = embedded.get(filename)\n        if source is None:\n            source = (ROOT / "scripts" / filename).read_text(encoding="utf-8")\n        if hashlib.sha256(source.encode()).hexdigest() != FROZEN_SOURCES[filename]:\n            raise ValueError("Frozen feature source changed: " + filename)\n        functions = {node.name: node for node in ast.parse(source).body if isinstance(node, ast.FunctionDef)}\n        tree = ast.Module(body=[functions[name] for name in names], type_ignores=[])\n        exec(compile(tree, filename, "exec"), namespace)\n    return namespace["expanded_features"], namespace["recording_features"]\n\n\ndef recording_features(raw):\n    dense_function, global_function = recipe_functions()\n    grid, local, names = dense_function(raw)\n    whole, whole_names = global_function(raw)\n    x = np.column_stack([local, np.broadcast_to(whole, (len(grid), 112))]).astype(np.float32)\n    names = [*names, *["file_" + name for name in whole_names]]\n    assert x.shape == (len(grid), 325) and len(set(names)) == 325 and np.isfinite(x).all()\n    return grid, x, names\n\n\ndef prepare_arrays(manifest):\n    local_function, global_function = recipe_functions()\n    parts = {key: [] for key in ("x", "y", "fold", "record", "grid")}\n    lengths, names = [], None\n    for i, row in enumerate(manifest.itertuples(index=False)):\n        frame = pd.read_csv(row.path, usecols=[*base.AXES, *base.TARGETS])\n        raw = frame.loc[:, list(base.AXES)].to_numpy()\n        labels = frame.loc[:, list(base.TARGETS)].to_numpy(np.uint8)\n        assert np.isin(labels, [0, 1]).all() and (labels.sum(1) <= 1).all()\n        grid, local, local_names = local_function(raw)\n        whole, whole_names = global_function(raw)\n        current_names = [*local_names, *["file_" + name for name in whole_names]]\n        assert names is None or names == current_names\n        names = current_names\n        x = np.column_stack([local, np.broadcast_to(whole, (len(grid), 112))]).astype(np.float32)\n        assert x.shape[1] == 325 and np.isfinite(x).all()\n        parts["x"].append(x); parts["y"].append(labels[grid][:, [0, 2]])\n        parts["fold"].append(np.full(len(grid), row.fold, np.int8))\n        parts["record"].append(np.full(len(grid), i, np.int16)); parts["grid"].append(grid)\n        lengths.append(len(frame))\n        if (i + 1) % 100 == 0:\n            print(f"Frozen engineered sensor features: {i + 1}/815 recordings", flush=True)\n    arrays = {key: np.concatenate(value) for key, value in parts.items()}\n    arrays.update(lengths=np.asarray(lengths, np.int32), feature_names=np.asarray(names),\n        recording_ids=manifest.recording_id.to_numpy(str),\n        subject=manifest.subject.to_numpy(str)[arrays["record"]])\n    assert arrays["x"].shape == (436128, 325) and int(arrays["lengths"].sum()) == 6959762\n    return arrays\n\n\ndef load_local_arrays(manifest):\n    """Read exact, already-validated local caches; this does not fit a model."""\n    for path, name in ((DENSE, "dense"), (GLOBAL, "global")):\n        assert hashlib.sha256(path.read_bytes()).hexdigest() == CACHE_HASHES[name]\n    with np.load(DENSE, allow_pickle=False) as payload:\n        local = {key: payload[key].copy() for key in payload.files}\n    with np.load(GLOBAL, allow_pickle=False) as payload:\n        whole = {key: payload[key].copy() for key in payload.files}\n    ids = manifest.recording_id.to_numpy(str)\n    assert np.array_equal(local["recording_ids"], ids) and np.array_equal(whole["recording_ids"], ids)\n    rows = local["record"]\n    result = {"x": np.column_stack([local["x"], whole["x"][rows]]).astype(np.float32),\n        "y": local["y"][:, [0, 2]].copy(), "fold": local["fold"], "record": rows,\n        "grid": local["grid"], "lengths": local["row_counts"], "recording_ids": ids,\n        "feature_names": np.r_[local["feature_names"], np.char.add("file_", whole["feature_names"])],\n        "subject": manifest.subject.to_numpy(str)[rows]}\n    assert result["x"].shape == (436128, 325) and np.isfinite(result["x"]).all()\n    return result\n\n\ndef feature_digest(arrays):\n    value = hashlib.sha256()\n    for key in ("x", "y", "fold", "record", "grid", "lengths"):\n        a = np.ascontiguousarray(arrays[key])\n        value.update(key.encode()); value.update(a.dtype.str.encode())\n        value.update(str(a.shape).encode()); value.update(a.tobytes())\n    value.update("\\n".join(arrays["feature_names"].tolist()).encode())\n    value.update("\\n".join(arrays["recording_ids"].tolist()).encode())\n    return value.hexdigest()\n', 'tdcs_gpu_boosted_context': '"""Fixed-budget GPU boosted trees on the frozen TDCS sensor/context features."""\nfrom __future__ import annotations\nimport argparse\nimport hashlib\nimport json\nimport time\nfrom dataclasses import asdict, dataclass, replace\nfrom importlib.metadata import version\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\n\nimport tdcs_gpu_boosted_features as features\n\ncommon = features.native.common\nROOT = Path(__file__).resolve().parents[1]\nTARGETS = ("StartHesitation", "Walking")\nLEARNERS = ("catboost", "xgboost")\nPARAMETERS = {\n    "catboost": {"iterations": 1000, "depth": 6, "learning_rate": .04,\n        "l2_leaf_reg": 20., "loss_function": "Logloss", "bootstrap_type": "Bernoulli",\n        "subsample": .85, "border_count": 128, "boosting_type": "Plain",\n        "random_strength": 1., "allow_writing_files": False, "thread_count": 2},\n    "xgboost": {"num_boost_round": 900, "max_depth": 4, "learning_rate": .035,\n        "min_child_weight": 30., "reg_lambda": 30., "subsample": .85,\n        "colsample_bytree": .85, "max_bin": 128, "objective": "binary:logistic",\n        "eval_metric": "logloss", "tree_method": "hist", "nthread": 2}}\n\n\n@dataclass(frozen=True)\nclass Config:\n    name: str = "tdcs_gpu_boosted_context_v8"\n    targets: tuple = TARGETS\n    folds: tuple = (0, 1, 2, 3, 4)\n    negative_divisor: int = 4\n    seed: int = 3107\n    smoke_rounds: int = 12\n    max_seconds: int = 21600\n\n\nCFG = Config()\n\n\ndef check_config(cfg):\n    if tuple(cfg.targets) != TARGETS or tuple(cfg.folds) != tuple(range(5)):\n        raise ValueError("Two difficult targets and all five frozen subject folds required")\n    if min(cfg.negative_divisor, cfg.smoke_rounds, cfg.max_seconds) < 1:\n        raise ValueError("Invalid sampling or budget")\n\n\ndef source_hashes():\n    modules = {"source": None, "feature_helper": features, "multiscale": features.base,\n        "native": features.native, "common": common, "legacy": features.native.legacy,\n        "recovery": features.native.safe}\n    result = {}\n    for name, module in modules.items():\n        if module is None:\n            digest = globals().get("SOURCE_SHA256") or hashlib.sha256(Path(__file__).read_bytes()).hexdigest()\n        else:\n            digest = getattr(module, "SOURCE_SHA256", None) or hashlib.sha256(Path(module.__file__).read_bytes()).hexdigest()\n        result[name + "_sha256"] = digest\n    result["frozen_feature_recipes"] = features.FROZEN_SOURCES\n    return result\n\n\ndef training_plan(y, folds, subjects, fold, seed, divisor=4):\n    y, folds, subjects = np.asarray(y), np.asarray(folds), np.asarray(subjects, str)\n    if y.ndim != 1 or len(y) != len(folds) or len(y) != len(subjects) or not np.isin(y, [0, 1]).all():\n        raise ValueError("Aligned binary labels and split metadata required")\n    fitting = np.flatnonzero(folds != fold)\n    if not len(fitting) or divisor < 1:\n        raise ValueError("Nonempty fitting split and positive negative-sampling divisor required")\n    positives, negatives = fitting[y[fitting] == 1], fitting[y[fitting] == 0]\n    rng = np.random.default_rng(seed)\n    count = max(1, int(np.ceil(len(negatives) / divisor))) if len(negatives) else 0\n    sampled = rng.choice(negatives, size=count, replace=False) if count else np.empty(0, np.int64)\n    chosen = np.r_[positives, sampled]; rng.shuffle(chosen)\n    _, inverse, counts = np.unique(subjects[fitting], return_inverse=True, return_counts=True)\n    weights = np.zeros(len(y), np.float64)\n    weights[fitting] = 1. / counts[inverse]\n    probability = count / len(negatives) if len(negatives) else 1.\n    weights[sampled] /= probability\n    selected_weights = weights[chosen]; selected_weights /= selected_weights.mean()\n    assert not (folds[chosen] == fold).any() and set(positives) <= set(chosen)\n    return chosen, selected_weights, probability\n\n\ndef fit_model(learner, x, y, weights, seed, gpu=True, rounds=None):\n    if learner not in LEARNERS:\n        raise ValueError("Unknown learner")\n    if gpu and not torch.cuda.is_available():\n        raise RuntimeError("GPU fitting requires CUDA; no CPU fallback")\n    params = dict(PARAMETERS[learner])\n    if learner == "catboost":\n        from catboost import CatBoostClassifier\n        params.update(task_type="GPU" if gpu else "CPU", random_seed=seed)\n        if gpu:\n            params.update(devices="0", gpu_ram_part=.6)\n        if rounds is not None:\n            params["iterations"] = rounds\n        model = CatBoostClassifier(**params)\n        model.fit(x, y, sample_weight=weights, verbose=200 if rounds is None else False)\n        actual = model.get_all_params()["task_type"]\n        if gpu and actual != "GPU":\n            raise RuntimeError("CatBoost did not use its GPU backend")\n        assert model.tree_count_ == params["iterations"]\n        backend = {"actual_training_backend": actual, "trees": model.tree_count_}\n    else:\n        import xgboost as xgb\n        params.update(device="cuda:0" if gpu else "cpu", seed=seed)\n        count = rounds if rounds is not None else params.pop("num_boost_round")\n        params.pop("num_boost_round", None)\n        data = xgb.QuantileDMatrix(x, label=y, weight=weights, max_bin=params["max_bin"], nthread=2)\n        model = xgb.train(params, data, num_boost_round=count, verbose_eval=False)\n        actual = json.loads(model.save_config())["learner"]["generic_param"]["device"]\n        if gpu and not actual.startswith("cuda"):\n            raise RuntimeError("XGBoost silently fell back from CUDA")\n        assert model.num_boosted_rounds() == count\n        backend = {"actual_training_backend": actual, "trees": model.num_boosted_rounds()}\n    return model, backend\n\n\ndef predict_model(model, learner, x):\n    """CPU scoring of GPU-trained models is explicit and portable."""\n    if learner == "catboost":\n        p = model.predict_proba(x, task_type="CPU", thread_count=2)[:, 1]\n    else:\n        import xgboost as xgb\n        model.set_param({"device": "cpu", "nthread": 2})\n        p = model.predict(xgb.DMatrix(x, nthread=2))\n    p = np.asarray(p, np.float32)\n    assert p.shape == (len(x),) and np.isfinite(p).all() and ((p >= 0) & (p <= 1)).all()\n    return p\n\n\ndef save_model(model, learner, path):\n    temporary = path.with_name(path.stem + ".tmp" + path.suffix)\n    model.save_model(str(temporary))\n    temporary.replace(path)\n\n\ndef load_model(learner, path):\n    if learner == "catboost":\n        from catboost import CatBoostClassifier\n        model = CatBoostClassifier()\n    else:\n        from xgboost import Booster\n        model = Booster(params={"nthread": 2, "device": "cpu"})\n    if learner == "catboost":\n        model.load_model(str(path))\n    else:\n        # Native buffer loading also avoids Windows filesystem-canonicalization\n        # failures in older XGBoost builds; the model bytes are unchanged.\n        model.load_model(bytearray(path.read_bytes()))\n    return model\n\n\ndef expand_native(grid, probability, length):\n    if length < 1 or not len(grid) or grid[0] != 0 or grid[-1] >= length or np.any(np.diff(grid) <= 0):\n        raise ValueError("Ordered in-record grid beginning at zero required")\n    if probability.shape != (len(grid), 2):\n        raise ValueError("Two sampled output columns required")\n    return np.column_stack([np.interp(np.arange(length), grid, probability[:, i]) for i in range(2)]).astype(np.float32)\n\n\ndef run_experiment(learner, cfg=CFG, output_root=None, synthetic=None):\n    check_config(cfg)\n    if learner not in LEARNERS:\n        raise ValueError("Unknown learner")\n    smoke = synthetic is not None\n    if not smoke and not torch.cuda.is_available():\n        raise RuntimeError("Real-data boosted-tree training requires Kaggle CUDA; no CPU fallback")\n    output_root = Path(output_root or ROOT / "results/tdcs_gpu_boosted_context" / learner)\n    output_root.mkdir(parents=True, exist_ok=True)\n    raw, metadata = common.locate_data() if not smoke else (None, None)\n    manifest = common.build_manifest(raw, metadata) if not smoke else synthetic[1]\n    population = (manifest[["recording_id", "subject", "fold", "source_bytes"]].to_csv(index=False, lineterminator="\\n")\n                  if not smoke else "synthetic wiring only; not TDCS performance")\n    settings = {"configuration": asdict(cfg), "learner": learner, "targets": TARGETS,\n        "parameters": PARAMETERS[learner], "sources": source_hashes(),\n        "population_sha256": hashlib.sha256(population.encode()).hexdigest(), "smoke_only": smoke,\n        "negative_sampling": "all positives + seeded one-quarter negatives with inverse sampling correction",\n        "subject_weights": "inverse full-fitting-row count per training subject, then normalized",\n        "positive_class_weights": [1., 1.], "early_stopping": False,\n        "outer_labels_used_for_fitting_or_stopping": False, "base_OOF_predictions_used_as_features": False,\n        "file_gate_applied": False, "retained_heads_overwritten": False, "competition_submission_created": False}\n    common.write_json(output_root / "predeclared_settings.json", settings)\n    if not smoke:\n        # Fail before expensive feature preparation if this installed library\n        # cannot really train on CUDA. This tiny random-data check is not an AP trial.\n        rng = np.random.default_rng(9)\n        probe, backend = fit_model(learner, rng.normal(size=(128, 16)).astype(np.float32),\n            np.tile([0, 1], 64), np.ones(128), 9, gpu=True, rounds=2)\n        print("Verified GPU backend:", backend, flush=True); del probe\n        arrays = features.prepare_arrays(manifest)\n    else:\n        arrays = synthetic[0]\n    cache = output_root / "engineered_features.npz"\n    np.savez_compressed(cache, **arrays)\n    config = {**settings, "feature_count": arrays["x"].shape[1],\n        "feature_array_sha256": features.feature_digest(arrays), "feature_cache_file": cache.name,\n        "feature_cache_sha256": hashlib.sha256(cache.read_bytes()).hexdigest(),\n        "runtime": {name: version(name) for name in ("numpy", "pandas", "torch", learner)},\n        "visible_gpu_count": torch.cuda.device_count(), "used_gpu_indices": [0] if not smoke else [],\n        "prediction_backend": "CPU scoring of final GPU-trained model; all native rows by in-record interpolation",\n        "reused_development_folds": True, "confidence_calibrated": False}\n    fingerprint = hashlib.sha256(json.dumps(config, sort_keys=True).encode()).hexdigest()[:20]\n    run = output_root / fingerprint; run.mkdir(parents=True, exist_ok=True)\n    common.write_json(run / "configuration.json", config)\n    manifest.to_csv(run / "frozen_tdcs_manifest.csv", index=False)\n    (run / "oof").mkdir(exist_ok=True)\n    grid_scores = np.full(arrays["y"].shape, np.nan, np.float32)\n    deadline = time.monotonic() + cfg.max_seconds\n    for fold in cfg.folds:\n        heldout = arrays["fold"] == fold\n        training_subjects = sorted(set(arrays["subject"][~heldout]))\n        heldout_subjects = sorted(set(arrays["subject"][heldout]))\n        assert set(training_subjects).isdisjoint(heldout_subjects)\n        directory = run / f"fold_{fold}"; directory.mkdir(exist_ok=True)\n        for index, target in enumerate(TARGETS):\n            if time.monotonic() >= deadline:\n                raise TimeoutError("Boosted-tree safety budget reached")\n            seed = cfg.seed + 1009 * fold + index\n            chosen, weights, probability = training_plan(arrays["y"][:, index], arrays["fold"],\n                arrays["subject"], fold, seed, cfg.negative_divisor)\n            sample_path = directory / (target + "_training_sample.npz")\n            np.savez_compressed(sample_path, indices=chosen, weights=weights)\n            path = directory / (target + (".cbm" if learner == "catboost" else ".ubj"))\n            receipt_path = directory / (target + "_training_receipt.json")\n            if path.is_file() and receipt_path.is_file():\n                receipt = json.loads(receipt_path.read_text()); assert receipt["fingerprint"] == fingerprint\n                model = load_model(learner, path)\n            else:\n                print(f"{learner} fold {fold} {target}: {len(chosen):,} fitting grid rows", flush=True)\n                started = time.monotonic()\n                model, actual = fit_model(learner, arrays["x"][chosen], arrays["y"][chosen, index],\n                    weights, seed, gpu=not smoke, rounds=cfg.smoke_rounds if smoke else None)\n                save_model(model, learner, path)\n                common.write_json(receipt_path, {"fingerprint": fingerprint, "target": target, "learner": learner,\n                    "training_subjects": training_subjects, "held_out_subjects": heldout_subjects,\n                    "training_grid_rows_before_sampling": int((~heldout).sum()), "sampled_grid_rows": len(chosen),\n                    "all_positive_training_rows_kept": True, "negative_sampling_probability": probability,\n                    "inverse_negative_sampling_correction": True, "subject_weights_training_only": True,\n                    "outer_labels_used_for_fitting_or_stopping": False, "early_stopping": False,\n                    "base_OOF_predictions_used_as_features": False, "extra_positive_class_weight": False,\n                    "parameters": PARAMETERS[learner], "seed": seed, "seconds": time.monotonic() - started,\n                    "training_sample_file": sample_path.name,\n                    "training_sample_sha256": hashlib.sha256(sample_path.read_bytes()).hexdigest(),\n                    "smoke_only": smoke, **actual})\n            grid_scores[heldout, index] = predict_model(model, learner, arrays["x"][heldout])\n            del model\n    assert np.isfinite(grid_scores).all()\n    np.savez_compressed(run / "grid_scores.npz", prediction=grid_scores, targets=np.asarray(TARGETS))\n    ys, ps = [], []\n    for i, row in enumerate(manifest.itertuples(index=False)):\n        selected = arrays["record"] == i\n        p = expand_native(arrays["grid"][selected], grid_scores[selected], int(arrays["lengths"][i]))\n        if smoke:\n            y = synthetic[2][i]\n        else:\n            frame = pd.read_csv(row.path, usecols=list(TARGETS))\n            y = frame.loc[:, list(TARGETS)].to_numpy(np.uint8)\n        assert p.shape == y.shape and np.isin(y, [0, 1]).all()\n        np.savez_compressed(run / "oof" / (row.recording_id + ".npz"), prediction=p, targets=np.asarray(TARGETS))\n        ys.append(y); ps.append(p)\n    y, p = np.concatenate(ys), np.concatenate(ps)\n    scores = common.average_precision(y, p)\n    pd.DataFrame([scores]).to_csv(run / "native_oof_metrics.csv", index=False)\n    decision = {"status": "complete", "run_dir": str(run), "learner": learner, **scores,\n        "native_rows": len(y), "smoke_only": smoke, "targets": TARGETS,\n        "retained_heads_unchanged": True, "goal_two_target_mAP": .6,\n        "goal_reached": not smoke and scores["two_target_mAP"] >= .6,\n        "independent_generalization_established": False, "competition_submission_created": False}\n    common.write_json(run / "decision.json", decision)\n    common.write_json(output_root / "latest_run.json", decision)\n    print(json.dumps(decision, indent=2), flush=True)\n    return decision\n\n\ndef smoke_experiment(learner, output_root):\n    rng, xs, labels, folds, records, grids, native_labels = np.random.default_rng(1), [], [], [], [], [], []\n    for fold in range(5):\n        x = rng.normal(size=(48, 12)).astype(np.float32)\n        y = np.column_stack([np.arange(48) % 7 == 0, np.arange(48) % 11 == 0]).astype(np.uint8)\n        xs.append(x); labels.append(y); folds.extend([fold] * 48); records.extend([fold] * 48)\n        grids.extend(range(48)); native_labels.append(y)\n    manifest = pd.DataFrame({"recording_id": [f"smoke_{i}" for i in range(5)],\n        "subject": [f"subject_{i}" for i in range(5)], "fold": range(5)})\n    arrays = {"x": np.concatenate(xs), "y": np.concatenate(labels), "fold": np.asarray(folds, np.int8),\n        "record": np.asarray(records, np.int16), "grid": np.asarray(grids, np.int32),\n        "lengths": np.full(5, 48, np.int32), "recording_ids": manifest.recording_id.to_numpy(str),\n        "subject": manifest.subject.to_numpy(str)[records],\n        "feature_names": np.asarray([f"feature_{i}" for i in range(12)])}\n    return run_experiment(learner, replace(CFG, smoke_rounds=4), output_root, (arrays, manifest, native_labels))\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--learner", choices=LEARNERS, required=True)\n    parser.add_argument("--output-root", type=Path, required=True)\n    parser.add_argument("--smoke", action="store_true")\n    args = parser.parse_args()\n    if args.smoke:\n        smoke_experiment(args.learner, args.output_root)\n    else:\n        run_experiment(args.learner, output_root=args.output_root)\n', 'tdcs_walking_conditional_gpu': '"""Walking-only conditional CatBoost trials, with a frozen cross-fitted file gate."""\nfrom __future__ import annotations\nimport argparse\nimport hashlib\nimport json\nimport time\nfrom dataclasses import asdict, dataclass, replace\nfrom importlib.metadata import version\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom sklearn.metrics import average_precision_score\n\nimport tdcs_gpu_boosted_context as shared\n\nROOT = Path(__file__).resolve().parents[1]\nTARGETS = ("Walking",)\nVARIANTS = ("local_213", "context_325")\nPARENT_FINGERPRINT = "7a2aa76886a59c9dc66c"\nPARENT_LOCAL = ROOT / "kaggle-output/tdcs-gpu-catboost-context-v8-version-1/tdcs_gpu_catboost_v8" / PARENT_FINGERPRINT\nGATE_FILE = ROOT / "results/tdcs_walking_conditional_gpu_gate_payload.json"\nFLOOR = .05\n\n\n@dataclass(frozen=True)\nclass Config:\n    name: str = "tdcs_walking_conditional_gpu_v9"\n    targets: tuple = TARGETS\n    folds: tuple = (0, 1, 2, 3, 4)\n    negative_divisor: int = 4\n    seed: int = 4107\n    smoke_rounds: int = 4\n    max_seconds: int = 21600\n\n\nCFG = Config()\n\n\ndef digest(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef check_config(cfg):\n    if tuple(cfg.targets) != TARGETS or tuple(cfg.folds) != tuple(range(5)):\n        raise ValueError("Walking only and all five frozen folds required")\n    if min(cfg.negative_divisor, cfg.smoke_rounds, cfg.max_seconds) < 1:\n        raise ValueError("Invalid sampling or training budget")\n\n\ndef source_hashes():\n    return {"source_sha256": globals().get("SOURCE_SHA256") or digest(__file__),\n            "shared_sources": shared.source_hashes()}\n\n\ndef gate_payload():\n    payload = globals().get("FROZEN_GATE_PAYLOAD")\n    return payload if payload is not None else json.loads(GATE_FILE.read_text())\n\n\ndef payload_digest(payload):\n    return hashlib.sha256(json.dumps(payload, sort_keys=True).encode()).hexdigest()\n\n\ndef load_parent():\n    if Path("/kaggle/input").exists():\n        matches = []\n        for path in Path("/kaggle/input").rglob("configuration.json"):\n            config = json.loads(path.read_text())\n            if path.parent.name == PARENT_FINGERPRINT and config.get("learner") == "catboost":\n                matches.append(path.parent)\n        if len(matches) != 1:\n            raise ValueError("Attach exactly the completed GPU CatBoost v8 notebook output")\n        parent = matches[0]\n    else:\n        parent = PARENT_LOCAL\n    config = json.loads((parent / "configuration.json").read_text())\n    assert not config["smoke_only"] and config["sources"] == shared.source_hashes()\n    cache = parent.parent / config["feature_cache_file"]\n    assert digest(cache) == config["feature_cache_sha256"]\n    with np.load(cache, allow_pickle=False) as payload:\n        arrays = {key: payload[key].copy() for key in payload.files}\n    assert shared.features.feature_digest(arrays) == config["feature_array_sha256"]\n    raw, metadata = shared.common.locate_data()\n    manifest = shared.common.build_manifest(raw, metadata)\n    assert arrays["recording_ids"].tolist() == manifest.recording_id.tolist()\n    assert arrays["x"].shape == (436128, 325)\n    return parent, config, arrays, manifest\n\n\ndef native_presence(arrays, manifest):\n    presence, positives, rows = [], 0, 0\n    for i, row in enumerate(manifest.itertuples(index=False)):\n        y = pd.read_csv(row.path, usecols=["Walking"]).Walking.to_numpy(np.uint8)\n        assert np.isin(y, [0, 1]).all() and len(y) == arrays["lengths"][i]\n        selected = arrays["record"] == i\n        np.testing.assert_array_equal(y[arrays["grid"][selected]], arrays["y"][selected, 1])\n        presence.append(int(y.any())); positives += int(y.sum()); rows += len(y)\n    assert rows == 6959762 and positives == 207515 and sum(presence) == 66\n    return np.asarray(presence, np.uint8)\n\n\ndef conditional_plan(y, folds, subjects, presence, fold, seed, divisor=4):\n    y, folds, subjects, presence = map(np.asarray, (y, folds, subjects, presence))\n    if y.ndim != 1 or any(len(a) != len(y) for a in (folds, subjects, presence)):\n        raise ValueError("Aligned grid labels, folds, subjects and file presence required")\n    if not np.isin(y, [0, 1]).all() or not np.isin(presence, [0, 1]).all() or divisor < 1:\n        raise ValueError("Binary labels and positive sampling divisor required")\n    training = folds != fold\n    if ((y[training] == 1) & (presence[training] == 0)).any():\n        raise ValueError("Positive training rows in a file marked absent")\n    eligible = np.flatnonzero(training & (presence == 1))\n    if not len(eligible):\n        raise ValueError("No Walking-positive training files")\n    positive, negative = eligible[y[eligible] == 1], eligible[y[eligible] == 0]\n    count = max(1, int(np.ceil(len(negative) / divisor))) if len(negative) else 0\n    rng = np.random.default_rng(seed)\n    sampled = rng.choice(negative, count, replace=False) if count else np.empty(0, np.int64)\n    chosen = np.r_[positive, sampled]; rng.shuffle(chosen)\n    _, inverse, counts = np.unique(subjects[eligible], return_inverse=True, return_counts=True)\n    weights = np.zeros(len(y), np.float64); weights[eligible] = 1. / counts[inverse]\n    probability = count / len(negative) if len(negative) else 1.\n    weights[sampled] /= probability\n    values = weights[chosen]; values /= values.mean()\n    assert not (folds[chosen] == fold).any()\n    return chosen, values, probability, eligible\n\n\ndef combine_native(grid, raw_probability, length, file_score):\n    if not 0 <= file_score <= 1 or not np.isfinite(file_score):\n        raise ValueError("Finite cross-fitted file score required")\n    if raw_probability.shape != (len(grid),):\n        raise ValueError("One raw Walking score per grid row required")\n    if grid[0] != 0 or grid[-1] >= length or np.any(np.diff(grid) <= 0):\n        raise ValueError("Ordered in-record grid required")\n    if not np.isfinite(raw_probability).all() or ((raw_probability < 0) | (raw_probability > 1)).any():\n        raise ValueError("Invalid local score")\n    raw = np.interp(np.arange(length), grid, raw_probability).astype(np.float32)\n    # A one-dimensional float32 gate matches the original gate implementation\n    # and avoids NumPy 1.x/2.x differences in zero-dimensional scalar promotion.\n    factor = (FLOOR + (1 - FLOOR) * np.asarray([file_score], np.float32)).astype(np.float32)[0]\n    return (raw * factor).astype(np.float32)[:, None]\n\n\ndef run_experiment(variant, cfg=CFG, output_root=None, synthetic=None):\n    check_config(cfg)\n    if variant not in VARIANTS:\n        raise ValueError("Unknown feature comparison")\n    smoke = synthetic is not None\n    if not smoke and not torch.cuda.is_available():\n        raise RuntimeError("Real-data Walking fitting requires Kaggle CUDA; no CPU fallback")\n    columns = 213 if variant == "local_213" else 325\n    output_root = Path(output_root or ROOT / "results/tdcs_walking_conditional_gpu" / variant)\n    output_root.mkdir(parents=True, exist_ok=True)\n    if not smoke:\n        parent, parent_config, arrays, manifest = load_parent()\n        presence = native_presence(arrays, manifest)\n        gates = gate_payload()\n        assert gates["recording_ids"] == manifest.recording_id.tolist()\n        assert gates["folds"] == manifest.fold.tolist()\n        q = np.asarray(gates["scores"], np.float32)\n    else:\n        arrays, manifest, presence, q, _ = synthetic\n        parent, parent_config = Path("synthetic"), {}\n        gates = {"recording_ids": manifest.recording_id.tolist(), "folds": manifest.fold.tolist(), "scores": q.tolist()}\n    assert q.shape == (len(manifest),) and np.isfinite(q).all() and ((q >= 0) & (q <= 1)).all()\n    population = ("synthetic wiring only; not TDCS performance" if smoke else\n        manifest[["recording_id", "subject", "fold", "source_bytes"]].to_csv(index=False, lineterminator="\\n"))\n    config = {"configuration": asdict(cfg), "targets": TARGETS, "variant": variant,\n        "feature_count": columns, "sources": source_hashes(), "smoke_only": smoke,\n        "population_sha256": hashlib.sha256(population.encode()).hexdigest(),\n        "parent_fingerprint": parent.name, "parent_feature_array_sha256": parent_config.get("feature_array_sha256"),\n        "parent_cache_sha256": parent_config.get("feature_cache_sha256"),\n        "gate_payload_sha256": payload_digest(gates), "gate_floor": FLOOR,\n        "runtime": {name: version(name) for name in ("catboost", "numpy", "torch")},\n        "used_gpu_indices": [] if smoke else [0],\n        "parameters": shared.PARAMETERS["catboost"], "training_scope": "Walking-positive training files only",\n        "gate_used_as_fitting_feature": False, "base_OOF_predictions_used_as_features": False,\n        "outer_labels_used_for_fitting_or_stopping": False, "early_stopping": False,\n        "raw_local_scores_gated_exactly_once": True, "retained_heads_overwritten": False,\n        "competition_submission_created": False}\n    fingerprint = hashlib.sha256(json.dumps(config, sort_keys=True).encode()).hexdigest()[:20]\n    run = output_root / fingerprint; run.mkdir(parents=True, exist_ok=True)\n    shared.common.write_json(run / "configuration.json", config)\n    shared.common.write_json(run / "frozen_gate_payload.json", gates)\n    manifest.to_csv(run / "frozen_tdcs_manifest.csv", index=False)\n    (run / "oof").mkdir(exist_ok=True)\n    raw_scores = np.full(len(arrays["y"]), np.nan, np.float32)\n    deadline = time.monotonic() + cfg.max_seconds\n    for fold in cfg.folds:\n        selected, weights, probability, eligible = conditional_plan(arrays["y"][:, 1],\n            arrays["fold"], arrays["subject"], presence[arrays["record"]], fold, cfg.seed + 1009 * fold,\n            cfg.negative_divisor)\n        heldout = arrays["fold"] == fold\n        training_subjects = set(arrays["subject"][eligible]); test_subjects = set(arrays["subject"][heldout])\n        assert training_subjects.isdisjoint(test_subjects)\n        if time.monotonic() >= deadline:\n            raise TimeoutError("Walking safety budget reached")\n        directory = run / f"fold_{fold}"; directory.mkdir(exist_ok=True)\n        model_path = directory / "Walking.cbm"; receipt_path = directory / "Walking_training_receipt.json"\n        np.savez_compressed(directory / "Walking_training_sample.npz", indices=selected, weights=weights)\n        if model_path.is_file() and receipt_path.is_file():\n            assert json.loads(receipt_path.read_text())["fingerprint"] == fingerprint\n            model = shared.load_model("catboost", model_path)\n        else:\n            print(f"{variant} fold {fold}: {len(selected):,} rows from {len(training_subjects)} Walking-positive training subjects", flush=True)\n            model, backend = shared.fit_model("catboost", arrays["x"][selected, :columns],\n                arrays["y"][selected, 1], weights, cfg.seed + 1009 * fold, gpu=not smoke,\n                rounds=cfg.smoke_rounds if smoke else None)\n            shared.save_model(model, "catboost", model_path)\n            shared.common.write_json(receipt_path, {"fingerprint": fingerprint, "target": "Walking",\n                "training_subjects": sorted(training_subjects), "held_out_subjects": sorted(test_subjects),\n                "eligible_training_grid_rows": len(eligible), "sampled_grid_rows": len(selected),\n                "training_recording_ids": arrays["recording_ids"][np.unique(arrays["record"][eligible])].tolist(),\n                "all_positive_eligible_rows_kept": True, "negative_sampling_probability": probability,\n                "inverse_sampling_correction": True, "subject_weights_from_eligible_training_rows_only": True,\n                "outer_labels_used_for_fitting_or_stopping": False, "early_stopping": False,\n                "gate_used_as_fitting_feature": False, "base_OOF_predictions_used_as_features": False,\n                "feature_count": columns, "seed": cfg.seed + 1009 * fold, "smoke_only": smoke,\n                "sample_sha256": digest(directory / "Walking_training_sample.npz"), **backend})\n        raw_scores[heldout] = shared.predict_model(model, "catboost", arrays["x"][heldout, :columns])\n        del model\n    assert np.isfinite(raw_scores).all()\n    np.savez_compressed(run / "grid_scores.npz", prediction=raw_scores[:, None], targets=np.asarray(TARGETS))\n    ys, ps = [], []\n    for i, row in enumerate(manifest.itertuples(index=False)):\n        selected = arrays["record"] == i\n        p = combine_native(arrays["grid"][selected], raw_scores[selected], int(arrays["lengths"][i]), q[i])\n        y = (synthetic[4][i] if smoke else\n             pd.read_csv(row.path, usecols=["Walking"]).loc[:, ["Walking"]].to_numpy(np.uint8))\n        assert p.shape == y.shape\n        np.savez_compressed(run / "oof" / (row.recording_id + ".npz"), prediction=p, targets=np.asarray(TARGETS))\n        ys.append(y); ps.append(p)\n    y, p = np.concatenate(ys), np.concatenate(ps)\n    ap = float(average_precision_score(y[:, 0], p[:, 0]))\n    pd.DataFrame([{"Walking": ap}]).to_csv(run / "native_oof_metrics.csv", index=False)\n    result = {"status": "complete", "run_dir": str(run), "variant": variant, "Walking": ap,\n        "native_rows": len(y), "targets": TARGETS, "smoke_only": smoke,\n        "no_two_or_three_target_mAP_claim": True, "retained_heads_unchanged": True,\n        "independent_generalization_established": False, "competition_submission_created": False}\n    shared.common.write_json(run / "decision.json", result)\n    shared.common.write_json(output_root / "latest_run.json", result)\n    print(json.dumps(result, indent=2), flush=True)\n    return result\n\n\ndef smoke_experiment(variant, output_root):\n    rng = np.random.default_rng(1); folds = np.repeat(np.arange(5), 64).astype(np.int8)\n    records = folds.astype(np.int16); y = np.zeros((320, 2), np.uint8)\n    native = []\n    for fold in range(5):\n        labels = ((np.arange(64) % 11 == 0) if fold != 4 else np.zeros(64, bool)).astype(np.uint8)\n        y[fold * 64:(fold + 1) * 64, 1] = labels; native.append(labels[:, None])\n    manifest = pd.DataFrame({"recording_id": [f"smoke_{i}" for i in range(5)],\n        "subject": [f"subject_{i}" for i in range(5)], "fold": range(5)})\n    arrays = {"x": rng.normal(size=(320, 325)).astype(np.float32), "y": y, "fold": folds,\n        "subject": manifest.subject.to_numpy(str)[records], "record": records,\n        "grid": np.tile(np.arange(64, dtype=np.int32), 5), "lengths": np.full(5, 64),\n        "recording_ids": manifest.recording_id.to_numpy(str)}\n    return run_experiment(variant, cfg=replace(CFG, smoke_rounds=4), output_root=output_root,\n        synthetic=(arrays, manifest, np.asarray([1, 1, 1, 1, 0], np.uint8), np.full(5, .5, np.float32), native))\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--variant", choices=VARIANTS, required=True)\n    parser.add_argument("--output-root", type=Path, required=True)\n    parser.add_argument("--smoke", action="store_true")\n    args = parser.parse_args()\n    (smoke_experiment if args.smoke else run_experiment)(args.variant, output_root=args.output_root)\n', 'tdcs_gpu_regularized_context': '"""Two fixed lower-capacity GPU CatBoost trials on the full TDCS training cohort.\n\nThe v8 feature cache and helpers are immutable. This module does not use the\nconditional v9 training population or its file gate; only its parent-cache loader.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nimport time\nfrom dataclasses import asdict, dataclass, replace\nfrom importlib.metadata import version\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\n\nimport tdcs_walking_conditional_gpu as parent_loader\n\nshared = parent_loader.shared\nROOT = Path(__file__).resolve().parents[1]\nTARGETS = ("StartHesitation", "Walking")\nVARIANTS = ("shallow_depth4", "regularized_depth6")\nPARAMETERS = {\n    variant: {**shared.PARAMETERS["catboost"], "depth": depth, "l2_leaf_reg": 100.}\n    for variant, depth in zip(VARIANTS, (4, 6))\n}\n\n\n@dataclass(frozen=True)\nclass Config:\n    name: str = "tdcs_gpu_regularized_context_v10"\n    targets: tuple = TARGETS\n    folds: tuple = (0, 1, 2, 3, 4)\n    negative_divisor: int = 4\n    seed: int = 5107\n    smoke_rounds: int = 4\n    max_seconds: int = 21600\n\n\nCFG = Config()\n\n\ndef digest(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef check_config(cfg):\n    if tuple(cfg.targets) != TARGETS or tuple(cfg.folds) != tuple(range(5)):\n        raise ValueError("Both difficult targets and all five frozen subject folds required")\n    if min(cfg.negative_divisor, cfg.smoke_rounds, cfg.max_seconds) < 1:\n        raise ValueError("Positive sampling divisor and training budget required")\n\n\ndef source_hashes():\n    return {"source_sha256": globals().get("SOURCE_SHA256") or digest(__file__),\n        "parent_loader_sha256": getattr(parent_loader, "SOURCE_SHA256", None) or digest(parent_loader.__file__),\n        "shared_sources": shared.source_hashes()}\n\n\ndef fit_model(variant, x, y, weights, seed, gpu=True, rounds=None):\n    if variant not in VARIANTS:\n        raise ValueError("Unknown fixed regularization trial")\n    if gpu and not torch.cuda.is_available():\n        raise RuntimeError("Real-data regularized fitting requires Kaggle CUDA; no CPU fallback")\n    from catboost import CatBoostClassifier\n    params = {**PARAMETERS[variant], "task_type": "GPU" if gpu else "CPU", "random_seed": seed}\n    if gpu:\n        params.update(devices="0", gpu_ram_part=.6)\n    if rounds is not None:\n        params["iterations"] = rounds\n    model = CatBoostClassifier(**params)\n    model.fit(x, y, sample_weight=weights, verbose=200 if rounds is None else False)\n    actual = model.get_all_params()["task_type"]\n    if gpu and actual != "GPU":\n        raise RuntimeError("CatBoost did not use the requested GPU backend")\n    assert model.tree_count_ == params["iterations"]\n    return model, {"actual_training_backend": actual, "trees": model.tree_count_}\n\n\ndef run_experiment(variant, cfg=CFG, output_root=None, synthetic=None):\n    check_config(cfg)\n    if variant not in VARIANTS:\n        raise ValueError("Unknown fixed regularization trial")\n    smoke = synthetic is not None\n    if not smoke and not torch.cuda.is_available():\n        raise RuntimeError("Real-data regularized fitting requires Kaggle CUDA; no CPU fallback")\n    output_root = Path(output_root or ROOT / "results/tdcs_gpu_regularized_context" / variant)\n    output_root.mkdir(parents=True, exist_ok=True)\n    if smoke:\n        arrays, manifest, native_labels = synthetic\n        parent, parent_config = Path("synthetic"), {}\n    else:\n        parent, parent_config, arrays, manifest = parent_loader.load_parent()\n        native_labels = None\n    population = ("synthetic wiring only; not TDCS performance" if smoke else\n        manifest[["recording_id", "subject", "fold", "source_bytes"]].to_csv(index=False, lineterminator="\\n"))\n    config = {"configuration": asdict(cfg), "variant": variant, "targets": TARGETS,\n        "learner": "catboost", "feature_count": 325, "parameters": PARAMETERS[variant],\n        "sources": source_hashes(), "smoke_only": smoke,\n        "population_sha256": hashlib.sha256(population.encode()).hexdigest(),\n        "parent_fingerprint": parent.name,\n        "parent_feature_array_sha256": parent_config.get("feature_array_sha256"),\n        "parent_cache_sha256": parent_config.get("feature_cache_sha256"),\n        "runtime": {name: version(name) for name in ("catboost", "numpy", "torch")},\n        "training_scope": "all training recordings; positive and negative files eligible",\n        "negative_sampling": "all positives plus seeded one-quarter negatives with inverse correction",\n        "subject_weights": "inverse full-fitting-row counts from training subjects only; normalized",\n        "conditional_training": False, "file_gate_applied": False, "early_stopping": False,\n        "outer_labels_used_for_fitting_or_stopping": False, "base_OOF_predictions_used_as_features": False,\n        "extra_positive_class_weights": False, "used_gpu_indices": [] if smoke else [0],\n        "prediction_backend": "CPU scoring of saved GPU models; in-record interpolation to native rows",\n        "reused_development_folds": True, "confidence_calibrated": False,\n        "retained_heads_overwritten": False, "competition_submission_created": False}\n    fingerprint = hashlib.sha256(json.dumps(config, sort_keys=True).encode()).hexdigest()[:20]\n    run = output_root / fingerprint; run.mkdir(parents=True, exist_ok=True)\n    shared.common.write_json(run / "configuration.json", config)\n    manifest.to_csv(run / "frozen_tdcs_manifest.csv", index=False)\n    (run / "oof").mkdir(exist_ok=True)\n    grid_scores = np.full(arrays["y"].shape, np.nan, np.float32)\n    deadline = time.monotonic() + cfg.max_seconds\n    for fold in cfg.folds:\n        heldout = arrays["fold"] == fold\n        training_subjects = sorted(set(arrays["subject"][~heldout]))\n        heldout_subjects = sorted(set(arrays["subject"][heldout]))\n        assert training_subjects and heldout_subjects and set(training_subjects).isdisjoint(heldout_subjects)\n        directory = run / f"fold_{fold}"; directory.mkdir(exist_ok=True)\n        for index, target in enumerate(TARGETS):\n            if time.monotonic() >= deadline:\n                raise TimeoutError("Regularized CatBoost safety budget reached")\n            seed = cfg.seed + 1009 * fold + index\n            chosen, weights, probability = shared.training_plan(arrays["y"][:, index], arrays["fold"],\n                arrays["subject"], fold, seed, cfg.negative_divisor)\n            sample_path = directory / (target + "_training_sample.npz")\n            model_path = directory / (target + ".cbm")\n            receipt_path = directory / (target + "_training_receipt.json")\n            existing = (model_path.is_file(), receipt_path.is_file(), sample_path.is_file())\n            if any(existing):\n                # Never rewrite sampling provenance underneath an existing model.\n                if not all(existing):\n                    raise ValueError("Incomplete model/sample/receipt; preserve artifacts and use a fresh output root")\n                receipt = json.loads(receipt_path.read_text())\n                assert receipt["fingerprint"] == fingerprint\n                assert receipt["target"] == target and receipt["variant"] == variant\n                assert receipt["seed"] == seed and receipt["parameters"] == PARAMETERS[variant]\n                assert receipt["trees"] == (cfg.smoke_rounds if smoke else PARAMETERS[variant]["iterations"])\n                assert receipt["actual_training_backend"] == ("CPU" if smoke else "GPU")\n                assert receipt["smoke_only"] == smoke\n                assert receipt["training_sample_sha256"] == digest(sample_path)\n                with np.load(sample_path, allow_pickle=False) as saved_sample:\n                    np.testing.assert_array_equal(saved_sample["indices"], chosen)\n                    np.testing.assert_array_equal(saved_sample["weights"], weights)\n                model = shared.load_model("catboost", model_path)\n            else:\n                np.savez_compressed(sample_path, indices=chosen, weights=weights)\n                print(f"{variant} fold {fold} {target}: {len(chosen):,} fitting grid rows", flush=True)\n                started = time.monotonic()\n                model, backend = fit_model(variant, arrays["x"][chosen], arrays["y"][chosen, index],\n                    weights, seed, gpu=not smoke, rounds=cfg.smoke_rounds if smoke else None)\n                shared.save_model(model, "catboost", model_path)\n                shared.common.write_json(receipt_path, {"fingerprint": fingerprint, "target": target,\n                    "variant": variant, "parameters": PARAMETERS[variant], "seed": seed,\n                    "training_subjects": training_subjects, "held_out_subjects": heldout_subjects,\n                    "training_grid_rows_before_sampling": int((~heldout).sum()),\n                    "sampled_grid_rows": len(chosen), "all_positive_training_rows_kept": True,\n                    "negative_sampling_probability": probability, "inverse_negative_sampling_correction": True,\n                    "subject_weights_training_only": True, "conditional_training": False,\n                    "outer_labels_used_for_fitting_or_stopping": False, "early_stopping": False,\n                    "base_OOF_predictions_used_as_features": False, "file_gate_applied": False,\n                    "extra_positive_class_weights": False, "training_sample_file": sample_path.name,\n                    "training_sample_sha256": digest(sample_path), "smoke_only": smoke,\n                    "seconds": time.monotonic() - started, **backend})\n            grid_scores[heldout, index] = shared.predict_model(model, "catboost", arrays["x"][heldout])\n            del model\n    assert np.isfinite(grid_scores).all()\n    np.savez_compressed(run / "grid_scores.npz", prediction=grid_scores, targets=np.asarray(TARGETS))\n    ys, ps = [], []\n    for i, row in enumerate(manifest.itertuples(index=False)):\n        selected = arrays["record"] == i\n        p = shared.expand_native(arrays["grid"][selected], grid_scores[selected], int(arrays["lengths"][i]))\n        y = (native_labels[i] if smoke else pd.read_csv(row.path, usecols=list(TARGETS)).loc[:, list(TARGETS)].to_numpy(np.uint8))\n        assert p.shape == y.shape and np.isin(y, [0, 1]).all()\n        np.savez_compressed(run / "oof" / (row.recording_id + ".npz"), prediction=p, targets=np.asarray(TARGETS))\n        ys.append(y); ps.append(p)\n    y, p = np.concatenate(ys), np.concatenate(ps)\n    scores = shared.common.average_precision(y, p)\n    pd.DataFrame([scores]).to_csv(run / "native_oof_metrics.csv", index=False)\n    result = {"status": "complete", "run_dir": str(run), "variant": variant, "targets": TARGETS,\n        **scores, "native_rows": len(y), "smoke_only": smoke, "retained_heads_unchanged": True,\n        "independent_generalization_established": False, "competition_submission_created": False}\n    shared.common.write_json(run / "decision.json", result)\n    shared.common.write_json(output_root / "latest_run.json", result)\n    print(json.dumps(result, indent=2), flush=True)\n    return result\n\n\ndef smoke_experiment(variant, output_root):\n    rng = np.random.default_rng(1)\n    folds = np.repeat(np.arange(5), 64).astype(np.int8); record = folds.astype(np.int16)\n    y = np.zeros((320, 2), np.uint8)\n    y[:, 0] = (np.arange(320) % 13 == 0).astype(np.uint8)\n    y[:, 1] = (np.arange(320) % 11 == 1).astype(np.uint8)\n    y[folds == 4] = 0\n    manifest = pd.DataFrame({"recording_id": [f"smoke_{i}" for i in range(5)],\n        "subject": [f"subject_{i}" for i in range(5)], "fold": range(5)})\n    arrays = {"x": rng.normal(size=(320, 325)).astype(np.float32), "y": y, "fold": folds,\n        "subject": manifest.subject.to_numpy(str)[record], "record": record,\n        "grid": np.tile(np.arange(64, dtype=np.int32), 5), "lengths": np.full(5, 64),\n        "recording_ids": manifest.recording_id.to_numpy(str)}\n    native = [y[folds == i].copy() for i in range(5)]\n    return run_experiment(variant, cfg=replace(CFG, smoke_rounds=4), output_root=output_root,\n        synthetic=(arrays, manifest, native))\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--variant", choices=VARIANTS, required=True)\n    parser.add_argument("--output-root", type=Path, required=True)\n    parser.add_argument("--smoke", action="store_true")\n    args = parser.parse_args()\n    (smoke_experiment if args.smoke else run_experiment)(args.variant, output_root=args.output_root)\n', 'tdcs_subject_weight_gpu': '"""Matched GPU trials of weaker training-subject balancing on TDCS.\n\nThe only difference from the v10 regularized recipe is the subject-count\nexponent: 0 (row-uniform) or 0.5 (square-root balancing), rather than 1.\nSampling, seeds, model parameters, features and evaluation population stay fixed.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nimport time\nfrom dataclasses import asdict, dataclass, replace\nfrom importlib.metadata import version\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\n\nimport tdcs_gpu_regularized_context as parent\n\nshared = parent.shared\nROOT = Path(__file__).resolve().parents[1]\nTARGETS = parent.TARGETS\nEXPONENTS = {"row_uniform": 0., "subject_sqrt": .5}\nVARIANTS = tuple(EXPONENTS)\nPARAMETERS = parent.PARAMETERS["regularized_depth6"]\n\n\n@dataclass(frozen=True)\nclass Config:\n    name: str = "tdcs_subject_weight_gpu_v12"\n    targets: tuple = TARGETS\n    folds: tuple = (0, 1, 2, 3, 4)\n    negative_divisor: int = 4\n    seed: int = 5107\n    smoke_rounds: int = 4\n    max_seconds: int = 21600\n\n\nCFG = Config()\n\n\ndef digest(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef source_hashes():\n    return {"source_sha256": globals().get("SOURCE_SHA256") or digest(__file__),\n        "parent_sources": parent.source_hashes()}\n\n\ndef check_config(cfg):\n    if tuple(cfg.targets) != TARGETS or tuple(cfg.folds) != tuple(range(5)):\n        raise ValueError("Both difficult targets and all five frozen folds required")\n    if min(cfg.negative_divisor, cfg.smoke_rounds, cfg.max_seconds) < 1:\n        raise ValueError("Positive sampling and budget required")\n\n\ndef training_plan(y, folds, subjects, fold, seed, divisor, exponent):\n    if exponent not in (0., .5, 1.):\n        raise ValueError("Only fixed exponents 0, 0.5 and control 1 are supported")\n    chosen, _, probability = shared.training_plan(y, folds, subjects, fold, seed, divisor)\n    y, folds, subjects = np.asarray(y), np.asarray(folds), np.asarray(subjects, str)\n    names, counts = np.unique(subjects[folds != fold], return_counts=True)\n    lookup = dict(zip(names, counts))\n    weights = np.asarray([float(lookup[s]) ** -exponent for s in subjects[chosen]], np.float64)\n    weights[y[chosen] == 0] /= probability\n    weights /= weights.mean()\n    assert np.isfinite(weights).all() and (weights > 0).all()\n    return chosen, weights, probability\n\n\ndef synthetic_inputs():\n    rng = np.random.default_rng(71)\n    lengths = np.asarray([32, 48, 64, 80, 96], np.int32)\n    record = np.repeat(np.arange(5), lengths).astype(np.int16)\n    grid = np.concatenate([np.arange(n, dtype=np.int32) for n in lengths])\n    y = np.column_stack([grid % 13 == 0, grid % 11 == 1]).astype(np.uint8)\n    y[record == 4] = 0\n    manifest = pd.DataFrame({"recording_id": [f"smoke_{i}" for i in range(5)],\n        "subject": [f"subject_{i}" for i in range(5)], "fold": range(5)})\n    arrays = {"x": rng.normal(size=(len(record), 325)).astype(np.float32),\n        "y": y, "fold": record.astype(np.int8), "record": record, "grid": grid,\n        "subject": manifest.subject.to_numpy(str)[record], "lengths": lengths,\n        "recording_ids": manifest.recording_id.to_numpy(str)}\n    return arrays, manifest, [y[record == i].copy() for i in range(5)]\n\n\ndef run_experiment(variant, cfg=CFG, output_root=None, synthetic=None):\n    check_config(cfg)\n    if variant not in VARIANTS:\n        raise ValueError("Unknown weighting trial")\n    smoke = synthetic is not None\n    if not smoke and not torch.cuda.is_available():\n        raise RuntimeError("Real-data weighting fitting requires Kaggle CUDA; no CPU fallback")\n    output_root = Path(output_root or ROOT / "results/tdcs_subject_weight_gpu" / variant)\n    output_root.mkdir(parents=True, exist_ok=True)\n    if smoke:\n        arrays, manifest, native_labels = synthetic\n        parent_run, parent_config = Path("synthetic"), {}\n    else:\n        parent_run, parent_config, arrays, manifest = parent.parent_loader.load_parent()\n        native_labels = None\n    population = ("synthetic wiring only; not TDCS performance" if smoke else\n        manifest[["recording_id", "subject", "fold", "source_bytes"]].to_csv(index=False, lineterminator="\\n"))\n    exponent = EXPONENTS[variant]\n    config = {"configuration": asdict(cfg), "variant": variant, "targets": TARGETS,\n        "learner": "catboost", "parameters": PARAMETERS, "sources": source_hashes(),\n        "smoke_only": smoke, "feature_count": 325, "subject_weight_exponent": exponent,\n        "weight_formula": "training_subject_full_grid_row_count ** (-exponent), times inverse negative-sampling probability, normalized to mean 1",\n        "parent_fingerprint": parent_run.name,\n        "parent_feature_array_sha256": parent_config.get("feature_array_sha256"),\n        "parent_cache_sha256": parent_config.get("feature_cache_sha256"),\n        "population_sha256": hashlib.sha256(population.encode()).hexdigest(),\n        "runtime": {name: version(name) for name in ("catboost", "numpy", "torch")},\n        "used_gpu_indices": [] if smoke else [0],\n        "prediction_backend": "CPU saved-model scoring; in-record interpolation to native rows",\n        "training_scope": "all training recordings, including negative files",\n        "negative_sampling": "all positives plus seeded quarter negatives with inverse correction",\n        "conditional_training": False, "file_gate_applied": False, "early_stopping": False,\n        "extra_positive_class_weights": False, "base_OOF_predictions_used_as_features": False,\n        "event_boundaries_used_as_features": False, "task_intervals_used_as_features": False,\n        "IDs_or_subjects_used_as_features": False, "metadata_used_as_features": False,\n        "outer_labels_used_for_fitting_or_stopping": False, "retained_heads_overwritten": False,\n        "competition_submission_created": False, "confidence_calibrated": False,\n        "independent_generalization_established": False}\n    fingerprint = hashlib.sha256(json.dumps(config, sort_keys=True).encode()).hexdigest()[:20]\n    run = output_root / fingerprint\n    run.mkdir(parents=True, exist_ok=True)\n    shared.common.write_json(run / "configuration.json", config)\n    manifest.to_csv(run / "frozen_tdcs_manifest.csv", index=False)\n    (run / "oof").mkdir(exist_ok=True)\n    scores = np.full(arrays["y"].shape, np.nan, np.float32)\n    deadline = time.monotonic() + cfg.max_seconds\n    for fold in cfg.folds:\n        heldout = arrays["fold"] == fold\n        training_subjects = sorted(set(arrays["subject"][~heldout]))\n        heldout_subjects = sorted(set(arrays["subject"][heldout]))\n        assert training_subjects and heldout_subjects and set(training_subjects).isdisjoint(heldout_subjects)\n        directory = run / f"fold_{fold}"\n        directory.mkdir(exist_ok=True)\n        for index, target in enumerate(TARGETS):\n            if time.monotonic() >= deadline:\n                raise TimeoutError("Subject-weight experiment safety budget reached")\n            seed = cfg.seed + 1009 * fold + index\n            chosen, weights, probability = training_plan(arrays["y"][:, index], arrays["fold"],\n                arrays["subject"], fold, seed, cfg.negative_divisor, exponent)\n            sample_path = directory / (target + "_training_sample.npz")\n            model_path = directory / (target + ".cbm")\n            receipt_path = directory / (target + "_training_receipt.json")\n            existing = [p.is_file() for p in (sample_path, model_path, receipt_path)]\n            if any(existing):\n                if not all(existing):\n                    raise ValueError("Incomplete model/sample/receipt; preserve artifacts and choose a fresh root")\n                receipt = json.loads(receipt_path.read_text())\n                assert receipt["fingerprint"] == fingerprint and receipt["target"] == target\n                assert receipt["variant"] == variant and receipt["seed"] == seed\n                assert receipt["subject_weight_exponent"] == exponent and receipt["parameters"] == PARAMETERS\n                assert receipt["smoke_only"] == smoke and receipt["sample_sha256"] == digest(sample_path)\n                assert receipt["trees"] == (cfg.smoke_rounds if smoke else PARAMETERS["iterations"])\n                assert receipt["actual_training_backend"] == ("CPU" if smoke else "GPU")\n                with np.load(sample_path, allow_pickle=False) as saved:\n                    np.testing.assert_array_equal(saved["indices"], chosen)\n                    np.testing.assert_array_equal(saved["weights"], weights)\n                model = shared.load_model("catboost", model_path)\n            else:\n                np.savez_compressed(sample_path, indices=chosen, weights=weights)\n                print(f"{variant} fold {fold} {target}: {len(chosen):,} rows; exponent {exponent}", flush=True)\n                started = time.monotonic()\n                model, backend = parent.fit_model("regularized_depth6", arrays["x"][chosen],\n                    arrays["y"][chosen, index], weights, seed, gpu=not smoke,\n                    rounds=cfg.smoke_rounds if smoke else None)\n                shared.save_model(model, "catboost", model_path)\n                shared.common.write_json(receipt_path, {"fingerprint": fingerprint, "variant": variant,\n                    "target": target, "seed": seed, "parameters": PARAMETERS,\n                    "feature_count": 325, "subject_weight_exponent": exponent,\n                    "training_subjects": training_subjects, "held_out_subjects": heldout_subjects,\n                    "training_grid_rows_before_sampling": int((~heldout).sum()),\n                    "sampled_grid_rows": len(chosen), "negative_sampling_probability": probability,\n                    "all_positive_training_rows_kept": True, "inverse_sampling_correction": True,\n                    "subject_counts_from_full_training_rows_only": True,\n                    "outer_labels_used_for_fitting_or_stopping": False,\n                    "base_OOF_predictions_used_as_features": False, "early_stopping": False,\n                    "file_gate_applied": False, "extra_positive_class_weights": False,\n                    "sample_sha256": digest(sample_path), "smoke_only": smoke,\n                    "seconds": time.monotonic() - started, **backend})\n            scores[heldout, index] = shared.predict_model(model, "catboost", arrays["x"][heldout])\n            del model\n    assert np.isfinite(scores).all()\n    np.savez_compressed(run / "grid_scores.npz", prediction=scores, targets=np.asarray(TARGETS))\n    ys, ps = [], []\n    for i, row in enumerate(manifest.itertuples(index=False)):\n        selected = arrays["record"] == i\n        p = shared.expand_native(arrays["grid"][selected], scores[selected], int(arrays["lengths"][i]))\n        y = (native_labels[i] if smoke else\n            pd.read_csv(row.path, usecols=list(TARGETS)).loc[:, list(TARGETS)].to_numpy(np.uint8))\n        assert p.shape == y.shape and np.isin(y, [0, 1]).all()\n        np.savez_compressed(run / "oof" / (row.recording_id + ".npz"), prediction=p, targets=np.asarray(TARGETS))\n        ys.append(y); ps.append(p)\n    y, p = np.concatenate(ys), np.concatenate(ps)\n    values = shared.common.average_precision(y, p)\n    pd.DataFrame([values]).to_csv(run / "native_oof_metrics.csv", index=False)\n    result = {"status": "complete", "run_dir": str(run), "variant": variant,\n        "targets": TARGETS, **values, "native_rows": len(y), "smoke_only": smoke,\n        "retained_heads_unchanged": True, "confidence_calibrated": False,\n        "independent_generalization_established": False, "competition_submission_created": False}\n    shared.common.write_json(run / "decision.json", result)\n    shared.common.write_json(output_root / "latest_run.json", result)\n    print(json.dumps(result, indent=2), flush=True)\n    return result\n\n\ndef smoke_experiment(variant, output_root):\n    return run_experiment(variant, replace(CFG, smoke_rounds=4), output_root, synthetic_inputs())\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--variant", choices=VARIANTS, required=True)\n    parser.add_argument("--output-root", type=Path, required=True)\n    parser.add_argument("--smoke", action="store_true")\n    args = parser.parse_args()\n    (smoke_experiment if args.smoke else run_experiment)(args.variant, output_root=args.output_root)\n', 'tdcs_record_rank_gpu': '"""Fixed per-record rank feature trials; fit only on Kaggle CUDA.\n\nCompare replacing local features by their within-record mid-percentiles with\nappending those percentiles. A frozen cross-fitted Walking file gate is applied\nexactly once after native interpolation. No gate score is a fitting feature.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nimport time\nfrom dataclasses import asdict, dataclass, replace\nfrom importlib.metadata import version\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom scipy.stats import rankdata\n\nimport tdcs_subject_weight_gpu as parent\n\nshared = parent.shared\ngate_helper = parent.parent.parent_loader\nROOT = Path(__file__).resolve().parents[1]\nTARGETS = parent.TARGETS\nVARIANTS = ("rank_replace", "rank_append")\nFEATURE_COUNTS = {"rank_replace": 325, "rank_append": 538}\nPARAMETERS = parent.PARAMETERS\nLOCAL_COLUMNS = 213\nGATE_FLOOR = .05\n\n\n@dataclass(frozen=True)\nclass Config:\n    name: str = "tdcs_record_rank_gpu_v16"\n    targets: tuple = TARGETS\n    folds: tuple = (0, 1, 2, 3, 4)\n    negative_divisor: int = 4\n    subject_weight_exponent: float = .5\n    seed: int = 5107\n    smoke_rounds: int = 4\n    max_seconds: int = 21600\n\n\nCFG = Config()\n\n\ndef digest(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef array_digest(value):\n    value = np.ascontiguousarray(value)\n    result = hashlib.sha256()\n    result.update(value.dtype.str.encode())\n    result.update(str(value.shape).encode())\n    result.update(memoryview(value).cast("B"))\n    return result.hexdigest()\n\n\ndef source_hashes():\n    return {"source_sha256": globals().get("SOURCE_SHA256") or digest(__file__),\n        "parent_sources": parent.source_hashes()}\n\n\ndef check_config(cfg):\n    if tuple(cfg.targets) != TARGETS or tuple(cfg.folds) != tuple(range(5)):\n        raise ValueError("Both targets and all five frozen subject folds required")\n    if cfg.subject_weight_exponent != .5:\n        raise ValueError("Fixed square-root training-subject weights required")\n    if min(cfg.negative_divisor, cfg.smoke_rounds, cfg.max_seconds) < 1:\n        raise ValueError("Positive sampling and training budget required")\n\n\ndef record_percentiles(x, records):\n    """Label-free (average rank - .5)/n, independently within each recording."""\n    x, records = np.asarray(x), np.asarray(records)\n    if x.ndim != 2 or x.shape[1] != 325 or len(x) == 0 or not np.isfinite(x).all():\n        raise ValueError("Nonempty finite 325-channel feature matrix required")\n    if records.shape != (len(x),) or not np.issubdtype(records.dtype, np.integer):\n        raise ValueError("One integer recording index per feature row required")\n    names, counts = np.unique(records, return_counts=True)\n    if not np.array_equal(names, np.arange(len(names))) or np.any(np.diff(records) < 0):\n        raise ValueError("Contiguous ordered recording blocks numbered from zero required")\n    result = np.empty((len(x), LOCAL_COLUMNS), np.float32)\n    begin = 0\n    for count in counts:\n        end = begin + int(count)\n        ranks = rankdata(x[begin:end, :LOCAL_COLUMNS], method="average", axis=0, nan_policy="raise")\n        result[begin:end] = ((ranks - .5) / count).astype(np.float32)\n        begin = end\n    assert begin == len(x) and np.isfinite(result).all()\n    assert ((result > 0) & (result < 1)).all()\n    return result\n\n\ndef transform_features(arrays, variant):\n    if variant not in VARIANTS:\n        raise ValueError("Unknown fixed rank-feature trial")\n    names = np.asarray(arrays["feature_names"], str)\n    if names.shape != (325,) or len(set(names)) != 325:\n        raise ValueError("325 unique frozen feature names required")\n    ranks = record_percentiles(arrays["x"], arrays["record"])\n    rank_names = np.char.add("record_midpercentile_", names[:LOCAL_COLUMNS])\n    if variant == "rank_replace":\n        x = np.column_stack([ranks, arrays["x"][:, LOCAL_COLUMNS:]]).astype(np.float32)\n        output_names = np.r_[rank_names, names[LOCAL_COLUMNS:]]\n    else:\n        x = np.column_stack([arrays["x"], ranks]).astype(np.float32)\n        output_names = np.r_[names, rank_names]\n    assert x.shape == (len(ranks), FEATURE_COUNTS[variant]) and np.isfinite(x).all()\n    assert len(set(output_names)) == x.shape[1]\n    return x, output_names, array_digest(ranks)\n\n\ndef gate_scores(payload, manifest):\n    if payload["recording_ids"] != manifest.recording_id.tolist() or payload["folds"] != manifest.fold.tolist():\n        raise ValueError("Frozen gate IDs and folds must match the full evaluation cohort")\n    q = np.asarray(payload["scores"], np.float32)\n    if q.shape != (len(manifest),) or not np.isfinite(q).all() or ((q < 0) | (q > 1)).any():\n        raise ValueError("One finite frozen file-gate score in [0,1] required per recording")\n    return q\n\n\ndef combine_native(grid, raw_scores, length, file_score):\n    if not np.isfinite(file_score) or not 0 <= file_score <= 1:\n        raise ValueError("Finite frozen file-gate score required")\n    raw_scores = np.asarray(raw_scores)\n    if raw_scores.shape != (len(grid), 2) or not np.isfinite(raw_scores).all() or ((raw_scores < 0) | (raw_scores > 1)).any():\n        raise ValueError("Finite raw SH/Walking grid scores in [0,1] required")\n    p = shared.expand_native(grid, raw_scores, length)\n    factor = (GATE_FLOOR + (1 - GATE_FLOOR) * np.asarray([file_score], np.float32)).astype(np.float32)[0]\n    p[:, 1] = np.asarray(p[:, 1] * factor, np.float32)\n    return p\n\n\ndef synthetic_inputs():\n    arrays, manifest, labels = parent.synthetic_inputs()\n    arrays["feature_names"] = np.asarray([f"local_{i}" for i in range(213)] + [f"file_{i}" for i in range(112)])\n    q = np.asarray([.2, .4, .6, .8, .1], np.float32)\n    gates = {"recording_ids": manifest.recording_id.tolist(), "folds": manifest.fold.tolist(), "scores": q.tolist()}\n    return arrays, manifest, labels, gates\n\n\ndef run_experiment(variant, cfg=CFG, output_root=None, synthetic=None):\n    check_config(cfg)\n    if variant not in VARIANTS:\n        raise ValueError("Unknown rank-feature variant")\n    smoke = synthetic is not None\n    if not smoke and not torch.cuda.is_available():\n        raise RuntimeError("Real-data rank-feature fitting requires Kaggle CUDA; no CPU fallback")\n    output_root = Path(output_root or ROOT / "results/tdcs_record_rank_gpu" / variant)\n    output_root.mkdir(parents=True, exist_ok=True)\n    if smoke:\n        arrays, manifest, native_labels, gates = synthetic\n        parent_run, parent_config = Path("synthetic"), {}\n    else:\n        parent_run, parent_config, arrays, manifest = gate_helper.load_parent()\n        native_labels, gates = None, gate_helper.gate_payload()\n    q = gate_scores(gates, manifest)\n    started = time.monotonic()\n    print(f"{variant}: calculating per-record label-free percentiles", flush=True)\n    x, names, rank_hash = transform_features(arrays, variant)\n    print(f"{variant}: {x.shape[0]:,} rows x {x.shape[1]} features; {x.nbytes / 2**20:.1f} MiB", flush=True)\n    population = ("synthetic wiring only; not TDCS performance" if smoke else\n        manifest[["recording_id", "subject", "fold", "source_bytes"]].to_csv(index=False, lineterminator="\\n"))\n    config = {"configuration": asdict(cfg), "variant": variant, "targets": TARGETS,\n        "learner": "catboost", "parameters": PARAMETERS, "sources": source_hashes(), "smoke_only": smoke,\n        "feature_count": len(names), "feature_names": names.tolist(), "feature_array_sha256": array_digest(x),\n        "rank_feature_sha256": rank_hash, "local_columns_ranked": LOCAL_COLUMNS,\n        "rank_formula": "(within-record average rank - 0.5) / number of grid rows in that recording; ties averaged",\n        "rank_scope": "Only the current recording\'s sensor-derived features, including its full offline time extent; no labels, other records or subjects",\n        "parent_fingerprint": parent_run.name,\n        "parent_feature_array_sha256": parent_config.get("feature_array_sha256"),\n        "parent_cache_sha256": parent_config.get("feature_cache_sha256"),\n        "population_sha256": hashlib.sha256(population.encode()).hexdigest(),\n        "gate_payload_sha256": gate_helper.payload_digest(gates), "walking_gate_floor": GATE_FLOOR,\n        "Walking_raw_scores_gated_exactly_once_after_native_interpolation": True,\n        "StartHesitation_gate_applied": False, "gate_used_as_fitting_feature": False,\n        "runtime": {name: version(name) for name in ("catboost", "numpy", "scipy", "torch")},\n        "used_gpu_indices": [] if smoke else [0], "subject_weight_exponent": cfg.subject_weight_exponent,\n        "training_scope": "All training recordings including negative files; no conditional file exclusion",\n        "negative_sampling": "All positive grid rows and a seeded quarter of negatives, inverse sampling correction",\n        "prediction_backend": "CPU scoring of saved GPU models; native in-record interpolation, then frozen Walking-only gate",\n        "early_stopping": False, "outer_labels_used_for_fitting_or_stopping": False,\n        "base_OOF_predictions_used_as_fitting_features": False, "extra_class_weights": False,\n        "IDs_or_subjects_used_as_features": False, "metadata_used_as_features": False,\n        "event_boundaries_used_as_features": False, "task_intervals_used_as_features": False,\n        "retained_heads_overwritten": False, "competition_submission_created": False,\n        "confidence_calibrated": False, "independent_generalization_established": False}\n    fingerprint = hashlib.sha256(json.dumps(config, sort_keys=True).encode()).hexdigest()[:20]\n    run = output_root / fingerprint\n    run.mkdir(parents=True, exist_ok=True)\n    shared.common.write_json(run / "configuration.json", config)\n    shared.common.write_json(run / "frozen_gate_payload.json", gates)\n    manifest.to_csv(run / "frozen_tdcs_manifest.csv", index=False)\n    (run / "oof").mkdir(exist_ok=True)\n    scores = np.full(arrays["y"].shape, np.nan, np.float32)\n    deadline = started + cfg.max_seconds\n    for fold in cfg.folds:\n        heldout = arrays["fold"] == fold\n        training_subjects = sorted(set(arrays["subject"][~heldout]))\n        heldout_subjects = sorted(set(arrays["subject"][heldout]))\n        assert training_subjects and heldout_subjects and set(training_subjects).isdisjoint(heldout_subjects)\n        directory = run / f"fold_{fold}"\n        directory.mkdir(exist_ok=True)\n        for index, target in enumerate(TARGETS):\n            if time.monotonic() >= deadline:\n                raise TimeoutError("Rank-feature experiment safety budget reached")\n            seed = cfg.seed + 1009 * fold + index\n            chosen, weights, probability = parent.training_plan(arrays["y"][:, index], arrays["fold"],\n                arrays["subject"], fold, seed, cfg.negative_divisor, cfg.subject_weight_exponent)\n            sample_path, model_path = directory / (target + "_training_sample.npz"), directory / (target + ".cbm")\n            receipt_path = directory / (target + "_training_receipt.json")\n            existing = [p.is_file() for p in (sample_path, model_path, receipt_path)]\n            if any(existing):\n                if not all(existing):\n                    raise ValueError("Incomplete model/sample/receipt; preserve artifacts and choose a fresh output root")\n                receipt = json.loads(receipt_path.read_text())\n                assert receipt["fingerprint"] == fingerprint and receipt["variant"] == variant and receipt["target"] == target\n                assert receipt["seed"] == seed and receipt["parameters"] == PARAMETERS\n                assert receipt["smoke_only"] == smoke and receipt["sample_sha256"] == digest(sample_path)\n                assert receipt["feature_count"] == len(names)\n                assert receipt["trees"] == (cfg.smoke_rounds if smoke else PARAMETERS["iterations"])\n                assert receipt["actual_training_backend"] == ("CPU" if smoke else "GPU")\n                with np.load(sample_path, allow_pickle=False) as saved:\n                    np.testing.assert_array_equal(saved["indices"], chosen)\n                    np.testing.assert_array_equal(saved["weights"], weights)\n                model = shared.load_model("catboost", model_path)\n            else:\n                np.savez_compressed(sample_path, indices=chosen, weights=weights)\n                print(f"{variant} fold {fold} {target}: {len(chosen):,} sampled rows", flush=True)\n                fit_started = time.monotonic()\n                model, backend = parent.parent.fit_model("regularized_depth6", x[chosen], arrays["y"][chosen, index],\n                    weights, seed, gpu=not smoke, rounds=cfg.smoke_rounds if smoke else None)\n                shared.save_model(model, "catboost", model_path)\n                shared.common.write_json(receipt_path, {"fingerprint": fingerprint, "variant": variant,\n                    "target": target, "seed": seed, "parameters": PARAMETERS, "feature_count": len(names),\n                    "feature_array_sha256": config["feature_array_sha256"], "subject_weight_exponent": .5,\n                    "training_subjects": training_subjects, "held_out_subjects": heldout_subjects,\n                    "training_grid_rows_before_sampling": int((~heldout).sum()), "sampled_grid_rows": len(chosen),\n                    "negative_sampling_probability": probability, "all_positive_training_rows_kept": True,\n                    "all_training_recordings_eligible": True, "inverse_sampling_correction": True,\n                    "subject_counts_from_full_training_rows_only": True,\n                    "outer_labels_used_for_fitting_or_stopping": False, "gate_used_as_fitting_feature": False,\n                    "early_stopping": False, "extra_class_weights": False,\n                    "sample_sha256": digest(sample_path), "smoke_only": smoke,\n                    "seconds": time.monotonic() - fit_started, **backend})\n            scores[heldout, index] = shared.predict_model(model, "catboost", x[heldout])\n            del model\n    assert np.isfinite(scores).all()\n    np.savez_compressed(run / "grid_scores.npz", prediction=scores, targets=np.asarray(TARGETS))\n    ys, ps, raw_ps = [], [], []\n    for i, row in enumerate(manifest.itertuples(index=False)):\n        selected = arrays["record"] == i\n        raw = shared.expand_native(arrays["grid"][selected], scores[selected], int(arrays["lengths"][i]))\n        p = combine_native(arrays["grid"][selected], scores[selected], int(arrays["lengths"][i]), q[i])\n        y = (native_labels[i] if smoke else\n            pd.read_csv(row.path, usecols=list(TARGETS)).loc[:, list(TARGETS)].to_numpy(np.uint8))\n        assert p.shape == y.shape and np.isin(y, [0, 1]).all()\n        np.savez_compressed(run / "oof" / (row.recording_id + ".npz"), prediction=p, targets=np.asarray(TARGETS))\n        ys.append(y)\n        ps.append(p)\n        raw_ps.append(raw)\n    y, p, raw = np.concatenate(ys), np.concatenate(ps), np.concatenate(raw_ps)\n    values = shared.common.average_precision(y, p)\n    raw_values = shared.common.average_precision(y, raw)\n    pd.DataFrame([values]).to_csv(run / "native_oof_metrics.csv", index=False)\n    pd.DataFrame([raw_values]).to_csv(run / "native_raw_oof_metrics.csv", index=False)\n    result = {"status": "complete", "run_dir": str(run), "variant": variant, "targets": TARGETS,\n        **values, "raw_ungated_scores_diagnostic_only": raw_values, "native_rows": len(y), "smoke_only": smoke,\n        "retained_heads_unchanged": True, "confidence_calibrated": False,\n        "independent_generalization_established": False, "competition_submission_created": False}\n    shared.common.write_json(run / "decision.json", result)\n    shared.common.write_json(output_root / "latest_run.json", result)\n    print(json.dumps(result, indent=2), flush=True)\n    return result\n\n\ndef smoke_experiment(variant, output_root):\n    return run_experiment(variant, replace(CFG, smoke_rounds=4), output_root, synthetic_inputs())\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--variant", choices=VARIANTS, required=True)\n    parser.add_argument("--output-root", type=Path, required=True)\n    parser.add_argument("--smoke", action="store_true")\n    args = parser.parse_args()\n    (smoke_experiment if args.smoke else run_experiment)(args.variant, output_root=args.output_root)\n', 'tdcs_metadata_context_gpu': '"""Matched sensor-only versus inference-available TDCS metadata GPU trials.\n\nEvent annotations and task intervals are not features. The only added fields\nare Visit, Test and Medication; IDs/subjects are join/split keys only.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nimport time\nfrom dataclasses import asdict, dataclass, replace\nfrom importlib.metadata import version\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\n\nimport tdcs_gpu_regularized_context as parent\n\nshared = parent.shared\nROOT = Path(__file__).resolve().parents[1]\nTARGETS = parent.TARGETS\nVARIANTS = (\'sensor_only\', \'sensor_metadata\')\nPARAMETERS = parent.PARAMETERS[\'regularized_depth6\']\nFIELDS = (\'Visit\', \'Test\', \'Medication\')\nMETADATA_NAMES = (\'metadata_Visit\', \'metadata_Visit_missing\', \'metadata_Test_1\',\n    \'metadata_Test_2\', \'metadata_Test_3\', \'metadata_Test_unknown\',\n    \'metadata_Medication_off\', \'metadata_Medication_on\', \'metadata_Medication_unknown\')\nLOCAL_METADATA = ROOT / \'data/metadata/kaggle_20261107/tdcsfog_metadata.csv\'\n\n\n@dataclass(frozen=True)\nclass Config:\n    name: str = \'tdcs_metadata_context_gpu_v11\'\n    targets: tuple = TARGETS\n    folds: tuple = (0, 1, 2, 3, 4)\n    negative_divisor: int = 4\n    seed: int = 5107\n    smoke_rounds: int = 4\n    max_seconds: int = 21600\n\n\nCFG = Config()\n\n\ndef digest(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef source_hashes():\n    return {\'source_sha256\': globals().get(\'SOURCE_SHA256\') or digest(__file__),\n        \'parent_sources\': parent.source_hashes()}\n\n\ndef check_config(cfg):\n    if tuple(cfg.targets) != TARGETS or tuple(cfg.folds) != tuple(range(5)):\n        raise ValueError(\'Both difficult targets and all five frozen folds required\')\n    if min(cfg.negative_divisor, cfg.smoke_rounds, cfg.max_seconds) < 1:\n        raise ValueError(\'Positive sampling and budget required\')\n\n\ndef locate_metadata():\n    if Path(\'/kaggle/input\').exists():\n        return shared.common.locate_data()[1]\n    if not LOCAL_METADATA.is_file():\n        raise FileNotFoundError(\'Download the TDCS metadata into data/metadata/kaggle_20261107\')\n    return LOCAL_METADATA\n\n\ndef metadata_matrix(recording_ids, frame, require_all=False):\n    """Fixed, label-free encoding; unknown IDs/conditions never need labels."""\n    required = {\'Id\', *FIELDS}\n    if not required <= set(frame):\n        raise ValueError(\'Need Id, Visit, Test and Medication columns\')\n    data = frame.loc[:, [\'Id\', *FIELDS]].copy()\n    data[\'Id\'] = data.Id.astype(str)\n    if data.Id.duplicated().any():\n        raise ValueError(\'Duplicate metadata recording IDs\')\n    ids = np.asarray(recording_ids, str)\n    if ids.ndim != 1 or len(np.unique(ids)) != len(ids):\n        raise ValueError(\'Unique recording IDs required\')\n    if require_all and not set(ids) <= set(data.Id):\n        raise ValueError(\'Training/evaluation recording metadata is missing\')\n    ordered = data.set_index(\'Id\').reindex(ids)\n    visit = pd.to_numeric(ordered.Visit, errors=\'coerce\').to_numpy(np.float64)\n    valid_visit = np.isfinite(visit) & (visit >= 0)\n    test = pd.to_numeric(ordered.Test, errors=\'coerce\').to_numpy(np.float64)\n    known_test = np.isin(test, [1., 2., 3.])\n    medication = ordered.Medication.astype(\'string\').str.strip().str.lower().fillna(\'unknown\').to_numpy(str)\n    known_medication = np.isin(medication, [\'off\', \'on\'])\n    x = np.column_stack([np.where(valid_visit, visit, -1.), ~valid_visit,\n        test == 1, test == 2, test == 3, ~known_test,\n        medication == \'off\', medication == \'on\', ~known_medication]).astype(np.float32)\n    assert x.shape == (len(ids), len(METADATA_NAMES)) and np.isfinite(x).all()\n    return x\n\n\ndef metadata_digest(x, recording_ids):\n    return hashlib.sha256(np.ascontiguousarray(x).tobytes() +\n        json.dumps({\'ids\': list(map(str, recording_ids)), \'names\': METADATA_NAMES}, sort_keys=True).encode()).hexdigest()\n\n\ndef model_features(arrays, metadata, variant):\n    if variant not in VARIANTS:\n        raise ValueError(\'Unknown metadata comparison\')\n    x = arrays[\'x\']\n    if variant == \'sensor_metadata\':\n        x = np.column_stack([x, metadata[arrays[\'record\']]]).astype(np.float32)\n    assert x.shape == (len(arrays[\'y\']), 334 if variant == \'sensor_metadata\' else 325)\n    assert np.isfinite(x).all()\n    return x\n\n\ndef synthetic_inputs():\n    rng = np.random.default_rng(1)\n    folds = np.repeat(np.arange(5), 64).astype(np.int8)\n    record = folds.astype(np.int16)\n    y = np.zeros((320, 2), np.uint8)\n    y[:, 0] = (np.arange(320) % 13 == 0).astype(np.uint8)\n    y[:, 1] = (np.arange(320) % 11 == 1).astype(np.uint8)\n    y[folds == 4] = 0\n    manifest = pd.DataFrame({\'recording_id\': [f\'smoke_{i}\' for i in range(5)],\n        \'subject\': [f\'subject_{i}\' for i in range(5)], \'fold\': range(5)})\n    arrays = {\'x\': rng.normal(size=(320, 325)).astype(np.float32), \'y\': y, \'fold\': folds,\n        \'subject\': manifest.subject.to_numpy(str)[record], \'record\': record,\n        \'grid\': np.tile(np.arange(64, dtype=np.int32), 5), \'lengths\': np.full(5, 64),\n        \'recording_ids\': manifest.recording_id.to_numpy(str)}\n    metadata = pd.DataFrame({\'Id\': manifest.recording_id, \'Visit\': [2, 3, 4, 5, 13],\n        \'Test\': [1, 2, 3, 1, 2], \'Medication\': [\'on\', \'off\', \'on\', \'off\', \'on\']})\n    return arrays, manifest, [y[folds == i].copy() for i in range(5)], metadata\n\n\ndef run_experiment(variant, cfg=CFG, output_root=None, synthetic=None):\n    check_config(cfg)\n    if variant not in VARIANTS:\n        raise ValueError(\'Unknown metadata comparison\')\n    smoke = synthetic is not None\n    if not smoke and not torch.cuda.is_available():\n        raise RuntimeError(\'Real-data metadata fitting requires Kaggle CUDA; no CPU fallback\')\n    output_root = Path(output_root or ROOT / \'results/tdcs_metadata_context_gpu\' / variant)\n    output_root.mkdir(parents=True, exist_ok=True)\n    if smoke:\n        arrays, manifest, native_labels, metadata_frame = synthetic\n        parent_run, parent_config = Path(\'synthetic\'), {}\n        metadata_bytes = metadata_frame.to_csv(index=False, lineterminator=\'\\n\').encode()\n    else:\n        parent_run, parent_config, arrays, manifest = parent.parent_loader.load_parent()\n        path = locate_metadata()\n        metadata_bytes = path.read_bytes()\n        metadata_frame = pd.read_csv(path, dtype={\'Id\': str})\n        native_labels = None\n        if \'Subject\' not in metadata_frame:\n            raise ValueError(\'Need subject metadata to audit split alignment\')\n        subjects = metadata_frame.set_index(\'Id\').Subject.reindex(manifest.recording_id).to_numpy(str)\n        np.testing.assert_array_equal(subjects, manifest.subject.to_numpy(str))\n    metadata = metadata_matrix(manifest.recording_id.to_numpy(str), metadata_frame, require_all=True)\n    x = model_features(arrays, metadata, variant)\n    population = (\'synthetic wiring only; not TDCS performance\' if smoke else\n        manifest[[\'recording_id\', \'subject\', \'fold\', \'source_bytes\']].to_csv(index=False, lineterminator=\'\\n\'))\n    config = {\'configuration\': asdict(cfg), \'variant\': variant, \'targets\': TARGETS,\n        \'parameters\': PARAMETERS, \'sources\': source_hashes(), \'smoke_only\': smoke,\n        \'feature_count\': x.shape[1], \'metadata_used_as_features\': variant == \'sensor_metadata\',\n        \'metadata_fields\': FIELDS, \'metadata_feature_names\': METADATA_NAMES,\n        \'metadata_csv_sha256\': hashlib.sha256(metadata_bytes).hexdigest(),\n        \'metadata_array_sha256\': metadata_digest(metadata, arrays[\'recording_ids\']),\n        \'metadata_snapshot_file\': \'metadata_snapshot.csv\', \'metadata_transform_fitted\': False,\n        \'parent_fingerprint\': parent_run.name, \'parent_feature_array_sha256\': parent_config.get(\'feature_array_sha256\'),\n        \'parent_cache_sha256\': parent_config.get(\'feature_cache_sha256\'),\n        \'population_sha256\': hashlib.sha256(population.encode()).hexdigest(),\n        \'runtime\': {name: version(name) for name in (\'catboost\', \'numpy\', \'torch\')},\n        \'used_gpu_indices\': [] if smoke else [0],\n        \'prediction_backend\': \'CPU saved-model scoring followed by within-record native interpolation\',\n        \'sampling\': \'All training files eligible; all positives plus one-quarter negatives with inverse correction\',\n        \'subject_weights\': \'Inverse full-fitting-row counts per training subject, normalized\',\n        \'file_gate_applied\': False, \'early_stopping\': False,\n        \'event_boundaries_used_as_features\': False, \'task_intervals_used_as_features\': False,\n        \'IDs_or_subjects_used_as_features\': False, \'base_OOF_predictions_used_as_features\': False,\n        \'outer_labels_used_for_fitting_or_stopping\': False,\n        \'retained_heads_overwritten\': False, \'competition_submission_created\': False,\n        \'confidence_calibrated\': False, \'independent_generalization_established\': False}\n    fingerprint = hashlib.sha256(json.dumps(config, sort_keys=True).encode()).hexdigest()[:20]\n    run = output_root / fingerprint\n    run.mkdir(parents=True, exist_ok=True)\n    shared.common.write_json(run / \'configuration.json\', config)\n    (run / \'metadata_snapshot.csv\').write_bytes(metadata_bytes)\n    np.savez_compressed(run / \'metadata_features.npz\', x=metadata,\n        recording_ids=arrays[\'recording_ids\'], feature_names=np.asarray(METADATA_NAMES))\n    manifest.to_csv(run / \'frozen_tdcs_manifest.csv\', index=False)\n    (run / \'oof\').mkdir(exist_ok=True)\n    scores = np.full(arrays[\'y\'].shape, np.nan, np.float32)\n    deadline = time.monotonic() + cfg.max_seconds\n    for fold in cfg.folds:\n        heldout = arrays[\'fold\'] == fold\n        training_subjects = sorted(set(arrays[\'subject\'][~heldout]))\n        heldout_subjects = sorted(set(arrays[\'subject\'][heldout]))\n        assert set(training_subjects).isdisjoint(heldout_subjects)\n        directory = run / f\'fold_{fold}\'\n        directory.mkdir(exist_ok=True)\n        for index, target in enumerate(TARGETS):\n            if time.monotonic() >= deadline:\n                raise TimeoutError(\'Metadata experiment safety budget reached\')\n            seed = cfg.seed + 1009 * fold + index\n            chosen, weights, probability = shared.training_plan(arrays[\'y\'][:, index], arrays[\'fold\'],\n                arrays[\'subject\'], fold, seed, cfg.negative_divisor)\n            sample_path = directory / (target + \'_training_sample.npz\')\n            model_path, receipt_path = directory / (target + \'.cbm\'), directory / (target + \'_training_receipt.json\')\n            existing = [p.is_file() for p in (sample_path, model_path, receipt_path)]\n            if any(existing):\n                if not all(existing):\n                    raise ValueError(\'Incomplete model/sample/receipt; preserve artifacts and choose a fresh root\')\n                receipt = json.loads(receipt_path.read_text())\n                assert receipt[\'fingerprint\'] == fingerprint and receipt[\'target\'] == target\n                assert receipt[\'variant\'] == variant and receipt[\'seed\'] == seed\n                assert receipt[\'parameters\'] == PARAMETERS and receipt[\'smoke_only\'] == smoke\n                assert receipt[\'trees\'] == (cfg.smoke_rounds if smoke else PARAMETERS[\'iterations\'])\n                assert receipt[\'actual_training_backend\'] == (\'CPU\' if smoke else \'GPU\')\n                assert receipt[\'sample_sha256\'] == digest(sample_path)\n                with np.load(sample_path, allow_pickle=False) as saved:\n                    np.testing.assert_array_equal(saved[\'indices\'], chosen)\n                    np.testing.assert_array_equal(saved[\'weights\'], weights)\n                model = shared.load_model(\'catboost\', model_path)\n            else:\n                np.savez_compressed(sample_path, indices=chosen, weights=weights)\n                print(f\'{variant} fold {fold} {target}: {len(chosen):,} rows, {x.shape[1]} features\', flush=True)\n                model, backend = parent.fit_model(\'regularized_depth6\', x[chosen], arrays[\'y\'][chosen, index],\n                    weights, seed, gpu=not smoke, rounds=cfg.smoke_rounds if smoke else None)\n                shared.save_model(model, \'catboost\', model_path)\n                shared.common.write_json(receipt_path, {\'fingerprint\': fingerprint, \'variant\': variant,\n                    \'target\': target, \'parameters\': PARAMETERS, \'seed\': seed, \'feature_count\': x.shape[1],\n                    \'training_subjects\': training_subjects, \'held_out_subjects\': heldout_subjects,\n                    \'training_grid_rows_before_sampling\': int((~heldout).sum()), \'sampled_grid_rows\': len(chosen),\n                    \'negative_sampling_probability\': probability, \'all_positive_training_rows_kept\': True,\n                    \'inverse_sampling_correction\': True, \'subject_weights_from_training_rows_only\': True,\n                    \'outer_labels_used_for_fitting_or_stopping\': False, \'base_OOF_predictions_used_as_features\': False,\n                    \'event_boundaries_used_as_features\': False, \'task_intervals_used_as_features\': False,\n                    \'IDs_or_subjects_used_as_features\': False, \'early_stopping\': False,\n                    \'sample_sha256\': digest(sample_path), \'smoke_only\': smoke, **backend})\n            scores[heldout, index] = shared.predict_model(model, \'catboost\', x[heldout])\n            del model\n    assert np.isfinite(scores).all()\n    np.savez_compressed(run / \'grid_scores.npz\', prediction=scores, targets=np.asarray(TARGETS))\n    ys, ps = [], []\n    for i, row in enumerate(manifest.itertuples(index=False)):\n        selected = arrays[\'record\'] == i\n        p = shared.expand_native(arrays[\'grid\'][selected], scores[selected], int(arrays[\'lengths\'][i]))\n        y = (native_labels[i] if smoke else\n            pd.read_csv(row.path, usecols=list(TARGETS)).loc[:, list(TARGETS)].to_numpy(np.uint8))\n        assert p.shape == y.shape and np.isin(y, [0, 1]).all()\n        np.savez_compressed(run / \'oof\' / (row.recording_id + \'.npz\'), prediction=p, targets=np.asarray(TARGETS))\n        ys.append(y); ps.append(p)\n    y, p = np.concatenate(ys), np.concatenate(ps)\n    values = shared.common.average_precision(y, p)\n    pd.DataFrame([values]).to_csv(run / \'native_oof_metrics.csv\', index=False)\n    result = {\'status\': \'complete\', \'run_dir\': str(run), \'variant\': variant, \'targets\': TARGETS,\n        **values, \'native_rows\': len(y), \'smoke_only\': smoke, \'retained_heads_unchanged\': True,\n        \'confidence_calibrated\': False, \'independent_generalization_established\': False,\n        \'competition_submission_created\': False}\n    shared.common.write_json(run / \'decision.json\', result)\n    shared.common.write_json(output_root / \'latest_run.json\', result)\n    print(json.dumps(result, indent=2), flush=True)\n    return result\n\n\ndef smoke_experiment(variant, output_root):\n    return run_experiment(variant, replace(CFG, smoke_rounds=4), output_root, synthetic_inputs())\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--variant\', choices=VARIANTS, required=True)\n    parser.add_argument(\'--output-root\', type=Path, required=True)\n    parser.add_argument(\'--smoke\', action=\'store_true\')\n    args = parser.parse_args()\n    (smoke_experiment if args.smoke else run_experiment)(args.variant, output_root=args.output_root)\n', 'tdcs_suite_data': '"""Data contracts for the two comprehensive TDCS suites; never fit models."""\nfrom __future__ import annotations\n\nimport hashlib\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom scipy.signal import resample_poly\n\nimport tdcs_record_rank_gpu as frozen\nimport tdcs_metadata_context_gpu as recording_metadata\n\nROOT = Path(__file__).resolve().parents[1]\nTARGETS = ("StartHesitation", "Walking")\nAXES = ("AccV", "AccML", "AccAP")\nCLINICAL_NUMERIC = ("Age", "YearsSinceDx", "UPDRSIII_On", "UPDRSIII_Off", "NFOGQ")\nCLINICAL_NAMES = tuple(name for field in CLINICAL_NUMERIC for name in ("clinical_" + field, "clinical_" + field + "_missing")) + (\n    "clinical_Sex_female", "clinical_Sex_male", "clinical_Sex_unknown")\n\n\ndef digest(path):\n    result = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for block in iter(lambda: stream.read(1024 * 1024), b\'\'):\n            result.update(block)\n    return result.hexdigest()\n\n\ndef array_digest(value):\n    return frozen.array_digest(value)\n\n\ndef csv_path(name):\n    if Path("/kaggle/input").exists():\n        _, metadata = frozen.shared.common.locate_data()\n        path = Path(metadata).parent / name\n    else:\n        fresh = ROOT / "data/metadata/kaggle_20261119" / name\n        path = fresh if fresh.is_file() else ROOT / "data/metadata" / name\n    if not path.is_file():\n        raise FileNotFoundError(f"Required competition input missing: {name}")\n    return path\n\n\ndef clinical_features(ids, recording_frame, subjects):\n    """Exact visit first; unique subject-level (blank Visit) fallback, never nearest visit."""\n    required = {"Subject", "Visit", "Sex", *CLINICAL_NUMERIC}\n    if not required <= set(subjects) or not {"Id", "Subject", "Visit"} <= set(recording_frame):\n        raise ValueError("Unexpected clinical/recording metadata schema")\n    table = subjects.copy()\n    table["Subject"] = table.Subject.astype(str).str.strip()\n    table["Visit"] = pd.to_numeric(table.Visit, errors="coerce")\n    specific, general = {}, {}\n    for row in table.to_dict("records"):\n        key = row["Subject"] if pd.isna(row["Visit"]) else (row["Subject"], float(row["Visit"]))\n        lookup = general if pd.isna(row["Visit"]) else specific\n        if key in lookup:\n            raise ValueError("Clinical key is not unique; refusing a many-to-many join")\n        lookup[key] = row\n    records = recording_frame.copy()\n    if records.Id.astype(str).duplicated().any():\n        raise ValueError("Duplicate recording metadata keys")\n    records["Id"] = records.Id.astype(str)\n    ordered = records.set_index("Id").reindex(list(map(str, ids)))\n    chosen, kinds = [], []\n    for row in ordered.to_dict("records"):\n        subject = str(row.get("Subject", "")).strip()\n        visit = pd.to_numeric(row.get("Visit"), errors="coerce")\n        value = specific.get((subject, float(visit))) if pd.notna(visit) else None\n        kind = "subject_visit"\n        if value is None:\n            value, kind = general.get(subject), "subject_level"\n        chosen.append(value or {})\n        kinds.append(kind if value is not None else "missing")\n    selected = pd.DataFrame(chosen).reindex(columns=["Sex", *CLINICAL_NUMERIC])\n    columns = []\n    for field in CLINICAL_NUMERIC:\n        values = pd.to_numeric(selected[field], errors="coerce").to_numpy(float)\n        missing = ~np.isfinite(values)\n        columns.extend((np.where(missing, 0., values), missing.astype(float)))\n    sex = selected.Sex.astype("string").str.strip().str.lower().fillna("")\n    female, male = sex.isin(["f", "female"]).to_numpy(), sex.isin(["m", "male"]).to_numpy()\n    columns.extend((female, male, ~(female | male)))\n    x = np.column_stack(columns).astype(np.float32)\n    assert x.shape == (len(ids), 13) and np.isfinite(x).all()\n    return x, {"rows": len(ids), "subject_visit_matches": kinds.count("subject_visit"),\n        "subject_level_matches": kinds.count("subject_level"), "missing": kinds.count("missing"),\n        "rule": "Exact subject/visit or unique blank-Visit subject-level row; no arbitrary visit fallback"}\n\n\ndef kinetic_labels(times, labels, events):\n    """Times/boundaries are seconds. Auxiliary FOG target only, never an input."""\n    times, labels = np.asarray(times, float), np.asarray(labels)\n    values, mask = np.zeros(len(times), np.float32), np.zeros(len(times), np.float32)\n    conflict = np.zeros(len(times), bool)\n    for row in events.itertuples(index=False):\n        value = pd.to_numeric(getattr(row, "Kinetic"), errors="coerce")\n        begin = pd.to_numeric(getattr(row, "Init"), errors="coerce")\n        end = pd.to_numeric(getattr(row, "Completion"), errors="coerce")\n        if value not in (0., 1.) or not np.isfinite(begin) or not np.isfinite(end) or end <= begin:\n            continue\n        event_type = getattr(row, \'Type\', None)\n        column = {\'StartHesitation\': 0, \'Turn\': 1, \'Walking\': 2}.get(event_type)\n        fog = labels[:, column] > 0 if column is not None else labels.any(1)\n        selected = (times >= begin) & (times < end) & fog\n        conflict |= selected & (mask > 0) & (values != value)\n        values[selected], mask[selected] = value, 1.\n    mask[conflict] = 0.\n    return np.column_stack([values, mask]).astype(np.float32)\n\n\ndef synthetic_context():\n    arrays, manifest, native, gate = frozen.synthetic_inputs()\n    rng = np.random.default_rng(1607)\n    ids = manifest.recording_id.to_numpy(str)\n    frame = pd.DataFrame({"Id": ids, "Subject": manifest.subject, "Visit": np.arange(5) + 1,\n        "Test": [1, 2, 3, 1, 2], "Medication": ["on", "off", "on", "off", "on"]})\n    clinical = pd.DataFrame({"Subject": manifest.subject, "Visit": [np.nan] * 5,\n        "Age": np.arange(5) + 60, "Sex": ["M", "F", "M", "F", "M"],\n        "YearsSinceDx": 5., "UPDRSIII_On": 20., "UPDRSIII_Off": 30., "NFOGQ": 10.})\n    m = recording_metadata.metadata_matrix(ids, frame, require_all=True)\n    c, audit = clinical_features(ids, frame, clinical)\n    k = np.column_stack([np.arange(len(arrays["y"])) % 2, arrays["y"].any(1)]).astype(np.float32)\n    u = {"x": rng.normal(size=(256, 325)).astype(np.float32),\n        "subject": np.repeat(np.r_[manifest.subject.to_numpy(str), ["external"]], [40, 40, 40, 40, 40, 56]),\n        "record": np.repeat(np.arange(6), [40, 40, 40, 40, 40, 56]).astype(np.int32)}\n    return {"arrays": arrays, "manifest": manifest, "native_y": np.concatenate(native),\n        "record_metadata": m, "clinical": c, "clinical_audit": audit, "frame": frame,\n        "kinetic": k, "gates": frozen.gate_scores(gate, manifest), "gate_payload": gate,\n        "native_offsets": np.r_[0, np.cumsum(arrays["lengths"])], "external": u, "smoke_only": True,\n        "source_inputs": {}, "parent": "synthetic"}\n\n\ndef load_context():\n    parent, config, arrays, manifest = frozen.gate_helper.load_parent()\n    ids = manifest.recording_id.to_numpy(str)\n    paths = {name: csv_path(name) for name in ("tdcsfog_metadata.csv", "subjects.csv", "events.csv", "daily_metadata.csv")}\n    frame = pd.read_csv(paths["tdcsfog_metadata.csv"], dtype={"Id": str, "Subject": str})\n    np.testing.assert_array_equal(frame.set_index("Id").Subject.reindex(ids).to_numpy(str), manifest.subject.to_numpy(str))\n    meta = recording_metadata.metadata_matrix(ids, frame, require_all=True)\n    clinical, audit = clinical_features(ids, frame, pd.read_csv(paths["subjects.csv"], dtype={"Subject": str}))\n    if audit["missing"]:\n        raise ValueError("Some TDCS subjects lack clinical metadata; review the explicit join audit")\n    events = pd.read_csv(paths["events.csv"], dtype={"Id": str})\n    groups = dict(tuple(events.groupby("Id", sort=False)))\n    y, kinetic = [], np.zeros((len(arrays["y"]), 2), np.float32)\n    for i, row in enumerate(manifest.itertuples(index=False)):\n        raw = pd.read_csv(row.path, usecols=["Time", "StartHesitation", "Turn", "Walking"])\n        labels = raw[["StartHesitation", "Turn", "Walking"]].to_numpy(np.uint8)\n        assert np.isin(labels, [0, 1]).all() and (labels.sum(1) <= 1).all()\n        chosen = arrays["record"] == i\n        grid = arrays["grid"][chosen]\n        np.testing.assert_array_equal(labels[grid][:, [0, 2]], arrays["y"][chosen])\n        y.append(labels[:, [0, 2]])\n        if row.recording_id in groups:\n            np.testing.assert_array_equal(raw.Time.to_numpy(), np.arange(len(raw)))\n            kinetic[chosen] = kinetic_labels(raw.Time.to_numpy()[grid] / 128., labels[grid], groups[row.recording_id])\n        if (i + 1) % 200 == 0:\n            print(f"Audited native labels and auxiliary annotations: {i + 1}/815", flush=True)\n    gate = frozen.gate_helper.gate_payload()\n    return {"arrays": arrays, "manifest": manifest, "native_y": np.concatenate(y),\n        "record_metadata": meta, "clinical": clinical, "clinical_audit": audit, "frame": frame,\n        "kinetic": kinetic, "gates": frozen.gate_scores(gate, manifest), "gate_payload": gate,\n        "native_offsets": np.r_[0, np.cumsum(arrays["lengths"])], "external": None, "smoke_only": False,\n        "source_inputs": {name: {"path": str(path), "sha256": digest(path)} for name, path in paths.items()},\n        "parent": str(parent), "parent_config": config}\n\n\ndef input_features(context, encoding):\n    a = context["arrays"]\n    if encoding == "rank_metadata":\n        x, _, _ = frozen.transform_features(a, "rank_replace")\n        return np.column_stack([x, context["record_metadata"][a["record"]]]).astype(np.float32)\n    if encoding in ("local_metadata", "local_rank_metadata"):\n        x = a["x"][:, :213] if encoding == "local_metadata" else frozen.record_percentiles(a["x"], a["record"])\n        return np.column_stack([x, context["record_metadata"][a["record"]]]).astype(np.float32)\n    fields = {"record_metadata": context["record_metadata"],\n        "clinical_metadata": np.column_stack([context["record_metadata"], context["clinical"]]),\n        "metadata_only": np.column_stack([context["record_metadata"], context["clinical"]]),\n        "visit_only": context["record_metadata"][:, :2], "test_only": context["record_metadata"][:, 2:6],\n        "medication_only": context["record_metadata"][:, 6:9]}\n    if encoding == "sensor":\n        return a["x"]\n    if encoding not in fields:\n        raise ValueError("Unknown fixed feature encoding")\n    values = fields[encoding][a["record"]]\n    return values.astype(np.float32) if encoding == "metadata_only" else np.column_stack([a["x"], values]).astype(np.float32)\n\n\ndef external_pool(context, output):\n    """Bounded label-free competition daily records, with known subject mapping."""\n    if context["external"] is not None:\n        return context["external"]\n    cache = Path(output) / "unlabeled_feature_pool.npz"\n    if cache.is_file():\n        provenance = json.loads((Path(output) / "unlabeled_pool_provenance.json").read_text())\n        if provenance["metadata_sha256"] != digest(csv_path("daily_metadata.csv")):\n            raise ValueError("Cached external metadata changed")\n        if provenance["cache_sha256"] != digest(cache):\n            raise ValueError("Cached external feature pool changed")\n        with np.load(cache, allow_pickle=False) as saved:\n            context["external"] = {key: saved[key] for key in saved.files}\n        return context["external"]\n    daily_path = csv_path("daily_metadata.csv")\n    daily = pd.read_csv(daily_path, dtype={"Id": str, "Subject": str})\n    if daily.Id.duplicated().any() or daily.Subject.isna().any() or daily.Subject.str.strip().eq(\'\').any():\n        raise ValueError("Unlabeled records need unique IDs and known subject mappings")\n    roots = [daily_path.parent / "unlabeled"]\n    paths = {path.stem: path for root in roots if root.exists() for path in root.rglob("*.csv")}\n    eligible = daily[daily.Id.isin(paths)].sort_values("Id").head(64)\n    if eligible.empty:\n        raise FileNotFoundError("No mapped competition unlabeled CSVs are attached; external-data tests cannot run")\n    xs, ss, rr = [], [], []\n    rows = []\n    for i, row in enumerate(eligible.itertuples(index=False)):\n        raw = pd.read_csv(paths[row.Id], nrows=32768, usecols=["Time", *AXES])\n        prefix_sha = array_digest(raw[["Time", *AXES]].to_numpy(np.float64))\n        signal = daily_signal(raw)\n        _, x, _ = frozen.shared.features.recording_features(signal)\n        xs.append(x)\n        ss.append(np.full(len(x), row.Subject))\n        rr.append(np.full(len(x), i, np.int32))\n        rows.append({"Id": row.Id, "Subject": row.Subject, "source_bytes": paths[row.Id].stat().st_size,\n            "used_prefix_sha256": prefix_sha, "source_rows_read": len(raw),\n            "documented_input_rate_hz": 100., "output_rate_hz": 128.,\n            "acceleration_conversion": "g * 9.80665 to m/s^2",\n            "resampled_native_rows": len(signal), "feature_rows": len(x)})\n        if (i + 1) % 8 == 0:\n            print(f"Unlabeled pool: {i + 1}/{len(eligible)} bounded recordings", flush=True)\n    pool = {"x": np.concatenate(xs), "subject": np.concatenate(ss), "record": np.concatenate(rr)}\n    np.savez_compressed(cache, **pool)\n    frozen.shared.common.write_json(Path(output) / "unlabeled_pool_provenance.json", {\n        "metadata_sha256": digest(daily_path), "cache_sha256": digest(cache),\n        "records": rows, "bounded_prefix_sampling": True,\n        "no_unlabeled_targets_read": True, "heldout_subjects_excluded_separately_per_fold": True})\n    context["external"] = pool\n    return pool\n\n\ndef daily_signal(raw):\n    """Official daily 100-Hz sample index/g units -> TDCS 128-Hz m/s^2.\n\n    The Time column is an integer timestep, not seconds. Do not infer 1 Hz from\n    adjacent sample indices. Reject unexpected spacing rather than guess units.\n    """\n    times = raw.Time.to_numpy(np.float64)\n    if len(times) < 1024 or not np.isfinite(times).all() or not np.equal(times, np.floor(times)).all() or not np.equal(np.diff(times), 1).all():\n        raise ValueError(\'Daily Time must be consecutive finite integer sample indices\')\n    signal = raw[list(AXES)].to_numpy(np.float64)\n    if not np.isfinite(signal).all():\n        raise ValueError(\'Nonfinite daily accelerometer values\')\n    return resample_poly(signal * 9.80665, 32, 25, axis=0)[:16384]\n\n\ndef external_training_indices(pool, heldout_subjects):\n    if len(pool["x"]) != len(pool["subject"]) or pd.isna(pool["subject"]).any():\n        raise ValueError("External rows require aligned known subjects")\n    for record in np.unique(pool[\'record\']):\n        if len(np.unique(pool[\'subject\'][pool[\'record\'] == record])) != 1:\n            raise ValueError(\'An external recording must map to exactly one subject\')\n    result = np.flatnonzero(~np.isin(pool["subject"].astype(str), list(map(str, heldout_subjects))))\n    assert set(pool["subject"][result]).isdisjoint(set(heldout_subjects))\n    if not len(result):\n        raise ValueError("No external rows remain after held-out-subject exclusion")\n    return result\n', 'tdcs_suite_models': '"""Bounded tree backends for suite experiments; CPU prediction is explicit."""\nfrom __future__ import annotations\n\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom scipy.special import expit\n\nimport tdcs_suite_data as data\n\n\ndef fit_tree(kind, x, y, weights, seed, cfg, gpu, policy="SymmetricTree", groups=None):\n    if gpu and not torch.cuda.is_available():\n        raise RuntimeError("Requested Kaggle CUDA backend is unavailable")\n    if len(np.unique(y)) < 2:\n        return {"constant": float(np.mean(y))}, {"backend": "constant", "actual_rounds": 0}\n    if kind == "catboost":\n        from catboost import CatBoostClassifier\n        params = {"iterations": cfg.tree_rounds, "depth": 6, "learning_rate": .04,\n            "l2_leaf_reg": 100., "loss_function": "Logloss", "boosting_type": "Plain",\n            "bootstrap_type": "Bernoulli", "subsample": .85, "border_count": 128,\n            "random_strength": 1., "thread_count": 2, "allow_writing_files": False,\n            "grow_policy": policy, "random_seed": seed, "task_type": "GPU" if gpu else "CPU"}\n        if policy != "SymmetricTree":\n            params.update(score_function="L2", min_data_in_leaf=20)\n        if policy == "Lossguide":\n            params["max_leaves"] = 64\n        if gpu:\n            params.update(devices="0", gpu_ram_part=.55)\n        model = CatBoostClassifier(**params).fit(x, y, sample_weight=weights, verbose=False)\n        actual = model.get_all_params()\n        assert actual["task_type"] == ("GPU" if gpu else "CPU")\n        return model, {"backend": actual["task_type"], "actual_rounds": model.tree_count_,\n            "configured": params, "resolved": actual}\n    if kind in ("xgboost", "rank_map"):\n        import xgboost as xgb\n        params = {"objective": "rank:map" if kind == "rank_map" else "binary:logistic",\n            "tree_method": "hist", "device": "cuda:0" if gpu else "cpu", "max_depth": 6,\n            "eta": .035, "min_child_weight": 30 if not cfg.smoke else 1, "lambda": 30,\n            "subsample": .85, "colsample_bytree": .85, "max_bin": 128,\n            "seed": seed, "nthread": 2, "base_score": .5, "verbosity": 0}\n        if kind == "rank_map":\n            params.update(lambdarank_pair_method="mean", lambdarank_num_pair_per_sample=4)\n        matrix = xgb.DMatrix(x, label=y, nthread=2)\n        if kind == "rank_map":\n            if groups is None or int(np.sum(groups)) != len(y) or len(weights) != len(groups):\n                raise ValueError("Ranking requires group counts and one weight per group")\n            matrix.set_group(groups)\n        matrix.set_weight(weights)\n        model = xgb.train(params, matrix, num_boost_round=cfg.tree_rounds, verbose_eval=False)\n        actual = json.loads(model.save_config())\n        if gpu and not actual["learner"]["generic_param"]["device"].startswith("cuda"):\n            raise RuntimeError("XGBoost silently fell back from CUDA")\n        return model, {"backend": actual["learner"]["generic_param"]["device"],\n            "actual_rounds": model.num_boosted_rounds(), "configured": params, "resolved": actual}\n    if kind == "lightgbm":\n        import lightgbm as lgb\n        # Intentionally CPU on Kaggle, not an undisclosed GPU fallback.\n        params = {"objective": "binary", "metric": "binary_logloss", "device_type": "cpu",\n            "learning_rate": .04, "num_leaves": 31, "max_depth": 6,\n            "min_data_in_leaf": 5 if cfg.smoke else 50, "min_data_in_bin": 1,\n            "lambda_l2": 30., "feature_fraction": .85, "bagging_fraction": .85,\n            "bagging_freq": 1, "max_bin": 127, "num_threads": 2, "seed": seed,\n            "deterministic": True, "force_col_wise": True, "verbosity": -1}\n        model = lgb.train(params, lgb.Dataset(x, label=y, weight=weights), num_boost_round=cfg.tree_rounds)\n        return model, {"backend": "CPU_explicit_LightGBM", "actual_rounds": model.current_iteration(),\n            "configured": params, "resolved": model.params}\n    raise ValueError("Unknown declared learner")\n\n\ndef predict_tree(model, kind, x):\n    if isinstance(model, dict) and "constant" in model:\n        return np.full(len(x), model["constant"], np.float32)\n    if kind == "catboost":\n        p = model.predict_proba(x, task_type="CPU", thread_count=2)[:, 1]\n    elif kind in ("xgboost", "rank_map"):\n        import xgboost as xgb\n        model.set_param({"device": "cpu", "nthread": 2})\n        p = model.predict(xgb.DMatrix(x, nthread=2))\n        if kind == "rank_map":\n            p = expit(np.clip(p, -40, 40))\n    elif kind == "lightgbm":\n        p = model.predict(x, num_threads=2)\n    else:\n        raise ValueError("Unknown model kind")\n    p = np.asarray(p, np.float32)\n    if p.shape != (len(x),) or not np.isfinite(p).all():\n        raise ValueError("Invalid saved-model predictions")\n    return np.clip(p, 0, 1)\n\n\ndef save_tree(model, kind, stem):\n    stem = Path(stem)\n    if isinstance(model, dict):\n        path = stem.with_suffix(".json")\n        data.frozen.shared.common.write_json(path, model)\n    elif kind == "catboost":\n        path = stem.with_suffix(".cbm")\n        model.save_model(str(path))\n    elif kind in ("xgboost", "rank_map"):\n        path = stem.with_suffix(".ubj")\n        path.write_bytes(model.save_raw(raw_format="ubj"))\n    else:\n        path = stem.with_suffix(".txt")\n        model.save_model(str(path))\n    return path\n\n\ndef load_tree(kind, path):\n    path = Path(path)\n    if path.suffix == ".json":\n        return json.loads(path.read_text())\n    if kind == "catboost":\n        from catboost import CatBoostClassifier\n        return CatBoostClassifier().load_model(str(path))\n    if kind in ("xgboost", "rank_map"):\n        import xgboost as xgb\n        model = xgb.Booster()\n        model.load_model(bytearray(path.read_bytes()))\n        return model\n    if kind == "lightgbm":\n        import lightgbm as lgb\n        return lgb.Booster(model_str=path.read_text())\n    raise ValueError("Unknown model kind")\n\n\ndef ranking_plan(chosen, row_weights, records):\n    order = np.argsort(records[chosen], kind="stable")\n    selected, weights = chosen[order], row_weights[order]\n    _, start, counts = np.unique(records[selected], return_index=True, return_counts=True)\n    group_weights = np.asarray([weights[s:s+n].mean() for s, n in zip(start, counts)], np.float64)\n    group_weights /= group_weights.mean()\n    return selected, counts.astype(np.uint32), group_weights\n', 'tdcs_suite_neural': '"""Metadata-aware sequence models and fold-safe masked reconstruction pretraining."""\nfrom __future__ import annotations\n\nimport copy\nimport math\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\nimport tdcs_suite_data as data\n\n\nclass Network(nn.Module):\n    def __init__(self, features, architecture, width=64, auxiliary=False):\n        super().__init__()\n        self.features, self.architecture, self.width, self.auxiliary = features, architecture, width, auxiliary\n        self.input = nn.Sequential(nn.Linear(features, width), nn.LayerNorm(width), nn.GELU())\n        if architecture == "tcn":\n            self.context = nn.ModuleList([nn.Conv1d(width, width, 5, padding=2*d, dilation=d) for d in (1, 2, 4, 8)])\n            size = width\n        elif architecture in ("bigru", "bilstm"):\n            cls = nn.GRU if architecture == "bigru" else nn.LSTM\n            self.context = cls(width, width, 2, batch_first=True, bidirectional=True, dropout=.15)\n            size = width * 2\n        elif architecture == "attention":\n            block = nn.TransformerEncoderLayer(width, 4, width * 2, .15, batch_first=True)\n            self.context = nn.TransformerEncoder(block, 2)\n            size = width\n        else:\n            raise ValueError("Unknown sequence architecture")\n        self.output = nn.Linear(size, 3 if auxiliary else 2)\n        self.decoder = nn.Linear(size, features)\n\n    def encode(self, x):\n        h = self.input(x)\n        if self.architecture == "tcn":\n            for layer in self.context:\n                h = (h + F.dropout(F.gelu(layer(h.transpose(1, 2)).transpose(1, 2)), .15, self.training)) / math.sqrt(2)\n        elif self.architecture in ("bigru", "bilstm"):\n            h = self.context(h)[0]\n        else:\n            positions = torch.arange(h.shape[1], device=h.device, dtype=h.dtype)[:, None]\n            frequencies = torch.exp(torch.arange(0, self.width, 2, device=h.device, dtype=h.dtype) * (-math.log(10000.) / self.width))\n            position = torch.zeros(h.shape[1], self.width, device=h.device, dtype=h.dtype)\n            position[:, 0::2], position[:, 1::2] = torch.sin(positions * frequencies), torch.cos(positions * frequencies)\n            h = self.context(h + position)\n        return h\n\n    def forward(self, x):\n        return self.output(self.encode(x))\n\n\ndef preprocessing(x, y, fitting):\n    count, mean, m2 = 0, np.zeros(x.shape[1], np.float64), np.zeros(x.shape[1], np.float64)\n    rows = np.flatnonzero(fitting)\n    for start in range(0, len(rows), 4096):\n        block = x[rows[start:start+4096]].astype(np.float64)\n        n = len(block); delta = block.mean(0) - mean; total = count + n\n        m2 += ((block - block.mean(0)) ** 2).sum(0) + delta ** 2 * count * n / total\n        mean += delta * n / total; count = total\n    scale = np.sqrt(np.maximum(m2 / max(count, 1), 0))\n    scale[scale < 1e-5] = 1\n    positive = y[rows].sum(0)\n    pw = np.clip(np.sqrt((len(rows) - positive) / np.maximum(positive, 1)), 1, 8)\n    return {"mean": mean.astype(np.float32), "scale": scale.astype(np.float32), "positive_weight": pw.astype(np.float32)}\n\n\ndef transform(x, prep):\n    return np.clip((x - prep["mean"]) / prep["scale"], -8, 8).astype(np.float32)\n\n\ndef sampling_groups(arrays, fitting):\n    records = []\n    for record in np.unique(arrays["record"][fitting]):\n        rows = np.flatnonzero((arrays["record"] == record) & fitting)\n        records.append(rows)\n    subjects = np.asarray([str(arrays["subject"][r[0]]) for r in records])\n    totals = {s: sum(len(r) for r, subject in zip(records, subjects) if subject == s) for s in set(subjects)}\n    probability = np.asarray([len(r) / math.sqrt(totals[s]) for r, s in zip(records, subjects)], float)\n    probability /= probability.sum()\n    return records, probability\n\n\ndef batch(records, probability, rng, x, labels, cfg):\n    values, ys, masks, trace = [], [], [], []\n    for _ in range(cfg.batch_size):\n        i = int(rng.choice(len(records), p=probability))\n        rows = records[i]\n        start = int(rng.integers(len(rows))) - cfg.window // 2\n        positions = start + np.arange(cfg.window)\n        valid = (positions >= 0) & (positions < len(rows))\n        chosen = rows[np.clip(positions, 0, len(rows) - 1)]\n        values.append(x[chosen]); ys.append(labels[chosen]); masks.append(valid)\n        trace.append((int(rows[0]), start))\n    return np.stack(values), np.stack(ys), np.stack(masks), trace\n\n\ndef train(context, x, fold, spec, cfg, directory, device, fingerprint, external=None, deadline=float("inf")):\n    a = context["arrays"]\n    fitting = a["fold"] != fold\n    train_subjects, heldout_subjects = set(a["subject"][fitting]), set(a["subject"][~fitting])\n    assert train_subjects.isdisjoint(heldout_subjects)\n    seed = cfg.seed + 1009 * fold\n    torch.manual_seed(seed)\n    if device.type == "cuda":\n        torch.cuda.manual_seed_all(seed)\n    prep = preprocessing(x, a["y"], fitting)\n    np.savez_compressed(directory / "preprocessing.npz", **prep)\n    xx = transform(x, prep)\n    model = Network(x.shape[1], spec["architecture"], cfg.width, spec.get("kinetic", False)).to(device)\n    labels = np.column_stack([a["y"], context["kinetic"]]).astype(np.float32)\n    records, probability = sampling_groups(a, fitting)\n    rng = np.random.default_rng(seed)\n    pretrain_indices = np.empty(0, np.int64)\n    if spec.get("pretrain"):\n        if external is None:\n            raise ValueError("Self-supervised method requires mapped external unlabeled records")\n        pretrain_indices = data.external_training_indices(external, heldout_subjects)\n        ux = transform(external["x"], prep)\n        urecords = [np.flatnonzero((external["record"] == i) & np.isin(np.arange(len(ux)), pretrain_indices))\n            for i in np.unique(external["record"][pretrain_indices])]\n        urecords = [r for r in urecords if len(r)]\n        up = np.asarray([len(r) for r in urecords], float); up /= up.sum()\n        optimizer = torch.optim.AdamW(model.parameters(), lr=.0008, weight_decay=.01)\n        ulabels = np.zeros((len(ux), 4), np.float32)\n        for _ in range(cfg.pretrain_steps):\n            if time.monotonic() >= deadline:\n                raise TimeoutError("Suite runtime budget reached during pretraining")\n            bx, _, mask, _ = batch(urecords, up, rng, ux, ulabels, cfg)\n            values = torch.from_numpy(bx).to(device)\n            selected = torch.rand_like(values) < .2\n            reconstruction = model.decoder(model.encode(values.masked_fill(selected, 0.)))\n            valid = selected & torch.from_numpy(mask).to(device)[..., None]\n            loss = ((reconstruction - values) ** 2)[valid].mean()\n            optimizer.zero_grad(set_to_none=True); loss.backward()\n            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.); optimizer.step()\n        torch.save({"model": model.state_dict(), "pretraining_steps": cfg.pretrain_steps,\n            "fitting_external_indices": pretrain_indices, "fingerprint": fingerprint}, directory / "pretrained.pt")\n        del ux\n    ema = copy.deepcopy(model).eval()\n    optimizer = torch.optim.AdamW(model.parameters(), lr=.0008, weight_decay=.01)\n    positive_weight = torch.from_numpy(prep["positive_weight"]).to(device)\n    traces = []\n    model.train()\n    for epoch in range(cfg.neural_epochs):\n        for _ in range(cfg.steps_per_epoch):\n            if time.monotonic() >= deadline:\n                raise TimeoutError("Suite runtime budget reached during supervised training")\n            bx, by, mask, trace = batch(records, probability, rng, xx, labels, cfg)\n            traces.extend(trace)\n            inputs = torch.from_numpy(bx).to(device)\n            truth = torch.from_numpy(by).to(device)\n            valid = torch.from_numpy(mask).to(device).float()\n            logits = model(inputs)\n            losses = F.binary_cross_entropy_with_logits(logits[..., :2], truth[..., :2], pos_weight=positive_weight, reduction="none")\n            loss = (losses * valid[..., None]).sum() / (valid.sum().clamp_min(1) * 2)\n            if spec.get("kinetic"):\n                kmask = valid * truth[..., 3]\n                auxiliary = F.binary_cross_entropy_with_logits(logits[..., 2], truth[..., 2], reduction="none")\n                loss = loss + .2 * (auxiliary * kmask).sum() / kmask.sum().clamp_min(1)\n            if not torch.isfinite(loss):\n                raise FloatingPointError("Nonfinite neural loss")\n            optimizer.zero_grad(set_to_none=True); loss.backward()\n            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.); optimizer.step()\n            with torch.no_grad():\n                for dst, src in zip(ema.parameters(), model.parameters()):\n                    dst.lerp_(src, .005)\n        torch.save({"model": model.state_dict(), "ema": ema.state_dict(), "optimizer": optimizer.state_dict(),\n            "completed_epochs": epoch + 1, "fingerprint": fingerprint}, directory / "epoch_checkpoint.pt")\n        print(f"{spec[\'id\']} fold {fold}: epoch {epoch+1}/{cfg.neural_epochs}", flush=True)\n    np.savez_compressed(directory / "training_trace.npz", crops=np.asarray(traces, np.int64), external_indices=pretrain_indices)\n    torch.save({"model": ema.state_dict(), "features": x.shape[1], "architecture": spec["architecture"],\n        "width": cfg.width, "auxiliary": spec.get("kinetic", False), "fingerprint": fingerprint,\n        "fold": fold}, directory / "model.pt")\n    receipt = {"backend": str(device), "optimizer_updates": cfg.neural_epochs * cfg.steps_per_epoch,\n        "fixed_epochs": cfg.neural_epochs, "final_weights": "EMA_0.995", "seed": seed,\n        "fitting_subjects": sorted(map(str, train_subjects)), "heldout_subjects": sorted(map(str, heldout_subjects)),\n        "external_pretraining_rows": len(pretrain_indices),\n        "kinetic_supervised_fitting_grid_rows": int(context["kinetic"][fitting, 1].sum()) if spec.get("kinetic") else 0,\n        "outer_labels_used_for_training_or_stopping": False}\n    return ema.eval(), prep, receipt\n\n\ndef predict(model, x, prep, cfg, device):\n    model.eval()\n    xx = transform(x, prep)\n    starts = list(range(-cfg.window // 2, len(x), cfg.window // 2))\n    result, counts = np.zeros((len(x), 2), np.float64), np.zeros(len(x), np.float64)\n    blend = np.maximum(np.hanning(cfg.window), .1)\n    with torch.no_grad():\n        for offset in range(0, len(starts), cfg.batch_size):\n            group = starts[offset:offset + cfg.batch_size]\n            windows = np.stack([xx[np.clip(s + np.arange(cfg.window), 0, len(xx)-1)] for s in group])\n            values = torch.sigmoid(model(torch.from_numpy(windows).to(device))[..., :2]).cpu().numpy()\n            for start, values in zip(group, values):\n                left, right = max(0, start), min(len(x), start + cfg.window)\n                if right <= left:\n                    continue\n                local = left - start; weights = blend[local:local+right-left]\n                result[left:right] += values[local:local+right-left] * weights[:, None]\n                counts[left:right] += weights\n    assert (counts > 0).all()\n    return (result / counts[:, None]).astype(np.float32)\n\n\ndef load(path, device):\n    payload = torch.load(path, map_location=device, weights_only=False)\n    model = Network(payload["features"], payload["architecture"], payload["width"], payload["auxiliary"]).to(device)\n    model.load_state_dict(payload["model"])\n    return model.eval()\n', 'tdcs_suite_runner': '"""Two labelled suites in two Kaggle runs; never updates best-model registries."""\nfrom __future__ import annotations\n\nimport gc\nimport hashlib\nimport json\nimport time\nimport traceback\nfrom dataclasses import asdict, dataclass\nfrom importlib.metadata import version\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom sklearn.metrics import average_precision_score\n\nimport tdcs_suite_data as data\nimport tdcs_suite_models as trees\nimport tdcs_suite_neural as neural\n\nROOT = Path(__file__).resolve().parents[1]\nTARGETS = data.TARGETS\n\n\ndef method(id, title, suite, kind="catboost", encoding="record_metadata", **options):\n    return {"id": id, "title": title, "suite": suite, "kind": kind, "encoding": encoding, **options}\n\n\nCATALOG = [\n    method("M00", "Control: all three recording metadata fields", "metadata"),\n    method("M01", "Within-record ranks plus recording metadata", "metadata", encoding="rank_metadata"),\n    method("M02", "Metadata-aware temporal CNN", "metadata", "neural", architecture="tcn"),\n    method("M03", "Metadata-aware bidirectional GRU", "metadata", "neural", architecture="bigru"),\n    method("M04", "Metadata-aware bidirectional LSTM", "metadata", "neural", architecture="bilstm"),\n    method("M05", "Metadata-aware attention encoder", "metadata", "neural", architecture="attention"),\n    method("M06", "XGBoost plus recording metadata", "metadata", "xgboost"),\n    method("M07", "Depthwise CatBoost plus metadata", "metadata", policy="Depthwise"),\n    method("M08", "Lossguide CatBoost plus metadata", "metadata", policy="Lossguide"),\n    method("M09", "213 local sensor columns plus metadata", "metadata", encoding="local_metadata"),\n    method("M10", "213 local rank columns plus metadata", "metadata", encoding="local_rank_metadata"),\n    method("M11", "Recording and clinical metadata plus sensors", "metadata", encoding="clinical_metadata"),\n    method("M12", "Metadata-only baseline: no sensor predictors or sensor gate", "metadata", encoding="metadata_only", gate=False),\n    method("M13", "Field ablation: Visit only plus sensors", "metadata", encoding="visit_only"),\n    method("M14", "Field ablation: Test only plus sensors", "metadata", encoding="test_only"),\n    method("M15", "Field ablation: Medication only plus sensors", "metadata", encoding="medication_only"),\n    method("M16", "Medication-specific experts with a global fallback", "metadata", experts="medication"),\n    method("M17", "Test-specific experts with a global fallback", "metadata", experts="test"),\n    method("M18", "Auxiliary Kinetic training label with metadata-aware CNN", "metadata", "neural", architecture="tcn", kinetic=True),\n    method("S00", "Sensor-only CatBoost control", "sensor", encoding="sensor"),\n    method("S01", "LightGBM: explicit Kaggle CPU training", "sensor", "lightgbm", encoding="sensor"),\n    method("S02", "Supervised sensor CNN control for pretraining", "sensor", "neural", "sensor", architecture="tcn"),\n    method("S03", "Masked reconstruction pretraining on mapped unlabeled recordings", "sensor", "neural", "sensor", architecture="tcn", pretrain=True),\n    method("S04", "Teacher-student pseudo-label training", "sensor", "pseudo", "sensor"),\n    method("S05", "Sensor-only binary XGBoost control for ranking", "sensor", "xgboost", "sensor"),\n    method("S06", "Per-record MAP-focused tree ranking", "sensor", "rank_map", "sensor"),\n]\n\n\n@dataclass(frozen=True)\nclass Config:\n    seed: int = 27103\n    tree_rounds: int = 1000\n    neural_epochs: int = 12\n    steps_per_epoch: int = 64\n    pretrain_steps: int = 512\n    batch_size: int = 16\n    window: int = 256\n    width: int = 64\n    max_seconds: int = 21600\n    folds: tuple = (0, 1, 2, 3, 4)\n    smoke: bool = False\n\n\ndef smoke_config():\n    return Config(tree_rounds=4, neural_epochs=1, steps_per_epoch=1, pretrain_steps=1,\n        batch_size=4, window=32, width=16, max_seconds=600, smoke=True)\n\n\ndef source_hashes():\n    modules = {"runner": __file__, "data": data.__file__, "trees": trees.__file__, "neural": neural.__file__}\n    result = {}\n    import sys\n    for name, path in modules.items():\n        module = sys.modules[__name__] if name == "runner" else {"data": data, "trees": trees, "neural": neural}[name]\n        result[name] = getattr(module, "SOURCE_SHA256", None) or data.digest(path)\n    result["frozen_rank_helpers"] = data.frozen.source_hashes()\n    return result\n\n\ndef write(path, value):\n    data.frozen.shared.common.write_json(Path(path), value)\n\n\ndef initialize(suite, output, cfg=None):\n    if suite not in ("metadata", "sensor"):\n        raise ValueError("Unknown suite")\n    cfg = cfg or Config()\n    if cfg.folds != (0, 1, 2, 3, 4) or min(cfg.tree_rounds, cfg.neural_epochs, cfg.steps_per_epoch) < 1:\n        raise ValueError("All five folds and positive fixed training budgets are required")\n    if not cfg.smoke and (not Path("/kaggle/working").exists() or not torch.cuda.is_available()):\n        raise RuntimeError("Real-data fitting is authorized only on Kaggle CUDA; local execution must be synthetic")\n    torch.set_num_threads(2)\n    context = data.synthetic_context() if cfg.smoke else data.load_context()\n    output = Path(output); output.mkdir(parents=True, exist_ok=True)\n    session = {"suite": suite, "output": output, "cfg": cfg, "context": context,\n        "deadline": time.monotonic() + cfg.max_seconds, "results": []}\n    versions = {}\n    for name in ("numpy", "torch", "catboost", "xgboost", "lightgbm"):\n        try:\n            versions[name] = version(name)\n        except Exception:\n            versions[name] = None\n    session["runtime"] = versions\n    write(output / "suite_manifest.json", {"suite": suite, "methods": [s for s in CATALOG if s["suite"] == suite],\n        "configuration": asdict(cfg), "sources": source_hashes(), "runtime": versions,\n        "clinical_join_audit": context["clinical_audit"], "input_sources": context["source_inputs"],\n        "all_negative_recordings_retained": True, "outer_stopping": False,\n        "independent_generalization_established": False, "retained_heads_changed": False,\n        "independent_validation_required_before_promotion": True, "competition_submission_created": False})\n    return session\n\n\ndef condition_values(context, field):\n    m = context["record_metadata"]\n    if field == "medication":\n        return np.where(m[:, 6] == 1, 0, np.where(m[:, 7] == 1, 1, 2))\n    return np.where(m[:, 2] == 1, 0, np.where(m[:, 3] == 1, 1, np.where(m[:, 4] == 1, 2, 3)))\n\n\ndef fit_tree_fold(session, spec, x, fold, directory, fingerprint, external):\n    c, cfg = session["context"], session["cfg"]\n    a = c["arrays"]\n    heldout = a["fold"] == fold\n    fitting_subjects = sorted(set(a["subject"][~heldout]))\n    heldout_subjects = sorted(set(a["subject"][heldout]))\n    assert set(fitting_subjects).isdisjoint(heldout_subjects)\n    predictions, models = np.empty((int(heldout.sum()), 2), np.float32), []\n    for target in range(2):\n        seed = cfg.seed + 1009 * fold + target\n        chosen, weights, probability = data.frozen.parent.training_plan(a["y"][:, target], a["fold"],\n            a["subject"], fold, seed, 4, .5)\n        kind = "catboost" if spec["kind"] == "pseudo" else spec["kind"]\n        groups = None\n        if kind == "rank_map":\n            chosen, groups, weights = trees.ranking_plan(chosen, weights, a["record"])\n        np.savez_compressed(directory / (TARGETS[target] + "_sample.npz"), indices=chosen, weights=weights,\n            groups=np.asarray([] if groups is None else groups))\n        model, backend = trees.fit_tree(kind, x[chosen], a["y"][chosen, target], weights, seed,\n            cfg, not cfg.smoke, spec.get("policy", "SymmetricTree"), groups)\n        info = {"target": TARGETS[target], "kind": kind, "training_sample_probability": probability,\n            "sample_sha256": data.digest(directory / (TARGETS[target] + "_sample.npz")), "backend": backend,\n            "pseudo_rows_added": 0, "experts": [], "expert_routing": []}\n        if spec["kind"] == "pseudo":\n            teacher_path = trees.save_tree(model, kind, directory / (TARGETS[target] + "_teacher"))\n            info["teacher_file"], info["teacher_sha256"] = teacher_path.name, data.digest(teacher_path)\n            eligible = data.external_training_indices(external, heldout_subjects)\n            probabilities = trees.predict_tree(model, kind, external["x"][eligible])\n            selected = np.flatnonzero((probabilities >= .90) | (probabilities <= .01))\n            if len(selected) > 40000:\n                selected = np.random.default_rng(seed).choice(selected, 40000, replace=False)\n            indices = eligible[selected]\n            pseudo = (probabilities[selected] >= .90).astype(np.uint8)\n            np.savez_compressed(directory / (TARGETS[target] + "_pseudo.npz"), indices=indices, labels=pseudo,\n                teacher_probabilities=probabilities[selected], weights=np.full(len(indices), .1))\n            info["pseudo_rows_added"] = len(indices)\n            if len(indices):\n                combined_weights = np.r_[weights, np.full(len(indices), .1)]\n                combined_weights /= combined_weights.mean()\n                model, backend = trees.fit_tree(kind, np.vstack([x[chosen], external["x"][indices]]),\n                    np.r_[a["y"][chosen, target], pseudo], combined_weights, seed, cfg, not cfg.smoke)\n                info["student_backend"] = backend\n        path = trees.save_tree(model, kind, directory / TARGETS[target])\n        info["model_file"], info["model_sha256"] = path.name, data.digest(path)\n        model = trees.load_tree(kind, path)\n        predictions[:, target] = trees.predict_tree(model, kind, x[heldout])\n        if spec.get("experts"):\n            conditions = condition_values(c, spec["experts"])[a["record"]]\n            for value in np.unique(conditions):\n                mask = conditions[chosen] == value\n                labels = a["y"][chosen[mask], target]\n                routing = {"condition": int(value), "training_rows": int(mask.sum()),\n                    "heldout_rows": int((conditions[heldout] == value).sum())}\n                # Fixed small-group fallback; no held-out labels decide routing.\n                if mask.sum() < 128 or labels.sum() < 5 or (1 - labels).sum() < 5:\n                    info[\'expert_routing\'].append({**routing, \'route\': \'global_small_fitting_group\'})\n                    continue\n                expert, actual = trees.fit_tree(kind, x[chosen[mask]], labels, weights[mask], seed,\n                    cfg, not cfg.smoke, spec.get("policy", "SymmetricTree"))\n                expert_path = trees.save_tree(expert, kind, directory / (TARGETS[target] + f"_expert_{value}"))\n                selected = conditions[heldout] == value\n                if selected.any():\n                    predictions[selected, target] = trees.predict_tree(expert, kind, x[heldout][selected])\n                info[\'expert_routing\'].append({**routing, \'route\': \'fitting_group_expert\'})\n                info["experts"].append({"condition": int(value), "model_file": expert_path.name,\n                    "model_sha256": data.digest(expert_path), "training_rows": int(mask.sum()), "backend": actual})\n        models.append(info)\n    return predictions, {"fingerprint": fingerprint, "fold": fold, "models": models,\n        "fitting_subjects": list(map(str, fitting_subjects)), "heldout_subjects": list(map(str, heldout_subjects)),\n        "outer_labels_used_for_training_or_stopping": False,\n        "ranking_weights_are_per_query_not_per_row": spec["kind"] == "rank_map"}\n\n\ndef evaluate(context, scores, apply_gate=True):\n    a = context["arrays"]\n    native = np.empty_like(context["native_y"], dtype=np.float32)\n    native_folds = np.empty(len(native), np.int8)\n    for i, row in enumerate(context["manifest"].itertuples(index=False)):\n        selected = a["record"] == i\n        left, right = context["native_offsets"][i:i+2]\n        for target in range(2):\n            native[left:right, target] = np.interp(np.arange(right-left), a["grid"][selected], scores[selected, target]).astype(np.float32)\n        if apply_gate:\n            factor = (np.asarray([context["gates"][i]], np.float32) * .95 + .05)[0]\n            native[left:right, 1] *= factor\n        native_folds[left:right] = row.fold\n    assert np.isfinite(native).all() and ((native >= 0) & (native <= 1)).all()\n    values = {target: float(average_precision_score(context["native_y"][:, j], native[:, j]))\n        for j, target in enumerate(TARGETS)}\n    folds = []\n    for fold in range(5):\n        selected = native_folds == fold\n        row = {"fold": fold}\n        for j, target in enumerate(TARGETS):\n            row[target] = (float(average_precision_score(context["native_y"][selected, j], native[selected, j]))\n                if context["native_y"][selected, j].any() else None)\n        folds.append(row)\n    values["two_target_mAP"] = float(np.mean(list(values.values())))\n    return values, folds, data.array_digest(native)\n\n\ndef run_method(session, method_id):\n    spec = next(s for s in CATALOG if s["id"] == method_id and s["suite"] == session["suite"])\n    c, cfg, output = session["context"], session["cfg"], session["output"]\n    status = {"id": method_id, "title": spec["title"], "suite": spec["suite"], "status": "pending"}\n    try:\n        if time.monotonic() >= session["deadline"]:\n            raise TimeoutError("Suite runtime budget reached before this test")\n        x = data.input_features(c, spec["encoding"])\n        a = c["arrays"]\n        external = data.external_pool(c, output) if spec.get("pretrain") or spec["kind"] == "pseudo" else None\n        if spec.get("kinetic") and not c["kinetic"][:, 1].any():\n            raise ValueError("No mapped Kinetic auxiliary labels are available")\n        config = {"method": spec, "configuration": asdict(cfg), "sources": source_hashes(),\n            "runtime": session["runtime"],\n            "input_feature_count": x.shape[1], "input_feature_sha256": data.array_digest(x),\n            "population_recordings": c["manifest"].recording_id.tolist(),\n            "population_subjects": c["manifest"].subject.tolist(), "population_folds": c["manifest"].fold.tolist(),\n            "source_inputs": c["source_inputs"], "gate_applied": spec.get("gate", True),\n            "external_pool_sha256": ({key: data.array_digest(value) for key, value in external.items()}\n                if external is not None else None),\n            "smoke_only": cfg.smoke, "outer_stopping": False, "automatic_promotion": False}\n        fingerprint = hashlib.sha256(json.dumps(config, sort_keys=True).encode()).hexdigest()[:20]\n        run = output / "methods" / method_id / fingerprint\n        run.mkdir(parents=True, exist_ok=True)\n        write(run / "configuration.json", config)\n        scores = np.full((len(x), 2), np.nan, np.float32)\n        for fold in cfg.folds:\n            if time.monotonic() >= session["deadline"]:\n                raise TimeoutError("Suite runtime budget reached between folds")\n            directory = run / f"fold_{fold}"\n            directory.mkdir(exist_ok=True)\n            saved, receipt_path = directory / "heldout_grid.npz", directory / "training_receipt.json"\n            heldout = a["fold"] == fold\n            if saved.is_file() and receipt_path.is_file():\n                receipt = json.loads(receipt_path.read_text())\n                assert receipt["fingerprint"] == fingerprint and receipt["fold"] == fold\n                assert receipt["grid_sha256"] == data.digest(saved)\n                for filename, expected in receipt["artifact_sha256"].items():\n                    assert data.digest(directory / filename) == expected\n                with np.load(saved, allow_pickle=False) as payload:\n                    np.testing.assert_array_equal(payload["indices"], np.flatnonzero(heldout))\n                    scores[heldout] = payload["prediction"]\n                print(f"{method_id} fold {fold}: reused verified complete checkpoint", flush=True)\n                continue\n            if any(directory.iterdir()):\n                raise ValueError("Incomplete fold artifacts: preserve them and use a fresh output root; no silent overwrite")\n            if spec["kind"] == "neural":\n                device = torch.device("cpu" if cfg.smoke else "cuda:0")\n                model, prep, receipt = neural.train(c, x, fold, spec, cfg, directory, device, fingerprint,\n                    external=external, deadline=session["deadline"])\n                parts = np.empty((int(heldout.sum()), 2), np.float32)\n                location = np.full(len(x), -1, np.int64); location[heldout] = np.arange(heldout.sum())\n                for record in np.unique(a["record"][heldout]):\n                    rows = np.flatnonzero((a["record"] == record) & heldout)\n                    parts[location[rows]] = neural.predict(model, x[rows], prep, cfg, device)\n                receipt.update(fingerprint=fingerprint, fold=fold)\n                del model\n            else:\n                parts, receipt = fit_tree_fold(session, spec, x, fold, directory, fingerprint, external)\n            assert parts.shape == (int(heldout.sum()), 2) and np.isfinite(parts).all()\n            scores[heldout] = parts\n            np.savez_compressed(saved, indices=np.flatnonzero(heldout), prediction=parts)\n            receipt["grid_sha256"] = data.digest(saved)\n            receipt["artifact_sha256"] = {p.name: data.digest(p) for p in directory.iterdir()\n                if p.is_file() and p.name not in ("training_receipt.json", "heldout_grid.npz")}\n            write(receipt_path, receipt)\n            print(f"{method_id} fold {fold}: training and held-out prediction complete", flush=True)\n            gc.collect()\n            if torch.cuda.is_available():\n                torch.cuda.empty_cache()\n        np.savez_compressed(run / "grid_scores.npz", prediction=scores, targets=np.asarray(TARGETS))\n        values, folds, native_hash = evaluate(c, scores, spec.get("gate", True))\n        status.update(status="complete", **values, run_directory=str(run), native_prediction_sha256=native_hash,\n            independent_validation_pending=True, smoke_only=cfg.smoke, actual_grid_rows=len(scores))\n        write(run / "decision.json", {**status, "folds": folds, "all_negative_rows_retained": True,\n            "retained_heads_changed": False, "confidence_calibrated": False,\n            "independent_generalization_established": False, "competition_submission_created": False})\n        del x\n    except Exception as error:\n        status.update(status="not_run_budget" if isinstance(error, TimeoutError) else "failed",\n            error_type=type(error).__name__, error=str(error), traceback=traceback.format_exc())\n        print(f"{method_id}: {status[\'status\']}: {error}", flush=True)\n    session["results"].append(status)\n    write(output / (method_id + "_status.json"), status)\n    pd.DataFrame([{key: value for key, value in row.items() if key != "traceback"} for row in session["results"]]).to_csv(output / "suite_summary.csv", index=False)\n    return {key: value for key, value in status.items() if key != "traceback"}\n\n\ndef finish(session):\n    expected = {s["id"] for s in CATALOG if s["suite"] == session["suite"]}\n    actual = {s["id"] for s in session["results"]}\n    complete = [s["id"] for s in session["results"] if s["status"] == "complete"]\n    result = {"status": "all_methods_completed_validation_pending" if len(complete) == len(expected) else "partial_coverage",\n        "expected_tests": sorted(expected), "attempted_tests": sorted(actual), "completed_tests": complete,\n        "missing_tests": sorted(expected - actual), "smoke_only": session["cfg"].smoke,\n        "all_complete_results_require_independent_replay_before_promotion": True,\n        "retained_heads_changed": False, "competition_submission_created": False,\n        "confidence_calibrated": False, "independent_generalization_established": False}\n    write(session["output"] / "suite_completion.json", result)\n    return result\n'}
HASHES = {'tdcs_multiscale_signal_experiment': '6606266f2147cd76a66474cdf27aeb123cab32d55022f4272e05afe46f408d51', 'tdcs_two_target_neural': 'f8db1ad2934adf9c55cf466e36fa9c5e512b62acf2590f5808a4c765eaff5064', 'tdcs_alternative_architectures': '1001e07b13366ecb099e174a09a65568afbf0170a6e917d34dce9ace67e95ce8', 'tdcs_resource_safe_experiment': 'f578ec5ef35030069bd81bda5536774b2588e560da6343dd3a01225c452a4ae6', 'tdcs_native_context_experiment': '9b1fe009f6d0e8deeb212d4f6f53626a9d4c685ea3df3003ebdd0a4fcde6eb5c', 'tdcs_gpu_boosted_features': '6c32f83a678599539d84f85cdb607c3d6464b88dbe6ca134d1e263e8eeac4230', 'tdcs_gpu_boosted_context': 'f6f6aaed59553dba84b4665519316f2d9c02d0727350cb741465e3f1a65d86c9', 'tdcs_walking_conditional_gpu': '33a8493299a93ba343a0b03052a504bf3d6012d3eccd2bf13cfe5f5f9598a1ba', 'tdcs_gpu_regularized_context': 'ec6a7c348428a945e1bccbc88d40afd3121dbf7bc30b60a42d28317446ce3bac', 'tdcs_subject_weight_gpu': 'bbf6dea3fbfa0378aee8954c8a3d1f98bc08ed68297a92b7edeb28de4f3b2cb3', 'tdcs_record_rank_gpu': '4a6b0ea5bc210d4a165d9ff23774be615efe09adbaf2868996009a1cc2dc2ae9', 'tdcs_metadata_context_gpu': '4fd8ecb8658da95d521c6ce582a26e45d7b6831d5389b636a58a3c6eba62c071', 'tdcs_suite_data': '369ad332c7dbe9f24a2ee7e25345ba6d2c31752fa615d18477a0e0a9c2c911cd', 'tdcs_suite_models': '003f729d9df5b33e5700e66ed59179bdd6aba5924cc8c1801474f0d256683482', 'tdcs_suite_neural': 'ddffef1b593e20e3f59a9b3d07b6bd785a3e66845e4a3aa77ae80543cbb64c76', 'tdcs_suite_runner': 'cc8e3b7cd7336a68843a47a68fdfdec3f2a7195acecf3dc35976b5f7adee3da1'}
RECIPES = {'tdcs_dense_context_experiment.py': '"""TDCS-only dense temporal/spectral HGB experiment; preserve previous winners.\n\nOne new feature/model configuration and four fixed candidate comparisons against\nthe retained 0.3800 blend. Settings are written before evaluation. No leaderboard\ncalls, learned OOF stacker, outer-label early stopping, or DeFOG mutations.\n"""\n\nfrom __future__ import annotations\n\nimport gc\nimport hashlib\nimport json\nimport time\nfrom dataclasses import asdict, replace\nfrom pathlib import Path\n\nimport joblib\nimport matplotlib\nmatplotlib.use("Agg")\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nimport sklearn\nfrom numpy.lib.stride_tricks import sliding_window_view\nfrom sklearn.ensemble import HistGradientBoostingClassifier\nfrom threadpoolctl import threadpool_limits\n\nimport tdcs_multiscale_signal_experiment as base\nimport tdcs_candidate_sweep as previous\n\n\nROOT = Path(__file__).resolve().parents[1]\nOUTPUT_ROOT = ROOT / "results" / "tdcs_dense_context"\nTARGETS = base.TARGETS\nFEATURE_CFG = replace(base.CFG, feature_version="dense_temporal_spectral_v2", training_stride=16)\nMODEL_PARAMS = {\n    "learning_rate": 0.07, "max_iter": 260, "max_leaf_nodes": 31,\n    "min_samples_leaf": 200, "l2_regularization": 20.0,\n    "max_bins": 128, "early_stopping": False,\n}\nFFT_WINDOWS = (256, 1024)\nCONTEXT_SHIFTS = (-256, -64, 64, 256)\nFIXED_BLENDS = {\n    "neural_dense_equal": {"previous_tdcs_blend": 0.5, "dense_context_hgb": 0.5},\n    "retained_dense_quarter": {"retained_neural_hgb": 0.75, "dense_context_hgb": 0.25},\n    "retained_dense_equal": {"retained_neural_hgb": 0.5, "dense_context_hgb": 0.5},\n}\nPLOT_LABELS = {\n    "retained_neural_hgb": "Retained neural + HGB",\n    "dense_context_hgb": "Dense spectral/context HGB",\n    "neural_dense_equal": "Neural + dense HGB (equal)",\n    "retained_dense_quarter": "Retained + dense HGB (25%)",\n    "retained_dense_equal": "Retained + dense HGB (50%)",\n}\n\n\ndef spectral_features(raw: np.ndarray, grid: np.ndarray) -> tuple[np.ndarray, list[str]]:\n    """Centered Hann-window spectra; real accelerometer/timing inputs only."""\n    raw = np.asarray(raw, np.float64)\n    center = raw.mean(axis=0)\n    z = (raw - center) / np.maximum(raw.std(axis=0), 0.05)\n    dynamic = np.linalg.norm(raw - center, axis=1)\n    dynamic /= max(float(np.sqrt(np.mean(dynamic ** 2))), 0.05)\n    channels = np.column_stack([z, dynamic])\n    channel_names = [*base.AXES, "orientation_invariant_dynamic_magnitude"]\n    statistics = ["log_power", "entropy", "centroid_hz", "spread_hz", "peak_hz", "peak_fraction",\n                  "low_fraction", "mid_fraction", "high_fraction"]\n    names, parts = [], []\n    for window in FFT_WINDOWS:\n        left = window // 2\n        padded = np.pad(channels, ((left, window - 1 - left), (0, 0)), mode="edge")\n        views = sliding_window_view(padded, window_shape=window, axis=0)\n        frequency = np.fft.rfftfreq(window, d=1 / FEATURE_CFG.sample_hz)\n        selected = (frequency >= 0.5) & (frequency <= 15.0)\n        frequency = frequency[selected]\n        band_masks = [(frequency < 3), ((frequency >= 3) & (frequency < 8)), (frequency >= 8)]\n        taper = np.hanning(window)\n        result = np.empty((len(grid), len(statistics) * 4), np.float32)\n        for start in range(0, len(grid), 128):\n            rows = grid[start:start + 128]\n            segments = views[rows].copy()\n            segments -= segments.mean(axis=-1, keepdims=True)\n            spectrum = np.fft.rfft(segments * taper, axis=-1)\n            power = np.abs(spectrum[..., selected]) ** 2 / np.sum(taper ** 2)\n            total = power.sum(axis=-1)\n            normalized = power / np.maximum(total[..., None], 1e-12)\n            entropy = -(normalized * np.log(np.maximum(normalized, 1e-12))).sum(axis=-1) / np.log(len(frequency))\n            centroid = (normalized * frequency).sum(axis=-1)\n            spread = np.sqrt(np.maximum((normalized * (frequency - centroid[..., None]) ** 2).sum(axis=-1), 0))\n            peak = frequency[np.argmax(power, axis=-1)]\n            peak = np.where(total > 1e-12, peak, 0)\n            values = [np.log1p(total), entropy, centroid, spread, peak, normalized.max(axis=-1)]\n            values.extend(normalized[..., mask].sum(axis=-1) for mask in band_masks)\n            result[start:start + len(rows)] = np.stack(values, axis=1).reshape(len(rows), -1)\n        parts.append(result)\n        names.extend(f"fft_{statistic}_{channel}_w{window}" for statistic in statistics for channel in channel_names)\n    return np.concatenate(parts, axis=1), names\n\n\ndef temporal_context(features: np.ndarray, names: list[str], grid: np.ndarray) -> tuple[np.ndarray, list[str]]:\n    """Look backward/forward inside one recording, never across its boundary."""\n    selected_names = [f"std_{axis}_w256" for axis in [*base.AXES, "dynamic_magnitude"]]\n    selected_names += [f"mean_{axis}_w256" for axis in base.AXES]\n    selected_names += [f"log_mid_low_energy_ratio_{axis}_w256" for axis in base.AXES]\n    indices = [names.index(name) for name in selected_names]\n    shifted, shifted_names = [], []\n    for shift in CONTEXT_SHIFTS:\n        shifted.append(np.column_stack([np.interp(grid + shift, grid, features[:, index]) for index in indices]))\n        shifted_names.extend(f"context_shift_{shift}_{name}" for name in selected_names)\n    return np.concatenate(shifted, axis=1).astype(np.float32), shifted_names\n\n\ndef expanded_features(raw: np.ndarray) -> tuple[np.ndarray, np.ndarray, list[str]]:\n    grid, original, names = base.sensor_features(raw, cfg=FEATURE_CFG)\n    spectrum, spectrum_names = spectral_features(raw, grid)\n    context, context_names = temporal_context(original, names, grid)\n    x = np.concatenate([original, spectrum, context], axis=1)\n    names = [*names, *spectrum_names, *context_names]\n    if x.shape[1] != len(names) or len(set(names)) != len(names) or not np.isfinite(x).all():\n        raise ValueError("Invalid expanded sensor features")\n    return grid, x, names\n\n\ndef prepare_arrays(manifest: pd.DataFrame, run_dir: Path) -> dict:\n    cache = run_dir / "dense_context_features.npz"\n    if cache.exists():\n        with np.load(cache, allow_pickle=False) as payload:\n            arrays = {name: payload[name].copy() for name in payload.files}\n        if not np.array_equal(arrays["recording_ids"], manifest.recording_id.to_numpy(str)):\n            raise ValueError("Cached feature recording order differs")\n        return arrays\n    parts = {name: [] for name in ["x", "y", "fold", "record", "grid"]}\n    row_counts, feature_names = [], None\n    for record_index, row in enumerate(manifest.itertuples(index=False)):\n        frame = pd.read_csv(row.path, usecols=[*base.AXES, *TARGETS])\n        labels = frame.loc[:, list(TARGETS)].to_numpy()\n        if not np.isin(labels, [0, 1]).all():\n            raise ValueError(f"Invalid labels: {row.recording_id}")\n        grid, x, names = expanded_features(frame.loc[:, list(base.AXES)].to_numpy())\n        if feature_names is not None and feature_names != names:\n            raise ValueError("Feature names changed")\n        feature_names = names\n        parts["x"].append(x)\n        parts["y"].append(labels[grid].astype(np.uint8))\n        parts["fold"].append(np.full(len(grid), row.fold, np.int8))\n        parts["record"].append(np.full(len(grid), record_index, np.int16))\n        parts["grid"].append(grid)\n        row_counts.append(len(frame))\n        if (record_index + 1) % 100 == 0:\n            print(f"Dense spectral/context features: {record_index + 1}/{len(manifest)}", flush=True)\n    arrays = {name: np.concatenate(values) for name, values in parts.items()}\n    arrays.update(row_counts=np.asarray(row_counts, np.int32),\n                  feature_names=np.asarray(feature_names, dtype=str), recording_ids=manifest.recording_id.to_numpy(str))\n    print(f"Saving cache: {len(arrays[\'x\']):,} rows x {arrays[\'x\'].shape[1]} features", flush=True)\n    np.savez_compressed(cache, **arrays)\n    return arrays\n\n\ndef train_models(arrays: dict, run_dir: Path) -> np.ndarray:\n    x, y, folds = arrays["x"], arrays["y"], arrays["fold"]\n    predictions = np.full(y.shape, np.nan, np.float32)\n    training_log = []\n    with threadpool_limits(limits=4):\n        for fold in range(5):\n            fitting, validation = folds != fold, folds == fold\n            fold_dir = run_dir / "dense_hgb" / f"fold_{fold}"\n            fold_dir.mkdir(parents=True, exist_ok=True)\n            for target_index, target in enumerate(TARGETS):\n                start = time.monotonic()\n                model_path = fold_dir / f"{target}.joblib"\n                prediction_path = fold_dir / f"{target}_sampled_prediction.npy"\n                labels = y[fitting, target_index]\n                positives = int(labels.sum())\n                if positives == 0 or positives == len(labels):\n                    raise ValueError("Both target classes required in training")\n                weight = float(min(8.0, np.sqrt((len(labels) - positives) / positives)))\n                reused = model_path.exists() and prediction_path.exists()\n                if reused:\n                    p = np.load(prediction_path, allow_pickle=False)\n                else:\n                    print(f"Training dense HGB fold {fold}, {target} ...", flush=True)\n                    model = HistGradientBoostingClassifier(**MODEL_PARAMS, random_state=42 + fold)\n                    model.fit(x[fitting], labels, sample_weight=np.where(labels == 1, weight, 1.0))\n                    p = model.predict_proba(x[validation])[:, 1].astype(np.float32)\n                    joblib.dump(model, model_path)\n                    np.save(prediction_path, p, allow_pickle=False)\n                    del model\n                    gc.collect()\n                if len(p) != int(validation.sum()) or not np.isfinite(p).all() or not ((p >= 0) & (p <= 1)).all():\n                    raise ValueError("Invalid sampled OOF predictions")\n                predictions[validation, target_index] = p\n                training_log.append({"fold": fold, "target": target, "positive_weight": weight,\n                                     "training_rows": int(fitting.sum()), "validation_rows": int(validation.sum()),\n                                     "elapsed_seconds": time.monotonic() - start, "reused": reused})\n                pd.DataFrame(training_log).to_csv(run_dir / "training_log.csv", index=False)\n                print(f"Dense HGB fold {fold} / {target}: complete ({training_log[-1][\'elapsed_seconds\']:.1f}s)", flush=True)\n    if not np.isfinite(predictions).all():\n        raise ValueError("Incomplete sampled OOF predictions")\n    np.save(run_dir / "dense_hgb_sampled_oof.npy", predictions, allow_pickle=False)\n    return predictions\n\n\ndef expand_native(manifest: pd.DataFrame, arrays: dict, sampled: np.ndarray, run_dir: Path) -> np.ndarray:\n    output = run_dir / "dense_hgb" / "oof"\n    output.mkdir(exist_ok=True)\n    parts = []\n    for index, row in enumerate(manifest.itertuples(index=False)):\n        mask = arrays["record"] == index\n        grid, sparse = arrays["grid"][mask], sampled[mask]\n        n_rows = int(arrays["row_counts"][index])\n        p = np.column_stack([np.interp(np.arange(n_rows), grid, sparse[:, target]) for target in range(3)]).astype(np.float32)\n        np.savez_compressed(output / f"{row.recording_id}.npz", prediction=p)\n        parts.append(p)\n    return np.concatenate(parts)\n\n\ndef get_prediction(method: str, predictions: dict) -> np.ndarray:\n    if method in predictions:\n        return predictions[method]\n    result = np.zeros_like(predictions["retained_neural_hgb"])\n    weights = FIXED_BLENDS[method]\n    if abs(sum(weights.values()) - 1.0) > 1e-12:\n        raise ValueError("Blend weights must sum to one")\n    for name, weight in weights.items():\n        result += np.float32(weight) * predictions[name]\n    return result\n\n\ndef evaluate_candidates(y: np.ndarray, folds: np.ndarray, predictions: dict, run_dir: Path) -> pd.DataFrame:\n    rows, fold_rows = [], []\n    for method in ["retained_neural_hgb", "dense_context_hgb", *FIXED_BLENDS]:\n        p = get_prediction(method, predictions)\n        previous.validate_prediction(y, p)\n        rows.append({"method": method, **base.score_targets(y, p)})\n        for fold in range(5):\n            mask = folds == fold\n            fold_rows.append({"method": method, "fold": fold, **base.score_targets(y[mask], p[mask])})\n        print(f"{method}: TDCS native-row mAP {rows[-1][\'mAP\']:.6f}", flush=True)\n    metrics = pd.DataFrame(rows)\n    if abs(float(metrics.iloc[0].mAP) - 0.3800059184564884) > 1e-8:\n        raise ValueError("The retained 0.3800 reference did not reproduce")\n    metrics.to_csv(run_dir / "candidate_metrics.csv", index=False)\n    pd.DataFrame(fold_rows).to_csv(run_dir / "candidate_fold_metrics.csv", index=False)\n    return metrics\n\n\ndef save_selected(manifest: pd.DataFrame, arrays: dict, p: np.ndarray, metric: pd.Series, run_dir: Path) -> Path:\n    selected_dir = run_dir / "selected_candidate"\n    prediction_dir = selected_dir / "oof"\n    prediction_dir.mkdir(parents=True, exist_ok=True)\n    manifest.to_csv(selected_dir / "frozen_tdcs_manifest.csv", index=False)\n    pd.DataFrame([{"domain": "tdcsfog", **metric.to_dict()}]).to_csv(selected_dir / "native_oof_metrics.csv", index=False)\n    offset = 0\n    for index, row in enumerate(manifest.itertuples(index=False)):\n        n_rows = int(arrays["row_counts"][index])\n        np.savez_compressed(prediction_dir / f"{row.recording_id}.npz", prediction=p[offset:offset + n_rows])\n        offset += n_rows\n    if offset != len(p):\n        raise ValueError("Selected prediction row alignment changed")\n    return selected_dir\n\n\ndef make_figures(metrics: pd.DataFrame, run_dir: Path) -> list[Path]:\n    plt.rcParams.update({"font.size": 11, "axes.spines.top": False, "axes.spines.right": False})\n    ordered = metrics.sort_values("mAP")\n    fig, ax = plt.subplots(figsize=(10.5, 5.2), layout="constrained")\n    colors = ["#8b929c" if method == "retained_neural_hgb" else "#2463a0" for method in ordered.method]\n    bars = ax.barh([PLOT_LABELS[name] for name in ordered.method], ordered.mAP, color=colors)\n    ax.bar_label(bars, fmt="%.4f", padding=5)\n    ax.set_xlim(0, 1)\n    ax.set_xlabel("Pooled native-row TDCS OOF mAP")\n    ax.set_title("Dense temporal/spectral TDCS candidates: 61 subjects, 5 folds")\n    ax.grid(axis="x", alpha=0.2)\n    first = run_dir / "tdcs_dense_candidate_map.png"\n    fig.savefig(first, dpi=150)\n    plt.close(fig)\n    reference = metrics.loc[metrics.method.eq("retained_neural_hgb")].iloc[0]\n    selected = metrics.sort_values("mAP", ascending=False).iloc[0]\n    fig, ax = plt.subplots(figsize=(10, 4.8), layout="constrained")\n    positions = np.arange(3)\n    for i, (row, label, color) in enumerate([\n        (reference, "Retained neural + HGB", "#8b929c"),\n        (selected, f"Selected: {PLOT_LABELS[selected.method]}", "#2463a0"),\n    ]):\n        bars = ax.bar(positions + (i - 0.5) * 0.34, row.loc[list(TARGETS)].to_numpy(float), 0.34, label=label, color=color)\n        ax.bar_label(bars, fmt="%.3f", padding=4)\n    ax.set_xticks(positions, ["Start hesitation", "Turn", "Walking"])\n    ax.set_ylim(0, 1)\n    ax.set_ylabel("Pooled native-row average precision")\n    ax.set_title("TDCS per-class tradeoffs against the retained 0.3800 blend")\n    ax.legend(loc="upper left", frameon=False, fontsize=10)\n    ax.grid(axis="y", alpha=0.2)\n    second = run_dir / "tdcs_dense_candidate_classes.png"\n    fig.savefig(second, dpi=150)\n    plt.close(fig)\n    return [first, second]\n\n\ndef run_experiment() -> dict:\n    sensor_run, manifest, old_arrays, old_audit = previous.load_inputs()\n    del old_arrays\n    gc.collect()\n    source_hashes = {path.name: hashlib.sha256(path.read_bytes()).hexdigest() for path in\n                     [Path(__file__), ROOT / "scripts" / "tdcs_multiscale_signal_experiment.py",\n                      ROOT / "scripts" / "tdcs_candidate_sweep.py"]}\n    configuration = {\n        "feature_configuration": asdict(FEATURE_CFG), "model_parameters": MODEL_PARAMS,\n        "fft_windows": FFT_WINDOWS, "fft_range_hz": [0.5, 15], "context_shifts_native_rows": CONTEXT_SHIFTS,\n        "fixed_blends": FIXED_BLENDS, "positive_weight": "sqrt(negative/positive), cap 8, training labels only",\n        "sensor_input_fingerprint": old_audit["fingerprint"], "source_hashes": source_hashes,\n        "sklearn_version": sklearn.__version__, "numpy_version": np.__version__,\n        "reference_mAP": 0.3800059184564884,\n        "selection_caveat": "development folds reused across experiments; exploratory selection",\n    }\n    fingerprint = hashlib.sha256(json.dumps(configuration, sort_keys=True).encode()).hexdigest()[:20]\n    run_dir = OUTPUT_ROOT / fingerprint\n    run_dir.mkdir(parents=True, exist_ok=True)\n    previous.write_json(run_dir / "predeclared_configuration.json", configuration)\n    manifest.to_csv(run_dir / "frozen_tdcs_manifest.csv", index=False)\n    print(f"Dense TDCS artifacts: {run_dir}", flush=True)\n    arrays = prepare_arrays(manifest, run_dir)\n    if int(arrays["row_counts"].sum()) != 6959762:\n        raise ValueError("Native population changed")\n    audit = {\n        "domain": "tdcsfog", "recordings": len(manifest), "subjects": int(manifest.subject.nunique()),\n        "native_rows": int(arrays["row_counts"].sum()), "sampled_training_rows": len(arrays["x"]),\n        "feature_count": int(arrays["x"].shape[1]), "training_stride": FEATURE_CFG.training_stride,\n        "validation": "same five frozen subject-disjoint folds; outer labels used for scoring only",\n        "feature_inputs": "accelerometer and recording timing only",\n        "offline_assumption": "centered spectra, past/future context and whole-record normalization",\n        "retained_defog_reported_oof_map": 0.817, "competition_submission_created": False,\n        "model_selection_caveat": configuration["selection_caveat"],\n    }\n    previous.write_json(run_dir / "audit.json", audit)\n    sampled = train_models(arrays, run_dir)\n    del arrays["x"]\n    gc.collect()\n    dense = expand_native(manifest, arrays, sampled, run_dir)\n    y, folds, original = previous.load_native_inputs(sensor_run, manifest)\n    predictions = {"previous_tdcs_blend": original["previous_tdcs_blend"],\n                   "retained_neural_hgb": previous.blended_prediction("neural_hgb_equal", original),\n                   "dense_context_hgb": dense}\n    del original\n    gc.collect()\n    metrics = evaluate_candidates(y, folds, predictions, run_dir)\n    best = metrics.sort_values("mAP", ascending=False).iloc[0]\n    reference = float(metrics.iloc[0].mAP)\n    selected_dir = save_selected(manifest, arrays, get_prediction(str(best.method), predictions), best, run_dir)\n    figures = make_figures(metrics, run_dir)\n    decision = {"run_dir": str(run_dir), "selected_candidate_dir": str(selected_dir),\n                "selected_method": str(best.method), "candidate_tdcs_oof_map": float(best.mAP),\n                "retained_reference_tdcs_oof_map": reference, "delta": float(best.mAP - reference),\n                "decision": "retain new TDCS candidate" if best.mAP > reference else "retain previous 0.3800 blend",\n                "retained_defog_reported_oof_map": 0.817, "competition_submission_created": False,\n                "selection_scope": "one new model and three fixed blends on reused development folds"}\n    previous.write_json(run_dir / "decision.json", decision)\n    previous.write_json(OUTPUT_ROOT / "latest_run.json", decision)\n    print(metrics.to_string(index=False), flush=True)\n    print(json.dumps(decision, indent=2), flush=True)\n    return {"run_dir": run_dir, "metrics": metrics, "decision": decision, "figures": figures}\n\n\nif __name__ == "__main__":\n    run_experiment()\n', 'tdcs_recording_context.py': '"""Two fixed recording-context gates for retained two-target native-row scores.\n\nOnly raw-sensor summaries train the gates. Outer-fold subjects, labels and base\nOOF predictions never enter gate fitting. A gated score is not a calibrated\nprobability; retained predictors are never overwritten by this experiment.\n"""\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nfrom pathlib import Path\n\nimport joblib\nimport numpy as np\nimport pandas as pd\nimport scipy\nimport sklearn\nfrom sklearn.ensemble import HistGradientBoostingClassifier\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import average_precision_score\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.preprocessing import StandardScaler\nfrom threadpoolctl import threadpool_limits\n\nimport tdcs_native_context_experiment as native\nfrom tdcs_bounded_head_fusion import read_prediction\nfrom validate_tdcs_experiment import independent_ap\n\ncommon = native.common\nROOT = Path(__file__).resolve().parents[1]\nTARGETS = common.TARGETS\nLEARNERS = ("linear", "hgb")\nFLOOR = .05\nPARAMETERS = {\n    "linear": {"C": .1, "max_iter": 2000, "solver": "lbfgs"},\n    "hgb": {"max_iter": 180, "learning_rate": .05, "max_leaf_nodes": 7,\n            "min_samples_leaf": 20, "l2_regularization": 10., "early_stopping": False},\n}\n\n\ndef recording_features(raw):\n    """112 label-free full-record summaries; no IDs or demographics."""\n    raw = np.asarray(raw, np.float64)\n    if raw.ndim != 2 or raw.shape[1] != 3 or len(raw) < 2 or not np.isfinite(raw).all():\n        raise ValueError("A finite recording with at least two rows and three axes is required")\n    values, names = [np.log1p(len(raw) / 128.)], ["log_duration_seconds"]\n\n    def add(value, name):\n        values.append(float(value)); names.append(name)\n\n    for i, axis in enumerate(common.AXES):\n        add(raw[:, i].std(), f"raw_{axis}_std")\n        for q in (.05, .25, .5, .75, .95):\n            add(np.quantile(raw[:, i], q), f"raw_{axis}_q{q}")\n    centered = raw - np.median(raw, axis=0, keepdims=True)\n    dynamic = np.linalg.norm(centered, axis=1)\n    vectors = {"magnitude": np.linalg.norm(raw, axis=1), "dynamic": dynamic,\n               "jerk": np.linalg.norm(np.diff(raw, axis=0), axis=1)}\n    for label, vector in vectors.items():\n        add(np.log1p(vector.mean()), label + "_log_mean")\n        add(np.log1p(vector.std()), label + "_log_std")\n        for q in (.1, .5, .9, .99):\n            add(np.log1p(np.quantile(vector, q)), f"{label}_log_q{q}")\n    energy = native.spectral_features(raw)\n    for i in range(12):\n        add(energy[:, i].mean(), f"energy_{i}_mean")\n        add(energy[:, i].std(), f"energy_{i}_std")\n        for q in (.1, .5, .9):\n            add(np.quantile(energy[:, i], q), f"energy_{i}_q{q}")\n    standardized = (raw - raw.mean(0)) / np.maximum(raw.std(0), 1e-12)\n    for i, j in ((0, 1), (0, 2), (1, 2)):\n        add(np.clip(np.mean(standardized[:, i] * standardized[:, j]), -1, 1), f"axis_corr_{i}_{j}")\n    boundaries = np.linspace(0, len(raw), 5).astype(int)\n    for quarter in range(4):\n        begin, end = boundaries[quarter:quarter + 2]\n        # Edge fallback is only for very short synthetic recordings.\n        rows = np.arange(begin, end) if end > begin else np.asarray([min(begin, len(raw) - 1)])\n        add(np.log1p(dynamic[rows].mean()), f"quarter_{quarter}_dynamic_log_mean")\n        add(energy[rows, 3].mean(), f"quarter_{quarter}_freeze_ratio_w128_mean")\n        add(energy[rows, 9].mean(), f"quarter_{quarter}_freeze_ratio_w512_mean")\n    x = np.asarray(values, np.float32)\n    assert x.shape == (112,) and len(names) == len(set(names)) == 112 and np.isfinite(x).all()\n    return x, names\n\n\ndef subject_weights(subjects):\n    subjects = np.asarray(subjects, str)\n    keys, counts = np.unique(subjects, return_counts=True)\n    lookup = dict(zip(keys, counts))\n    result = np.asarray([1 / lookup[value] for value in subjects], np.float64)\n    return result / result.mean()\n\n\ndef fit_gate(x, y, subjects, learner, seed):\n    if learner not in LEARNERS:\n        raise ValueError("Unknown context learner")\n    weight = subject_weights(subjects)\n    if len(np.unique(y)) == 1:\n        return float(np.average(y, weights=weight))\n    if learner == "linear":\n        model = Pipeline([("scale", StandardScaler()),\n                          ("model", LogisticRegression(**PARAMETERS[learner], random_state=seed))])\n        model.fit(x, y, model__sample_weight=weight)\n    else:\n        model = HistGradientBoostingClassifier(**PARAMETERS[learner], random_state=seed)\n        model.fit(x, y, sample_weight=weight)\n    return model\n\n\ndef predict_gate(model, x):\n    if isinstance(model, float):\n        return np.full(len(x), model, np.float64)\n    return model.predict_proba(x)[:, list(model.classes_).index(1)]\n\n\ndef train_gates(arrays, learner, directory=None):\n    prediction = np.full(arrays["presence"].shape, np.nan, np.float32)\n    with threadpool_limits(limits=2):\n        for fold in range(5):\n            training = arrays["fold"] != fold\n            heldout = ~training\n            train_subjects = set(arrays["subject"][training])\n            test_subjects = set(arrays["subject"][heldout])\n            assert train_subjects and test_subjects and train_subjects.isdisjoint(test_subjects)\n            if directory is not None:\n                fold_dir = directory / f"fold_{fold}"\n                fold_dir.mkdir(parents=True, exist_ok=True)\n                common.write_json(fold_dir / "training_receipt.json", {\n                    "targets": TARGETS, "learner": learner, "fold": fold,\n                    "training_subjects": sorted(train_subjects), "held_out_subjects": sorted(test_subjects),\n                    "training_recordings": arrays["recording_ids"][training].tolist(),\n                    "held_out_recordings": arrays["recording_ids"][heldout].tolist(),\n                    "outer_labels_used_for_fitting_or_stopping": False,\n                    "base_predictions_used_as_features": False, "early_stopping": False,\n                    "subject_balanced_training_weights": True, "parameters": PARAMETERS[learner]})\n            for index, target in enumerate(TARGETS):\n                model = fit_gate(arrays["x"][training], arrays["presence"][training, index],\n                                 arrays["subject"][training], learner, 731 + 1009 * fold + index)\n                prediction[heldout, index] = predict_gate(model, arrays["x"][heldout])\n                if directory is not None:\n                    joblib.dump(model, fold_dir / f"{target}.joblib")\n    assert np.isfinite(prediction).all() and ((prediction >= 0) & (prediction <= 1)).all()\n    return prediction\n\n\ndef apply_gate(reference, probability, floor=FLOOR):\n    reference, probability = np.asarray(reference), np.asarray(probability)\n    if reference.ndim != 2 or reference.shape[1] != 2 or probability.shape != (2,) or not 0 <= floor <= 1:\n        raise ValueError("Two native-row columns and two recording scores are required")\n    if not np.isfinite(probability).all() or ((probability < 0) | (probability > 1)).any():\n        raise ValueError("Invalid recording score")\n    if not np.isfinite(reference).all() or ((reference < 0) | (reference > 1)).any():\n        raise ValueError("Invalid native-row score")\n    factor = (floor + (1 - floor) * probability).astype(reference.dtype)\n    return np.asarray(reference * factor[None], dtype=reference.dtype)\n\n\ndef prepare_arrays(manifest, directory):\n    path = directory / "recording_features.npz"\n    if path.is_file():\n        with np.load(path, allow_pickle=False) as payload:\n            arrays = {key: payload[key].copy() for key in payload.files}\n        assert arrays["recording_ids"].tolist() == manifest.recording_id.tolist()\n        return arrays\n    xs, presence, fractions, lengths, feature_names = [], [], [], [], None\n    for number, row in enumerate(manifest.itertuples(index=False), 1):\n        frame = pd.read_csv(row.path, usecols=[*common.AXES, *TARGETS])\n        y = frame.loc[:, list(TARGETS)].to_numpy(np.uint8)\n        assert np.isin(y, [0, 1]).all()\n        x, names = recording_features(frame.loc[:, list(common.AXES)].to_numpy())\n        assert feature_names is None or names == feature_names\n        feature_names = names\n        xs.append(x); presence.append(y.any(0)); fractions.append(y.mean(0)); lengths.append(len(y))\n        if number % 100 == 0:\n            print(f"Recording-context features: {number}/{len(manifest)}", flush=True)\n    arrays = {"x": np.stack(xs), "presence": np.asarray(presence, np.uint8),\n        "fractions": np.asarray(fractions), "lengths": np.asarray(lengths),\n        "fold": manifest.fold.to_numpy(), "subject": manifest.subject.to_numpy(str),\n        "recording_ids": manifest.recording_id.to_numpy(str), "feature_names": np.asarray(feature_names)}\n    assert sum(lengths) == 6959762\n    np.savez_compressed(path, **arrays)\n    return arrays\n\n\ndef validate(run):\n    run = Path(run).resolve()\n    config = json.loads((run / "configuration.json").read_text())\n    assert config["source_sha256"] == hashlib.sha256(Path(__file__).read_bytes()).hexdigest()\n    assert config["native_source_sha256"] == hashlib.sha256(Path(native.__file__).read_bytes()).hexdigest()\n    assert config["common_source_sha256"] == hashlib.sha256(Path(common.__file__).read_bytes()).hexdigest()\n    assert run.name == hashlib.sha256(json.dumps(config, sort_keys=True).encode()).hexdigest()[:20]\n    raw, metadata = common.locate_data(); manifest = common.build_manifest(raw, metadata)\n    cols = ["recording_id", "subject", "fold", "source_bytes"]\n    saved = pd.read_csv(run / "frozen_tdcs_manifest.csv", dtype={"recording_id": str, "subject": str})\n    pd.testing.assert_frame_equal(saved[cols], manifest[cols], check_dtype=False)\n    assert config["population_sha256"] == hashlib.sha256(manifest[cols].to_csv(index=False, lineterminator="\\n").encode()).hexdigest()\n    assert config["retained_validation_sha256"] == hashlib.sha256((Path(config["retained_two"]) / "validation.json").read_bytes()).hexdigest()\n    reports = {}\n    for learner in LEARNERS:\n        candidate = run / learner\n        specification = json.loads((candidate / "configuration.json").read_text())\n        assert specification == {**config, "learner": learner, "parent_run": str(run)}\n        gate = pd.read_csv(candidate / "recording_scores.csv", dtype={"recording_id": str})\n        assert gate.recording_id.tolist() == manifest.recording_id.tolist()\n        assert gate.fold.tolist() == manifest.fold.tolist()\n        q = gate.loc[:, list(TARGETS)].to_numpy(np.float32)\n        assert np.isfinite(q).all() and ((q >= 0) & (q <= 1)).all()\n        assert {p.stem for p in (candidate / "oof").glob("*.npz")} == set(manifest.recording_id)\n        for fold in range(5):\n            receipt = json.loads((candidate / f"fold_{fold}/training_receipt.json").read_text())\n            train, heldout = manifest.loc[manifest.fold.ne(fold)], manifest.loc[manifest.fold.eq(fold)]\n            assert set(receipt["training_subjects"]) == set(train.subject)\n            assert set(receipt["held_out_subjects"]) == set(heldout.subject)\n            assert set(receipt["training_recordings"]) == set(train.recording_id)\n            assert set(receipt["held_out_recordings"]) == set(heldout.recording_id)\n            assert set(train.subject).isdisjoint(heldout.subject)\n            assert not any(receipt[key] for key in ("outer_labels_used_for_fitting_or_stopping", "base_predictions_used_as_features", "early_stopping"))\n            assert receipt["parameters"] == PARAMETERS[learner]\n        ys, ps = [], []\n        for index, row in enumerate(manifest.itertuples(index=False)):\n            reference = read_prediction(Path(config["retained_two"]), row.recording_id, TARGETS)\n            p = read_prediction(candidate, row.recording_id, TARGETS)\n            expected = apply_gate(reference, q[index], config["gate_floor"])\n            assert p.dtype == reference.dtype and p.tobytes() == expected.tobytes()\n            y = pd.read_csv(row.path, usecols=list(TARGETS)).loc[:, list(TARGETS)].to_numpy(np.uint8)\n            assert p.shape == y.shape and np.isin(y, [0, 1]).all()\n            ys.append(y); ps.append(p)\n        y, p = np.concatenate(ys), np.concatenate(ps)\n        assert len(y) == 6959762\n        values = {target: independent_ap(y[:, i], p[:, i]) for i, target in enumerate(TARGETS)}\n        values["two_target_mAP"] = float(np.mean(list(values.values())))\n        reported = pd.read_csv(candidate / "native_oof_metrics.csv").iloc[0]\n        assert all(abs(value - reported[key]) < 1e-10 for key, value in values.items())\n        report = {"status": "passed", "native_rows": len(y), "recordings": 815, "subjects": 61,\n            "subject_disjoint_folds": 5, "targets": TARGETS, "independent_rank_based_AP": values,\n            "all_negative_rows_retained": True, "outer_labels_not_used_for_fitting": True,\n            "base_OOF_predictions_not_used_in_gate_fitting": True,\n            "no_three_target_mAP_claim": True, "confidence_calibrated": False,\n            "independent_generalization_established": False, "retained_heads_overwritten": False,\n            "competition_submission_created": False}\n        common.write_json(candidate / "validation.json", report)\n        reports[learner] = report\n    common.write_json(run / "validation.json", {"status": "passed", "candidates": reports})\n    print(json.dumps(reports, indent=2), flush=True)\n    return reports\n\n\ndef run_experiment(output_root=None):\n    raw, metadata = common.locate_data(); manifest = common.build_manifest(raw, metadata)\n    registry = json.loads((ROOT / "results/tdcs_two_target_current_best.json").read_text())\n    reference = Path(registry["native_oof_directory"])\n    assert json.loads((reference / "validation.json").read_text())["status"] == "passed"\n    cols = ["recording_id", "subject", "fold", "source_bytes"]\n    config = {"targets": TARGETS, "learners": LEARNERS, "parameters": PARAMETERS,\n        "source_sha256": hashlib.sha256(Path(__file__).read_bytes()).hexdigest(),\n        "native_source_sha256": hashlib.sha256(Path(native.__file__).read_bytes()).hexdigest(),\n        "common_source_sha256": hashlib.sha256(Path(common.__file__).read_bytes()).hexdigest(),\n        "population_sha256": hashlib.sha256(manifest[cols].to_csv(index=False, lineterminator="\\n").encode()).hexdigest(),\n        "runtime": {"sklearn": sklearn.__version__, "numpy": np.__version__, "scipy": scipy.__version__},\n        "retained_two": str(reference),\n        "retained_validation_sha256": hashlib.sha256((reference / "validation.json").read_bytes()).hexdigest(),\n        "gate_floor": FLOOR, "score_rule": "retained_row_score * (0.05 + 0.95 * file_presence_score)",\n        "seed_rule": "731 + 1009 * outer_fold + target_index",\n        "training_features": "112 label-free raw-sensor recording summaries; no IDs, demographics or base OOF scores",\n        "subject_balanced_training_weights": True, "outer_labels_used_for_fitting_or_stopping": False,\n        "feature_normalization_fitted_only_on_training_subjects": True,\n        "prior_models_unchanged": True, "competition_submission_created": False,\n        "selection_scope": "Two fixed file classifiers; subsequent fixed coarse blends are exploratory on reused development folds"}\n    fingerprint = hashlib.sha256(json.dumps(config, sort_keys=True).encode()).hexdigest()[:20]\n    output_root = Path(output_root or ROOT / "results/tdcs_recording_context").resolve()\n    run = output_root / fingerprint; run.mkdir(parents=True, exist_ok=True)\n    common.write_json(run / "configuration.json", config)\n    manifest.to_csv(run / "frozen_tdcs_manifest.csv", index=False)\n    arrays = prepare_arrays(manifest, run)\n    metrics = []\n    for learner in LEARNERS:\n        candidate = run / learner; (candidate / "oof").mkdir(parents=True, exist_ok=True)\n        common.write_json(candidate / "configuration.json", {**config, "learner": learner, "parent_run": str(run)})\n        manifest.to_csv(candidate / "frozen_tdcs_manifest.csv", index=False)\n        q = train_gates(arrays, learner, candidate)\n        gate = manifest[["recording_id", "fold"]].copy()\n        for index, target in enumerate(TARGETS):\n            gate[target] = q[:, index]\n        gate.to_csv(candidate / "recording_scores.csv", index=False, float_format="%.9g")\n        ys, ps = [], []\n        for index, row in enumerate(manifest.itertuples(index=False)):\n            p = apply_gate(read_prediction(reference, row.recording_id, TARGETS), q[index])\n            np.savez_compressed(candidate / "oof" / f"{row.recording_id}.npz", prediction=p, targets=np.asarray(TARGETS))\n            ys.append(pd.read_csv(row.path, usecols=list(TARGETS)).loc[:, list(TARGETS)].to_numpy(np.uint8))\n            ps.append(p)\n        y, p = np.concatenate(ys), np.concatenate(ps)\n        values = {target: float(average_precision_score(y[:, i], p[:, i])) for i, target in enumerate(TARGETS)}\n        values["two_target_mAP"] = float(np.mean(list(values.values())))\n        pd.DataFrame([values]).to_csv(candidate / "native_oof_metrics.csv", index=False)\n        metrics.append({"learner": learner, **values})\n        print(json.dumps(metrics[-1]), flush=True)\n    pd.DataFrame(metrics).to_csv(run / "candidate_metrics.csv", index=False)\n    common.write_json(output_root / "latest_run.json", {"run_dir": str(run), "status": "awaiting_independent_validation"})\n    reports = validate(run)\n    common.write_json(run / "decision.json", {"run_dir": str(run), "status": "complete", "candidates": reports,\n        "retained_heads_unchanged": True, "confidence_calibrated": False,\n        "independent_generalization_established": False, "competition_submission_created": False})\n    common.write_json(output_root / "latest_run.json", {"run_dir": str(run), "status": "complete"})\n    return run\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--output-root", type=Path)\n    parser.add_argument("--validate", type=Path)\n    args = parser.parse_args()\n    validate(args.validate) if args.validate else run_experiment(args.output_root)\n'}
FROZEN_GATE = None
if RUN_FULL:
    gate_path = Path(os.environ.get(
        'TDCS_FROZEN_GATE_FILE', str(PROJECT_ROOT / 'results' / 'tdcs_walking_conditional_gpu_gate_payload.json')
    )).expanduser()
    if not gate_path.is_file():
        raise FileNotFoundError(
            'Set TDCS_FROZEN_GATE_FILE to the authorized private gate JSON (see release notes).'
        )
    FROZEN_GATE = json.loads(gate_path.read_text(encoding='utf-8'))
    if hashlib.sha256(json.dumps(FROZEN_GATE, sort_keys=True).encode()).hexdigest() != 'bedc03165f2a8724e775c2f3e1fd4ab13bf5cf23e677710fbf3aa5fa6a601da0':
        raise ValueError('The private gate JSON does not match the pinned v31 payload.')
for name, source in SOURCES.items():
    assert hashlib.sha256(source.encode()).hexdigest() == HASHES[name]
    module = types.ModuleType(name)
    module.__file__ = str(PROJECT_ROOT / 'scripts' / (name + '.py'))
    module.SOURCE_SHA256 = HASHES[name]
    module.FROZEN_FEATURE_SOURCES = RECIPES
    module.FROZEN_GATE_PAYLOAD = FROZEN_GATE
    sys.modules[name] = module
    exec(compile(source, module.__file__, 'exec'), module.__dict__)
runner = sys.modules['tdcs_suite_runner']
print('Pinned source modules:', len(HASHES), flush=True)

Pinned source modules: 16


In [3]:
RECOVERY_SOURCE = '"""v26 reader-only recovery: bounded Parquet batches, frozen training recipes.\n\nDo not edit the original four suite modules: their hashes identify completed v1\nmodels. The explicit recovery source hash is added to each new configuration.\n"""\nfrom __future__ import annotations\n\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nimport tdcs_suite_data as data\nimport tdcs_suite_runner as runner\n\n\ndef read_prefix(path, limit=32768):\n    path=Path(path)\n    if limit<1024 or limit>32768: raise ValueError(\'Bounded daily prefix required\')\n    if path.suffix.lower()==\'.csv\':\n        return pd.read_csv(path,nrows=limit,usecols=[\'Time\',*data.AXES]), {\'format\':\'csv\',\'time_source\':\'Time_column\'}\n    if path.suffix.lower()!=\'.parquet\': raise ValueError(\'Unsupported unlabeled format\')\n    import pyarrow.parquet as pq\n    parquet=pq.ParquetFile(path); names=parquet.schema_arrow.names\n    if not set(data.AXES)<=set(names): raise ValueError(\'Parquet accelerometer columns are missing\')\n    metadata=parquet.schema_arrow.pandas_metadata or {}\n    time_name=\'Time\' if \'Time\' in names else None\n    if time_name is None:\n        for field in metadata.get(\'columns\',[]):\n            if field.get(\'name\')==\'Time\' and field.get(\'field_name\') in names:\n                time_name=field[\'field_name\']\n    columns=[*data.AXES,*([time_name] if time_name else [])]\n    batch=next(parquet.iter_batches(batch_size=limit,columns=columns),None)\n    if batch is None: raise ValueError(\'Empty unlabeled Parquet recording\')\n    frame=batch.to_pandas(ignore_metadata=True).iloc[:limit].copy()\n    if time_name is not None:\n        frame=frame.rename(columns={time_name:\'Time\'}); source=\'Time_column\'\n    else:\n        ranges=[value for value in metadata.get(\'index_columns\',[]) if isinstance(value,dict) and value.get(\'kind\')==\'range\']\n        if len(ranges)!=1 or ranges[0].get(\'step\')!=1:\n            raise ValueError(\'Parquet needs Time or an explicit consecutive RangeIndex; refusing guessed timing\')\n        start=ranges[0][\'start\']; frame[\'Time\']=start+np.arange(len(frame)); source=\'declared_Parquet_RangeIndex\'\n    return frame[[\'Time\',*data.AXES]], {\'format\':\'parquet\',\'time_source\':source,\n        \'total_parquet_rows\':parquet.metadata.num_rows,\'rows_read\':len(frame)}\n\n\ndef mapped_paths(root, daily):\n    paths={}\n    for path in sorted(Path(root).rglob(\'*\')):\n        if path.suffix.lower() not in (\'.csv\',\'.parquet\') or path.stem not in set(daily.Id): continue\n        if path.stem in paths: raise ValueError(\'Ambiguous unlabeled ID across files/formats\')\n        paths[path.stem]=path\n    eligible=daily[daily.Id.isin(paths)].sort_values(\'Id\').head(64)\n    if eligible.empty: raise FileNotFoundError(\'No mapped CSV/Parquet daily records in attached competition input\')\n    return paths,eligible\n\n\ndef external_pool(context, output):\n    if context[\'smoke_only\']:\n        return context[\'external\']\n    if context[\'external\'] is not None: return context[\'external\']\n    path=data.csv_path(\'daily_metadata.csv\'); daily=pd.read_csv(path,dtype={\'Id\':str,\'Subject\':str})\n    if daily.Id.duplicated().any() or daily.Subject.isna().any() or daily.Subject.str.strip().eq(\'\').any():\n        raise ValueError(\'Unique daily IDs and known subject mappings are required\')\n    paths,eligible=mapped_paths(path.parent/\'unlabeled\',daily)\n    print(\'Mapped daily recordings:\',len(paths),\'bounded records selected:\',len(eligible),flush=True)\n    xs=[]; subjects=[]; records=[]; provenance=[]\n    for i,row in enumerate(eligible.itertuples(index=False)):\n        frame,info=read_prefix(paths[row.Id]); signal=data.daily_signal(frame)\n        _,features,_=data.frozen.shared.features.recording_features(signal)\n        xs.append(features); subjects.append(np.full(len(features),row.Subject)); records.append(np.full(len(features),i,np.int32))\n        provenance.append({\'Id\':row.Id,\'Subject\':row.Subject,\'source_bytes\':paths[row.Id].stat().st_size,\n            \'used_prefix_sha256\':data.array_digest(frame[[\'Time\',*data.AXES]].to_numpy(np.float64)),\n            \'feature_rows\':len(features),\'resampled_native_rows\':len(signal),**info})\n        if (i+1)%8==0: print(f\'Parquet recovery prefixes: {i+1}/{len(eligible)}\',flush=True)\n    pool={\'x\':np.concatenate(xs),\'subject\':np.concatenate(subjects),\'record\':np.concatenate(records)}\n    output=Path(output); output.mkdir(parents=True,exist_ok=True)\n    cache=output/\'unlabeled_feature_pool.npz\'; np.savez_compressed(cache,**pool)\n    recovery_hash=globals().get(\'SOURCE_SHA256\') or data.digest(__file__)\n    context[\'source_inputs\'][\'unlabeled_reader_recovery\']={\'reader_sha256\':recovery_hash,\n        \'metadata_sha256\':data.digest(path),\'cache_sha256\':data.digest(cache),\'input_rate_hz\':100,\n        \'output_rate_hz\':128,\'g_to_ms2\':9.80665,\'bounded_input_rows\':32768,\'bounded_output_rows\':16384,\n        \'only_reader_changed_training_settings_frozen\':True}\n    runner.write(output/\'unlabeled_pool_provenance.json\',{\'metadata_sha256\':data.digest(path),\n        \'cache_sha256\':data.digest(cache),\'records\':provenance,\'reader_sha256\':recovery_hash,\n        \'input_rate_hz\':100,\'output_rate_hz\':128,\'g_to_ms2\':9.80665,\n        \'heldout_subjects_excluded_separately_per_fold\':True,\'no_unlabeled_targets_read\':True})\n    context[\'external\']=pool\n    return pool\n\n\ndef reuse_v1(session,id,status):\n    assert id not in (\'S03\',\'S04\') and status[\'id\']==id and status[\'status\']==\'complete\' and not status[\'smoke_only\']\n    row={**status,\'status\':\'complete_reused_v1\',\'source_version\':1,\n        \'artifacts_origin\':\'private Kaggle sensor suite version 1; preserved locally\',\n        \'not_retrained_in_reader_recovery\':True}\n    session[\'results\'].append(row); runner.write(session[\'output\']/(id+\'_status.json\'),row)\n    return row\n\n\ndef finish(session):\n    rows=session[\'results\']; expected={s[\'id\'] for s in runner.CATALOG if s[\'suite\']==\'sensor\'}\n    done=[r[\'id\'] for r in rows if r[\'status\'] in (\'complete\',\'complete_reused_v1\')]\n    result={\'status\':\'all_methods_completed_validation_pending\' if set(done)==expected else \'partial_coverage\',\n        \'expected_tests\':sorted(expected),\'attempted_tests\':[r[\'id\'] for r in rows],\'completed_tests\':done,\n        \'newly_executed_tests\':[r[\'id\'] for r in rows if r[\'status\']==\'complete\'],\n        \'reused_v1_tests\':[r[\'id\'] for r in rows if r[\'status\']==\'complete_reused_v1\'],\n        \'missing_tests\':sorted(expected-{r[\'id\'] for r in rows}),\'smoke_only\':session[\'cfg\'].smoke,\n        \'retained_heads_changed\':False,\'competition_submission_created\':False,\'automatic_promotion\':False,\n        \'confidence_calibrated\':False,\'independent_generalization_established\':False}\n    runner.write(session[\'output\']/\'suite_completion.json\',result)\n    pd.DataFrame([{k:v for k,v in r.items() if k!=\'traceback\'} for r in rows]).to_csv(session[\'output\']/\'suite_summary.csv\',index=False)\n    return result\n'
RECOVERY_SHA256 = '19105d2af597c40c6a56016b92a23b16c2b1434ec1cee327b1a8d94285f31440'
assert hashlib.sha256(RECOVERY_SOURCE.encode()).hexdigest() == RECOVERY_SHA256
recovery = types.ModuleType('tdcs_suite_unlabeled_parquet')
recovery.__file__ = str(PROJECT_ROOT / 'scripts/tdcs_suite_unlabeled_parquet.py')
recovery.SOURCE_SHA256 = RECOVERY_SHA256
sys.modules[recovery.__name__] = recovery
exec(compile(RECOVERY_SOURCE, recovery.__file__, 'exec'), recovery.__dict__)
runner.data.external_pool = recovery.external_pool
PRIOR_V1_RESULTS = {'S00': {'id': 'S00', 'title': 'Sensor-only CatBoost control', 'suite': 'sensor', 'status': 'complete', 'StartHesitation': 0.5465950473623613, 'Walking': 0.1227910261941098, 'two_target_mAP': 0.33469303677823553, 'run_directory': '/kaggle/working/tdcs_sensor_comprehensive_v26/methods/S00/fc78883f66e9f95a589e', 'native_prediction_sha256': '0ebc9b3e2b85da20ddedca5643cbf4482a4e7cb9667803f7135c5c6bb1fa6da7', 'independent_validation_pending': True, 'smoke_only': False, 'actual_grid_rows': 436128}, 'S01': {'id': 'S01', 'title': 'LightGBM: explicit Kaggle CPU training', 'suite': 'sensor', 'status': 'complete', 'StartHesitation': 0.2276165393770844, 'Walking': 0.1225309632076005, 'two_target_mAP': 0.17507375129234246, 'run_directory': '/kaggle/working/tdcs_sensor_comprehensive_v26/methods/S01/f03562c5c87f7ce2fb73', 'native_prediction_sha256': '6360a939decb77908eab5460e464be2d1a79f7ad4da5f36f4f5ebf95c315bc7f', 'independent_validation_pending': True, 'smoke_only': False, 'actual_grid_rows': 436128}, 'S02': {'id': 'S02', 'title': 'Supervised sensor CNN control for pretraining', 'suite': 'sensor', 'status': 'complete', 'StartHesitation': 0.3227282610066557, 'Walking': 0.1295775166094419, 'two_target_mAP': 0.2261528888080488, 'run_directory': '/kaggle/working/tdcs_sensor_comprehensive_v26/methods/S02/cc787152e2838047b8ff', 'native_prediction_sha256': '99822e3e64527d0e8f8038c3688d141db7a60964bea97f837d32d796b5e8e05b', 'independent_validation_pending': True, 'smoke_only': False, 'actual_grid_rows': 436128}, 'S05': {'id': 'S05', 'title': 'Sensor-only binary XGBoost control for ranking', 'suite': 'sensor', 'status': 'complete', 'StartHesitation': 0.38910059070965347, 'Walking': 0.13148538852861444, 'two_target_mAP': 0.26029298961913394, 'run_directory': '/kaggle/working/tdcs_sensor_comprehensive_v26/methods/S05/d029e82637632e3018fa', 'native_prediction_sha256': '40b5ac3dab705ef2bf906c8afba7067f46ff4ecdbe384cc926dc1c098745e3df', 'independent_validation_pending': True, 'smoke_only': False, 'actual_grid_rows': 436128}, 'S06': {'id': 'S06', 'title': 'Per-record MAP-focused tree ranking', 'suite': 'sensor', 'status': 'complete', 'StartHesitation': 0.054938761623870236, 'Walking': 0.09960435074066686, 'two_target_mAP': 0.07727155618226855, 'run_directory': '/kaggle/working/tdcs_sensor_comprehensive_v26/methods/S06/19dc71c060502b1b36a4', 'native_prediction_sha256': 'ea7c7832c65fe2af8cb0983ccce74d90c3f922f161d035135b36c62a8d496dcf', 'independent_validation_pending': True, 'smoke_only': False, 'actual_grid_rows': 436128}}
print('Reader recovery pinned:', RECOVERY_SHA256)

Reader recovery pinned: 19105d2af597c40c6a56016b92a23b16c2b1434ec1cee327b1a8d94285f31440


### 3. Load and audit inputs
Recording fields are Visit, Test and Medication. Clinical fields are joined
on an exact specified visit, or the unique subject-level row with blank Visit.
IDs map records and folds only; they are not fitting predictors. Events supply
the optional Kinetic training target, never event start/end input features.
`tasks.csv` contains no TDCS intervals and is intentionally not invented.
TDCS integer Time indices are divided by 128 before joining event seconds.
External stages read at most 64 sorted mapped daily records, 32,768 CSV rows
each; convert daily g units to m/s², resample the documented 100 Hz to 128 Hz
and retain at most 16,384 native rows per record. Time indices are not seconds.
Subject mappings exclude outer-held-out subjects even from unlabeled data.
Sources: [official acquisition documentation](https://www.kaggle.com/competitions/tlvmc-parkinsons-freezing-gait-prediction/overview/evaluation)
and [original fourth-place preprocessing](https://github.com/Zinxira/tlvmc-parkinsons-fog-prediction-4th-place-solution/blob/main/preprocessing.py).

The preceding legacy setup describes reused v26/v27 stages. The new v28 pair reads labelled train/defog CSVs, not the bounded daily unlabeled pool; its rate/unit and annotation-mask contract is specified above.

In [4]:
FOLLOWUP_SOURCE = '"""Single-head v27 followups; frozen v26 data/recipes remain unchanged.\n\nWalking: true one-output GRUs, all versus local sensors plus recording fields.\nHesitation: three pseudo-row weights with the exact same cached teacher/rows.\nReal fitting is Kaggle CUDA only. Scores never promote a retained predictor.\n"""\nfrom __future__ import annotations\n\nimport copy\nimport gc\nimport hashlib\nimport json\nimport time\nimport traceback\nfrom dataclasses import asdict\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom sklearn.metrics import average_precision_score\n\nimport tdcs_suite_data as data\nimport tdcs_suite_models as trees\nimport tdcs_suite_neural as neural\nimport tdcs_suite_runner as base\n\nSPECS={\n    \'M19\':{\'id\':\'M19\',\'suite\':\'metadata\',\'target\':\'Walking\',\'kind\':\'walking_gru\',\'encoding\':\'record_metadata\'},\n    \'M20\':{\'id\':\'M20\',\'suite\':\'metadata\',\'target\':\'Walking\',\'kind\':\'walking_gru\',\'encoding\':\'local_metadata\'},\n    \'S07\':{\'id\':\'S07\',\'suite\':\'sensor\',\'target\':\'StartHesitation\',\'kind\':\'pseudo\',\'encoding\':\'sensor\',\'pseudo_weight\':.03},\n    \'S08\':{\'id\':\'S08\',\'suite\':\'sensor\',\'target\':\'StartHesitation\',\'kind\':\'pseudo\',\'encoding\':\'sensor\',\'pseudo_weight\':.10},\n    \'S09\':{\'id\':\'S09\',\'suite\':\'sensor\',\'target\':\'StartHesitation\',\'kind\':\'pseudo\',\'encoding\':\'sensor\',\'pseudo_weight\':.30},\n}\n\n\ndef source_hash():\n    return globals().get(\'SOURCE_SHA256\') or data.digest(__file__)\n\n\ndef fingerprint(config):\n    return hashlib.sha256(json.dumps(config,sort_keys=True).encode()).hexdigest()[:20]\n\n\ndef check_time(session):\n    if time.monotonic()>=session[\'deadline\']: raise TimeoutError(\'Followup suite runtime budget reached\')\n\n\ndef assert_fit_scope(session):\n    if not session[\'cfg\'].smoke and (not Path(\'/kaggle/working\').exists() or not torch.cuda.is_available()):\n        raise RuntimeError(\'Real followup fitting requires Kaggle CUDA; no local or CPU fallback\')\n\n\ndef walking_model(features,width):\n    model=neural.Network(features,\'bigru\',width,False)\n    # Preserve the initial Walking row of the original two-output model. This\n    # avoids an unrelated random output initialization in the loss-scope test.\n    original=model.output\n    with torch.random.fork_rng(devices=[]): head=nn.Linear(original.in_features,1)\n    with torch.no_grad():\n        head.weight.copy_(original.weight[1:2]); head.bias.copy_(original.bias[1:2])\n    model.output=head\n    return model\n\n\ndef load_walking(path,device):\n    saved=torch.load(path,map_location=device,weights_only=False)\n    assert saved[\'targets\']==[\'Walking\']\n    model=walking_model(saved[\'features\'],saved[\'width\']).to(device)\n    model.load_state_dict(saved[\'model\'])\n    return model.eval()\n\n\ndef predict_walking(model,x,prep,cfg,device):\n    model.eval(); values=neural.transform(x,prep)\n    starts=list(range(-cfg.window//2,len(x),cfg.window//2)); taper=np.maximum(np.hanning(cfg.window),.1)\n    total=np.zeros(len(x),np.float64); denominator=np.zeros(len(x),np.float64)\n    with torch.no_grad():\n        for offset in range(0,len(starts),cfg.batch_size):\n            group=starts[offset:offset+cfg.batch_size]\n            windows=np.stack([values[np.clip(s+np.arange(cfg.window),0,len(x)-1)] for s in group])\n            probability=torch.sigmoid(model(torch.from_numpy(windows).to(device)))[...,0].cpu().numpy()\n            for start,p in zip(group,probability):\n                inside=(start+np.arange(cfg.window)>=0)&(start+np.arange(cfg.window)<len(x))\n                rows=start+np.flatnonzero(inside)\n                total[rows]+=p[inside]*taper[inside]; denominator[rows]+=taper[inside]\n    assert (denominator>0).all()\n    return (total/denominator).astype(np.float32)\n\n\ndef train_walking(session,spec,x,fold,directory,key):\n    c,cfg=session[\'context\'],session[\'cfg\']; a=c[\'arrays\']; fitting=a[\'fold\']!=fold\n    device=torch.device(\'cpu\' if cfg.smoke else \'cuda:0\'); seed=cfg.seed+1009*fold\n    torch.manual_seed(seed)\n    if device.type==\'cuda\': torch.cuda.manual_seed_all(seed)\n    labels=a[\'y\'][:,[1]].astype(np.float32)\n    prep=neural.preprocessing(x,labels,fitting); np.savez_compressed(directory/\'preprocessing.npz\',**prep)\n    values=neural.transform(x,prep); model=walking_model(x.shape[1],cfg.width).to(device)\n    ema=copy.deepcopy(model).eval(); groups,probability=neural.sampling_groups(a,fitting)\n    rng=np.random.default_rng(seed); optimizer=torch.optim.AdamW(model.parameters(),lr=.0008,weight_decay=.01)\n    positive=torch.from_numpy(prep[\'positive_weight\']).to(device); trace=[]\n    model.train()\n    for epoch in range(cfg.neural_epochs):\n        for _ in range(cfg.steps_per_epoch):\n            check_time(session)\n            bx,by,mask,crops=neural.batch(groups,probability,rng,values,labels,cfg); trace.extend(crops)\n            logits=model(torch.from_numpy(bx).to(device))[...,0]\n            truth=torch.from_numpy(by[...,0]).to(device); valid=torch.from_numpy(mask).to(device).float()\n            loss=(F.binary_cross_entropy_with_logits(logits,truth,pos_weight=positive,reduction=\'none\')*valid).sum()/valid.sum().clamp_min(1)\n            if not torch.isfinite(loss): raise FloatingPointError(\'Nonfinite Walking loss\')\n            optimizer.zero_grad(set_to_none=True); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1.)\n            optimizer.step()\n            with torch.no_grad():\n                for dst,src in zip(ema.parameters(),model.parameters()): dst.lerp_(src,.005)\n        torch.save({\'model\':model.state_dict(),\'ema\':ema.state_dict(),\'optimizer\':optimizer.state_dict(),\n            \'completed_epochs\':epoch+1,\'fingerprint\':key},directory/\'epoch_checkpoint.pt\')\n        print(f"{spec[\'id\']} fold {fold}: epoch {epoch+1}/{cfg.neural_epochs}",flush=True)\n    np.savez_compressed(directory/\'training_trace.npz\',crops=np.asarray(trace,np.int64))\n    torch.save({\'model\':ema.state_dict(),\'features\':x.shape[1],\'width\':cfg.width,\'fingerprint\':key,\n        \'fold\':fold,\'targets\':[\'Walking\']},directory/\'model.pt\')\n    result=np.empty(int((~fitting).sum()),np.float32); where=np.full(len(x),-1,np.int64); where[~fitting]=np.arange((~fitting).sum())\n    for record in np.unique(a[\'record\'][~fitting]):\n        rows=np.flatnonzero((a[\'record\']==record)&~fitting)\n        result[where[rows]]=predict_walking(ema,x[rows],prep,cfg,device)\n    return result,{\'backend\':str(device),\'seed\':seed,\'optimizer_updates\':cfg.neural_epochs*cfg.steps_per_epoch,\n        \'fixed_epochs\':cfg.neural_epochs,\'final_weights\':\'EMA_0.995\',\'output_heads\':[\'Walking\'],\n        \'SH_or_Turn_losses_used\':False,\'initial_Walking_head_preserved_from_two_head_initialization\':True}\n\n\ndef shared_teacher(session,x,pool,fold):\n    c,cfg=session[\'context\'],session[\'cfg\']; a=c[\'arrays\']; seed=cfg.seed+1009*fold\n    config={\'scope\':\'one SH teacher shared by all pseudo-weight tests\',\'fold\':fold,\'seed\':seed,\n        \'tree_rounds\':cfg.tree_rounds,\'feature_sha256\':data.array_digest(x),\n        \'pool_sha256\':{k:data.array_digest(v) for k,v in pool.items()},\n        \'sources\':base.source_hashes(),\'followup_source_sha256\':source_hash(),\'smoke_only\':cfg.smoke}\n    root=session[\'output\']/\'shared_teachers\'/fingerprint(config)/f\'fold_{fold}\'; root.mkdir(parents=True,exist_ok=True)\n    receipt_path=root/\'teacher_receipt.json\'\n    if receipt_path.is_file():\n        receipt=json.loads(receipt_path.read_text()); assert receipt[\'configuration\']==config\n        for name,sha in receipt[\'artifact_sha256\'].items(): assert data.digest(root/name)==sha\n    else:\n        if any(root.iterdir()): raise ValueError(\'Incomplete shared teacher; preserve it and use a fresh output root\')\n        chosen,weights,probability=data.frozen.parent.training_plan(a[\'y\'][:,0],a[\'fold\'],a[\'subject\'],fold,seed,4,.5)\n        model,backend=trees.fit_tree(\'catboost\',x[chosen],a[\'y\'][chosen,0],weights,seed,cfg,not cfg.smoke)\n        model_path=trees.save_tree(model,\'catboost\',root/\'teacher\')\n        heldout_subjects=set(a[\'subject\'][a[\'fold\']==fold]); eligible=data.external_training_indices(pool,heldout_subjects)\n        p=trees.predict_tree(model,\'catboost\',pool[\'x\'][eligible])\n        selected=np.flatnonzero((p>=.90)|(p<=.01))\n        if len(selected)>40000: selected=np.random.default_rng(seed).choice(selected,40000,replace=False)\n        np.savez_compressed(root/\'rows.npz\',indices=chosen,weights=weights,pseudo_indices=eligible[selected],\n            pseudo_labels=(p[selected]>=.90).astype(np.uint8),teacher_probabilities=p[selected])\n        receipt={\'configuration\':config,\'backend\':backend,\'model_file\':model_path.name,\n            \'sample_probability\':probability,\'artifact_sha256\':{p.name:data.digest(p) for p in root.iterdir() if p.is_file()}}\n        base.write(receipt_path,receipt)\n    with np.load(root/\'rows.npz\',allow_pickle=False) as saved: rows={k:saved[k] for k in saved.files}\n    return root,receipt,rows\n\n\ndef train_pseudo(session,spec,x,pool,fold,directory,key):\n    c,cfg=session[\'context\'],session[\'cfg\']; a=c[\'arrays\']; seed=cfg.seed+1009*fold\n    teacher,receipt,rows=shared_teacher(session,x,pool,fold)\n    chosen,weights=rows[\'indices\'],rows[\'weights\']; selected=rows[\'pseudo_indices\']; pseudo=rows[\'pseudo_labels\']\n    assert (a[\'fold\'][chosen]!=fold).all()\n    assert not set(pool[\'subject\'][selected])&set(a[\'subject\'][a[\'fold\']==fold])\n    if len(selected):\n        w=np.r_[weights,np.full(len(selected),spec[\'pseudo_weight\'])]; w/=w.mean()\n        model,backend=trees.fit_tree(\'catboost\',np.vstack([x[chosen],pool[\'x\'][selected]]),\n            np.r_[a[\'y\'][chosen,0],pseudo],w,seed,cfg,not cfg.smoke)\n    else:\n        model=trees.load_tree(\'catboost\',teacher/receipt[\'model_file\']); backend={\'backend\':\'shared_teacher_no_qualified_rows\'}\n    path=trees.save_tree(model,\'catboost\',directory/\'StartHesitation\')\n    np.savez_compressed(directory/\'training_sample.npz\',**rows)\n    result=trees.predict_tree(trees.load_tree(\'catboost\',path),\'catboost\',x[a[\'fold\']==fold])\n    return result,{\'backend\':backend,\'seed\':seed,\'model_file\':path.name,\'output_heads\':[\'StartHesitation\'],\n        \'pseudo_weight\':spec[\'pseudo_weight\'],\'pseudo_rows_added\':len(selected),\'positive_pseudo_labels\':int(pseudo.sum()),\n        \'shared_teacher_directory\':str(teacher),\'shared_teacher_receipt_sha256\':data.digest(teacher/\'teacher_receipt.json\'),\n        \'shared_teacher_rows_sha256\':data.digest(teacher/\'rows.npz\'),\'teacher_changed_between_weight_tests\':False}\n\n\ndef native_scores(c,grid,target):\n    values=np.empty(len(c[\'native_y\']),np.float32); a=c[\'arrays\']\n    for i,row in enumerate(c[\'manifest\'].itertuples(index=False)):\n        mask=a[\'record\']==i; left,right=c[\'native_offsets\'][i:i+2]\n        values[left:right]=np.interp(np.arange(right-left),a[\'grid\'][mask],grid[mask]).astype(np.float32)\n        if target==\'Walking\': values[left:right]*=(np.asarray([c[\'gates\'][i]],np.float32)*.95+.05)[0]\n    ap=float(average_precision_score(c[\'native_y\'][:,data.TARGETS.index(target)],values))\n    return ap,data.array_digest(values)\n\n\ndef run_method(session,id):\n    spec=SPECS[id]; assert spec[\'suite\']==session[\'suite\']; cfg=session[\'cfg\']; c=session[\'context\']; a=c[\'arrays\']\n    status={\'id\':id,\'title\':id+\' \'+spec[\'target\']+\' \'+spec[\'kind\'],\'suite\':session[\'suite\'],\'status\':\'pending\',\'target_scope\':[spec[\'target\']]}\n    try:\n        assert_fit_scope(session); check_time(session)\n        x=data.input_features(c,spec[\'encoding\'])\n        pool=data.external_pool(c,session[\'output\']) if spec[\'kind\']==\'pseudo\' else None\n        config={\'specification\':spec,\'configuration\':asdict(cfg),\'base_sources\':base.source_hashes(),\n            \'followup_source_sha256\':source_hash(),\'feature_sha256\':data.array_digest(x),\'feature_count\':x.shape[1],\n            \'external_pool_sha256\':{k:data.array_digest(v) for k,v in pool.items()} if pool is not None else None,\n            \'source_inputs\':c[\'source_inputs\'],\'runtime\':session[\'runtime\'],\'targets\':[spec[\'target\']],\n            \'recordings\':c[\'manifest\'].recording_id.tolist(),\'subjects\':c[\'manifest\'].subject.tolist(),\n            \'folds\':c[\'manifest\'].fold.tolist(),\'smoke_only\':cfg.smoke,\'outer_stopping\':False,\'automatic_promotion\':False}\n        key=fingerprint(config); run=session[\'output\']/\'methods\'/id/key; run.mkdir(parents=True,exist_ok=True)\n        base.write(run/\'configuration.json\',config); grid=np.full(len(x),np.nan,np.float32)\n        for fold in cfg.folds:\n            check_time(session); directory=run/f\'fold_{fold}\'; directory.mkdir(exist_ok=True)\n            receipt_path=directory/\'training_receipt.json\'; saved=directory/\'heldout_grid.npz\'; heldout=a[\'fold\']==fold\n            if receipt_path.is_file() and saved.is_file():\n                receipt=json.loads(receipt_path.read_text()); assert receipt[\'fingerprint\']==key\n                for name,sha in receipt[\'artifact_sha256\'].items(): assert data.digest(directory/name)==sha\n                with np.load(saved,allow_pickle=False) as payload:\n                    np.testing.assert_array_equal(payload[\'indices\'],np.flatnonzero(heldout)); grid[heldout]=payload[\'prediction\']\n                continue\n            if any(directory.iterdir()): raise ValueError(\'Incomplete followup fold; preserve it and use a fresh output root\')\n            parts,receipt=(train_pseudo(session,spec,x,pool,fold,directory,key) if spec[\'kind\']==\'pseudo\' else\n                train_walking(session,spec,x,fold,directory,key))\n            assert parts.shape==(int(heldout.sum()),) and np.isfinite(parts).all() and ((parts>=0)&(parts<=1)).all()\n            np.savez_compressed(saved,indices=np.flatnonzero(heldout),prediction=parts); grid[heldout]=parts\n            receipt.update(fingerprint=key,fold=fold,fitting_subjects=sorted(map(str,set(a[\'subject\'][~heldout]))),\n                heldout_subjects=sorted(map(str,set(a[\'subject\'][heldout]))),outer_labels_used_for_training_or_stopping=False,\n                artifact_sha256={p.name:data.digest(p) for p in directory.iterdir() if p.is_file() and p.name!=\'training_receipt.json\'})\n            assert set(receipt[\'fitting_subjects\']).isdisjoint(receipt[\'heldout_subjects\'])\n            base.write(receipt_path,receipt); print(id,\'fold\',fold,\'complete\',flush=True)\n            gc.collect()\n            if torch.cuda.is_available(): torch.cuda.empty_cache()\n        np.savez_compressed(run/\'grid_scores.npz\',prediction=grid[:,None],targets=np.asarray([spec[\'target\']]))\n        ap,native_hash=native_scores(c,grid,spec[\'target\'])\n        status.update(status=\'complete\',run_directory=str(run),smoke_only=cfg.smoke,\n            native_prediction_sha256=native_hash,independent_validation_pending=True,**{spec[\'target\']:ap})\n        base.write(run/\'decision.json\',{**status,\'all_negative_rows_retained\':True,\'retained_heads_changed\':False,\n            \'standalone_two_target_mAP_defined\':False,\'confidence_calibrated\':False,\'independent_generalization_established\':False})\n    except Exception as error:\n        status.update(status=\'not_run_budget\' if isinstance(error,TimeoutError) else \'failed\',\n            error_type=type(error).__name__,error=str(error),traceback=traceback.format_exc())\n        print(id,status[\'status\'],str(error),flush=True)\n    session[\'results\'].append(status); base.write(session[\'output\']/(id+\'_status.json\'),status)\n    pd.DataFrame([{k:v for k,v in r.items() if k!=\'traceback\'} for r in session[\'results\']]).to_csv(session[\'output\']/\'suite_summary.csv\',index=False)\n    return {k:v for k,v in status.items() if k!=\'traceback\'}\n\n\ndef reuse_result(session,id,result):\n    assert result[\'id\']==id and result[\'status\'] in (\'complete\',\'complete_reused_v1\') and not result[\'smoke_only\']\n    row={**result,\'status\':\'complete_reused_v1\',\'not_retrained_in_v27\':True,\'artifacts_origin\':\'preserved completed v26 outputs\'}\n    session[\'results\'].append(row); base.write(session[\'output\']/(id+\'_status.json\'),row); return row\n\n\ndef finish(session):\n    expected={s[\'id\'] for s in base.CATALOG if s[\'suite\']==session[\'suite\']}|{id for id,s in SPECS.items() if s[\'suite\']==session[\'suite\']}\n    done=[r[\'id\'] for r in session[\'results\'] if r[\'status\'] in (\'complete\',\'complete_reused_v1\')]\n    result={\'status\':\'all_methods_completed_validation_pending\' if set(done)==expected else \'partial_coverage\',\n        \'expected_tests\':sorted(expected),\'attempted_tests\':[r[\'id\'] for r in session[\'results\']],\'completed_tests\':done,\n        \'newly_fitted_tests\':[r[\'id\'] for r in session[\'results\'] if r[\'status\']==\'complete\'],\n        \'reused_tests\':[r[\'id\'] for r in session[\'results\'] if r[\'status\']==\'complete_reused_v1\'],\n        \'missing_tests\':sorted(expected-{r[\'id\'] for r in session[\'results\']}),\'smoke_only\':session[\'cfg\'].smoke,\n        \'retained_heads_changed\':False,\'competition_submission_created\':False,\'automatic_promotion\':False,\n        \'confidence_calibrated\':False,\'independent_generalization_established\':False}\n    base.write(session[\'output\']/\'suite_completion.json\',result); return result\n'
FOLLOWUP_SHA256 = '1b78ed01e1de1a93cb93263739b35a301289f8195304ab7516da8254e4706c86'
assert hashlib.sha256(FOLLOWUP_SOURCE.encode()).hexdigest() == FOLLOWUP_SHA256
followups = types.ModuleType('tdcs_v27_followups')
followups.__file__ = str(PROJECT_ROOT / 'scripts/tdcs_v27_followups.py')
followups.SOURCE_SHA256 = FOLLOWUP_SHA256
sys.modules[followups.__name__] = followups
exec(compile(FOLLOWUP_SOURCE, followups.__file__, 'exec'), followups.__dict__)
print('Historical v27 helper loaded for synthetic checks only')

Historical v27 helper loaded for synthetic checks only


In [5]:
RAW_SOURCE = '"""Raw-signal v28: matched TDCS warmup versus independent DeFOG pretraining.\n\nBoth models use 50-Hz raw accelerometry, CNN/residual BiGRU context and a\nfour-state training objective. Only SH/Walking are evaluated. No validation\nlabels select epochs; no local real-data fitting or automatic promotion.\n"""\nfrom __future__ import annotations\n\nimport copy\nimport hashlib\nimport json\nimport time\nimport traceback\nfrom dataclasses import asdict, dataclass, replace\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\nfrom scipy.signal import resample_poly\nfrom sklearn.metrics import average_precision_score\n\nimport tdcs_suite_data as data\nimport tdcs_suite_runner as base\n\nLABELS=(\'StartHesitation\',\'Turn\',\'Walking\')\nSPECS={\'M21\':{\'suite\':\'metadata\',\'transfer\':False},\'S10\':{\'suite\':\'sensor\',\'transfer\':True}}\n\n\n@dataclass(frozen=True)\nclass Config:\n    sample_hz:int=50\n    train_length:int=1000\n    infer_length:int=5000\n    batch_size:int=8\n    micro_batch:int=2\n    conv_width:int=64\n    hidden:int=96\n    layers:int=3\n    dropout:float=.15\n    warm_steps:int=512\n    fine_steps:int=1536\n    warm_lr:float=.001\n    fine_lr:float=.0005\n    weight_decay:float=.001\n    ema_decay:float=.995\n    seed:int=2851\n    max_seconds:int=21600\n    smoke:bool=False\n\n\ndef smoke_config():\n    return replace(Config(),train_length=16,infer_length=32,batch_size=2,micro_batch=1,\n        conv_width=8,hidden=8,layers=1,dropout=0.,warm_steps=1,fine_steps=1,max_seconds=600,smoke=True)\n\n\ndef source_hash(): return globals().get(\'SOURCE_SHA256\') or data.digest(__file__)\ndef key(value): return hashlib.sha256(json.dumps(value,sort_keys=True).encode()).hexdigest()[:20]\n\n\ndef check_config(cfg):\n    if cfg.micro_batch<1 or cfg.batch_size<1 or cfg.sample_hz!=50 or cfg.batch_size%cfg.micro_batch or cfg.conv_width%8:\n        raise ValueError(\'Fixed 50-Hz rate, valid microbatch division and GroupNorm width required\')\n    if min(cfg.train_length,cfg.infer_length,cfg.hidden,cfg.layers,cfg.warm_steps,cfg.fine_steps)<1:\n        raise ValueError(\'Positive training sizes and budgets required\')\n    if not 0<=cfg.dropout<1 or not 0<cfg.ema_decay<1: raise ValueError(\'Invalid regularization\')\n\n\ndef boolean(values):\n    values=np.asarray(values)\n    if values.dtype.kind==\'b\': return values.astype(bool)\n    if values.dtype.kind in \'iuf\' and np.isin(values,[0,1]).all(): return values.astype(bool)\n    words=np.char.lower(np.char.strip(values.astype(str)))\n    if not np.isin(words,[\'true\',\'false\',\'0\',\'1\']).all(): raise ValueError(\'Invalid annotation mask values\')\n    return np.isin(words,[\'true\',\'1\'])\n\n\ndef make_record(frame,id,subject,fold,source_hz,domain):\n    if (domain,source_hz) not in ((\'tdcs\',128),(\'defog\',100)):\n        raise ValueError(\'Only the documented TDCS/DeFOG unit-rate contracts are supported\')\n    times=frame.Time.to_numpy(float)\n    if not np.isfinite(times).all() or not np.equal(np.diff(times),1).all():\n        raise ValueError(\'Consecutive sample-index Time is required\')\n    raw=frame[list(data.AXES)].to_numpy(np.float64)\n    if not np.isfinite(raw).all(): raise ValueError(\'Nonfinite accelerometer input\')\n    labels=frame[list(LABELS)].to_numpy()\n    if not np.isin(labels,[0,1]).all() or (labels.sum(1)>1).any(): raise ValueError(\'Exclusive binary class labels required\')\n    classes=(labels@np.asarray([1,2,3])).astype(np.uint8)\n    valid=np.ones(len(frame),bool) if domain==\'tdcs\' else boolean(frame.Valid)&boolean(frame.Task)\n    raw_g=raw/9.80665 if domain==\'tdcs\' else raw\n    signal=resample_poly(raw_g,25,64,axis=0,padtype=\'line\') if source_hz==128 else resample_poly(raw_g,1,2,axis=0,padtype=\'line\')\n    if len(signal)<2: raise ValueError(\'Recording is too short after resampling\')\n    positions=np.arange(len(signal),dtype=np.float64)*source_hz/50.\n    nearest=np.clip(np.rint(positions).astype(np.int64),0,len(frame)-1)\n    signal=signal.astype(np.float32)\n    record={\'id\':str(id),\'subject\':str(subject),\'fold\':int(fold),\'domain\':domain,\'signal\':signal,\n        \'classes\':classes[nearest],\'valid\':valid[nearest],\'native_length\':len(frame),\'source_hz\':source_hz,\n        \'mean\':signal.mean(0,dtype=np.float64),\'cov\':np.cov(signal.astype(np.float64),rowvar=False,bias=True)}\n    record[\'valid_indices\']=np.flatnonzero(record[\'valid\']).astype(np.int32)\n    return record\n\n\ndef features(record,positions,angle=0.,gain=1.,noise=None):\n    positions=np.asarray(positions,np.float64); n=len(record[\'signal\'])\n    clipped=np.clip(positions,0,n-1); left=np.floor(clipped).astype(np.int64); right=np.minimum(left+1,n-1)\n    fraction=(clipped-left)[:,None]\n    raw=(1-fraction)*record[\'signal\'][left]+fraction*record[\'signal\'][right]\n    cosine,sine=np.cos(angle),np.sin(angle)\n    rotation=np.asarray([[1,0,0],[0,cosine,-sine],[0,sine,cosine]],np.float64)\n    raw=raw@rotation.T; mean=record[\'mean\']@rotation.T\n    std=np.sqrt(np.maximum(np.diag(rotation@record[\'cov\']@rotation.T),.005**2))\n    raw=mean+gain*(raw-mean)\n    if noise is not None: raw=raw+noise\n    z=np.clip((raw-mean)/std,-8,8)\n    return np.column_stack([np.clip(raw,-4,4),z,np.minimum(np.linalg.norm(z,axis=1)/np.sqrt(3),8),\n        np.clip(positions/max(n-1,1),0,1)]).astype(np.float32)\n\n\ndef class_weights(records):\n    counts=sum((np.bincount(r[\'classes\'][r[\'valid\']],minlength=4) for r in records),np.zeros(4,np.int64))\n    if counts.sum()==0: raise ValueError(\'No annotated fitting samples\')\n    weights=np.clip(np.sqrt(max(int(counts[0]),1)/np.maximum(counts,1)),.5,6.)\n    return (weights/weights.mean()).astype(np.float32),counts\n\n\ndef sampling_plan(records):\n    records=[r for r in records if len(r[\'valid_indices\'])]\n    if not records: raise ValueError(\'No annotated fitting recordings\')\n    totals={s:sum(len(r[\'valid_indices\']) for r in records if r[\'subject\']==s) for s in {r[\'subject\'] for r in records}}\n    p=np.asarray([len(r[\'valid_indices\'])/np.sqrt(totals[r[\'subject\']]) for r in records]); p/=p.sum()\n    return records,p\n\n\ndef batch(records,probability,rng,cfg):\n    xs=[]; ys=[]; masks=[]; trace=[]\n    for _ in range(cfg.batch_size):\n        record=records[int(rng.choice(len(records),p=probability))]\n        center=int(rng.choice(record[\'valid_indices\'])); stretch=float(rng.uniform(.8,1.2))\n        positions=center+np.linspace(-cfg.train_length*stretch/2,cfg.train_length*stretch/2,cfg.train_length)\n        nearest=np.clip(np.rint(positions).astype(int),0,len(record[\'signal\'])-1)\n        inside=(positions>=0)&(positions<=len(record[\'signal\'])-1)\n        xs.append(features(record,positions,float(rng.uniform(-.2,.2)),float(rng.uniform(.9,1.1)),rng.normal(0,.001,(cfg.train_length,3))))\n        ys.append(record[\'classes\'][nearest]); masks.append(inside&record[\'valid\'][nearest])\n        trace.append((record[\'id\'],record[\'subject\'],center,stretch))\n    return np.stack(xs),np.stack(ys).astype(np.int64),np.stack(masks),trace\n\n\nclass Network(nn.Module):\n    def __init__(self,cfg):\n        super().__init__(); self.cfg=cfg; width=cfg.conv_width; size=2*cfg.hidden\n        self.stem=nn.Sequential(nn.Conv1d(8,width,7,padding=3),nn.GroupNorm(8,width),nn.GELU(),\n            nn.Conv1d(width,width,7,padding=3),nn.GroupNorm(8,width),nn.GELU())\n        self.project=nn.Sequential(nn.Linear(width,size),nn.LayerNorm(size),nn.GELU())\n        self.blocks=nn.ModuleList([nn.GRU(size,cfg.hidden,batch_first=True,bidirectional=True) for _ in range(cfg.layers)])\n        self.norms=nn.ModuleList([nn.LayerNorm(size) for _ in range(cfg.layers)])\n        self.head=nn.Sequential(nn.Linear(size,64),nn.GELU(),nn.Linear(64,4))\n\n    def forward(self,x):\n        h=self.project(self.stem(x.transpose(1,2)).transpose(1,2))\n        for block,norm in zip(self.blocks,self.norms):\n            h=norm((h+F.dropout(block(h)[0],self.cfg.dropout,self.training))/np.sqrt(2))\n        return self.head(h)\n\n\ndef train_phase(model,records,cfg,steps,lr,seed,deadline,ema=None):\n    torch.manual_seed(seed)\n    if next(model.parameters()).is_cuda: torch.cuda.manual_seed_all(seed)\n    device=next(model.parameters()).device; rng=np.random.default_rng(seed); records,p=sampling_plan(records)\n    weights,counts=class_weights(records); weight=torch.from_numpy(weights).to(device)\n    optimizer=torch.optim.AdamW(model.parameters(),lr=lr,weight_decay=cfg.weight_decay)\n    trace=[]; model.train(); started=time.monotonic()\n    for step in range(steps):\n        if time.monotonic()>=deadline: raise TimeoutError(\'Raw sequence runtime budget reached\')\n        bx,by,mask,crops=batch(records,p,rng,cfg); mass=float((weights[by]*mask).sum())\n        if mass<=0: raise ValueError(\'Sampled batch contains no annotated positions\')\n        trace.extend(crops); optimizer.zero_grad(set_to_none=True)\n        for start in range(0,cfg.batch_size,cfg.micro_batch):\n            stop=start+cfg.micro_batch; x=torch.from_numpy(bx[start:stop]).to(device)\n            y=torch.from_numpy(by[start:stop]).to(device); valid=torch.from_numpy(mask[start:stop]).to(device).float()\n            losses=F.cross_entropy(model(x).transpose(1,2),y,weight=weight,reduction=\'none\')\n            loss=(losses*valid).sum()/mass\n            if not torch.isfinite(loss): raise FloatingPointError(\'Nonfinite raw sequence loss\')\n            loss.backward()\n        torch.nn.utils.clip_grad_norm_(model.parameters(),1.); optimizer.step()\n        if ema is not None:\n            with torch.no_grad():\n                for dst,src in zip(ema.parameters(),model.parameters()): dst.lerp_(src,1-cfg.ema_decay)\n        if (step+1)%64==0: print(f\'Raw phase: {step+1}/{steps} updates; elapsed {time.monotonic()-started:.1f}s\',flush=True)\n    return {\'updates\':steps,\'seed\':seed,\'class_counts\':counts.tolist(),\'class_weights\':weights.tolist(),\n        \'subjects\':sorted({r[\'subject\'] for r in records}),\'elapsed_seconds\':time.monotonic()-started,\'trace\':trace}\n\n\ndef predict(model,record,cfg,device):\n    n=len(record[\'signal\']); total=np.zeros((n,2),np.float64); denominator=np.zeros(n,np.float64)\n    taper=np.maximum(np.hanning(cfg.infer_length),.1); model.eval()\n    with torch.no_grad():\n        for start in range(-cfg.infer_length//2,n,cfg.infer_length//2):\n            positions=start+np.arange(cfg.infer_length); x=features(record,positions)\n            p=torch.softmax(model(torch.from_numpy(x[None]).to(device)),dim=-1)[0][:,[1,3]].cpu().numpy()\n            inside=(positions>=0)&(positions<n); rows=positions[inside]\n            total[rows]+=p[inside]*taper[inside,None]; denominator[rows]+=taper[inside]\n    assert (denominator>0).all()\n    return (total/denominator[:,None]).astype(np.float32)\n\n\ndef synthetic():\n    main=[]; aux=[]; native=[]\n    for i in range(7):\n        hz=128 if i<5 else 100; n=128+i*32; t=np.arange(n)/hz\n        frame=pd.DataFrame({\'Time\':np.arange(n),\'AccV\':-1+.1*np.sin(2*np.pi*t),\n            \'AccML\':.2*np.cos(6*np.pi*t),\'AccAP\':.2*np.sin(4*np.pi*t)})\n        if i<5: frame[list(data.AXES)]*=9.80665\n        labels=(np.arange(n)//12)%4\n        for j,name in enumerate(LABELS,1): frame[name]=(labels==j).astype(np.uint8)\n        frame[\'Valid\']=np.arange(n)>4; frame[\'Task\']=np.arange(n)<n-4\n        record=make_record(frame,f\'toy_{i}\',f\'subject_{i}\',i if i<5 else -1,hz,\'tdcs\' if i<5 else \'defog\')\n        (main if i<5 else aux).append(record)\n        if i<5: native.append(frame[[\'StartHesitation\',\'Walking\']].to_numpy(np.uint8))\n    return main,aux,np.concatenate(native)\n\n\ndef prepare(context,root,transfer):\n    cache=Path(root)/\'raw_cache\'; cache.mkdir(parents=True,exist_ok=True); main=[]; aux=[]; provenance=[]\n    for i,row in enumerate(context[\'manifest\'].itertuples(index=False)):\n        frame=pd.read_csv(row.path,usecols=[\'Time\',*data.AXES,*LABELS])\n        left,right=context[\'native_offsets\'][i:i+2]\n        np.testing.assert_array_equal(frame[[\'StartHesitation\',\'Walking\']].to_numpy(),context[\'native_y\'][left:right])\n        assert len(frame)==right-left\n        record=make_record(frame,row.recording_id,row.subject,row.fold,128,\'tdcs\'); main.append(record)\n        provenance.append({\'id\':row.recording_id,\'subject\':row.subject,\'fold\':row.fold,\'domain\':\'tdcs\',\n            \'source_sha256\':data.digest(row.path),\'signal_sha256\':data.array_digest(record[\'signal\']),\n            \'class_sha256\':data.array_digest(record[\'classes\']),\'valid_sha256\':data.array_digest(record[\'valid\'])})\n        np.savez_compressed(cache/(row.recording_id+\'.npz\'),signal=record[\'signal\'],classes=record[\'classes\'],valid=record[\'valid\'])\n    tdcs_metadata=pd.read_csv(data.csv_path(\'tdcsfog_metadata.csv\'),dtype=str)\n    if tdcs_metadata.Subject.isna().any(): raise ValueError(\'Missing TDCS subject for auxiliary exclusion\')\n    excluded=set(tdcs_metadata.Subject)\n    if transfer:\n        metadata_path=data.csv_path(\'defog_metadata.csv\'); meta=pd.read_csv(metadata_path,dtype=str)\n        if meta.Id.duplicated().any() or meta.Id.isna().any() or meta.Subject.isna().any(): raise ValueError(\'Invalid auxiliary metadata\')\n        raw_root=data.frozen.shared.common.locate_data()[0].parent/\'defog\'\n        for row in meta.itertuples(index=False):\n            if row.Subject in excluded: continue\n            path=raw_root/(row.Id+\'.csv\')\n            frame=pd.read_csv(path,usecols=[\'Time\',*data.AXES,*LABELS,\'Valid\',\'Task\'])\n            record=make_record(frame,row.Id,row.Subject,-1,100,\'defog\'); aux.append(record)\n            provenance.append({\'id\':row.Id,\'subject\':row.Subject,\'domain\':\'defog\',\'source_sha256\':data.digest(path),\n                \'signal_sha256\':data.array_digest(record[\'signal\']),\'class_sha256\':data.array_digest(record[\'classes\']),\n                \'valid_sha256\':data.array_digest(record[\'valid\']),\'valid_rows\':int(record[\'valid\'].sum())})\n            np.savez_compressed(cache/(row.Id+\'.npz\'),signal=record[\'signal\'],classes=record[\'classes\'],valid=record[\'valid\'])\n        if not aux or set(r[\'subject\'] for r in aux)&excluded: raise ValueError(\'No independent auxiliary records\')\n    base.write(Path(root)/\'raw_dataset_manifest.json\',{\'records\':provenance,\'sample_hz\':50,\n        \'auxiliary_TDCS_subject_overlap\':0,\'auxiliary_loss_mask\':\'Valid AND Task\',\'clinical_or_ID_predictors\':False})\n    print(\'Prepared raw data:\',len(main),\'TDCS records;\',len(aux),\'auxiliary records\',flush=True)\n    return main,aux,context[\'native_y\']\n\n\ndef evaluate(main,native_y,parts,gates,apply_gate):\n    if not len(main)==len(parts)==len(gates): raise ValueError(\'Recording prediction alignment mismatch\')\n    predictions=[]\n    for record,p,q in zip(main,parts,gates):\n        if p.shape!=(len(record[\'signal\']),2): raise ValueError(\'Unexpected raw prediction length or heads\')\n        grid=np.arange(len(p))*record[\'source_hz\']/50.\n        native=np.column_stack([np.interp(np.arange(record[\'native_length\']),grid,p[:,j]) for j in range(2)]).astype(np.float32)\n        if apply_gate: native[:,1]*=(np.asarray([q],np.float32)*.95+.05)[0]\n        predictions.append(native)\n    p=np.concatenate(predictions)\n    assert p.shape==native_y.shape and np.isfinite(p).all() and ((p>=0)&(p<=1)).all()\n    values={t:float(average_precision_score(native_y[:,j],p[:,j])) for j,t in enumerate(data.TARGETS)}\n    values[\'two_target_mAP\']=float(np.mean(list(values.values())))\n    return values,data.array_digest(p)\n\n\ndef run(session,id):\n    spec=SPECS[id]; assert spec[\'suite\']==session[\'suite\']; cfg=smoke_config() if session[\'cfg\'].smoke else Config()\n    status={\'id\':id,\'title\':\'Raw 50-Hz CNN/residual GRU \'+(\'DeFOG transfer\' if spec[\'transfer\'] else \'TDCS matched control\'),\n        \'suite\':session[\'suite\'],\'status\':\'pending\'}\n    try:\n        check_config(cfg)\n        if not cfg.smoke and (not Path(\'/kaggle/working\').exists() or not torch.cuda.is_available()):\n            raise RuntimeError(\'Real raw fitting requires Kaggle CUDA\')\n        root=session[\'output\']/\'raw_v28\'; root.mkdir(parents=True,exist_ok=True)\n        main,aux,y=synthetic() if cfg.smoke else prepare(session[\'context\'],root,spec[\'transfer\'])\n        signature=[{\'id\':r[\'id\'],\'subject\':r[\'subject\'],\'fold\':r[\'fold\'],\'domain\':r[\'domain\'],\n            \'signal\':data.array_digest(r[\'signal\']),\'classes\':data.array_digest(r[\'classes\']),\'valid\':data.array_digest(r[\'valid\'])} for r in [*main,*aux]]\n        config={\'specification\':spec,\'id\':id,\'configuration\':asdict(cfg),\'source_sha256\':source_hash(),\n            \'base_sources\':base.source_hashes(),\'dataset_signature\':signature,\'targets\':list(data.TARGETS),\n            \'training_classes\':[\'NoFOG\',*LABELS],\'outer_stopping\':False,\'automatic_promotion\':False,\n            \'primary_postprocessing\':\'ungated\',\'secondary_postprocessing\':\'frozen_Walking_gate_once\',\n            \'runtime\':session[\'runtime\'],\'smoke_only\':cfg.smoke,\'warm_model_initialization_seed\':cfg.seed}\n        run_dir=session[\'output\']/\'methods\'/id/key(config); run_dir.mkdir(parents=True,exist_ok=True); base.write(run_dir/\'configuration.json\',config)\n        device=torch.device(\'cpu\' if cfg.smoke else \'cuda:0\'); torch.set_num_threads(2)\n        deadline=min(session[\'deadline\'],time.monotonic()+cfg.max_seconds)\n        initial=None; warm_receipt=None\n        if spec[\'transfer\']:\n            assert set(r[\'subject\'] for r in aux).isdisjoint({r[\'subject\'] for r in main})\n            if (run_dir/\'defog_pretrained.pt\').is_file():\n                stored=torch.load(run_dir/\'defog_pretrained.pt\',map_location=\'cpu\',weights_only=False)\n                assert stored[\'source_sha256\']==source_hash() and stored[\'configuration\']==asdict(cfg)\n                assert stored[\'updates\']==cfg.warm_steps\n                assert set(stored[\'training_subjects\'])=={r[\'subject\'] for r in aux if r[\'valid\'].any()}\n                initial=stored[\'model\']; warm_receipt=json.loads((run_dir/\'auxiliary_pretraining_receipt.json\').read_text())\n            else:\n                torch.manual_seed(cfg.seed); model=Network(cfg).to(device)\n                warm_receipt=train_phase(model,aux,cfg,cfg.warm_steps,cfg.warm_lr,cfg.seed+999,deadline)\n                torch.save({\'model\':model.state_dict(),\'configuration\':asdict(cfg),\'source_sha256\':source_hash(),\n                    \'training_subjects\':warm_receipt[\'subjects\'],\'updates\':cfg.warm_steps},run_dir/\'defog_pretrained.pt\')\n                initial={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}; del model\n                base.write(run_dir/\'auxiliary_pretraining_receipt.json\',{k:v for k,v in warm_receipt.items() if k!=\'trace\'})\n                base.write(run_dir/\'auxiliary_pretraining_trace.json\',{\'crops\':warm_receipt[\'trace\']})\n        parts=[None]*len(main); completed=[]\n        for fold in range(5):\n            if time.monotonic()>=deadline: raise TimeoutError(\'Raw suite budget reached between folds\')\n            directory=run_dir/f\'fold_{fold}\'; directory.mkdir(exist_ok=True)\n            fitting=[r for r in main if r[\'fold\']!=fold]; heldout=[i for i,r in enumerate(main) if r[\'fold\']==fold]\n            assert {r[\'subject\'] for r in fitting}.isdisjoint({main[i][\'subject\'] for i in heldout})\n            receipt_path=directory/\'training_receipt.json\'\n            if receipt_path.is_file():\n                receipt=json.loads(receipt_path.read_text()); assert receipt[\'fingerprint\']==run_dir.name\n                for name,sha in receipt[\'artifact_sha256\'].items(): assert data.digest(directory/name)==sha\n                with np.load(directory/\'heldout_grid.npz\',allow_pickle=False) as saved:\n                    np.testing.assert_array_equal(saved[\'record_indices\'],heldout)\n                    np.testing.assert_array_equal(saved[\'offsets\'],np.r_[0,np.cumsum([len(main[i][\'signal\']) for i in heldout])])\n                    for k,i in enumerate(saved[\'record_indices\']): parts[int(i)]=saved[\'prediction\'][saved[\'offsets\'][k]:saved[\'offsets\'][k+1]]\n                completed.append(fold); continue\n            if any(directory.iterdir()): raise ValueError(\'Incomplete raw fold; preserve it and use a fresh output root\')\n            torch.manual_seed(cfg.seed); model=Network(cfg).to(device)\n            if initial is not None: model.load_state_dict(initial)\n            else: warm_receipt=train_phase(model,fitting,cfg,cfg.warm_steps,cfg.warm_lr,cfg.seed+1009*fold+999,deadline)\n            torch.save({\'model\':model.state_dict(),\'warm_steps\':cfg.warm_steps,\'fingerprint\':run_dir.name},directory/\'warmup.pt\')\n            ema=copy.deepcopy(model).eval()\n            fine=train_phase(model,fitting,cfg,cfg.fine_steps,cfg.fine_lr,cfg.seed+1009*fold+200000,deadline,ema)\n            torch.save({\'model\':ema.state_dict(),\'configuration\':asdict(cfg),\'fingerprint\':run_dir.name,\n                \'fold\':fold,\'classes\':[\'NoFOG\',*LABELS]},directory/\'model.pt\')\n            payload=torch.load(directory/\'model.pt\',map_location=device,weights_only=False)\n            replay_model=Network(cfg).to(device); replay_model.load_state_dict(payload[\'model\']); replay_model.eval()\n            assert all(torch.equal(value,payload[\'model\'][name]) for name,value in ema.state_dict().items())\n            base.write(directory/\'training_trace.json\',{\'warmup\':warm_receipt[\'trace\'] if not spec[\'transfer\'] else \'shared_independent_DeFOG\',\n                \'fine_tune\':fine[\'trace\']})\n            for i in heldout: parts[i]=predict(replay_model,main[i],cfg,device)\n            offsets=np.r_[0,np.cumsum([len(parts[i]) for i in heldout])]\n            np.savez_compressed(directory/\'heldout_grid.npz\',record_indices=np.asarray(heldout),offsets=offsets,\n                prediction=np.concatenate([parts[i] for i in heldout]))\n            receipt={\'fingerprint\':run_dir.name,\'fold\':fold,\'fitting_subjects\':sorted({r[\'subject\'] for r in fitting}),\n                \'heldout_subjects\':sorted({main[i][\'subject\'] for i in heldout}),\'warmup_subjects\':warm_receipt[\'subjects\'],\n                \'warmup_updates\':cfg.warm_steps,\'fine_tune_updates\':cfg.fine_steps,\'fine_tune_class_counts\':fine[\'class_counts\'],\n                \'fine_tune_class_weights\':fine[\'class_weights\'],\'fixed_final_EMA\':cfg.ema_decay,\'backend\':str(device),\n                \'warmup_elapsed_seconds\':warm_receipt[\'elapsed_seconds\'],\'fine_tune_elapsed_seconds\':fine[\'elapsed_seconds\'],\n                \'outer_labels_used_for_training_or_stopping\':False,\'auxiliary_valid_mask_applied\':spec[\'transfer\'],\n                \'saved_model_reloaded_for_all_heldout_predictions\':True,\n                \'artifact_sha256\':{p.name:data.digest(p) for p in directory.iterdir() if p.is_file() and p.name!=\'training_receipt.json\'}}\n            assert not set(receipt[\'warmup_subjects\'])&set(receipt[\'heldout_subjects\'])\n            base.write(receipt_path,receipt); completed.append(fold); print(id,\'fold\',fold,\'complete\',flush=True)\n            del model,ema,replay_model,payload\n            if torch.cuda.is_available(): torch.cuda.empty_cache()\n        offsets=np.r_[0,np.cumsum([len(p) for p in parts])]\n        np.savez_compressed(run_dir/\'grid_scores.npz\',prediction=np.concatenate(parts),offsets=offsets,\n            recording_ids=np.asarray([r[\'id\'] for r in main]),native_lengths=np.asarray([r[\'native_length\'] for r in main]),\n            targets=np.asarray(data.TARGETS))\n        gates=np.ones(len(main),np.float32) if cfg.smoke else session[\'context\'][\'gates\']\n        values,native_hash=evaluate(main,y,parts,gates,False); gated,gated_hash=evaluate(main,y,parts,gates,True)\n        status.update(status=\'complete\',**values,run_directory=str(run_dir),smoke_only=cfg.smoke,\n            native_prediction_sha256=native_hash,independent_validation_pending=True)\n        base.write(run_dir/\'decision.json\',{**status,\'fixed_gate_ablation\':gated,\'fixed_gate_native_sha256\':gated_hash,\n            \'all_negative_native_rows_retained\':True,\'native_rows\':len(y),\'retained_heads_changed\':False,\n            \'independent_generalization_established\':False,\'confidence_calibrated\':False})\n    except Exception as error:\n        status.update(status=\'not_run_budget\' if isinstance(error,TimeoutError) else \'failed\',\n            error_type=type(error).__name__,error=str(error),traceback=traceback.format_exc())\n        print(id,status[\'status\'],str(error),flush=True)\n    session[\'results\'].append(status); base.write(session[\'output\']/(id+\'_status.json\'),status)\n    pd.DataFrame([{k:v for k,v in r.items() if k!=\'traceback\'} for r in session[\'results\']]).to_csv(session[\'output\']/\'suite_summary.csv\',index=False)\n    return {k:v for k,v in status.items() if k!=\'traceback\'}\n\n\ndef reuse_result(session,id,result):\n    assert result[\'id\']==id and result[\'status\'] in (\'complete\',\'complete_reused_v1\') and not result[\'smoke_only\']\n    row={**result,\'status\':\'complete_reused\',\'not_retrained_in_v28\':True,\n        \'artifacts_origin\':result[\'run_directory\'],\'reused_scores_are_not_a_new_fit\':True}\n    session[\'results\'].append(row); base.write(session[\'output\']/(id+\'_status.json\'),row)\n    return row\n\n\ndef finish(session):\n    import tdcs_v27_followups as earlier\n    expected={s[\'id\'] for s in base.CATALOG if s[\'suite\']==session[\'suite\']}\n    expected|={id for id,s in earlier.SPECS.items() if s[\'suite\']==session[\'suite\']}\n    expected|={id for id,s in SPECS.items() if s[\'suite\']==session[\'suite\']}\n    done=[r[\'id\'] for r in session[\'results\'] if r[\'status\'] in (\'complete\',\'complete_reused\')]\n    assert len({r[\'id\'] for r in session[\'results\']})==len(session[\'results\'])\n    result={\'status\':\'all_methods_completed_validation_pending\' if set(done)==expected else \'partial_coverage\',\n        \'expected_tests\':sorted(expected),\'attempted_tests\':[r[\'id\'] for r in session[\'results\']],\n        \'completed_tests\':done,\'newly_fitted_tests\':[r[\'id\'] for r in session[\'results\'] if r[\'status\']==\'complete\'],\n        \'reused_tests\':[r[\'id\'] for r in session[\'results\'] if r[\'status\']==\'complete_reused\'],\n        \'missing_tests\':sorted(expected-set(done)),\'smoke_only\':session[\'cfg\'].smoke,\n        \'retained_heads_changed\':False,\'competition_submission_created\':False,\'automatic_promotion\':False,\n        \'confidence_calibrated\':False,\'independent_generalization_established\':False}\n    base.write(session[\'output\']/\'suite_completion.json\',result)\n    pd.DataFrame([{k:v for k,v in r.items() if k!=\'traceback\'} for r in session[\'results\']]).to_csv(session[\'output\']/\'suite_summary.csv\',index=False)\n    return result\n'
RAW_SHA256 = '17268a0911be41d6064edd7f696dd406dbbf17759561b96c36275207a0781231'
assert hashlib.sha256(RAW_SOURCE.encode()).hexdigest() == RAW_SHA256
raw = types.ModuleType('tdcs_raw_transfer_v28')
raw.__file__ = str(PROJECT_ROOT / 'scripts/tdcs_raw_transfer_v28.py')
raw.SOURCE_SHA256 = RAW_SHA256
sys.modules[raw.__name__] = raw
exec(compile(RAW_SOURCE, raw.__file__, 'exec'), raw.__dict__)
PREVIOUS_RESULTS_V28 = {'S00': {'id': 'S00', 'title': 'Sensor-only CatBoost control', 'suite': 'sensor', 'status': 'complete_reused_v1', 'StartHesitation': 0.5465950473623613, 'Walking': 0.1227910261941098, 'two_target_mAP': 0.33469303677823553, 'run_directory': '/kaggle/working/tdcs_sensor_comprehensive_v26/methods/S00/fc78883f66e9f95a589e', 'native_prediction_sha256': '0ebc9b3e2b85da20ddedca5643cbf4482a4e7cb9667803f7135c5c6bb1fa6da7', 'independent_validation_pending': True, 'smoke_only': False, 'actual_grid_rows': 436128, 'source_version': 1, 'artifacts_origin': 'preserved completed v26 outputs', 'not_retrained_in_reader_recovery': True, 'not_retrained_in_v27': True}, 'S01': {'id': 'S01', 'title': 'LightGBM: explicit Kaggle CPU training', 'suite': 'sensor', 'status': 'complete_reused_v1', 'StartHesitation': 0.2276165393770844, 'Walking': 0.1225309632076005, 'two_target_mAP': 0.17507375129234246, 'run_directory': '/kaggle/working/tdcs_sensor_comprehensive_v26/methods/S01/f03562c5c87f7ce2fb73', 'native_prediction_sha256': '6360a939decb77908eab5460e464be2d1a79f7ad4da5f36f4f5ebf95c315bc7f', 'independent_validation_pending': True, 'smoke_only': False, 'actual_grid_rows': 436128, 'source_version': 1, 'artifacts_origin': 'preserved completed v26 outputs', 'not_retrained_in_reader_recovery': True, 'not_retrained_in_v27': True}, 'S02': {'id': 'S02', 'title': 'Supervised sensor CNN control for pretraining', 'suite': 'sensor', 'status': 'complete_reused_v1', 'StartHesitation': 0.3227282610066557, 'Walking': 0.1295775166094419, 'two_target_mAP': 0.2261528888080488, 'run_directory': '/kaggle/working/tdcs_sensor_comprehensive_v26/methods/S02/cc787152e2838047b8ff', 'native_prediction_sha256': '99822e3e64527d0e8f8038c3688d141db7a60964bea97f837d32d796b5e8e05b', 'independent_validation_pending': True, 'smoke_only': False, 'actual_grid_rows': 436128, 'source_version': 1, 'artifacts_origin': 'preserved completed v26 outputs', 'not_retrained_in_reader_recovery': True, 'not_retrained_in_v27': True}, 'S03': {'id': 'S03', 'title': 'Masked reconstruction pretraining on mapped unlabeled recordings', 'suite': 'sensor', 'status': 'complete_reused_v1', 'StartHesitation': 0.23812923824791613, 'Walking': 0.13161298201617527, 'two_target_mAP': 0.1848711101320457, 'run_directory': '/kaggle/working/tdcs_sensor_comprehensive_v26_parquet_recovery/methods/S03/19edaa0069ab7317834c', 'native_prediction_sha256': '2476d59a4eab6b9b99d56c8778f79ec96b7be403049cd6f09fce7724d14d88ee', 'independent_validation_pending': True, 'smoke_only': False, 'actual_grid_rows': 436128, 'not_retrained_in_v27': True, 'artifacts_origin': 'preserved completed v26 outputs'}, 'S04': {'id': 'S04', 'title': 'Teacher-student pseudo-label training', 'suite': 'sensor', 'status': 'complete_reused_v1', 'StartHesitation': 0.5855197454877096, 'Walking': 0.12172493864910547, 'two_target_mAP': 0.35362234206840754, 'run_directory': '/kaggle/working/tdcs_sensor_comprehensive_v26_parquet_recovery/methods/S04/cc96468c48350b327531', 'native_prediction_sha256': 'fe8812e6d9146520c3d067ac96d8fb40feb2f1043cc72b1fe4943c32d853e494', 'independent_validation_pending': True, 'smoke_only': False, 'actual_grid_rows': 436128, 'not_retrained_in_v27': True, 'artifacts_origin': 'preserved completed v26 outputs'}, 'S05': {'id': 'S05', 'title': 'Sensor-only binary XGBoost control for ranking', 'suite': 'sensor', 'status': 'complete_reused_v1', 'StartHesitation': 0.38910059070965347, 'Walking': 0.13148538852861444, 'two_target_mAP': 0.26029298961913394, 'run_directory': '/kaggle/working/tdcs_sensor_comprehensive_v26/methods/S05/d029e82637632e3018fa', 'native_prediction_sha256': '40b5ac3dab705ef2bf906c8afba7067f46ff4ecdbe384cc926dc1c098745e3df', 'independent_validation_pending': True, 'smoke_only': False, 'actual_grid_rows': 436128, 'source_version': 1, 'artifacts_origin': 'preserved completed v26 outputs', 'not_retrained_in_reader_recovery': True, 'not_retrained_in_v27': True}, 'S06': {'id': 'S06', 'title': 'Per-record MAP-focused tree ranking', 'suite': 'sensor', 'status': 'complete_reused_v1', 'StartHesitation': 0.054938761623870236, 'Walking': 0.09960435074066686, 'two_target_mAP': 0.07727155618226855, 'run_directory': '/kaggle/working/tdcs_sensor_comprehensive_v26/methods/S06/19dc71c060502b1b36a4', 'native_prediction_sha256': 'ea7c7832c65fe2af8cb0983ccce74d90c3f922f161d035135b36c62a8d496dcf', 'independent_validation_pending': True, 'smoke_only': False, 'actual_grid_rows': 436128, 'source_version': 1, 'artifacts_origin': 'preserved completed v26 outputs', 'not_retrained_in_reader_recovery': True, 'not_retrained_in_v27': True}, 'S07': {'id': 'S07', 'title': 'S07 StartHesitation pseudo', 'suite': 'sensor', 'status': 'complete', 'target_scope': ['StartHesitation'], 'run_directory': '/kaggle/working/tdcs_sensor_comprehensive_v26_v27_followups/methods/S07/64fa912cab40f0d998d9', 'smoke_only': False, 'native_prediction_sha256': '1b2d0f457dc79969eea1509555beda0432af0eacc0ab1a430922eb9e49b44412', 'independent_validation_pending': True, 'StartHesitation': 0.5938817930418117}, 'S08': {'id': 'S08', 'title': 'S08 StartHesitation pseudo', 'suite': 'sensor', 'status': 'complete', 'target_scope': ['StartHesitation'], 'run_directory': '/kaggle/working/tdcs_sensor_comprehensive_v26_v27_followups/methods/S08/028e2f5ca4894cd469d3', 'smoke_only': False, 'native_prediction_sha256': 'af4c6b81d6e05637a93c49ba49747a5822a423facbf283f20b408a90a1ff3e42', 'independent_validation_pending': True, 'StartHesitation': 0.5855197454877096}, 'S09': {'id': 'S09', 'title': 'S09 StartHesitation pseudo', 'suite': 'sensor', 'status': 'complete', 'target_scope': ['StartHesitation'], 'run_directory': '/kaggle/working/tdcs_sensor_comprehensive_v26_v27_followups/methods/S09/e57e8bfdeba3b54adb7f', 'smoke_only': False, 'native_prediction_sha256': 'c630b1a46316d1c3ef7417b883fd8ce2c8e423a69d222c52dc90a8fc099bd533', 'independent_validation_pending': True, 'StartHesitation': 0.5861296551125366}}
print('Pinned v28 raw experiment:', 'S10', '; real prior tests reused:', len(PREVIOUS_RESULTS_V28))

Pinned v28 raw experiment: S10 ; real prior tests reused: 10


In [6]:
READER_RECOVERY_SOURCE = '"""S10 reader-only recovery: join subtype-labelled train/defog files to metadata.\n\nThe 137 metadata IDs cover 91 typed DeFOG and 46 train/notype recordings.\nnotype has no subtype supervision for the fixed four-state objective. Do not\ninvent those labels or read test/defog, even when an ID also occurs there.\nThe frozen v28 model, optimizer, budgets, masks, units and TDCS folds do not\nchange. A composite source hash explicitly binds this reader replacement.\n"""\nfrom __future__ import annotations\n\nfrom contextlib import contextmanager\nimport hashlib\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nimport tdcs_raw_transfer_v28 as raw\n\nFROZEN_RAW_SHA256=\'17268a0911be41d6064edd7f696dd406dbbf17759561b96c36275207a0781231\'\nORIGINAL_PREPARE=raw.prepare\n\n\ndef source_hash():\n    return globals().get(\'SOURCE_SHA256\') or raw.data.digest(__file__)\n\n\ndef effective_source_hash():\n    return hashlib.sha256(json.dumps({\'frozen_raw_source\':FROZEN_RAW_SHA256,\n        \'labelled_file_manifest_reader\':source_hash()},sort_keys=True).encode()).hexdigest()\n\n\ndef auxiliary_inventory(metadata,raw_root,excluded_subjects,expected_files=91):\n    """Select actual typed training files, not all IDs in a wider dimension table."""\n    raw_root=Path(raw_root)\n    if not {\'Id\',\'Subject\'}.issubset(metadata.columns): raise ValueError(\'Required auxiliary metadata columns missing\')\n    if metadata.Id.isna().any() or metadata.Subject.isna().any() or metadata.Id.duplicated().any():\n        raise ValueError(\'Auxiliary metadata IDs/subjects must be nonmissing and IDs unique\')\n    files=sorted(raw_root.glob(\'*.csv\')); ids=[p.stem for p in files]\n    if not files or (expected_files is not None and len(files)!=expected_files):\n        raise ValueError(f\'Expected {expected_files} subtype-labelled train/defog CSVs, found {len(files)}\')\n    if len(ids)!=len(set(ids)): raise ValueError(\'Duplicate auxiliary training file ID\')\n    unknown=set(ids)-set(metadata.Id)\n    if unknown: raise ValueError(\'Auxiliary training files without metadata: \'+str(sorted(unknown)))\n    matched=metadata.set_index(\'Id\').loc[ids].reset_index()\n    selected=matched[~matched.Subject.isin(excluded_subjects)].copy()\n    selected[\'path\']=[str(raw_root/(id+\'.csv\')) for id in selected.Id]\n    if selected.empty: raise ValueError(\'No subject-independent subtype-labelled auxiliary training files\')\n    audit={\'metadata_rows\':len(metadata),\'metadata_subjects\':int(metadata.Subject.nunique()),\n        \'available_subtype_labelled_training_files\':len(files),\'available_subtype_labelled_subjects\':int(matched.Subject.nunique()),\n        \'metadata_IDs_outside_typed_training_folder\':sorted(set(metadata.Id)-set(ids)),\n        \'selected_training_files\':len(selected),\'selected_subjects\':int(selected.Subject.nunique()),\n        \'excluded_TDCS_subject_file_IDs\':matched.loc[matched.Subject.isin(excluded_subjects),\'Id\'].tolist(),\n        \'selected_IDs\':selected.Id.tolist(),\'subject_ID_overlap_after_exclusion\':0,\n        \'selection\':\'actual train/defog CSVs inner-joined to unique metadata; train/notype and all test files excluded\',\n        \'missing_subtype_labels_invented\':False,\'auxiliary_loss_mask\':\'Valid AND Task\'}\n    assert set(selected.Subject).isdisjoint(excluded_subjects)\n    return selected,audit\n\n\ndef prepare(context,root,transfer):\n    if not transfer: return ORIGINAL_PREPARE(context,root,False)\n    main,_,y=ORIGINAL_PREPARE(context,root,False)\n    excluded=set(pd.read_csv(raw.data.csv_path(\'tdcsfog_metadata.csv\'),dtype=str).Subject)\n    metadata=pd.read_csv(raw.data.csv_path(\'defog_metadata.csv\'),dtype=str)\n    raw_root=raw.data.frozen.shared.common.locate_data()[0].parent/\'defog\'\n    selected,audit=auxiliary_inventory(metadata,raw_root,excluded)\n    aux=[]; root=Path(root); manifest=json.loads((root/\'raw_dataset_manifest.json\').read_text())\n    for row in selected.itertuples(index=False):\n        frame=pd.read_csv(row.path,usecols=[\'Time\',*raw.data.AXES,*raw.LABELS,\'Valid\',\'Task\'])\n        record=raw.make_record(frame,row.Id,row.Subject,-1,100,\'defog\'); aux.append(record)\n        manifest[\'records\'].append({\'id\':row.Id,\'subject\':row.Subject,\'domain\':\'defog\',\n            \'source_sha256\':raw.data.digest(row.path),\'signal_sha256\':raw.data.array_digest(record[\'signal\']),\n            \'class_sha256\':raw.data.array_digest(record[\'classes\']),\'valid_sha256\':raw.data.array_digest(record[\'valid\']),\n            \'valid_rows\':int(record[\'valid\'].sum())})\n        np.savez_compressed(root/\'raw_cache\'/(row.Id+\'.npz\'),signal=record[\'signal\'],classes=record[\'classes\'],valid=record[\'valid\'])\n    manifest[\'auxiliary_inventory\']=audit\n    manifest[\'reader_recovery_source_sha256\']=source_hash()\n    manifest[\'effective_raw_source_sha256\']=effective_source_hash()\n    raw.base.write(root/\'raw_dataset_manifest.json\',manifest)\n    raw.base.write(root/\'reader_recovery_provenance.json\',{\n        \'frozen_raw_source_sha256\':FROZEN_RAW_SHA256,\'reader_source_sha256\':source_hash(),\n        \'effective_source_sha256\':effective_source_hash(),\'inventory\':audit,\n        \'model_optimizer_masks_units_budgets_and_TDCS_folds_changed\':False})\n    print(\'Recovered auxiliary inventory:\',audit,flush=True)\n    return main,aux,y\n\n\n@contextmanager\ndef applied():\n    """Replace the reader only in this process; always restore frozen module state."""\n    assert raw.source_hash()==FROZEN_RAW_SHA256, \'Unexpected base raw revision or nested recovery\'\n    previous_prepare=raw.prepare; had_override=\'SOURCE_SHA256\' in raw.__dict__\n    previous_override=raw.__dict__.get(\'SOURCE_SHA256\')\n    raw.prepare=prepare; raw.SOURCE_SHA256=effective_source_hash()\n    try: yield\n    finally:\n        raw.prepare=previous_prepare\n        if had_override: raw.SOURCE_SHA256=previous_override\n        else: raw.__dict__.pop(\'SOURCE_SHA256\',None)\n\n\ndef run(session,id=\'S10\'):\n    if id!=\'S10\' or session[\'suite\']!=\'sensor\': raise ValueError(\'Only the failed S10 reader is recovered\')\n    with applied(): return raw.run(session,id)\n'
READER_RECOVERY_SHA256 = '2db0005a56e638584c6b2dc21d969025d4be918663799b6b6c0a0c375258b0d9'
assert hashlib.sha256(READER_RECOVERY_SOURCE.encode()).hexdigest() == READER_RECOVERY_SHA256
reader_recovery = types.ModuleType('tdcs_raw_transfer_v28_recovery')
reader_recovery.__file__ = str(PROJECT_ROOT / 'scripts/tdcs_raw_transfer_v28_recovery.py')
reader_recovery.SOURCE_SHA256 = READER_RECOVERY_SHA256
sys.modules[reader_recovery.__name__] = reader_recovery
exec(compile(READER_RECOVERY_SOURCE, reader_recovery.__file__, 'exec'), reader_recovery.__dict__)
assert raw.source_hash() == reader_recovery.FROZEN_RAW_SHA256
print('Pinned S10 reader-only recovery:', reader_recovery.effective_source_hash())

Pinned S10 reader-only recovery: 77f3f17df1ece82ef2ad64a62d22d749472166484a33476524f9796cc40f26e3


In [7]:
WALKING_RANK_SOURCE = '"""S11: S10\'s raw transfer recipe plus a Walking-vs-other-FOG ranking term.\n\nSame typed input, warmup, architecture, optimizer, crop plan, seeds, fixed\nbudget and inference as S10. Only TDCS fine-tune loss changes. No held-out\nlabels, predictions, or thresholds participate in fitting. Preserve S10 and\nthe frozen v28 reader/model source by pinning this variant separately.\n"""\nfrom contextlib import contextmanager\nimport hashlib\nimport json\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom torch.nn import functional as F\n\nimport tdcs_raw_transfer_v28_recovery as typed\n\nraw = typed.raw\nORIGINAL_TRAIN = raw.train_phase\nMETHOD = \'S11\'\nTERM_WEIGHT, MARGIN, HARD_NEGATIVES = .2, .2, 16\nSPEC = {\'suite\':\'sensor\', \'transfer\':True, \'matched_control\':\'S10\', \'rank_term_weight\':TERM_WEIGHT,\n    \'rank_margin_log_odds\':MARGIN, \'hard_negatives_per_microbatch\':HARD_NEGATIVES,\n    \'ranking_positives\':\'all valid Walking rows in the microbatch\',\n    \'ranking_negatives\':\'at most 16 highest Walking log-odds among valid SH/Turn rows\',\n    \'normalization\':\'mean over positive-negative pairs; average over all microbatches, including zero terms\',\n    \'training_change\':\'add rank term to TDCS fine-tune cross entropy only\'}\n\n\ndef source_hash():\n    return globals().get(\'SOURCE_SHA256\') or raw.data.digest(__file__)\n\n\ndef effective_source_hash():\n    sources = {\'typed_control_source\':typed.effective_source_hash(), \'Walking_rank_source\':source_hash()}\n    return hashlib.sha256(json.dumps(sources, sort_keys=True).encode()).hexdigest()\n\n\ndef rank_term(logits, labels, valid):\n    """Return mean softplus ranking loss and its exact pair count."""\n    if logits.shape != (*labels.shape, 4) or labels.shape != valid.shape:\n        raise ValueError(\'Aligned four-state logits, labels and masks are required\')\n    walking = valid & labels.eq(3)\n    other_fog = valid & (labels.eq(1) | labels.eq(2))\n    score = logits[..., 3] - torch.logsumexp(logits[..., :3], dim=-1)\n    positives = score[walking]\n    negatives = score[other_fog]\n    if not positives.numel() or not negatives.numel():\n        return score.sum() * 0., 0\n    hard = torch.topk(negatives, min(negatives.numel(), HARD_NEGATIVES), sorted=False).values\n    return F.softplus(hard[None, :] - positives[:, None] + MARGIN).mean(), positives.numel() * hard.numel()\n\n\ndef train_fine(model, records, cfg, steps, lr, seed, deadline, ema):\n    """Keep S10\'s sample stream, weight mass, optimizer and EMA arithmetic."""\n    if ema is None or any(r[\'domain\'] != \'tdcs\' for r in records):\n        raise ValueError(\'Ranking is allowed only in the TDCS fine-tune phase\')\n    torch.manual_seed(seed)\n    if next(model.parameters()).is_cuda: torch.cuda.manual_seed_all(seed)\n    device = next(model.parameters()).device; rng = np.random.default_rng(seed)\n    records, p = raw.sampling_plan(records)\n    weights, counts = raw.class_weights(records); weight = torch.from_numpy(weights).to(device)\n    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=cfg.weight_decay)\n    trace = []; eligible_micros = 0; pairs = 0\n    micro_count = cfg.batch_size // cfg.micro_batch\n    model.train(); started = time.monotonic()\n    for step in range(steps):\n        if time.monotonic() >= deadline: raise TimeoutError(\'Raw sequence runtime budget reached\')\n        bx, by, mask, crops = raw.batch(records, p, rng, cfg)\n        mass = float((weights[by] * mask).sum())\n        if mass <= 0: raise ValueError(\'Sampled batch contains no annotated positions\')\n        trace.extend(crops); optimizer.zero_grad(set_to_none=True)\n        for start in range(0, cfg.batch_size, cfg.micro_batch):\n            stop = start + cfg.micro_batch; x = torch.from_numpy(bx[start:stop]).to(device)\n            y = torch.from_numpy(by[start:stop]).to(device); valid = torch.from_numpy(mask[start:stop]).to(device)\n            logits = model(x)\n            base_loss = F.cross_entropy(logits.transpose(1,2), y, weight=weight, reduction=\'none\')\n            rank, n_pairs = rank_term(logits, y, valid)\n            pairs += n_pairs; eligible_micros += int(n_pairs > 0)\n            loss = (base_loss * valid.float()).sum()/mass + (TERM_WEIGHT / micro_count) * rank\n            if not torch.isfinite(loss): raise FloatingPointError(\'Nonfinite Walking ranking loss\')\n            loss.backward()\n        torch.nn.utils.clip_grad_norm_(model.parameters(),1.); optimizer.step()\n        with torch.no_grad():\n            for dst, src in zip(ema.parameters(), model.parameters()): dst.lerp_(src, 1-cfg.ema_decay)\n        if (step+1)%128 == 0:\n            print(f\'Walking rank phase: {step+1}/{steps} updates; {eligible_micros} eligible microbatches\', flush=True)\n    if not cfg.smoke and not eligible_micros: raise ValueError(\'No valid Walking/other-FOG ranking pairs in the fit\')\n    return {\'updates\':steps,\'seed\':seed,\'class_counts\':counts.tolist(),\'class_weights\':weights.tolist(),\n        \'subjects\':sorted({r[\'subject\'] for r in records}),\'elapsed_seconds\':time.monotonic()-started,\'trace\':trace,\n        \'rank_usage\':{\'eligible_microbatches\':eligible_micros, \'total_microbatches\':steps*micro_count, \'pairs\':pairs}}\n\n\n@contextmanager\ndef applied(usage=None):\n    assert raw.source_hash() == typed.FROZEN_RAW_SHA256 and METHOD not in raw.SPECS\n    prior_prepare, prior_train = raw.prepare, raw.train_phase\n    had_override = \'SOURCE_SHA256\' in raw.__dict__; override = raw.__dict__.get(\'SOURCE_SHA256\')\n\n    def phase(model, records, cfg, steps, lr, seed, deadline, ema=None):\n        if ema is None: return ORIGINAL_TRAIN(model, records, cfg, steps, lr, seed, deadline)\n        result = train_fine(model, records, cfg, steps, lr, seed, deadline, ema)\n        if usage is not None: usage.append({\'seed\':seed, **result[\'rank_usage\']})\n        return result\n\n    raw.prepare = typed.prepare; raw.train_phase = phase\n    raw.SOURCE_SHA256 = effective_source_hash(); raw.SPECS[METHOD] = SPEC\n    try: yield\n    finally:\n        raw.prepare = prior_prepare; raw.train_phase = prior_train\n        if had_override: raw.SOURCE_SHA256 = override\n        else: raw.__dict__.pop(\'SOURCE_SHA256\', None)\n        raw.SPECS.pop(METHOD)\n\n\ndef run(session):\n    usage = []\n    with applied(usage): result = raw.run(session,METHOD)\n    if result[\'status\'] == \'complete\':\n        cfg = raw.smoke_config() if session[\'cfg\'].smoke else raw.Config()\n        # A rerun can reuse existing folds; record exact usage only on the first full fit.\n        destination = Path(result[\'run_directory\'])/\'walking_rank_usage.json\'\n        if len(usage) == 5:\n            for fold, info in enumerate(usage):\n                info[\'fold\'] = fold\n                assert info[\'seed\'] == cfg.seed+1009*fold+200000\n            raw.base.write(destination, {\'source_sha256\':effective_source_hash(), \'folds\':usage,\n                \'config\':SPEC, \'warmup_received_ranking_term\':False, \'smoke_only\':cfg.smoke})\n        elif not destination.is_file():\n            raise ValueError(\'Rank usage receipt missing for reused folds; rerun in a fresh root\')\n    return result\n\n\ndef finish(session):\n    with applied(): return raw.finish(session)\n'
WALKING_RANK_SHA256 = '846bdebccc7aea9f6c1803e33f5a6510dbc91ad92096bad26ac3ea06b181df27'
assert hashlib.sha256(WALKING_RANK_SOURCE.encode()).hexdigest() == WALKING_RANK_SHA256
walking_rank = types.ModuleType('tdcs_raw_walking_rank_v30')
walking_rank.__file__ = str(PROJECT_ROOT / 'scripts/tdcs_raw_walking_rank_v30.py')
walking_rank.SOURCE_SHA256 = WALKING_RANK_SHA256
sys.modules[walking_rank.__name__] = walking_rank
exec(compile(WALKING_RANK_SOURCE, walking_rank.__file__, 'exec'), walking_rank.__dict__)
COMPLETED_S10 = {'id': 'S10', 'title': 'Raw 50-Hz CNN/residual GRU DeFOG transfer', 'suite': 'sensor', 'status': 'complete', 'StartHesitation': 0.30982186007000906, 'Walking': 0.21074198543961103, 'two_target_mAP': 0.26028192275481005, 'run_directory': '/kaggle/working/tdcs_sensor_comprehensive_v26_v28_defog_manifest_recovery/methods/S10/ff9607368b5902a1b8db', 'smoke_only': False, 'native_prediction_sha256': '91add969588fc700f0f0c797af9889600f3f69f43d813225359c870825cb5f6c', 'independent_validation_pending': True}
print('Pinned Walking ranking:', walking_rank.effective_source_hash())

Pinned Walking ranking: 44d7106dbd80ce779e2cd0bbf172071f86a3b1672f29b849e1aa6a0118720ecd


In [8]:
PAIRED_SOURCES = {'tdcs_raw_weak_transfer_v29': '"""M22: typed DeFOG + weak binary Event pretraining, then unchanged TDCS fits.\n\nNo subtype is invented for train/notype. Event=1 contributes -log P(any FOG),\nEvent=0 contributes the known NoFOG loss. Valid AND Task masks both. Compare\ntyped-only S10 at the same seeds/architecture/512 warmup/1536 fine-tune updates.\nNo real-data local fitting, test-file use, metric changes or automatic promotion.\n"""\nfrom __future__ import annotations\n\nfrom contextlib import contextmanager\nimport hashlib\nimport json\nfrom pathlib import Path\nimport time\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch.nn import functional as F\n\nimport tdcs_raw_transfer_v28 as raw\nimport tdcs_raw_transfer_v28_recovery as typed\n\nUNKNOWN_FOG=255\nORIGINAL_TRAIN=raw.train_phase\nORIGINAL_SYNTHETIC=raw.synthetic\nSPEC={\'suite\':\'metadata\',\'transfer\':True,\'weak_notype_Event_pretraining\':True,\n    \'unknown_subtype_positive_loss\':\'negative_log_sum_probability_of_SH_Turn_Walking\',\n    \'unknown_subtype_positive_weight\':\'mean of three fitting FOG-class weights\',\n    \'class_weights\':\'fitting known-state counts only; weak positives never assigned subtypes\',\n    \'fine_tune_changed\':False,\'warmup_updates\':512,\'fine_tune_updates\':1536}\n\n\ndef source_hash(): return globals().get(\'SOURCE_SHA256\') or raw.data.digest(__file__)\n\n\ndef effective_source_hash():\n    return hashlib.sha256(json.dumps({\'frozen_raw_source\':typed.FROZEN_RAW_SHA256,\n        \'typed_file_reader\':typed.source_hash(),\'weak_Event_training\':source_hash()},sort_keys=True).encode()).hexdigest()\n\n\ndef make_weak_record(frame,id,subject):\n    if not {\'Event\',\'Valid\',\'Task\'}.issubset(frame.columns): raise ValueError(\'Weak Event annotation schema missing\')\n    events=raw.boolean(frame.Event.to_numpy())\n    # Zero columns are only a temporary adapter for shared sensor preprocessing.\n    # They are replaced by the binary annotation, never used as subtype labels.\n    adapted=frame.copy()\n    for target in raw.LABELS: adapted[target]=0\n    record=raw.make_record(adapted,id,subject,-1,100,\'defog\')\n    nearest=np.clip(np.rint(np.arange(len(record[\'signal\']))*2).astype(int),0,len(frame)-1)\n    record[\'classes\']=np.where(events[nearest],UNKNOWN_FOG,0).astype(np.uint8)\n    record[\'label_kind\']=\'weak_binary_Event\'\n    return record\n\n\ndef weak_weights(records):\n    counts=np.zeros(4,np.int64); unknown=0\n    for record in records:\n        values=record[\'classes\'][record[\'valid\']]\n        if not np.isin(values,[0,1,2,3,UNKNOWN_FOG]).all(): raise ValueError(\'Unsupported partial label\')\n        counts+=np.bincount(values[values<4],minlength=4)\n        unknown+=int((values==UNKNOWN_FOG).sum())\n    if counts.sum()+unknown==0: raise ValueError(\'No annotated auxiliary rows\')\n    weights=np.clip(np.sqrt(max(int(counts[0]),1)/np.maximum(counts,1)),.5,6.)\n    weights=(weights/weights.mean()).astype(np.float32)\n    return weights,counts,unknown,float(weights[1:].mean())\n\n\ndef partial_losses(logits,labels,mask,weight,fog_weight):\n    """Unnormalized losses and weighted mass; differentiable without pseudo-types."""\n    if not torch.all(((labels>=0)&(labels<=3))|(labels==UNKNOWN_FOG)): raise ValueError(\'Unknown partial training state\')\n    logp=F.log_softmax(logits,dim=-1); weak=labels==UNKNOWN_FOG\n    fitting_weight=torch.where(weak,torch.as_tensor(fog_weight,device=logits.device),weight[labels.clamp_max(3)])\n    typed_loss=-logp.gather(-1,labels.clamp_max(3).unsqueeze(-1)).squeeze(-1)\n    weak_loss=-torch.logsumexp(logp[...,1:4],dim=-1)\n    loss=torch.where(weak,weak_loss,typed_loss)*fitting_weight*mask\n    return loss.sum(),(fitting_weight*mask).sum()\n\n\ndef train_phase(model,records,cfg,steps,lr,seed,deadline,ema=None):\n    if not any(r.get(\'label_kind\')==\'weak_binary_Event\' or np.any(r[\'classes\']==UNKNOWN_FOG) for r in records):\n        return ORIGINAL_TRAIN(model,records,cfg,steps,lr,seed,deadline,ema)\n    if ema is not None: raise ValueError(\'Weak Event loss is warmup-only; TDCS fine-tune must remain unchanged\')\n    torch.manual_seed(seed)\n    if next(model.parameters()).is_cuda: torch.cuda.manual_seed_all(seed)\n    device=next(model.parameters()).device; rng=np.random.default_rng(seed)\n    records,p=raw.sampling_plan(records); weights,counts,unknown,fog_weight=weak_weights(records)\n    weight=torch.from_numpy(weights).to(device)\n    optimizer=torch.optim.AdamW(model.parameters(),lr=lr,weight_decay=cfg.weight_decay)\n    trace=[]; model.train(); started=time.monotonic()\n    for step in range(steps):\n        if time.monotonic()>=deadline: raise TimeoutError(\'Weak pretraining runtime budget reached\')\n        bx,by,mask,crops=raw.batch(records,p,rng,cfg); trace.extend(crops)\n        row_weights=np.where(by==UNKNOWN_FOG,fog_weight,weights[np.minimum(by,3)])\n        mass=float((row_weights*mask).sum())\n        if mass<=0: raise ValueError(\'No annotated sampled auxiliary positions\')\n        optimizer.zero_grad(set_to_none=True)\n        for start in range(0,cfg.batch_size,cfg.micro_batch):\n            stop=start+cfg.micro_batch; x=torch.from_numpy(bx[start:stop]).to(device)\n            labels=torch.from_numpy(by[start:stop]).to(device); valid=torch.from_numpy(mask[start:stop]).to(device).float()\n            numerator,_=partial_losses(model(x),labels,valid,weight,fog_weight)\n            loss=numerator/mass\n            if not torch.isfinite(loss): raise FloatingPointError(\'Nonfinite weak pretraining loss\')\n            loss.backward()\n        torch.nn.utils.clip_grad_norm_(model.parameters(),1.); optimizer.step()\n        if (step+1)%64==0: print(\'Weak Event warmup:\',step+1,\'/\',steps,\'; elapsed\',round(time.monotonic()-started,1),flush=True)\n    return {\'updates\':steps,\'seed\':seed,\'class_counts\':counts.tolist(),\'class_weights\':weights.tolist(),\n        \'unknown_subtype_positive_rows\':unknown,\'unknown_subtype_positive_weight\':fog_weight,\n        \'subjects\':sorted({r[\'subject\'] for r in records}),\'elapsed_seconds\':time.monotonic()-started,\n        \'trace\':trace,\'loss\':\'typed CE plus binary Event partial-state likelihood; no invented subtype labels\'}\n\n\ndef prepare(context,root,transfer):\n    if not transfer: raise ValueError(\'M22 requires the declared auxiliary pretraining stage\')\n    main,aux,y=typed.prepare(context,root,True)\n    excluded=set(pd.read_csv(raw.data.csv_path(\'tdcsfog_metadata.csv\'),dtype=str).Subject)\n    metadata=pd.read_csv(raw.data.csv_path(\'defog_metadata.csv\'),dtype=str)\n    notype_root=raw.data.frozen.shared.common.locate_data()[0].parent/\'notype\'\n    selected,inventory=typed.auxiliary_inventory(metadata,notype_root,excluded,expected_files=46)\n    inventory[\'selection\']=\'actual train/notype CSVs joined to metadata; binary Event supervision only; no test files\'\n    typed_ids={r[\'id\'] for r in aux}; weak_ids=set(selected.Id)\n    if typed_ids&weak_ids: raise ValueError(\'Typed and weak training IDs overlap\')\n    available_typed={p.stem for p in notype_root.parent.joinpath(\'defog\').glob(\'*.csv\')}\n    available_weak={p.stem for p in notype_root.glob(\'*.csv\')}\n    if available_typed|available_weak!=set(metadata.Id): raise ValueError(\'Training file roles do not cover the metadata inventory\')\n    root=Path(root); manifest=json.loads((root/\'raw_dataset_manifest.json\').read_text())\n    for row in selected.itertuples(index=False):\n        frame=pd.read_csv(row.path,usecols=[\'Time\',*raw.data.AXES,\'Event\',\'Valid\',\'Task\'])\n        record=make_weak_record(frame,row.Id,row.Subject); aux.append(record)\n        manifest[\'records\'].append({\'id\':row.Id,\'subject\':row.Subject,\'domain\':\'defog_notype\',\n            \'source_sha256\':raw.data.digest(row.path),\'signal_sha256\':raw.data.array_digest(record[\'signal\']),\n            \'class_sha256\':raw.data.array_digest(record[\'classes\']),\'valid_sha256\':raw.data.array_digest(record[\'valid\']),\n            \'valid_rows\':int(record[\'valid\'].sum()),\'positive_partial_Event_rows\':int(((record[\'classes\']==UNKNOWN_FOG)&record[\'valid\']).sum())})\n        np.savez_compressed(root/\'raw_cache\'/(row.Id+\'.npz\'),signal=record[\'signal\'],classes=record[\'classes\'],valid=record[\'valid\'])\n    manifest[\'notype_inventory\']=inventory\n    manifest[\'effective_raw_source_sha256\']=effective_source_hash()\n    manifest[\'weak_Event_source_sha256\']=source_hash()\n    manifest[\'unknown_subtype_positive_encoding\']=UNKNOWN_FOG\n    manifest[\'unknown_subtype_positive_is_a_new_FOG_class\']=False\n    raw.base.write(root/\'raw_dataset_manifest.json\',manifest)\n    print(\'Auxiliary pool:\',len(aux),\'independent typed + weak records; no subtype pseudo-labels\',flush=True)\n    return main,aux,y\n\n\ndef synthetic():\n    main,aux,y=ORIGINAL_SYNTHETIC()\n    aux[-1][\'classes\']=np.where(aux[-1][\'classes\']>0,UNKNOWN_FOG,0).astype(np.uint8)\n    aux[-1][\'label_kind\']=\'weak_binary_Event\'\n    return main,aux,y\n\n\n@contextmanager\ndef applied():\n    assert raw.source_hash()==typed.FROZEN_RAW_SHA256\n    previous={name:raw.__dict__.get(name) for name in [\'prepare\',\'train_phase\',\'synthetic\',\'SOURCE_SHA256\']}\n    had_override=\'SOURCE_SHA256\' in raw.__dict__; previous_spec=raw.SPECS.get(\'M22\')\n    raw.prepare=prepare; raw.train_phase=train_phase; raw.synthetic=synthetic\n    raw.SOURCE_SHA256=effective_source_hash(); raw.SPECS[\'M22\']=SPEC\n    try: yield\n    finally:\n        for name in [\'prepare\',\'train_phase\',\'synthetic\']: setattr(raw,name,previous[name])\n        if had_override: raw.SOURCE_SHA256=previous[\'SOURCE_SHA256\']\n        else: raw.__dict__.pop(\'SOURCE_SHA256\',None)\n        if previous_spec is None: raw.SPECS.pop(\'M22\',None)\n        else: raw.SPECS[\'M22\']=previous_spec\n\n\ndef run(session,id=\'M22\'):\n    if id!=\'M22\' or session[\'suite\']!=\'metadata\': raise ValueError(\'Only the declared M22 weak-transfer test is supported\')\n    with applied(): return raw.run(session,id)\n\n\ndef reuse_result(session,id,result):\n    assert result[\'id\']==id and result[\'status\'] in (\'complete\',\'complete_reused_v1\',\'complete_reused\') and not result[\'smoke_only\']\n    row={**result,\'status\':\'complete_reused\',\'not_retrained_in_v29\':True,\'artifacts_origin\':result[\'run_directory\']}\n    session[\'results\'].append(row); raw.base.write(session[\'output\']/(id+\'_status.json\'),row); return row\n\n\ndef finish(session):\n    with applied(): return raw.finish(session)\n', 'tdcs_raw_walking_rank_v30': '"""S11: S10\'s raw transfer recipe plus a Walking-vs-other-FOG ranking term.\n\nSame typed input, warmup, architecture, optimizer, crop plan, seeds, fixed\nbudget and inference as S10. Only TDCS fine-tune loss changes. No held-out\nlabels, predictions, or thresholds participate in fitting. Preserve S10 and\nthe frozen v28 reader/model source by pinning this variant separately.\n"""\nfrom contextlib import contextmanager\nimport hashlib\nimport json\nimport time\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom torch.nn import functional as F\n\nimport tdcs_raw_transfer_v28_recovery as typed\n\nraw = typed.raw\nORIGINAL_TRAIN = raw.train_phase\nMETHOD = \'S11\'\nTERM_WEIGHT, MARGIN, HARD_NEGATIVES = .2, .2, 16\nSPEC = {\'suite\':\'sensor\', \'transfer\':True, \'matched_control\':\'S10\', \'rank_term_weight\':TERM_WEIGHT,\n    \'rank_margin_log_odds\':MARGIN, \'hard_negatives_per_microbatch\':HARD_NEGATIVES,\n    \'ranking_positives\':\'all valid Walking rows in the microbatch\',\n    \'ranking_negatives\':\'at most 16 highest Walking log-odds among valid SH/Turn rows\',\n    \'normalization\':\'mean over positive-negative pairs; average over all microbatches, including zero terms\',\n    \'training_change\':\'add rank term to TDCS fine-tune cross entropy only\'}\n\n\ndef source_hash():\n    return globals().get(\'SOURCE_SHA256\') or raw.data.digest(__file__)\n\n\ndef effective_source_hash():\n    sources = {\'typed_control_source\':typed.effective_source_hash(), \'Walking_rank_source\':source_hash()}\n    return hashlib.sha256(json.dumps(sources, sort_keys=True).encode()).hexdigest()\n\n\ndef rank_term(logits, labels, valid):\n    """Return mean softplus ranking loss and its exact pair count."""\n    if logits.shape != (*labels.shape, 4) or labels.shape != valid.shape:\n        raise ValueError(\'Aligned four-state logits, labels and masks are required\')\n    walking = valid & labels.eq(3)\n    other_fog = valid & (labels.eq(1) | labels.eq(2))\n    score = logits[..., 3] - torch.logsumexp(logits[..., :3], dim=-1)\n    positives = score[walking]\n    negatives = score[other_fog]\n    if not positives.numel() or not negatives.numel():\n        return score.sum() * 0., 0\n    hard = torch.topk(negatives, min(negatives.numel(), HARD_NEGATIVES), sorted=False).values\n    return F.softplus(hard[None, :] - positives[:, None] + MARGIN).mean(), positives.numel() * hard.numel()\n\n\ndef train_fine(model, records, cfg, steps, lr, seed, deadline, ema):\n    """Keep S10\'s sample stream, weight mass, optimizer and EMA arithmetic."""\n    if ema is None or any(r[\'domain\'] != \'tdcs\' for r in records):\n        raise ValueError(\'Ranking is allowed only in the TDCS fine-tune phase\')\n    torch.manual_seed(seed)\n    if next(model.parameters()).is_cuda: torch.cuda.manual_seed_all(seed)\n    device = next(model.parameters()).device; rng = np.random.default_rng(seed)\n    records, p = raw.sampling_plan(records)\n    weights, counts = raw.class_weights(records); weight = torch.from_numpy(weights).to(device)\n    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=cfg.weight_decay)\n    trace = []; eligible_micros = 0; pairs = 0\n    micro_count = cfg.batch_size // cfg.micro_batch\n    model.train(); started = time.monotonic()\n    for step in range(steps):\n        if time.monotonic() >= deadline: raise TimeoutError(\'Raw sequence runtime budget reached\')\n        bx, by, mask, crops = raw.batch(records, p, rng, cfg)\n        mass = float((weights[by] * mask).sum())\n        if mass <= 0: raise ValueError(\'Sampled batch contains no annotated positions\')\n        trace.extend(crops); optimizer.zero_grad(set_to_none=True)\n        for start in range(0, cfg.batch_size, cfg.micro_batch):\n            stop = start + cfg.micro_batch; x = torch.from_numpy(bx[start:stop]).to(device)\n            y = torch.from_numpy(by[start:stop]).to(device); valid = torch.from_numpy(mask[start:stop]).to(device)\n            logits = model(x)\n            base_loss = F.cross_entropy(logits.transpose(1,2), y, weight=weight, reduction=\'none\')\n            rank, n_pairs = rank_term(logits, y, valid)\n            pairs += n_pairs; eligible_micros += int(n_pairs > 0)\n            loss = (base_loss * valid.float()).sum()/mass + (TERM_WEIGHT / micro_count) * rank\n            if not torch.isfinite(loss): raise FloatingPointError(\'Nonfinite Walking ranking loss\')\n            loss.backward()\n        torch.nn.utils.clip_grad_norm_(model.parameters(),1.); optimizer.step()\n        with torch.no_grad():\n            for dst, src in zip(ema.parameters(), model.parameters()): dst.lerp_(src, 1-cfg.ema_decay)\n        if (step+1)%128 == 0:\n            print(f\'Walking rank phase: {step+1}/{steps} updates; {eligible_micros} eligible microbatches\', flush=True)\n    if not cfg.smoke and not eligible_micros: raise ValueError(\'No valid Walking/other-FOG ranking pairs in the fit\')\n    return {\'updates\':steps,\'seed\':seed,\'class_counts\':counts.tolist(),\'class_weights\':weights.tolist(),\n        \'subjects\':sorted({r[\'subject\'] for r in records}),\'elapsed_seconds\':time.monotonic()-started,\'trace\':trace,\n        \'rank_usage\':{\'eligible_microbatches\':eligible_micros, \'total_microbatches\':steps*micro_count, \'pairs\':pairs}}\n\n\n@contextmanager\ndef applied(usage=None):\n    assert raw.source_hash() == typed.FROZEN_RAW_SHA256 and METHOD not in raw.SPECS\n    prior_prepare, prior_train = raw.prepare, raw.train_phase\n    had_override = \'SOURCE_SHA256\' in raw.__dict__; override = raw.__dict__.get(\'SOURCE_SHA256\')\n\n    def phase(model, records, cfg, steps, lr, seed, deadline, ema=None):\n        if ema is None: return ORIGINAL_TRAIN(model, records, cfg, steps, lr, seed, deadline)\n        result = train_fine(model, records, cfg, steps, lr, seed, deadline, ema)\n        if usage is not None: usage.append({\'seed\':seed, **result[\'rank_usage\']})\n        return result\n\n    raw.prepare = typed.prepare; raw.train_phase = phase\n    raw.SOURCE_SHA256 = effective_source_hash(); raw.SPECS[METHOD] = SPEC\n    try: yield\n    finally:\n        raw.prepare = prior_prepare; raw.train_phase = prior_train\n        if had_override: raw.SOURCE_SHA256 = override\n        else: raw.__dict__.pop(\'SOURCE_SHA256\', None)\n        raw.SPECS.pop(METHOD)\n\n\ndef run(session):\n    usage = []\n    with applied(usage): result = raw.run(session,METHOD)\n    if result[\'status\'] == \'complete\':\n        cfg = raw.smoke_config() if session[\'cfg\'].smoke else raw.Config()\n        # A rerun can reuse existing folds; record exact usage only on the first full fit.\n        destination = Path(result[\'run_directory\'])/\'walking_rank_usage.json\'\n        if len(usage) == 5:\n            for fold, info in enumerate(usage):\n                info[\'fold\'] = fold\n                assert info[\'seed\'] == cfg.seed+1009*fold+200000\n            raw.base.write(destination, {\'source_sha256\':effective_source_hash(), \'folds\':usage,\n                \'config\':SPEC, \'warmup_received_ranking_term\':False, \'smoke_only\':cfg.smoke})\n        elif not destination.is_file():\n            raise ValueError(\'Rank usage receipt missing for reused folds; rerun in a fresh root\')\n    return result\n\n\ndef finish(session):\n    with applied(): return raw.finish(session)\n', 'tdcs_raw_paired_crops_v31': '"""Matched M22 follow-ups: deliberately co-sample rare and confusable subtypes.\n\nM23 and S12 share weak Event pretraining and TDCS crop streams. One of four\nfine-tune microbatches pairs Walking with a known other-FOG event. M23 changes\nonly crop sampling versus M22; S12 adds the fixed S11 rank term versus M23.\nNo evaluation labels, prediction artifacts, inputs or score changes.\n"""\nfrom contextlib import contextmanager\nfrom dataclasses import replace\nimport hashlib\nimport json\nfrom pathlib import Path\n\nimport numpy as np\n\nimport tdcs_raw_weak_transfer_v29 as weak\nimport tdcs_raw_walking_rank_v30 as rank\n\nraw=weak.raw\nORIGINAL_BATCH=raw.batch\nORIGINAL_SMOKE=raw.smoke_config\nPAIR_SEED_OFFSET=81731\nCOMMON={**weak.SPEC,\'fine_tune_changed\':True,\'matched_warmup_control\':\'M22\',\n    \'paired_crops_per_batch\':2,\'remaining_crops\':\'the unchanged final 6 M22 natural crops in each batch\',\n    \'pairing\':\'position 0 Walking; position 1 alternates Turn and StartHesitation\',\n    \'episode_sampling\':\'uniform subject then uniform contiguous event then uniform interior center, >=3 valid 50Hz positions\',\n    \'pair_rng_seed_offset\':PAIR_SEED_OFFSET}\nSPECS={\'M23\':{**COMMON,\'suite\':\'metadata\',\'rank_term_weight\':0.0},\n       \'S12\':{**COMMON,\'suite\':\'sensor\',\'matched_rank_control\':\'M23\',\'rank_term_weight\':rank.TERM_WEIGHT,\n              \'rank_margin_log_odds\':rank.MARGIN,\'hard_negatives_per_microbatch\':rank.HARD_NEGATIVES}}\n\n\ndef source_hash(): return globals().get(\'SOURCE_SHA256\') or raw.data.digest(__file__)\n\n\ndef effective_source_hash():\n    pins={\'M22_weak_source\':weak.effective_source_hash(),\'S11_rank_source\':rank.source_hash(),\'paired_source\':source_hash()}\n    return hashlib.sha256(json.dumps(pins,sort_keys=True).encode()).hexdigest()\n\n\ndef episode_pools(records):\n    pools={target:{} for target in (1,2,3)}\n    for record in records:\n        if record[\'domain\']!=\'tdcs\': raise ValueError(\'Only outer fitting TDCS records can be paired\')\n        for target in pools:\n            membership=record[\'valid\']&(record[\'classes\']==target)\n            changes=np.flatnonzero(np.diff(membership.astype(np.int8),prepend=0,append=0))\n            for left,right in zip(changes[::2],changes[1::2]):\n                if right-left<3: continue\n                pools[target].setdefault(record[\'subject\'],[]).append((record,int(left+1),int(right-1)))\n    if not all(pools.values()): raise ValueError(\'A fitting fold lacks an eligible subtype event\')\n    return pools\n\n\nclass PairedBatch:\n    def __init__(self,records,cfg,seed):\n        if cfg.batch_size!=8 or cfg.micro_batch!=2: raise ValueError(\'Pairing requires 8 crops / 2 per microbatch\')\n        self.pools=episode_pools(records)\n        self.subjects={target:sorted(values) for target,values in self.pools.items()}\n        self.rng=np.random.default_rng(seed+PAIR_SEED_OFFSET)\n        self.steps=self.eligible=self.total=self.pairs=0\n        self.targeted={1:0,2:0,3:0}\n\n    def crop(self,target,cfg):\n        rng=self.rng; subjects=self.subjects[target]\n        subject=subjects[int(rng.integers(len(subjects)))]; events=self.pools[target][subject]\n        record,low,high=events[int(rng.integers(len(events)))]; center=int(rng.integers(low,high))\n        stretch=float(rng.uniform(.8,1.2)); width=cfg.train_length\n        coords=center+np.linspace(-width*stretch/2,width*stretch/2,width)\n        n=len(record[\'signal\']); idx=np.clip(np.rint(coords).astype(int),0,n-1)\n        mask=(coords>=0)&(coords<=n-1)&record[\'valid\'][idx]\n        y=record[\'classes\'][idx]; assert np.any(mask&(y==target))\n        x=raw.features(record,coords,float(rng.uniform(-.2,.2)),float(rng.uniform(.9,1.1)),rng.normal(0,.001,(width,3)))\n        return x,y,mask,(record[\'id\'],record[\'subject\'],center,stretch)\n\n    def __call__(self,records,probability,rng,cfg):\n        # Consume the identical natural RNG stream before replacing two crops.\n        x,y,mask,trace=ORIGINAL_BATCH(records,probability,rng,cfg)\n        other=2 if self.steps%2==0 else 1\n        for slot,target in enumerate((3,other)):\n            x[slot],y[slot],mask[slot],trace[slot]=self.crop(target,cfg)\n            self.targeted[target]+=1\n        n=y.reshape(-1,cfg.micro_batch*cfg.train_length); m=mask.reshape(n.shape)\n        pos=np.count_nonzero(m&(n==3),axis=1); neg=np.count_nonzero(m&((n==1)|(n==2)),axis=1)\n        pairs=pos*np.minimum(rank.HARD_NEGATIVES,neg); assert pairs[0]>0\n        self.steps+=1; self.total+=len(pairs); self.eligible+=int(np.count_nonzero(pairs)); self.pairs+=int(pairs.sum())\n        return x,y,mask,trace\n\n    def receipt(self,seed):\n        return {\'seed\':seed,\'pair_rng_seed\':seed+PAIR_SEED_OFFSET,\'paired_batches\':self.steps,\n            \'eligible_microbatches\':self.eligible,\'total_microbatches\':self.total,\'pairs\':self.pairs,\n            \'targeted_centers\':{raw.LABELS[k-1]:v for k,v in self.targeted.items()},\n            \'natural_crops\':6*self.steps,\'pair_subject_counts\':{raw.LABELS[k-1]:len(v) for k,v in self.pools.items()}}\n\n\ndef smoke_config(): return replace(ORIGINAL_SMOKE(),batch_size=8,micro_batch=2)\n\n\n@contextmanager\ndef applied(id,usage=None):\n    if id not in SPECS: raise ValueError(\'Unexpected paired-crop test ID\')\n    with weak.applied():\n        prior_source=raw.SOURCE_SHA256; prior_train=raw.train_phase; prior_smoke=raw.smoke_config\n        assert rank.METHOD not in raw.SPECS and not set(SPECS)&set(raw.SPECS)\n\n        def phase(model,records,cfg,steps,lr,seed,deadline,ema=None):\n            if ema is None: return prior_train(model,records,cfg,steps,lr,seed,deadline)\n            sampler=PairedBatch(records,cfg,seed)\n            prior_batch=raw.batch; raw.batch=sampler\n            try:\n                fine=rank.train_fine if id==\'S12\' else prior_train\n                out=fine(model,records,cfg,steps,lr,seed,deadline,ema)\n            finally: raw.batch=prior_batch\n            row=sampler.receipt(seed); assert row[\'paired_batches\']==steps\n            if id==\'S12\': assert all(row[k]==v for k,v in out[\'rank_usage\'].items())\n            if usage is not None: usage.append(row)\n            return out\n\n        raw.train_phase=phase; raw.SOURCE_SHA256=effective_source_hash(); raw.smoke_config=smoke_config\n        raw.SPECS[id]=SPECS[id]; raw.SPECS[rank.METHOD]=rank.SPEC\n        try: yield\n        finally:\n            raw.train_phase=prior_train; raw.SOURCE_SHA256=prior_source; raw.smoke_config=prior_smoke\n            raw.SPECS.pop(id); raw.SPECS.pop(rank.METHOD)\n\n\ndef run(session,id):\n    usage=[]\n    with applied(id,usage): result=raw.run(session,id)\n    if result[\'status\']==\'complete\':\n        destination=Path(result[\'run_directory\'])/\'paired_crop_usage.json\'\n        if len(usage)==5:\n            cfg=smoke_config() if session[\'cfg\'].smoke else raw.Config()\n            for fold,row in enumerate(usage):\n                row[\'fold\']=fold; assert row[\'seed\']==cfg.seed+1009*fold+200000\n            raw.base.write(destination,{\'id\':id,\'source_sha256\':effective_source_hash(),\'folds\':usage,\n                \'specification\':SPECS[id],\'smoke_only\':cfg.smoke,\'ranking_used\':id==\'S12\',\n                \'auxiliary_pretraining_crops_changed\':False,\'outer_labels_used_for_pairing\':False})\n        elif not destination.is_file(): raise ValueError(\'Paired-crop usage missing for reused folds\')\n    return result\n\n\ndef finish(session,id):\n    with applied(id): return raw.finish(session)\n'}
PAIRED_HASHES = {'tdcs_raw_weak_transfer_v29': '814c12d94984ec3ad84558767ac0ebc8d3d3827a5f2190a31864bef6066423b2', 'tdcs_raw_walking_rank_v30': '846bdebccc7aea9f6c1803e33f5a6510dbc91ad92096bad26ac3ea06b181df27', 'tdcs_raw_paired_crops_v31': '2db78215471a32e3c7305421fd5cc5b9cfc2a22442a2b3cc66cc99c8504f2a34'}
for name, source in PAIRED_SOURCES.items():
    assert hashlib.sha256(source.encode()).hexdigest() == PAIRED_HASHES[name]
    if name in sys.modules:
        assert sys.modules[name].SOURCE_SHA256 == PAIRED_HASHES[name]
        continue
    module = types.ModuleType(name)
    module.__file__ = str(PROJECT_ROOT / 'scripts' / (name + '.py'))
    module.SOURCE_SHA256 = PAIRED_HASHES[name]
    sys.modules[name] = module
    exec(compile(source, module.__file__, 'exec'), module.__dict__)
paired = sys.modules['tdcs_raw_paired_crops_v31']
PAIRED_PRIOR_RESULT = {'id': 'S11', 'title': 'Raw 50-Hz CNN/residual GRU DeFOG transfer', 'suite': 'sensor', 'status': 'complete', 'StartHesitation': 0.32211904879289177, 'Walking': 0.2074019739975308, 'two_target_mAP': 0.2647605113952113, 'run_directory': '/kaggle/working/tdcs_sensor_comprehensive_v26_v30_walking_rank/methods/S11/e67f20b6f8f21f5cf916', 'smoke_only': False, 'native_prediction_sha256': '1bb0470998e5eed44fa44ac077d6dfc30ca0fe2b445f3daf6ee26ed7a6f6d74a', 'independent_validation_pending': True}
print('Pinned paired-crop test:', 'S12', paired.effective_source_hash())

Pinned paired-crop test: S12 f25b3b8328204ea0ddccd3795a1373a7f36559e594e0a5d186c6b25be7b23a1b


In [9]:
session = runner.initialize(SUITE, WORK_ROOT, runner.Config() if RUN_FULL else runner.smoke_config())
print('Clinical join:', session['context']['clinical_audit'], flush=True)
display(pd.DataFrame([{'ID': t['id'], 'Experiment': t['title'], 'Learner': t['kind']}
    for t in runner.CATALOG if t['suite'] == SUITE]))

Clinical join: {'rows': 5, 'subject_visit_matches': 0, 'subject_level_matches': 5, 'missing': 0, 'rule': 'Exact subject/visit or unique blank-Visit subject-level row; no arbitrary visit fallback'}


,ID,Experiment,Learner
0,S00,Sensor-only CatBoost control,catboost
1,S01,LightGBM: explicit Kaggle CPU training,lightgbm
2,S02,Supervised sensor CNN control for pretraining,neural
3,S03,Masked reconstruction pretraining on mapped un...,neural
4,S04,Teacher-student pseudo-label training,pseudo
5,S05,Sensor-only binary XGBoost control for ranking,xgboost
6,S06,Per-record MAP-focused tree ranking,rank_map


### S00 — Sensor-only CatBoost control

Sensor-only 325-column CatBoost control. No clinical/Visit/Test/Medication predictor; the frozen sensor-derived Walking gate is common except in M12.

In [10]:
display(raw.reuse_result(session, 'S00', PREVIOUS_RESULTS_V28['S00']) if RUN_FULL else runner.run_method(session, 'S00'))

S00 fold 0: training and held-out prediction complete
S00 fold 1: training and held-out prediction complete
S00 fold 2: training and held-out prediction complete
S00 fold 3: training and held-out prediction complete
S00 fold 4: training and held-out prediction complete


{'id': 'S00',
 'title': 'Sensor-only CatBoost control',
 'suite': 'sensor',
 'status': 'complete',
 'StartHesitation': 0.051623749607711086,
 'Walking': 0.11925172644409271,
 'two_target_mAP': 0.0854377380259019,
 'run_directory': 'D:\\Dissertation\\tmp\\tdcs_sensor_comprehensive_v26_v31_paired_crops_smoke_14b34482\\methods\\S00\\49c38fa77050d4f2a02f',
 'native_prediction_sha256': '1140ecd60ec40ff750ee0e377fe83b8ce21603819d8b25e669d72c74e6d1ed75',
 'independent_validation_pending': True,
 'smoke_only': True,
 'actual_grid_rows': 320}

### S01 — LightGBM: explicit Kaggle CPU training

LightGBM binary classifier on the same sampled sensor rows. CPU training on Kaggle is deliberate, not fallback. The sensor suite permits internet solely for the official LightGBM package if absent.

In [11]:
display(raw.reuse_result(session, 'S01', PREVIOUS_RESULTS_V28['S01']) if RUN_FULL else runner.run_method(session, 'S01'))

S01 fold 0: training and held-out prediction complete
S01 fold 1: training and held-out prediction complete
S01 fold 2: training and held-out prediction complete
S01 fold 3: training and held-out prediction complete
S01 fold 4: training and held-out prediction complete


{'id': 'S01',
 'title': 'LightGBM: explicit Kaggle CPU training',
 'suite': 'sensor',
 'status': 'complete',
 'StartHesitation': 0.044414128550842175,
 'Walking': 0.09371457505468342,
 'two_target_mAP': 0.0690643518027628,
 'run_directory': 'D:\\Dissertation\\tmp\\tdcs_sensor_comprehensive_v26_v31_paired_crops_smoke_14b34482\\methods\\S01\\a5062c49e66fbf9cc4e9',
 'native_prediction_sha256': 'afcdea118fa026e4fe4a768154632fbe96625715e5760d7db4528a87e745b36d',
 'independent_validation_pending': True,
 'smoke_only': True,
 'actual_grid_rows': 320}

### S02 — Supervised sensor CNN control for pretraining

A supervised-only temporal CNN control with the same initialization and supervised budget as S03.

In [12]:
display(raw.reuse_result(session, 'S02', PREVIOUS_RESULTS_V28['S02']) if RUN_FULL else runner.run_method(session, 'S02'))

S02 fold 0: epoch 1/1
S02 fold 0: training and held-out prediction complete
S02 fold 1: epoch 1/1
S02 fold 1: training and held-out prediction complete
S02 fold 2: epoch 1/1
S02 fold 2: training and held-out prediction complete
S02 fold 3: epoch 1/1
S02 fold 3: training and held-out prediction complete
S02 fold 4: epoch 1/1
S02 fold 4: training and held-out prediction complete


{'id': 'S02',
 'title': 'Supervised sensor CNN control for pretraining',
 'suite': 'sensor',
 'status': 'complete',
 'StartHesitation': 0.10458936937488264,
 'Walking': 0.11640760053092744,
 'two_target_mAP': 0.11049848495290504,
 'run_directory': 'D:\\Dissertation\\tmp\\tdcs_sensor_comprehensive_v26_v31_paired_crops_smoke_14b34482\\methods\\S02\\42051925ce271c72251f',
 'native_prediction_sha256': '72d91d0c43328ced6e48cb5c3abbca50be8339059d5ce36e42e533567f31201f',
 'independent_validation_pending': True,
 'smoke_only': True,
 'actual_grid_rows': 320}

### S03 — Masked reconstruction pretraining on mapped unlabeled recordings

Pretrain a masked-feature reconstruction encoder using bounded prefixes from mapped competition unlabeled recordings, then fine-tune on fitting labels. Exclude every held-out subject from pretraining separately in each fold. Missing external inputs are a failed test, not substituted training data.

In [13]:
display(raw.reuse_result(session, 'S03', PREVIOUS_RESULTS_V28['S03']) if RUN_FULL else runner.run_method(session, 'S03'))

S03 fold 0: epoch 1/1
S03 fold 0: training and held-out prediction complete
S03 fold 1: epoch 1/1
S03 fold 1: training and held-out prediction complete
S03 fold 2: epoch 1/1
S03 fold 2: training and held-out prediction complete
S03 fold 3: epoch 1/1
S03 fold 3: training and held-out prediction complete
S03 fold 4: epoch 1/1
S03 fold 4: training and held-out prediction complete


{'id': 'S03',
 'title': 'Masked reconstruction pretraining on mapped unlabeled recordings',
 'suite': 'sensor',
 'status': 'complete',
 'StartHesitation': 0.10208015756622665,
 'Walking': 0.12013400969130812,
 'two_target_mAP': 0.1111070836287674,
 'run_directory': 'D:\\Dissertation\\tmp\\tdcs_sensor_comprehensive_v26_v31_paired_crops_smoke_14b34482\\methods\\S03\\2d89d2b481c697ad2f7f',
 'native_prediction_sha256': 'abe851a695803caa529ed6c88dee3535c65d78d376efa03248fc2ef2cd7e6c77',
 'independent_validation_pending': True,
 'smoke_only': True,
 'actual_grid_rows': 320}

### S04 — Teacher-student pseudo-label training

Outer-training-only CatBoost teacher; take external probabilities ≥0.90 or ≤0.01, cap at 40,000 rows per head/fold, pseudo-weight 0.1, then fit a student. Exclude held-out subjects. Report zero selected rows explicitly if no pseudo-labels qualify.

In [14]:
display(raw.reuse_result(session, 'S04', PREVIOUS_RESULTS_V28['S04']) if RUN_FULL else runner.run_method(session, 'S04'))

S04 fold 0: training and held-out prediction complete
S04 fold 1: training and held-out prediction complete
S04 fold 2: training and held-out prediction complete
S04 fold 3: training and held-out prediction complete
S04 fold 4: training and held-out prediction complete


{'id': 'S04',
 'title': 'Teacher-student pseudo-label training',
 'suite': 'sensor',
 'status': 'complete',
 'StartHesitation': 0.051623749607711086,
 'Walking': 0.11925172644409271,
 'two_target_mAP': 0.0854377380259019,
 'run_directory': 'D:\\Dissertation\\tmp\\tdcs_sensor_comprehensive_v26_v31_paired_crops_smoke_14b34482\\methods\\S04\\1f144d3ea31763c7611f',
 'native_prediction_sha256': '1140ecd60ec40ff750ee0e377fe83b8ce21603819d8b25e669d72c74e6d1ed75',
 'independent_validation_pending': True,
 'smoke_only': True,
 'actual_grid_rows': 320}

### S05 — Sensor-only binary XGBoost control for ranking

Binary-logistic sensor XGBoost control for S06, with matching fixed rounds/depth/seed and CPU scoring of saved CUDA models.

In [15]:
display(raw.reuse_result(session, 'S05', PREVIOUS_RESULTS_V28['S05']) if RUN_FULL else runner.run_method(session, 'S05'))

S05 fold 0: training and held-out prediction complete
S05 fold 1: training and held-out prediction complete
S05 fold 2: training and held-out prediction complete
S05 fold 3: training and held-out prediction complete
S05 fold 4: training and held-out prediction complete


{'id': 'S05',
 'title': 'Sensor-only binary XGBoost control for ranking',
 'suite': 'sensor',
 'status': 'complete',
 'StartHesitation': 0.04728310486602491,
 'Walking': 0.09806315419654973,
 'two_target_mAP': 0.07267312953128732,
 'run_directory': 'D:\\Dissertation\\tmp\\tdcs_sensor_comprehensive_v26_v31_paired_crops_smoke_14b34482\\methods\\S05\\b52fae0c1cf227d0eb7a',
 'native_prediction_sha256': '6f333cb8ec022f6d043ffbe8fc9c2a7c81078d35ff676329647f1389dc455eb9',
 'independent_validation_pending': True,
 'smoke_only': True,
 'actual_grid_rows': 320}

### S06 — Per-record MAP-focused tree ranking

XGBoost rank:map with one query per recording and binary relevance. Rank weights are per query, not illegal per-row ranking weights. Per-record MAP is a surrogate and is not the pooled native-row AP used for decisions; a fixed sigmoid makes scores bounded but not calibrated.

In [16]:
display(raw.reuse_result(session, 'S06', PREVIOUS_RESULTS_V28['S06']) if RUN_FULL else runner.run_method(session, 'S06'))

S06 fold 0: training and held-out prediction complete
S06 fold 1: training and held-out prediction complete
S06 fold 2: training and held-out prediction complete
S06 fold 3: training and held-out prediction complete
S06 fold 4: training and held-out prediction complete


{'id': 'S06',
 'title': 'Per-record MAP-focused tree ranking',
 'suite': 'sensor',
 'status': 'complete',
 'StartHesitation': 0.12971966156598486,
 'Walking': 0.1001705455691078,
 'two_target_mAP': 0.11494510356754634,
 'run_directory': 'D:\\Dissertation\\tmp\\tdcs_sensor_comprehensive_v26_v31_paired_crops_smoke_14b34482\\methods\\S06\\09994c697e7b0b3aad17',
 'native_prediction_sha256': '3e9c548a1ed9fad167ab7f03defd04e07a5e6dd06ff6da1c3a77b1fe72e68743',
 'independent_validation_pending': True,
 'smoke_only': True,
 'actual_grid_rows': 320}

### S07 — Hesitation pseudo-label weight 0.03

Reduce each extra pseudo-row weight from 0.10 to 0.03 before mean-one normalization. The five outer-training-only teachers, actual pseudo-row IDs/labels/probabilities and original fitting rows are shared exactly across S07/S08/S09.

In [17]:
display(raw.reuse_result(session, 'S07', PREVIOUS_RESULTS_V28['S07']) if RUN_FULL else followups.run_method(session, 'S07'))

S07 fold 0 complete
S07 fold 1 complete
S07 fold 2 complete
S07 fold 3 complete
S07 fold 4 complete


{'id': 'S07',
 'title': 'S07 StartHesitation pseudo',
 'suite': 'sensor',
 'status': 'complete',
 'target_scope': ['StartHesitation'],
 'run_directory': 'D:\\Dissertation\\tmp\\tdcs_sensor_comprehensive_v26_v31_paired_crops_smoke_14b34482\\methods\\S07\\43ea5c94f3db2ae8e1e3',
 'smoke_only': True,
 'native_prediction_sha256': '660c53a37abaf459ba3c505aa972a93138eb76600774519adb0039c8d2991fe2',
 'independent_validation_pending': True,
 'StartHesitation': 0.051623749607711086}

### S08 — Hesitation pseudo-label weight 0.10 — matched refit control

Refit the SH component of recovered S04 as a one-head experiment using the same recipe and the shared teachers/rows. This tests the paired control; compare its actual results with historical S04 rather than assuming GPU fits are bitwise identical.

In [18]:
display(raw.reuse_result(session, 'S08', PREVIOUS_RESULTS_V28['S08']) if RUN_FULL else followups.run_method(session, 'S08'))

S08 fold 0 complete
S08 fold 1 complete
S08 fold 2 complete
S08 fold 3 complete
S08 fold 4 complete


{'id': 'S08',
 'title': 'S08 StartHesitation pseudo',
 'suite': 'sensor',
 'status': 'complete',
 'target_scope': ['StartHesitation'],
 'run_directory': 'D:\\Dissertation\\tmp\\tdcs_sensor_comprehensive_v26_v31_paired_crops_smoke_14b34482\\methods\\S08\\00c652c8899ea0728ef0',
 'smoke_only': True,
 'native_prediction_sha256': '660c53a37abaf459ba3c505aa972a93138eb76600774519adb0039c8d2991fe2',
 'independent_validation_pending': True,
 'StartHesitation': 0.051623749607711086}

### S09 — Hesitation pseudo-label weight 0.30

Raise each pseudo-row weight to 0.30 before mean-one normalization, with the exact same teachers/rows as S07/S08. Most selected rows are teacher-labelled background, not ground-truth negative annotations.

In [19]:
display(raw.reuse_result(session, 'S09', PREVIOUS_RESULTS_V28['S09']) if RUN_FULL else followups.run_method(session, 'S09'))

S09 fold 0 complete
S09 fold 1 complete
S09 fold 2 complete
S09 fold 3 complete
S09 fold 4 complete


{'id': 'S09',
 'title': 'S09 StartHesitation pseudo',
 'suite': 'sensor',
 'status': 'complete',
 'target_scope': ['StartHesitation'],
 'run_directory': 'D:\\Dissertation\\tmp\\tdcs_sensor_comprehensive_v26_v31_paired_crops_smoke_14b34482\\methods\\S09\\5aca5a82a016be20a7d1',
 'smoke_only': True,
 'native_prediction_sha256': '660c53a37abaf459ba3c505aa972a93138eb76600774519adb0039c8d2991fe2',
 'independent_validation_pending': True,
 'StartHesitation': 0.051623749607711086}

### S10 — Independent labelled-DeFOG pretraining, then raw TDCS fine-tuning

Warm up on labelled DeFOG accelerometry with loss masked by Valid AND Task. Exclude every subject ID in the complete TDCS metadata table from auxiliary fitting. Share that independent 512-update checkpoint across the five outer models; fine-tune each on its own outer-fitting TDCS subjects only. This is not reuse or modification of the retained DeFOG predictor.

In [20]:
display(raw.reuse_result(session, 'S10', COMPLETED_S10) if RUN_FULL else reader_recovery.run(session, 'S10'))

S10 fold 0 complete
S10 fold 1 complete
S10 fold 2 complete
S10 fold 3 complete
S10 fold 4 complete


{'id': 'S10',
 'title': 'Raw 50-Hz CNN/residual GRU DeFOG transfer',
 'suite': 'sensor',
 'status': 'complete',
 'StartHesitation': 0.38243594524468155,
 'Walking': 0.26489434545674145,
 'two_target_mAP': 0.3236651453507115,
 'run_directory': 'D:\\Dissertation\\tmp\\tdcs_sensor_comprehensive_v26_v31_paired_crops_smoke_14b34482\\methods\\S10\\65e2662217210c73ada6',
 'smoke_only': True,
 'native_prediction_sha256': 'c5243bdb7f4ba6e3173e8ca641bff72b83a362421ec31f51b745b1821b5b0d0f',
 'independent_validation_pending': True}

### S11 — Walking vs other-FOG hard-negative ranking (typed warmup)

Match S10 except for the predeclared fine-tuning objective. All previous
tests above are reused on Kaggle. No held-out stopping or auto-promotion.

In [21]:
display(raw.reuse_result(session, 'S11', PAIRED_PRIOR_RESULT) if RUN_FULL else walking_rank.run(session))

S11 fold 0 complete
S11 fold 1 complete
S11 fold 2 complete
S11 fold 3 complete
S11 fold 4 complete


{'id': 'S11',
 'title': 'Raw 50-Hz CNN/residual GRU DeFOG transfer',
 'suite': 'sensor',
 'status': 'complete',
 'StartHesitation': 0.38238463279208224,
 'Walking': 0.2648856856613796,
 'two_target_mAP': 0.3236351592267309,
 'run_directory': 'D:\\Dissertation\\tmp\\tdcs_sensor_comprehensive_v26_v31_paired_crops_smoke_14b34482\\methods\\S11\\826ebaa0374230c10c67',
 'smoke_only': True,
 'native_prediction_sha256': '8ca7e439690217f6a651f4e2ef1dee62a2b5e68be29a526a63b81da40982d246',
 'independent_validation_pending': True}

### S12 — paired crops plus the fixed ranking loss

Use only the current paired-crop protocol above.

In [22]:
display(paired.run(session, 'S12'))

S12 fold 0 complete
S12 fold 1 complete
S12 fold 2 complete
S12 fold 3 complete
S12 fold 4 complete


{'id': 'S12',
 'title': 'Raw 50-Hz CNN/residual GRU DeFOG transfer',
 'suite': 'sensor',
 'status': 'complete',
 'StartHesitation': 0.35569070539867387,
 'Walking': 0.24233112812046376,
 'two_target_mAP': 0.2990109167595688,
 'run_directory': 'D:\\Dissertation\\tmp\\tdcs_sensor_comprehensive_v26_v31_paired_crops_smoke_14b34482\\methods\\S12\\1dba19b18a07423178da',
 'smoke_only': True,
 'native_prediction_sha256': '9cc569ae753ec5cc3d58d23cf7793381274abe941322dfd82bc0da9775c568b3',
 'independent_validation_pending': True}

## Results and coverage
The table reports every attempted method, including failures and budget
exhaustion. Completed rows contain pooled native-row AP and two-target mAP,
not mean fold AP. Saved local outputs are synthetic only; charts must not be
interpreted as TDCS performance. Missing results are not zero scores.

In [23]:
completion = paired.finish(session, 'S12')
if not RUN_FULL:
    assert completion['status'] == 'all_methods_completed_validation_pending', 'Synthetic failure: inspect method statuses'
results = pd.DataFrame(session['results'])
columns = [c for c in ['id', 'title', 'status', 'StartHesitation', 'Walking',
    'two_target_mAP', 'error'] if c in results]
display(results[columns].round(6))
print(json.dumps(completion, indent=2))

{
  "status": "all_methods_completed_validation_pending",
  "expected_tests": [
    "S00",
    "S01",
    "S02",
    "S03",
    "S04",
    "S05",
    "S06",
    "S07",
    "S08",
    "S09",
    "S10",
    "S11",
    "S12"
  ],
  "attempted_tests": [
    "S00",
    "S01",
    "S02",
    "S03",
    "S04",
    "S05",
    "S06",
    "S07",
    "S08",
    "S09",
    "S10",
    "S11",
    "S12"
  ],
  "completed_tests": [
    "S00",
    "S01",
    "S02",
    "S03",
    "S04",
    "S05",
    "S06",
    "S07",
    "S08",
    "S09",
    "S10",
    "S11",
    "S12"
  ],
  "newly_fitted_tests": [
    "S00",
    "S01",
    "S02",
    "S03",
    "S04",
    "S05",
    "S06",
    "S07",
    "S08",
    "S09",
    "S10",
    "S11",
    "S12"
  ],
  "reused_tests": [],
  "missing_tests": [],
  "smoke_only": true,
  "retained_heads_changed": false,
  "competition_submission_created": false,
  "automatic_promotion": false,
  "confidence_calibrated": false,
  "independent_generalization_established": false

,id,title,status,StartHesitation,Walking,two_target_mAP
0,S00,Sensor-only CatBoost control,complete,0.051624,0.119252,0.085438
1,S01,LightGBM: explicit Kaggle CPU training,complete,0.044414,0.093715,0.069064
2,S02,Supervised sensor CNN control for pretraining,complete,0.104589,0.116408,0.110498
3,S03,Masked reconstruction pretraining on mapped un...,complete,0.102080,0.120134,0.111107
4,S04,Teacher-student pseudo-label training,complete,0.051624,0.119252,0.085438
5,S05,Sensor-only binary XGBoost control for ranking,complete,0.047283,0.098063,0.072673
6,S06,Per-record MAP-focused tree ranking,complete,0.129720,0.100171,0.114945
7,S07,S07 StartHesitation pseudo,complete,0.051624,NaN,NaN
8,S08,S08 StartHesitation pseudo,complete,0.051624,NaN,NaN
9,S09,S09 StartHesitation pseudo,complete,0.051624,NaN,NaN


### Compare completed tests only

In [24]:
import matplotlib.pyplot as plt
plt.rcParams.update({'font.size': 11, 'axes.spines.top': False, 'axes.spines.right': False})
completed = results[results.status.isin(['complete', 'complete_reused'])]
if len(completed):
    fig, axes = plt.subplots(1, 2, figsize=(12, max(4, len(completed) * .32)), sharey=True)
    for ax, target, color in zip(axes, ['StartHesitation', 'Walking'], ['#245B87', '#A75319']):
        ax.barh(completed.id, completed[target], color=color)
        ax.set(xlim=(0, 1), xlabel='Native-row average precision (0–1)', title=target)
        ax.grid(axis='x', alpha=.2); ax.set_axisbelow(True)
    axes[0].invert_yaxis()
    fig.suptitle(('SYNTHETIC WIRING ONLY — not TDCS performance' if not RUN_FULL else
        'TDCS v31 — S12; prior results reused'))
    fig.tight_layout(rect=(0, 0, 1, .94))
    fig.savefig(WORK_ROOT / 'completed_test_comparison.png', dpi=140, bbox_inches='tight')
    plt.show()
else:
    print('No completed results to plot; inspect failure statuses above.')
print(f"Completed {len(completed)}/{len(results)} attempted tests; no automatic promotion.")

Completed 13/13 attempted tests; no automatic promotion.


tdcs_sensor_comprehensive_v26_public.ipynb:code-24:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown


## Checks and next steps
Inspect `suite_completion.json`, `suite_summary.csv`, each method's
configuration, training receipts, saved models and held-out grid arrays.
A `complete` row means training/scoring finished, not that independent
replay or promotion passed. Use the separate saved-model replay tool after
download and review all fold/pooled tradeoffs before changing any retained
head. GPU training is not bitwise deterministic. A broad reused-fold search
needs fresh subject validation before a generalization claim.

Notebook execution: Run All once with the attached Kaggle inputs. The two
suite jobs have independent output roots and failure logs. If the safety
budget prevents full coverage, use the explicit unfinished-test list; do
not call skipped methods tested. No submission.csv is created.